# Docta · una IA que sabe lo que no sabe

Red neuronal transformer escrita a mano en numpy —estructura completa de
DeepSeek-V3 a escala de juguete: RoPE, RMSNorm, SwiGLU, atención agrupada y
mezcla de expertos con balanceo sin pérdida auxiliar— más una base de
conocimiento curada y una puerta de ignorancia.

**El objetivo no es que lo sepa todo.** Es que tenga la cultura general de una
persona media y que, cuando no la tenga, lo diga. Todo lo demás está al servicio
de eso.

---

### Cómo se ejecuta esto

**Run → Run all.** Las celdas PAQUETE llevan el proyecto comprimido dentro; no
hay que leerlas ni tocarlas, y la celda DEPLIEGUE lo monta todo en `./docta`.

Las cuatro primeras celdas útiles valen solas: despliegan, muestran el
comportamiento, pasan los tests y comprueban el backprop. Ninguna necesita GPU ni
conexión ni entrenar nada.

> ⚠️ **Si al abrir este notebook Kaggle muestra una página bloqueada** con un
> error `removeChild` en `vendor.js`, no es el notebook: es el editor atragantándose
> con las celdas del paquete. Dale a *Run all* igualmente, o ábrelo con
> `File → Open in → Colab`. Las celdas del paquete están troceadas a propósito
> para que esto no pase, pero el editor de Kaggle es delicado con celdas grandes.

### Qué se va a ver

| Celda | Qué hace | Tarda |
|---|---|---|
| PAQUETE + DEPLIEGUE | Monta el proyecto y comprueba que carga | segundos |
| PUERTA | Sabe, duda, calla y veta: 9 preguntas | segundos |
| TESTS | Las dos suites (46 + 13 casos) | segundos |
| GRADIENTE | Backprop contra derivada numérica | ~1 min |
| ENTRENAR | Pre-entrenamiento de verdad | ~2 min (tiny) |
| PREGUNTAR | Hablar con Docta | segundos |

### Lo importante

En la celda PUERTA hay nueve preguntas. Cinco las contesta y **cuatro las
rechaza**, y esas cuatro son el punto entero del proyecto. Un modelo de lenguaje
responde a todo con la misma fluidez; Docta tiene que distinguir entre lo que
tiene curado, lo que tiene a medias y lo que no tiene, y hablar de forma distinta
en cada caso.

Una de esas cuatro es una dosis de paracetamol para un niño. Aunque hubiera datos
en la base, de eso no se opina.

### Sin credenciales

Este notebook es autocontenido. No adjunta ningún Dataset, no descarga nada y no
necesita ningún token de API. Se sube a Kaggle desde el navegador y se ejecuta.


In [ ]:
# ============================================================================
# DOCTA · paquete autocontenido  (1/12)
# ============================================================================
# Las celdas marcadas como PAQUETE llevan el proyecto entero comprimido y en
# base64: los 9 módulos, los 5 scripts, los 2 tests y los 9 archivos de datos,
# con las 53 fichas curadas, el corpus y el tokenizer.
#
# NO HACE FALTA LEERLAS NI TOCARLAS. Ejecuta todas las celdas en orden
# (Run > Run all) y la celda DEPLIEGUE monta el proyecto en ./docta.
#
# Cada línea de abajo es una cadena de una línea a propósito. Un solo bloque de
# 174 KB entre comillas triples bloquea el editor de Kaggle: el resaltador de
# sintaxis intenta anidar una cadena de mil setecientas líneas y el DOM se
# desincroniza. Troceado y en cadenas simples, no.
#
# Lo que NO está aquí: los checkpoints (28 MB). Se regeneran entrenando, que es
# precisamente para lo que se viene a Kaggle.
# ============================================================================

PAQUETE = [
    'H4sIAFt642oC/+y9W28cV5Ym2s/8FdFp1BRpp9JM3iTRZtXQEl2lat1KlNxdRxZYOzMikyFFRmRFZNBKaVSYeuyHfhjMAAcYzIsw'
    'T0ZDwPT44QDuARpw/pP5JWd9a629Y0dkJEXLtGeqSkaVSGZG7Mvaa6/7ZRjPzDDO0t50/jc/1n+b9N/ezg7/pP8aP/e2r25u2c/k'
    '8/5mf2vzb4LNv/kJ/iuLmclp+r/56/yv0+msPYiKqUnCbD84i/J4FA8JIxbfpEEYBWYU5xNGkKgI5sGQsKXora09zOnPYVaYIgjj'
    'Yhan9HHwhzKiz9KzOEqjIM2CSfRimJh8f22t3wsO7957ePigF9wwoXGj6iSJCWi8aRnRUQQFBplMTW4w2Ix+0NfRWRxG6TA2a0GA'
    'GadZGg+SiFZEj09MPsTDk4yHoZ3QHN0gLENaYJAFRZzaLzJa+xYt4hav5XbGa6ZpF6+rV7MgSaIzWkhJn42inOd1D6dZF3OG8TDS'
    'v2nI7V7wxdGDW5/fuoFhf4cRM/onSLIgnkyzfGbSWbSPjZg0m+nGaRTagZkmBO/g4eGdz24t/vFuYBJscRY9n8l8YwKmwCCPwiCN'
    'yjxLTdIFfIuMxjdJMCxzWnUveJQGkyyM6EPANErHpXkaYTSCLH0yywhQQZn6wB5meR4N6Rc5hYEpAFOz+BcTxjmtrqTzMbSSOD2L'
    '6IzDDMPRsdBgM5p5lJR0Li96wTGBWAY0UwJ9lBDQi1k0MYwx+eKNCUYljUAwmRB60KkWGMkk5TBOCbg3aMgoSeTV+oxBYRI9Y/qD'
    'Tzn1DhrgvxkX0eJfsO0kHkQABsO6jsw09Y17d4+PHnxxePPeg0PCSFoOHiM8MYpDHUKVNTt03mEod6qpOgxjuzMCGV8HPiYags45'
    'GNAmMxy7oMaaQB74nGcDgj4tYhrRc/I1HTMdrozSxfv4mIEf8TkRLPOY0Y8mW8vjcfVm9ByroL3fWHwzwc6HdBC0NH5vlOMcG2ht'
    '9gniHwZ3F//1ztGDe8fBeoK7RpMlOJp8Fm30gqPndDIACl3uGKMoXsiAc8w6FrSnw+C76NZJd52uaVq7rIAf3fMkfmFSHGSEWx1m'
    'oB007iROmKZg7fRQRHQFAx77e+Br1rnWu7ZzncjJLM+KDr3hzwFggE4w0YnolXTx7STKM8zN4zXBQIQhHht7Rvw74VyOK4tLXjD6'
    'FRX+ESgMwbWIIzpEvk508dIgLSeLr/OYaGAXVIg+oREni9dFMFq8HsYJdjiNCaXzHsP9xr3Pjh48fPTgMLi9+Md/IELRC35bLr7G'
    'PoceIST8MIkZAPS4ZthZlMZh5h8Er4FhPiSAN2DeC+5mQUl0j/Yz/SrLw4KQdkIDJnS7nxK9CAgVp4tvinhGtwS7luHo4KdMZnjj'
    'dkXFECfGB4nvaM7Fa6whJtoWjRdvZnRysr/Pbv3qweGdwxpeAcAjk8zMwBBy3Qa5oes5K0HPFl8/B92jceWO0GXF9nkxeZRYlmOP'
    'wRT7HlrgQim3AA0EpBzYMOnx4p/k3AEeHjKxBIgeLJaYRWDKFO8xlU0cfTZYfUEgPXpKh5/gDYODZRJNl7eY7Qed32SnJqWlDsp8'
    'LEQf+6cd0XMAWGImNG/W4UOJng+TsojP+LPILZrH0xOWVdIfx2UIYANKc4zLawn+93/8L3YCULSYiFfIU90wA0VgYIYb0kenIhqW'
    'M5q8CIrFGz5PIGjqSE81fVqmNO3TkkBP0+gC3VDAOjoFQq4oJRKknPHuveDUDCNchykTqsXXIDl3M6F/ZTQwIFCWwNPqiWfePLwJ'
    '6u9/z/f1wdHx/cPb9O09HkxvK0Pe0l+58pb5HfNtqQskTCnkT8cDcCi0LZqf9plnWCHwijkfj2UUIYFeRLD/UMZnGahdL7ifgYBn'
    'TPs80UDFBWKVQ0PMngb8veU8YfZ7Xjdj6Rn42CwvlUwHp+WEpIGAQEpr82A/t/yCPuUNA1yWxUYySVf4KuEB89ZZRPeXOCGEuVGe'
    'TYKTk1FJ9yw6OVEBhAQPSB4zulXF2pp+RuSMn6YJDElqRYELJ1+5j7rBKI6SUB6MiOrZJ47od/l0Np/G6dh+fovAakg26wb3ppgO'
    'wsoxgZFgFa3JC8/S7KskComV6Du/joantKcHDqY6cPaMiN8Logz6nGMn+draGi8vsLLrOtazsS8s5FZwUOPb/On9wwc3bh3exld0'
    'Q2iShC+hfEnIe8Ds30mKHZri3zsw6GyHThyWmQjgjypm1ZBk50A2QoWIuNgM4tog57OvsS9c5qxgXO7h+AIr/e0TBc+VHFrx3G6W'
    'P6b9EgITXUwIwx/Tw09oC3xY62E0MmUyOxkZmjifH+CJDf+lEybL6fd711HvfRL8MjOjVzZ7m/WvTgbxODcTkOvmM4Le+w4tHvOp'
    'Y+K7ROoD998HwSm+4IsziZ7i9htHq72R7M7dMOesn4edZeDmSSY8whtUWMQkI9ooYAcu0Pnj038Pdkl7mytBHVlRNF8vomS0EVz5'
    'Bd4QdJDTIjikwUt7Vr3jW8Ql/vd/+8+drju/nuIiffFH/+O79/jR/7fz6jEG79mTf9KGi1848dZiYwNvfN1NYVXh7xMRjiwe1mC5'
    'aufpSRFXuyYau7Tropys94MRnRkEiYA3UVMh41Fg3LaCuLq/BKaN1fOm2Y827917K+fFB1mOEU4qWaFaCCN4tZSUsIa0rvWlyTea'
    'q5VHAM3gI7oeuyQ+vW0Da8FF/lu9S0W4jY3gY0Lg5+vE9Psr950QuY29jQ6yLKn2SVTqu387jpRNTssBFNg8mLEmapJfEitOoHRD'
    'tjs18+Du4c1DXwcnidgSOv8EBST0zgGRjFUrq7huUS2PMbfiHU9qS3W/Vw+IVkNkgOlMAcEshBiPNYYl6+MgFHPIOVbGqUZhlTl4'
    'eO8m6VDeIMLmhZCnIiIwV6g09SipSJobD7JIufgGiqXPOgS2ZprNTQ5VjhZGc9BtP20IOCwGuNF4HogzKheIkABdi9RSogG0hCQC'
    'WYTxgRYTimqOjU+zEKqo1VTdkBiN2KWIpCqGieVCBDVB0hR6eBvUz2itDFti2wSSA3p3tr7RDR4/cY+sRvr9tYshN5HNpfvBOmFa'
    'RrUv9KwOaCBrjqDJ1x/bP5/IJCqIRQlt9PGTjdoQWO0pVitjLU+MB3J+oOdh637rBSZCfhbRctbz3iyelQnJQHmvzJONtRW3W14g'
    'SQ4TCGz3V5IG+b5nwnCd39tY+aScTs9Mp1EarudLBEu+X3PXkBDGUsSscQ89DrNEpR+b70WisU2PPYAJfhBcubz/aDQ2eFT6N6v+'
    'lzzJyfHDe/chneTZiygF+uN+GBjuTDKGAW9MygD/a/RHIT8JU2E35H/oV7p5ifwLO6L81I+TbE1/8gdxAIuFwbCi14opYZJZiyro'
    'NP8DKy79oFdw+Us20+DHLFtTDZZnk6kwfiCTsN1hRnQQtkn6P/0Jqw7/TzRG1lWgLdH/C1am1vDPwPDvqVWvihk/OouszRS2lxHs'
    'BiXJqXSJDCuV+CfGr/gEBBKMJTFrWFYCasS2VqyMRE+idsEkpv/RbyUkSf4Xf8yDFIQ5Zd4E+NL/M1iVgizAPwSbNfzDv2aBfMWm'
    'aGJDGdHIYcbqqOrL8n+i+PIvLaKI2GxA5DdgmxGshWm2pqI/88UM0gr9ryCOORngZYCDACD6rDIQggWJq4b/KVhyVfF1VtL/iGel'
    'a8AXRhbClICxpSQ4hvIvrYV00jMzYUY2J2Uk41UBbmGWwiipp01oAb2xVxDjI9JMQsEJad6Hnz04JKTNox5YQpxE63nnsbnyYvF6'
    '8fXizeKbxbeLf1n8r80r15981NnAvbzB9sIuo9rbDH1An8oOjhu++JcMaEaEYUzfm97ayd1HMFM2l/Bl+PjLsNd98uGX4X/4MpSZ'
    'H6VEm0KeYWLGKdFR/MF8iq4ETWXYn7D4FwPSYy3a88DMSxL9A2PdDTnN+ujurZuHN4+OQZM7zyYkm9O/3/1P/JwE/K8YIfF5nCy+'
    'qf6kTSYgY/p7p1sntp1TM4jZDcBPYMP8yxhw4t9+hn/OoiE9gDOgAzj64tYhLIaymqKHB457h/wzmg35583eDf55qD8Nse/n/Bvs'
    'oL3mMngp08XrcU+2Jk/2BJD3fnNPKAWblAmVB2UxNGkvOEpIFI4n0MNhjpxBrhtCcPg9axxEtKcJXVJdY6/X2/h9YGjAKXwR7KIh'
    'AXHMRo5CpCKSRfAwCEZRDkhailI2qCt8MQz+GhqiJPIHDVdRKrVE0ckSQQnZ7lz0gvt5NoNR0lRGHiZ+UZ6D7vh27ISGc1hKoz/F'
    'A2ypz4kswVZZwDBD6pGBVlvwvWHrfAqXVgL9VoWhM+yUKCWTamfOxF1LItBaIjbh4s04JjjpqZ48OKITfUk6Vw2313/56d8+pht1'
    'eOX/wUX70+IfF/+0+M+L/7r4T4v/Vr92TzY6pDjQy7DPTqN1s3GeekBP0tgXH7qLkW/96u69B0c3Do+PGmM7Bl7D0FdMArL0aZkq'
    'Nxc5ERBORY8qYv0mBOdggzYoGFHpiR4MwefGvbu/qfHK9c6c0R3/gAYzjjNTw28grnzhDF0axuHPYSEqYHUZCHfDmsawH3eD41t3'
    'ibwmRB1gTcvPolRoOwvMoD+wH7KtlQby/XaFGQhBx5ND3aWYiHkA0OX6HlXkD1maICLEJuDC7jliK4yQv17whWEuoUuzvIVQuDJ/'
    'hYyweTQsp5GgbMKWbqIoXbUXgoARnIqO5fGdxdf0B7wjMHizQZcA/MXRg8/uLUkjuIbwwFlm7jHgImbPTc7c2+PdysgHNVaeG/2R'
    'rtW5Ge3DyL/0R3mW4Z+YxxcXGP0b5kZ/pGunltUPmOkP8NppOQDdGTATIz59Gr9gOhSlTjgY8nP0Ub42iZnxEK6RNBDSt/xvio/i'
    'jP/I5V9i+ARVE/C/qfwY6J9Yn/yyRioxATuQH+laCqE1wL+8CfolWjuLSTTHP5DMiXDwv/w1fsnX2JeFf4gDxU/FZ5evsRYmulgq'
    'PwYk1iRmArsy/Qu5NM0mcSoUSn5dw4nDaKY/U1zLiZF/9Q8aZlSm7M+Wn9DzZiV7VPknfGBDtqbC5W1/NWvQuOlQaLHut3TNJMQJ'
    'XkBA5Z9pXWRoQaszphRnTCnkW5LvzzZa6cTxraM794koWnqxdJ9EN2fvmtVQ+fLs02gk0cWQHod5PIiUSND9s1TCXnE6bwy7dBeh'
    'tBilPCefPzp68PCoQYDOJzvVFyw6Pbz3dyf3bx/eXZJcQH6J9oq4tAYl6sS6LU9ArbL1ynrXMLpYffq7f/s1cSESYPh5R9oy8cdA'
    'gqRVZ78UNe2IHVGLb1P4j6Zx9EL4q0kXr2nLJBrTJkh/H87iIcGLwwNCtlEQdOewETAhZA3/D+XijTiAhDSxQM4G5H0+F1YRUpYY'
    'hPmJgWJcxvkSuVOw1+yT+Jg09TPxXUYQaInPKmbmXflETwrvX9sRDK/IPBFBNUOnrHMMWEfgk6cdEWbMojx2tgsNuTDwh/ZqEFY9'
    '1aTz9ZmHucBk+dsdb28U051KkvXKJSEHuLFhzxdWkzg/KVU8bZ6vs7zXD/k+21o8L34a2BECcZNWMQ374jCucR6VagwzEgXKhLiE'
    'yEPMKxAT43ExtW7dV5+dd7zss7Ue1zwvpwz/hD2fOJXv/tnzUSrBt65HEkyIUorgf0EX6Xf/2nNGbRbq4Elas1ayxbcmgLrkeytB'
    'jMMqiMWQoFeA94mwB3/pvmpQ0Ldq3lbnHq553hC/4dm75K2cxEPV1hZfp7U1Whg3qBXvbwyqKyQ9YLdxMSxpAzkdgzhxa75WkHFB'
    'hBq0OuyEhdu+0+ZUrhbDXhLBFCaUsiz43UjxqVknJWTDu3uiPFfRW4KANNWMbYXiHCFemQ1zxceMCAuJK4o5v8PUBHYSQuDLhQTN'
    'WjJQrUDAAdQAgWnChFfVQNIBdna6e4FTnSS4wl8GzdY5LGKLXazl7uwgBqRTvYeT8rQx6AmhqlU8Igy5sywdive38s3mouBg9Rzt'
    'QiN3XXwNkz7oqSBkqT2snqWtzLw5WgkqQ+Tt93OSi2/dOAQEgYEwGBC/jc+sA9AqoCzkKX4rBmeieMg4Yl9R1YEtC1FKnw+ZQ4pJ'
    'yCksrGRZXyEsifOIozemdJ+LOolzgxC1ZxZF+pcoHwdfhhvd9V/iB7jal883t+inEDVZZV7ySyyXFif8Z7HuDSiPEZ+H5Y6/XjYB'
    '8njqYWLQwRz7eFPMwSC0cTdYn8JpHKcxnM/pBiiv8pBZtM7jblQD03Qx+wzqhtAl++8U2hyW71FsHutxHFwJ+k9oDRu9JPsqytcr'
    'lSeibfL5HgSf02FG/qy0SKxsvdMFuD7pbNQX4L36MPfWESV4VxbDr6uC03j9g4C+UJORaBkNOh+zTiqigJGLBe8m8L7XGOm7fxZk'
    'uhthzeyWfrh4M4hm3/3rPvRTift5wbq1mMf4kzAipGRRfN4YkGd1jNnn710VBBgvl9fRyi8uxCtoqU348Aidxuq7ogT5G+iycgN/'
    'TT2atTEeE3RxnngRK7jYRUk0BWFIjrwRAvE3LBvZqL7GeEpJ5IbuqzjA1gn4h9gi6Rw+GCAfgYlJYBDzocY5Vgd04N/ix3KLHl/p'
    'P9mnK/Ok9o49wvobdKn2nzR9K01x1E23QdQmXPpaR95Ydj9cEPF9iftiN4det1817zogYN0XBISNteWP4Zr1aZV6P51wdWB9Umzh'
    '6AYDLPJFPF2XYbo63OP+/hNv02UDtGZ/8KSyhykNJQqx0SPBhAbrBF2iFP6Wyvpe7HrssssNXzJ1q6U1OlLK0madKLfJmrNymkSQ'
    'OEFaZ/zPk4bw+ZAHERtuGQi14UstBEhdhHwDWAxEkHKWx2NSSRPxIhLCRHmSqWRQcbM5UygWQguVQguO7FXHhCyfLbbwaWlAJUhD'
    'GsbilmGVQeVY5XwQ2SpxVe00Ia04VPdkeg5HnQvvPRMTRXWIMBUZBMjpBXaRlnzZWxWGx+uT3jjPyun65kY3mPRgA2E356SHM9xY'
    'diFOAnaIsxoB7QQa4pdfnWuZ++g/PO5+8sSx5Cd1NUPMqt9LyYDZgN8SnwY80ZWSNorlhxjmbIwUVDxj5Z0OLPewg0DW1RjViZkv'
    'viVBNzGdrgbXQ7gU87HnPeYBs2EJX7phiXKuml+U5xmRy6hyDDAipNlkAF4BLhhn8G6w8F7Np/ghKoFduXeSnaMkOEI0FukubB2q'
    'RxWzKZnJMwOUtWqB2VHC8qb/bq/Xq4neSs7trMwDYWCUewOfU9dXoEtlADaoOBAMqwe1831ErN8wq4O9Ztw+M020Xpdw6c4G+Lg+'
    'H0YaBFu3andVL0hJyYD3LXHRzxyoAHMLDP55JBGzEhlQvwBCAw7kp9K4mhgoCLkkBqoE+AHOcxaNo4lGZimF0HhSmNfYHunCnuXx'
    'uFV4/bJXk14369Irk3Vsfsl4ve/JiNXozlb/mN96wnPV7jEpYQOkDhCSHfAz++zAVtrf02X0O0QH3MDKddhSU+ge2Iomu3jc+9tf'
    'fkIM5svio07trcpLX2dUbJaQ0aptIOpuVuNCgAWtaKP2Yd9+6B+bHh245Wwj+EXQr7OneqTArMacdH1LrMnGHIbRBcmT+BIrQZ1I'
    'ck+zAPgqXuu63/iXMBpA3zEcJwyHQmaNBAK3mLN2Ki1OCZNYDNlpHKuvQG+WWs1zUgFFO8dX1mxVRemwFYK92zKUmJ+YJuoFfhCN'
    'WQ6t9qKGGwmE1lwFlYWNJK28CHq0Nc4yMgWBrnKVquHCukvXebEHvPgu87gD1a034OAQ5cEpzjTmmfFItQz23DiZc9vREG9uJeG+'
    'r0ygVScDbagp6iIum/hynclMLmU9cI4ftjJSr1sXkUBH0reodyKbMkS6+gebug6A4lhFynIY/+aP7b3Hcm716lL0UdrLsQVc7Y3g'
    '0+pPWu1+JfYzmnZ3gd4MqAMee0lKxqbTBrXw72dXL2ddMiYVdP8CcYC0ClyRnrcK7Ohti+jKIuoSuw8gEJwV8CEMsbCuv8NRVDR0'
    '7WnmZgXPLuSPXt9owhs0SB7cgIK/xedTffi4/4Q/3+bP3We9uAhJIiXxa791v51O72kWVyMvw24Z2RuH4Ha4DIZVUAXK1GfRq+ru'
    'c4XJ+bw+5hk9dSDRpuvpxjnkeNR5Sc/uj1/5Z/h8GE1nwRcmKaMjFq3OGSBtoedOzBTjaQsVL6IGEb/fmsLUrRF0mKwQHuOylBoi'
    '9cuvmIh8xQREA1LOMbo3cecrGyQn4VcWc74CS9t6VYnO0fdgS/eNBrS4RBhSYfH/h0d3b928V8uw6brgUdZ2UjYlE0pEMzXg9Hw3'
    'jZcvJVZX054DxZZlsX17iWKsNDG/VGN5hGlJXG1a9msA74qDBwYK0pS+UXulJI+ygV8M1nkMdQ6OAJHhVWmK8kJsxxOxjw9sqFJt'
    'ZfMq6EKyl7775xb7jpp9W/wBSNaipdE3m8HPBPJijDfBtevBz/ZFZORjIWXkhJQOtep0EnOCETu1PLkqgETEVkw3KVOgpq/whTH4'
    'OfysKcvuwup+W3IwbW1/ZxwGU3AAibcWjGPjB2Dr7zz1Ny0LE4WAjfonnkGfnlaoteWBpSKjS16WuvR5nHtPJQsuYR/6zPe1qYqi'
    'Ry1atU3SYAh4YQWcRiVRngg1y9lF5gKOoZezQnP/0d2Hjw4f2HCIokwQh1AbeKjjOHSD7Q0AlRNGumr2iYzmMqebjomupG8lsGY3'
    'tA4217wzhXgbfXhS0+2JqppXJy8HrzpLpqEZ3aKZmIR+lOjUmrfikoevJ5GEljGwZ/okTuPZyQmH9pLgzOmjJ63pP7u7LRFmsyxB'
    'GMkwNics+9P4tXSgzbaXdJa2/KHtzY32mP6Wlblw3sniTRpLmvaKlFoht4MorS1GCUa+nFiL3YJiteTRsi9SKWJdWpAc4NVZtaRn'
    'zOKxuO+89NrK9NsKzInJx1HKTIJdSxKGxOn9Tqlgow6Ro8pvZf+7UcIRlUccCJpVqbywHrBH7gxetu7uNedi2+itrT6ndpA3eWUt'
    '//QPjcB80eBWg/6oEDOGeCQx7oOj24c3bt27e3RcZ/0er1Qn8flJx/vB2eJrdrQnDQUDBNkgbIBW4zI61dD2+/ZsBw5qbyIloXDz'
    'o/oLdMLsVVs659ZxHYk9aB6FNan8cOIQfFDRAvaR5rKmTCkC/cHSEtTsLMF3NsGML+2TlqyhZam2kmnLyYXlVSXKdWcca5sBJ7HY'
    '1TTlwTNoCsNlWV3Hq/kZRMseEuLSEpEsZQbF+tkGCQn0c0g/+9GV60sSJz8UXAmGSLHS1z89cCd8salr23Nn0EA+PQT+XY9Bk1Fc'
    'oq0m8vE5NHNWm0j73b8dibMbJg0R1rzMePZycTZCIEZ1l/L0yyor6tj3mCVV6AoCOILfVZ41G1JR82ROPc+ZVb26HMyN+AmQxYgl'
    'E85/tnYcngAXfBqNTVNwqBCDhU4iGzlsmRrxdQanVoiVcWURDsGXmBlbNUS/NizBuaoEtRT4uQqCIo7a5HmeDgFzRWzjj4ZZghSu'
    'dnqhto22VCLFhOrw1tWg7WU9dTVn9YATl6uc4qpATmejOiRJeA8UJU/E3p/bYWUN3uMCT1VmbLYUDDxLyVslX71GtBMPu9HcU7E0'
    'fWmn7iIS/KBculf0kk3IUpHNpv8up17JFy6Pyb25NKisQba4XrZYCrzde8N54PkAFjaH6oULxeQgLWIbS4FZHD0uyXKc+NOtaqjI'
    'eMBNsdbZaB2ExhZR0wX9yIYrq7qSIAx27I/UiIPyLuM+p+h4Rj+48fJyKMIQKlSUtojQmm+qcOWEPKReLg7U5YzFxWvSqBJmm5xF'
    '44JbqluEEjgHipBNq7MHeRLGD4Kt/SXnWc5KsX9GywfoDpmVcD684NPaR1jFRnv2nC4w92Q5+K7xqealVdUj6m/1rEOkclnUvpYL'
    'i4SOUWclRXTxR17xnqBzXow/6Sje8L795wMmVF4mItt2h+WkTDi22MuKb81k5Ry1MKqN18xsDf3UVpvSyjZuCZKzm9OqV5zWGnhi'
    '9we+bd5mt7oSF+fmtNaRqlfRKf2tyVk5TkOSiwuUwxhOotlpFlbijqUJTMfaonFb2acLvVsZZMd+S+PxzMPEutKr2DFxiBN0xxqk'
    'bF3QSjdSy5FhrQBgPMT/7p/pwePFmzCN5nD93YmSQUZbjr771y5HAAX9a7t9+QO5WAHi6vyQy5z97XDGF/GaR1I53qyWqElaAyhi'
    'LLDWJOJ68YxAkcRjnbUAFxuzWdW9Kr2NaxyAdb4ggYHZtLUq1Cv6gPImEUgjySFSoqolSHouqMxeTZnPNNl6XT/8xDqOCLu9lUV0'
    'T3LEdIa16h5c/KMCm0dz21m/4mMzrEbQaMMzdVdmV/1OqOK5SCwEjohGJT6sLJ1QL1Sx1Vakot8oUbH56rHxa1O4eS1TFwG1sqqu'
    'FFBb7AAkBXg1QrQ0CSqEnCfM3tCKgWWqpj2b6IoLI+VEgPNnhFZnzPSEVLnoklpFAigQImnDkN9wXm74SOOeWrKOV4+RduY95huc'
    'PWlCliP7XLtQqnkl7R0Ep8J1TvQiZOt1eWdJ9ZJQBFKVWP9iqc56F3jSlL1k1cbdXBv1qLG36Got0zxpeIGSE38fPhSrOWuvAJr4'
    'SisJOIBWj3tQVRUAVliaO63tzZ6wiuAiGNR13bTrL7+xd1+9X9ebqkP+u/rGoA/WHjgnW0/CSnUc9pz1e5sbjYnPoPFj2qpoCFsk'
    'K0yjYdwnWIddgnti5RLwphumWsCSK7QCICiVVhNaW/aluQQC1QAl0Fr1pdYifjD2aPHMlvGYfFsDciBjqt4Zxq6UpWdtk4SVeW9p'
    'rIp0H/jUbek5J7GNOi5Q/OXPu8HPxY0oW9945UWK1ZIKOsu+wwp1Pl1hMfow2Ozt7V/Gkm3+nzoHlGOuV1O9rIynvc2fvdpoXa9g'
    '3Ketdqi2Uz9KMGu22gJ3915lgYOujwcRg8SCSMuAnN4kKRWZMwP2zi++1zLMBcvxVVVh6wkiLSPW/TP/+z/+l2qvkA/6m5vBz8Sn'
    'Yyv47V94ja0F/m5kKcuMdqnir+J0q7b1dTyvHJLAikry4qyvTts3CG9uGYvYyyCGCFp0JKzSmj+apQX3tcps0jDHVEM5My3k4rAc'
    'Rp7nrLUK4SXc3/VRp+a8a8sA4jIPtSqQqAC7dp7itc4Kyku9JHyJqlKey/DZWArpuAhFuCAtUIntPIKwfBcH5qkJ1t9GB4rogms4'
    'vnXuESzP35i5+xZ4L0FaoXzeWx8hS8OWtLWRjMDjBi+TWJVOU4ZwZXp884FnmevQ4A4WGxvd9pVUO//IktRucAVsWXnIkpVKpMK4'
    'kAJ5kFd5Ib+QLx5vPlk+EitISsWfrm8/bF0VC44HJHNzbA/+6Dol7sDtqWulwYMKWO27rNcXPJAf3WrvB+639veXCwkeOFjJHT04'
    'dWZP+eGfluo0Ap/+kxYTeotu4hXBs8Urffv5cp09VTrsF77WIQK4VT/KVBMtRUEUOb6ma7iKVPCetAm9Nc2bpFhGwaZDYOTstywN'
    'jjyDrB8X7omwCih/b/LMgSKBP++B/4cHrwNnNL501/eDKA1pbS+k5ncqNbIveZbjW3eRf3KXM77ZfGFNUNmwnAh/6mioUG7Xk6+z'
    '09AHHFIpkjiN9tlKpFkrS+jN6cv5iV8TrPY8ITVstvkJ6pHXv6sXmXRRsmpwrtUcRZCFzctEUfXfy9J+3/VK0lX17bmUJXtXpXJ1'
    'pKXjI2QU8kenOHIbwC21jiB7DxrTnrHQC706E9GF440kZl1MmWLPRpDonL9cc2GAUlI/tqUgxKkTcthRbb/Yh2nm6XAdtChfUTYO'
    'uYFyNPXAbR7Ji5ozPVvU81VAf/AVqAXRLbG/+hD6ihIiFwzb+TLVYEN5XL/nA6avse5aJTUXOF9DBdawBCnW6mF7wUc8x5cpSZg0'
    '1XJFvcZb37+KXjXPCHVJkZ8Pub2qnU1X4bt/diD77l9JkDDWCr3R8xe1WozwJ/mjVI8Ulxxqr1xw9Kq+oAQ05jam3YK05fp5xHBk'
    'z8P7fq3mhBoVbYGT7gQ+V59Uc8ea0ZrXc1gxHApwtvBv7kNAg1ZSCgspea+KBFIxBeVsbty7c//Bvc8OHyxLP4xyLPIFweOX8asn'
    'wcu8BzLKGXRmnVRWKCYvecZXnTYHCtcibPeQyOh8ZF+m8tFLfnzVSAhi/z5D4fmWsSqw62t8ydpOZhkVVmPhB8HvXConiFViawBI'
    'iJD6hG1MHwfXa3FTdTN0g4az5wOu9FLrF+EKgUshC66Fpe4KfKU5AEhDWlvSCInkRqmGLNmrJxU7tIwb02Xeb+i3x/DiF9tum4+5'
    'wcuKH74CTyAqLq1cvN4gWiG9E3y5tko+ZxVKirKcGo+VMmvR3Dx4TM1w8U2vszLemHZQTuIWbrvEDOEPTRZvgA1ath2eSDYHJ9kY'
    'rnr53Ugy08i8cBKYLdnb5CC1MGgOq6YHUK/31ccv01c2a67yzjWUD3rh5z/nqBN6TYrpyp0d/Tz47v8LdLQ0e1VrTfPzV81R8LCr'
    'df3SLtQjURtgUj+s/w8JV6N4/GN2f3pb/6etre3+Tr3/0+bV3b299/2ffqr+TzcYB8qqRE+iMQBo7kDXJqSrNeSuPYmEaGuJoP3g'
    '9mFw+OC3j249PLrBvU3EAseBNrYXEWlxdOXX1m9G0fQ4ip5d+WJ7P3iQ3T8i2v3gzvHdLJ/Qb8dfxb+6/Yh++dVvD+nfO9kRi/0D'
    'k5h0GEkPpyQrijVTPo+T2ORIqgOlObqN3kmHi/90L5CAw6fluIxmkVIAtuRxguRcfubcJ0ayuUjCJZE2lQLIv7r/CO10UOVF/Mu/'
    'n2Xz35Mc9XvUPiknv+eGPdmQ6yOUPs2x0NoX8q7xlwUqqErKMBFTo6AZ50ba+UidMnqcHZthVISRVF9Cr4pp/D2aOVyghYMpoMbX'
    'WjnUmzZYf1tbnfk72JygyH4VYRigK8agTDhC8mOtPwatig1D+I7I5QsS4IjoEpXd2by+5/M02vgofmrqZe4+I6RwB5YiJ1Tk5ROG'
    'rx1qa3evaU4kHDmVMIRoMojCEDsjshqHpRoy05PEzKO8sGNck6I25vlJEf3hhCirN7gXRhmQsKZ+hHVCzH2JOSBVasAG07/7+Ate'
    '72+74jevUHzr4y+2Nxw40pPTyITV5C2mVR0xZJEjn+trz87qb26tfu1ZNP/4DI44rQY4i6E6EKvcea6r/rsvhKtl0+hkdhrNvJjs'
    '/uYJ0d3epr/1zz+/6zYwGqUnkzKZxdMkqt7a6u3t7V3VYCi5vzCqbxNcSeIaRsUnwbWP8Zd2Z/r89r37hRvvNA5DwN25elGPwDWD'
    '+ADOd/6Vq4cmyPDWW+Ivxl/wnfg5bkmQjYKj5yhdV7j1f0BHMzOk9qoFjUaxZ0WURtJIIn4nk0qIR8cPHxzd+DXaZFVrteWU1Ln7'
    'AXRkrizcCw4RkIHTkKR0Qt085lKSKZK8bBkXku5gA0Z13+nJMw7kYOTvrbmIsmlZxIuvc67b61bENbfMxGiPMz4aWk8hsQlKZ+np'
    'JJNeaz1FHxlgFd59UE2Ql7OIa95zKIqZigTKq2xHPYt/5eJ1qtWJ7N7crkQXPiXhNWyupN+2CJsMzwMhFN07og3t0BGdvB11ahRB'
    'zB46h92nv7KLoKJHXnQgd8dkrLKIiEbVDUB1SHFEL66jOztCxcXrIh5mbmtMok5GefSHNjjxRgTghkvmaFVago5/C/JozERZM42J'
    'NCOXw92EELFY5Wy5OwtnfBSzsPbN1pqm1p44ulosb/IDB47co7+h0HHDFh+/gLysc4mzB5azB76osOHd4SN6sBwLc55GUiBQyxsU'
    'Opx6c6Z0hRDYEayLAjfmZbi7ohb6Dxhtud1eGnSI7UvdII6GG8czzY4iaEEFMdXhs2GMm0ZoJwR4UNZccJxfSANBR0p+/ReKchBx'
    'WWW3bJIYxkyXzKmtMULwOAFggBF11OKv9dUTrPskyT1iHl3ZUr2x4CtAF6qscs9FSeJcLbVx2JFmSCqZeddAgvm9i8A/CYn7Hzvi'
    '4kX/nEyzYuanDrGqhndqFjm2YHtUVf0bTY18sviWaTxvob9XiXMJOyiFw3B5PY5ShDUgNEUjJ7k+0wGulThxVKZA5xX7kOUpwccf'
    '03wb9FV/b2nddRK0au1LhJWdvqSq57OquqBHx5dX3ZhH8hD6e91qB963tOAdf9UbS8teInbtK1/x7EETkkvj11HonMEbDx4gvIRo'
    'lHZ/sTjlnB8kz+YanGNP7Gf2YZaMuPJbN+jYb5FIFITxWYwCv8ZKXp3WEe0QbkQrbtlB7QNLg9oH28cV7m5zP9yuunDG45uqSDjn'
    'XcqQ+tDKrlOY7ySMJ2/pg+RD6uOPa/tcNTJt5mLj2iXYS1PBoSICs+wEqkY1Gv5aGk70EXlIl8W6RiOOEYqKDDdMCHzEm/AHD9vx'
    'tJJONT46ywFfX44qRxi92zs5cVrNCStBxcnJq+aq6MH1Dz98SSKPFC9+1pX6xWEvnkWTYn0D6P4Mn/A0r34Mv9d9JOxBUJK66CZH'
    'kW/SteEwdXm4VhjsXfL09x8cHR89PBYwS6Uw/AYO8NJj3be11qAWdSpU8FUFO5XCTCRBHxHbT8NB9jxY3wpu3H/UJXnmV58FDw7v'
    'CFfXIf949Y5kPmrtf+TZo3UFJAapdhRPI9D3yr0KGTtOS9f95gOc11cmD7vEzIbP5LfD0Ez+vgv5yEYY+ZLBAMkMzrSrniaC7rwj'
    '26/c5ZUyewAttmv10QPSFbtOszyg3z19kr+sIkrlmhxc63rE42CreqAirQe7/a1uQ9I96G/Rm0t0uTmHkhDMwoSGJmjI3wd9eZ4E'
    'IAv8/s4dlkRUi+hAtuKQHMRITW1F98CaSiygsneG07UfEU48048CJ2SfESZp4wCbBSNl9/eD7/6N0BQ2K+4pYqv/aydnwdqnMM3/'
    '0kHvBHw/+ilhuAQ7VVoOWDVp7hf3/GZknSJEsV2HC81u4+SeCi+CdSnt+5Q1VyLeMULoCx1Nfb3GFl4yCfcuRXy2mNukyxgHZnXF'
    'fyE1leLck9prRGN39w57V2C+WryexQnbC0lOJAqD9ODDO4qqREaT5DxAX+tf36oAzaBxgGYc8yCNby+OrRfDzsaYDgX3KvRsXtrt'
    '3c07kgj2q/uPAi5qkXCL2kMN+whCd3buaiPnZiKV4K22IgUTpW2un1aivfA4ywPVMkyq1TKIU+ZxdIYQAwaumEbPg25/b/vaTgXe'
    '/ubWjgdf/OHDl79eAnB/rwbhnZUQ5sNbhnBzVAvi7a1lENvaMCSdnohvZD01k8i1BQXl6wYffpgR48jjEGllJI4s2Ug7nc5Nm26g'
    'ZTLoOwayVDJkJZaZvfRMcuMF2XTI6ldURQohSs1MXAKk5dOVAGNiUiz/LppzFP76qCMjc6mBLM04VvolRvjb/FUvuOnSQ9mthMAj'
    'HXHDBj3gsk7gE3/54Yf63WO8/6S29Ve+m8wDAUlRMoAthY+i1KSycaUT+nh9OBrv+y80JEUX4cIZj75wQMdlCwUf3nh464t7x5W9'
    'iXEWoZazvPTphtbshsmfaTQXFXHNm6vi51xIVApUQjzIjaszmkfqbfQME+qv3bva/8wtih0SQvrZGBZsX/2saeWzeyMB5XY3+AKl'
    'z0ZjK7J3+Q97O+Sv6i7xe8/O9BUR2/mzU/2ooaCdRvp5/ZbId4V+t3RbhMhEkwE98AWJ+iKOZyWbhLgHIb1VNwmJc1MeVsriDOf7'
    'wR+6LEl3A9tpfgYJPcTTwUfBFv/8EPv6CP+4MbAiZpIBSnzJQ6eryp2NDTw+5ZQIDRSAdTFHqxuXLTg6pcDX1zPld3+SaHVZNdQn'
    'bbElEoCRLioTfVSfhEbUKT7yAV596+YtnCWQ0Kc5nqiO32MsB0T6ThAKVoQsskeo7F8d0xbY7i1agv8WL+n8N9Xo7CyZhTrXH8cS'
    'BcMVb006jtZvsya1ju63/Q1Svy2OVgZQ1sCfbLTOuKkT8azVVLeDK7XZ17QDQrV5xryPPMQSs+qUHSip63JPb8h56vP4o+50wRvO'
]


In [ ]:
# PAQUETE (2/12)
PAQUETE += [
    '7MqXH9nMdGM+4nvyUXNxH3rL+KgOog/dfB/pTePxFAP8MW/Ts+veFtwBbfCrtdppVZSqkqYOW5TX+S8vPrmjtvYTIlInTKT8BzGF'
    '/7C78PqQW11tRL74WXrCQ+iTvOwPg9v+g9iD/8wSzKpzWoIZ/eEPxSeCJ2go/8mlR+SY+Sl/MnkOnB910E/A6E5OgIKdk5OJidOT'
    'EzUxqLc08wwexM9OJoP1tobQS3l3X8WzU2LvdCc6H5OSNfx4EpEaO8o6GwGCgdp7ybLmi9jU/VVNYfGElJwuMMN65040OTwzJBJy'
    'BYjVDWIVXQB9GUP6OXFhxY8DSE3Nsiz3jlvy/KbwFTfGlHzCTmpSW4KiKkoB0qZQ05q+WMGoQzK7/5R0NapUDJJVqm/3e5ujV8Gd'
    'z75043NyH4SkNgGJiAjN2pDpvFRO3LVlGcXPop+yzfwjOPBt4jOYl8a7sgsWBoeu9lEiCEy3t6r5Ef2KDMfH7kY+YVMt4BxVZmZY'
    'UiBlHi/+icP+5L1PfbB8yB3DJQTw7r2OV7tYoHhAeMuy3ivC4AMviNU+4BzqBy890eNVIIRNPrQiCH3ajMEbdUQK1+f4j1d/+Fj/'
    'tEL6K2LlwXD2XB7zRPxX9dKuHnVvIJVbLXtoX9Z46qvKwr5+qlPUZJxXG+1pJCOYYK+8dAz1VTf46OUyJ33lOfhWjGPnXZKfXm2c'
    'Gzhc7avyBOpQ3hitx9YmBwcBwWb6+Of698+f7P+CdKD2961jtZKY3etLfMANtHT69M/6y/7m5oftr31cWwxu6c82WlcjqK0mw0nM'
    'UeNYDX++/4vresEDENbv/o0vBmu0v6RH6I/lLf7QSLi/zv9gF//4zuHdW5/fOjp+eK/3tMjSnzb+b3Orv7VXj//r97f6m+/j/36K'
    '/yArdricawbhr9Pv9XtbXJ20M0KDF3y2tbm1d6W/eWXrunxxyr3H8c3mjrlGBGtvtLt1dW+419fvOS2Ivt/d9v4G6/Xi7+nrTf6W'
    'iUc0wQciuHbGUUZsdsTP7FzrWlPTDH/v0l+v+D0vAI0+397c5Cc7Jh+egihVw4kJREjMx27sE9d1iHE+cc/TG4P5jGXm/tb2NVer'
    'skOkfmt3D9veG13fDkfXB9tbV69ev761LVTyVff8+bhkuVSoWjnj7la/3zLhte3+YHN7e7Q9oP92B+ZCE47iIh6a1XNd3dy93jJX'
    'f7i9fX179/pmeHVv8/rWzoXmOiUBxaTZKYmc5vwd7lzd3GqZdTQaXtuO9rZpyv72Zn/3nFmBCSfcifXceTZ32o4u3Ly6GUbh9au7'
    'O+HuztWlefJpWfRmz2dtQ271d661nY4ZhVvXos2hMUS3rkbX62MWo9k5GLZzbXdrrw3Fdq+HZmc0Gl29trO7szuqj+liJnnktoFp'
    '2J3tlnF3RjvG7G1vjfbCq/3+5lUdt7pVQgpOcIp0s3JLE/r+1a99G17fCU1IZHw43An3BnudtVfvOfGfF/+/GIX8kfg/3fadfoP/'
    'Ex3Yfs//fxL+34lD3GI69St03lc2pS2MMuSKFX/skIFbsSPMMGWG3Pnu326Ui9eJBrjrY9CHP5dytr/ECy5VFG/cR5RqwXmntquq'
    'NOEsbfdS0lEG8GhpRXh6GkGEpWQhoLRxgoLFQwTAEu4uvpnQ7yi0XyaIlEg0Upgt+kU8TpB03fXkksed220rxRZkbZ0n9PgQLUhf'
    'sKDSu34d8UDxWQQq3hkYBG5iSC3yQp9xalTWebW2DNGty4LoEfeKaIHoHRPmcchZW9K2FI1JBVaSvTWN0sUbVGo3QTxYfM3VSuaN'
    'CRjw3EhN3nmUcrzgUYmIIdOVPl1FsLe7qV2ztKMpGtkAMpIKYfLzYa17wNyyajwtv9mG09487MhM0NI0vNwz2b6sMzlEJW4S7FpO'
    '5bOSIz8P4xwZgpwikUilGS3rcljOCMKZlMz2n+ZO2Bx2BntVmYZxLn1x+Szz7CxONY2kPoe26+HmfHkezzLb9hrZ2hnyeHBJ5tUG'
    '6GE6nvsJcYHzj81tE6/4k+Kt2+csSh1luLx2ztTNeblnunNZZ/obM6VzWT7Qh9mzOLPVyBevc4In3wSiRwiJMMu0S9tL4OQmWoN4'
    '+2qtc/GUBK4sNdzRlTMs2eeJsj0Msywtc+12f/4ByZqxF14lnj1qXyQ9zI9o/CFHPDcWRXI8noUoeakntHt5lHDGCQuPOMxk+aT+'
    '3hSncTqeofFrL7jBZ2abC9KhcAfKxbep9HG21+YumtAHv8vyZ/ve73byFu409+gst7dAw7qs68oI2QWPIhSU5m14CxMm1b96ffNt'
    'FNPfa2MQ3h1e9xacZm8DGF7wBql68HqLutyT37s0eluiNkLSJlfcMOkADQOJesatpVCrhocIIgc45FQjLYChwRJc35s/kORFIsd2'
    'aG1PjhiD+eIbmKHRHdGWD+Pvc7tazQqTTP5JjKp2nIlo587jM8Ot2ziNnFAJvf64PXD/+ua1t5BjCwUuEqaLwxt22/Nq11KnN/Xg'
    'iAdX7Kh9yS2jXi56XL0s9PgM7TOSFl7Mn6OXhG2qSa8+WLzJhHamUYy0daDNf89ICCwRBGWRRYqjILKtBTPswIAjkYByXOZADBzi'
    '3ibH7KK5U2iz14a2NcftxbfonXojg49rHtxD5UnC22gSwU3MuEQnXIDO5P4eueu4aN+VdHw+ssgSmXHrYvF4c/MjLdZn57EqfrcC'
    'XuseLxcRrl0WIhB+m3DxehkT7s1m5ivDEhYzyzzjlo9EJlC0KcK9tQdPa2cu2DhyGUGl1WhZJrsnjUO6trYcZoi0vPBvS5Lqhuef'
    'ly4dy5Op8LRbaLZyr5d7Etcv6ySqAonLh/GWhvBdKdpMnBoKnpYYgki1j0j5GWdZ1qs+BuNsQEwYUHoqNffoHBpVFCVH3W81hhY4'
    'cSHNdECPP0uyaDLCscWpe5rbXz8tw5iOOeGWaG0yAS2nVpWya4t415islPI2qjilRkL+mmjxFujg6QYYnIytm2fUaW6//mi1dX64'
    'tvX6k3bveKyt8mYbOKQciL/7ZSy95mMpQkovjKT9zUvjGyQdn7UJFfqFHgDnch2XwzxqDMEslNRUOciAROgYbfVwdl2vuV0Nh4Bp'
    '0ITMi5VIPOc/afBxzs1PuGWTpTx2PIfpjhgtXg9nsP0HN6Q8DDq4cJ8TBQVzoFYAdOuFvaYoPSPVCyYS2KM98MJlm8oFwFLNo2rj'
    'eXv3Hl5CmavvjjKXZt+6D84LpoomecuIs/jTpJghu2TSeDUb8emjni1UkMR4UOK2d3xnPEh0gzqxmusZuX4v2n1mwjwaLMwrbkpA'
    '/rWZS5U36cehaaFuJ5zYFi7p/yvXv3ym0CBr0FDt0x1lGkW0+jRcfF1UbFNXdqmHe2mmtodljmiDFoJwmD4z0DHulMNTE4y1BQF0'
    'ZbovULMmgzKxXQ1wb+g5Ak08yXxBsi5H2mIm9u0WJs9q2fWt7a4t+YTgS2KFJIgxNkwX3w6SWMr56Sg0BOk2PA09wkzUEWnbn1JT'
    'UZmpYhSpHGGbynNHCOe27IotEGtH84iubdek2QMRo10e2WT5zxbfFCOSWs4XdiygsVoBLSPPOZBwr9QetB1T6juoLfhS5aP+pVkQ'
    '73Ah3KxFn7V82z7SlQ72MRjrjZt3/kFkC3vGUWGbbIFAs+WnqJt+ihpvLlZZqLY2Wy1UPTqfWTwuVd3lQpJmgtavN2PUWCPK9LkY'
    'Oro2kx071SVz5dubn0utsyw0ZzhzGiObR+cjiO69Vg3bfni+lWvpcd/i1djkj2Tx6l+aTfJBWbSJJ3eyYrj41mozxyYN7kdEtX3Z'
    's6HY4dDp9kO5C/IS4WfQcoi29K8RUePhGpSj3TGjd0tQhy32/e0VIFW5RMY5/7R5m+wc4JWwSaO+Kd6PfaNaO7sR3OrfturLPeRL'
    'M2veOG11JNyPni3ekGBQectw8Wg/n0Xx0zgd94JjksbHp4vX8bmbb/OlLdktmxT3LbrqqfoEZI3Bui5pg0/uQquSMS73RC7N3Hg0'
    'jqezFtp8BF0qzsUb0Nia8HS8vfjTSF1tFZMlYmUGkbbfJpYl5aHlYs7klO7G0nMbpfkR88l5YTTcr8r4hakXrifdL8EPWRkIr/X6'
    '0RG30MW3mJp5t4CC3Z9SWP695RSXtornvWWqzr6CQLtJLvXsL82WeCNL0PKgVSkcZ7S7rpXlhSPqp87j4Dkzt3p7O85tWvk0LWHM'
    '8pCOMcqNlaYPYWnq2tQ0URHO4ojrKac+m3e7qty3Eow+ES8vMIIWDsGbPlWNa87dzN8imNnds91Qdsb2Ad1k1LK9H8tb2780q+D9'
    'KF98u3yet+OJ8bznDO9CL+N9NHkfSbCBGoOKaDLIhia0bT5ymFIfLN5MzBD3ms5KyiXC2ju09RCJgcHgq/54UL+q3OIFjLi8cvbX'
    '0lpZe0Zlsah1ydlw8bVJs2rpl3sa1y+R3SVRi1EQFeMMxBffZ24/dS96B4Z+b1zfzXpYVl0p0Z0bXxqxwrsIh93NlqtaP1iT5CTq'
    'htKLun640JmwvPRtUSe8CZXaeGcq7Mgu7cm++34u9dC3Ls3SdisN20iq+C9vRslp7OIY0GHUqhahH85CwM0nER/S4htwFCnpR+/2'
    'iFZNBmZ+vsRB3/AyVBKyblorCpkVopALPHoLG5WxnUeZFya08+1Lk3ft9J7P7lJP89KMYIcJDZK28sgoR4nUXvBFlpwRpQtqzlHx'
    'Hl3fZCeXWDDKVIs8E1X8JABxxbXndO2zuKjTym5jGfTXA2cCsWpogAvxWZamglCn2Vxa8eRnRnpaczdU9v7GKap057Gk3Z/redUN'
    'M1uULcrRpumyplU9fW84hLnu0g/y0gxet2btTvUHGVjj50mmCfMt2mQexWKulTGUaPWv7e3S/epfu9oH7XxY5tAPEtHfjNiTQn6s'
    'T6iBp3sBJkM+ocWWhAVj2+ATMVAJt1p1ePKWezizLnIMjEfr+2h5tLl0dr3J0nXnUvaoiX36vq4tfBvx/V5Wza13NDVFhYt1qMRE'
    '5R5TNdeKQ6FgbR2ElaPzshYHuppQ9petKj3SSwxWlxj/W/dhL/g1TUK7K/aDh7TIYTw2ydRw2aFjk5yh78Y+my7sXwjgGJrcjEsa'
    '8Q4pC/RLT13lD+gbefo3WbH4GuWiUkNE1J9aP4L7Y0APs6F5AOXHJxI3M9z6oUllOGIx/MmY9nO/5BT/B7xfnqo0ac9JxPtWHkbz'
    '3DR6UUa8c1rwEFayo2EpWxIhWyS3/UBkTPVoWCeSdQD1RBrYd7JAr4q422+EBj7KS4LHfJ8d1tyMJ5OiWfLpYVFK6YWejbXcD2x8'
    '6O+gQqJqNdbIfsOEpVpV3vBggjZzjVv1a2JaW30vptDiDZt8BWOYonCXcKsUiYvHx71mlkgQp8OknNMCWkydTseqhzjKVroMTgKf'
    'LzvJ4pdJW8ue1aMs5hDDm0dHh/r6LtNTufUDDYEr7DlsUli+rDeldA+bAfywwmok6Uc90Zbb8GRkKxR1wi0Sc1u+6GoYIdFTrkON'
    '0aW3xQXCHWXUmhrk5DtUYCVlCvk9gQmFN+fsm4cLwhoj5iJXWXPIVOJhcG5iOkG1P9c1gkY649rIqdRsts43OFcJpyMO1zCof/QH'
    'LqGd2tYXCA+0lYmHfCCF9b+WKG+3dFcUJpcVIqp+0sojKyWz7Prt0Ube0WZqL3CBo0t4fG33YniMpzLBqOQciFaOTG2gxXGRoAqd'
    '96lAf875P82MxR8l/6e/1d/ebub/XN19n//70+f/0HmfkwBUYUM7r+I6GN5TKE9+Hmtylemk6BYKEvZQp8tmiMyDAZGTqmCydeaC'
    'AKccVQnSE5NUqeJL17N6HqZEotkT3FVntXMF3xtGJiXC1bVO+pqTWGxOYmOVIqwTNFAOuUMTBxcfpTUJ2qRIMDJPmST6axuSdCOE'
    'fcYO+/o0HGiLSYoM0uMR5ONYRre7IDZmMy7Yl1ZCpXVQY7E9Lqrx3Ws6tBVyqiNp8qqj59B6U1dYttCxCyl8xp2GG8d+x02+973g'
    '3nVg90axoNOy5LuBGeclgagOqSi10FkWyN5NcxY833pHPE+Uu1cP+r6QloARBoPzku7sdPdqItGzOFl8o/ZGqA5YMSIfSUR7HsPN'
    'J1Iazbi9GfxM9U1BRikcH1vnHkk/EfqTHcL1fkZwcQboJNir3vWkLggf8bKP71CdsCt3ief5IQmLW7GxyXf/0z24bBR/y5qWD3v3'
    'h5z29iWftm0G0hJg7OhLTfbWyHo7yr4A7Vp39+3IwB3nnI7OFQbp0Yk2dpcBodXUsgKQ6lck5sqKhfsoebV7dengevYWyvZt8/VV'
    'YGhikAXCW6CHt9yjHi4tgUVR6Xut6XLJxc5lIxDrAFk7vaATr0JVdnpXN1dFqzgfoSgsYmP1vRPVLeOcshbfk6Ua3HlT/flqhi5K'
    'sEIXF9S7trkiNOji9EO37ehCcbF9chxzY214UXhxv7ez8k3pnFKml8w9di8ZHUZwIrZ4Jo3HUm3qobRBqV7zz3FfvYEYQA45R1em'
    'L7Jilj0DEckR7432snT7r1y73t0KvvsfN8Qcf22b7eq0K4CWI7Z4Em4RDzURHSVLUvd1Dj4CGHsitg1x9B9ypYjuSRm02jZFax6W'
    'iAmdSRIMl8Y1U2mIFltZkBt6jJkOoJkKEo09JKmRSA6GRm22tqzjCnDnwZzJSuoBaBKHCIdpgY4GQ8hKjY5eDyuj5U28VS3v5ZLZ'
    '2t47oCEsL9yguS5nwVbsi++VVFhUVpZobFoTLu/b08zSM9szrQqB93Lzub43LZTOcCzhqqukbwjGDeFb11eKWCkC820vwwb9hmKN'
    'S0Z8bPxU8pQSFm15V49ySSyxbvtHvCbJXAfqTmOJt76xeF1yqPTcfnk3Gmu8D8dfOKfUaUTU1joWM0hIcTTOoBggWjIrrH7jGqFx'
    'ly1CtxKFoJdk8pXKSMXuS+MiwKxkLGYiP9XoTOP14TErNOL/0sCg81VAEOvuW4/7fBp8YYuq4P7Vd1ZUbdw2y8iEH9Jcy73XwpPr'
    'wvi+nI6IvTWu1XUxXxISWWNMTgkVqWvX/6ap9DG8b5gcvdalHsKe//jdjGBo7DvygM8CKzVXV1nJU3xlPNqIvpaKl04wIHpGeIet'
    'dkXeZGfpNBsihjhOGqnrEafKgdXChMQhJ4hCUg8u7eMMnDprlRL++Daub+H5xzosPWnwjz4kaxLlH71tK8peeONLqHr1+1tNhzF6'
    'BXuqHIp6swFXcU6NpC3Ife0HyhdhVMRVuzIXp7fSZdAQNB6l1QhFJB0dIiUiAefPcmBaUkZnEd9xzQAYmiTLu4K9Lk/VY5XIJxpE'
    '2hONs4vg/xDYM+9PguPF61POsDhyv3u6ClhyUzPoNvd8Y/Gn27du3vP3/gnbpN1C/LDlnbcP6AMRLH1Al6JcjqA7B2qGG18U0Qut'
    '/O9t3MGvLnstI+xFTlZlFAVc85UhIQiKmtZyT86F7btT7PfG9T9z+79fQPDHq//Z393batr/d/f2dt7b/39q+z+d92rzvyDD+TyH'
    'JUnNGsnH2XnsRqVol5BKygZyWMF6HlZCvMs0CqMh+LlyCMTHe2Rrr4daUc8mHBBahghP4EWAyGrimlSqYqV0wMUMYB/izwsWARzz'
    'OZyYFyLTFN5OapMhau3ZxFr3ZghZ4sRK1u9ANrM8JvGSxKzM5n7Di82Odtqkk5dbZsQsfnMSLcp14/DRzcPb947v7XNLF1qRKuYI'
    '+RBGLA10Z1zvrNQC5ZxZsGSbORL4Qc2N6vBT1uEWxE9IRNQey15zt/sW/3nz7SZKDE0ZknBQZHVedTfjNrYcmIdi2HJSQw+rqmO4'
    'DJGs8qQntIU3KPpuoyDoixkJZZ9AxDkFD5RUCA4CSJpimlyWrXe8LNBs4V+Bn8uvu7b6viTBEUnR9Kfg4rXetZ3r59dhW6o/IUd5'
    'N5qSTsgC2sPFm0E0Y30AsBMpzKYn0yxP5WpW77hsH84S3NrsIYMXrqYJKS0vROCLCsMaklwPxiHJXk0Mawk0DEcLasf75pJJWciS'
    'IWwNdwwoxN9FbG36tfkKeURJFIw59ERdd9Y0CXiSOBVq4oth5VMtTgPzlE1RuCyfIOK3WhgHl6LGB26badTMe8gmhy5PyMrYaTwZ'
    'ZDmhN7ukJMrLraSywNnUiOGpQc0lp3lPs6T1Nuq5yor0XFccqd4x+4YfI750rPqsB8UqRyQTa549BoIYn0xRDmgSJGNp1o+3Y76K'
    'QuJ0lOipCeUyt4DtLWaui2r6csu2350l2VSIcz10R7Usj0ox2Ntt0Qx4JJvF+hZXHJNg2KiELksy1OEsgQ1CMkqSYPFPaWh/VwdO'
    'kDmpf6iWl8Wf+I8W9HErX71jPc7qSc/T0rZJfT7L0f0XxjdtPs/1MR2c/H24TegOunbBl2XuFETY+f6IwAZMhcmKkIQbcNR7UUz3'
    'iL+mrl/2r+Mwp4kWr2HVCG6BxNl6LRUVY/BskfC6/1YIKWBg4bOwauR1+qffC26qGUXi1maIRcu8OAMYqtlCg+hGTuJ09MhiU5qJ'
    '3Z/FfIhRftZaW6wnRy5UQNM3beG4owrH7AwIja6eqmfFEQoBMJfkfRE82H13gpAgueaCNpGjujnUEYbtq/0ebUn9pMfc052Pjh/n'
    'ToqluzScYsKCWkHiD1F1W+BDJGB4h9mhEqLRHIdvgJyERpowFtwHN0iNFLY1DvZeuOJs8fUQkcwKb86bSTiG+Ej3izQmrCAsk2HN'
    'j6+LdW4cgRH7HuNMJIiapfETtQOzJy/PRux/rXJb5fXPTPxMA4uP4wGyKgR2/d7ezpaVWeZeHMCWeCgTf5HJ4s0fymZkwzL1q87n'
    'ree74gAVpesDeSejx+FgqKeor/F0Fl6+wFplxnqbqkPbH0Jg5g9gwVuD3eXYz+US7b277OolnDnp/Fzx9XaV0wqsIE3ARQxVUQh6'
    'JExJ8jyOuMoJKzUaa7Vfxdt3XTR+txLFONK+W5Uykgy7eRVKL1W7Gotn0VsA/+sYCQtzw6sUP/zvWLasNunCrbzS1YXEqMFPqUJA'
    'oYCiZRUorRwVmbHMgEYXv2eI6DZVA1ASxolfHN2hku3eroJlycHpcgAhODZgyLZD9wAxBnMWQzCvw1MR0O56xeECPtbf07YdF3Df'
    '3E8Fta4TwiHELuHw9rsj8dXvh8Q3iaP7BbQ8T3pBWHcuM0hrllQkd80MEV1BFkjqmcYpanbqrznVb2ajpYnuRVluHWE2g0Lc6Rzb'
    '4zzzhFxJUjIGz533mPNbCZxJdEbHGBLJ5xIu4BHApnE2E4t5UcI5jbrL9BihZcQcWUupcCSlZpTfPTw+hKaL6AAuN0kEZ4C+zCVx'
    'fPR4bdNVWrZfS/Svw0Bd6/bB0zpAfACcwqOZncWh9Szy6qZ5GQ2MroavpI6kLNZtDzSEN3eZ9PHauwsZkldyQSmDS574asfVbr9F'
    '7xgirAIYmKENHA5Ryz76Zb81JKqndVSyYcneRvplZs4yPwroPK2FsI4wgY4FMXvBaYnWd6ekByLzx8aiaVQbi55tcSCMmWBtYPWw'
    'AhVdNR9oL3GUP2yWNk61dPkQJVABoKyqhua0EY5PZe8REvHolzRTp1oTYV0xmbYjeRvEtUiurcTpxZstDcXlVHmu1XC7XEXo+qXg'
    'pg2Ue4tf0Ksf+YWZcaqUAG+zu3OdYYVqZuzRA+OKE8+D6ypv80kHcSpdxWMxufl2g6FLAXsgFfELV1luqB0WbyPf+zUjQf/61vWu'
    'a2GgxQM5EXNMZBLxpma2+CZpSdU+qnYhJhe3C2WH7lvRkt3GRGSsO6f9x0XwJpkxMWf+3mzu6XHpti/m3Uopi6swlZW7fUs22vdD'
    'oFWFM1ch0G8RLmRhnSB/emJeEC+VYI+pDShCSp8rLNdq9OeQnjnINw+DzCgaI/HJj3cxd1vCQF2NN2f41JabIDpSqVglQRgsnZBo'
    'ylQ81tGYC6/CcW3loF5wmJCeMjaNyMm+i08uOKCLCFy9SDh4JSnqgwysmRReMyQRKFXxLoU7I7S1CrjiIj1Z09yIh4WgluVQbA30'
    'Pstd7EindRNhS2K6qmIJOOZUPQR5Ek6WoXgXiGuH5ZD1fadGZc8Xb0jizXzdR3VO3LORM42S3Lev5YtDuwauH5kTb+Zl7NGQc+E9'
    'DOaEmS3kXH1vGjuUSDhNLc5bW4CIX6AR5ru7Isz3ONJTtv6H2nHV3DmrTpwDTPVUK03z/BNV2UMgFLZCVIPT9bJP4ln2iVQvaABt'
    'Oefu2v+VfvPz/L/tTd0u3f/b7283+z9evbrz3v/7k/t/T8/z/zaR4Rxrq82MUs+qnyTOSRD1NPFlNvFFFBPthDHWyzjvuuI6pdQx'
    'iOnyCllA4KqdsyZg1CphEW3QeDGWfZaSwr08eAgW7A4hapzCtmvCOPerAlQit4g1SQx6ZAq6ReITaUi2LhW/W+W3V6zJmSecBcM3'
    'c0CR8mwcrqxCl6sbeNaO1toGXVd8oetqBHT9ygvdqtZC1yux0LVFFLq23gDBZAlmPb+dE7wUflp9xI2j2lwWfs0A626EmbhWYoD5'
    'ALQLO0ViJbIWJNIkMVR2lAJ0zbNdXt7KrH9UVa+OOliuW+AXByoV9eqId0kmjtPzfMwXvZA21rVe4MMvpr98Ax+5Mlj1sEwt25BK'
    'Mr6tYQwrlf+sn/aEwAocWuGOL5XzE2XK6DnaMor1Tz3LMkTMdPEtFL7WtXS1AABEZilsOzxFVZoJ6WeGy7fM/RIBUgU1HSeLrwuu'
    '0AFLo9j2JuwmssNrL+uu27PailgT9B7sBtWAwZmtEnMu8vty0NtAzUmTbbD2w3ARtsAv4nB54VIhuQWwAEULDLkYE1vpFGqq49iN'
    'iQZgK+C0juX62//wmgMaPYujuiUlxxHaifJNAvG3RdKenuc8vtj1qSvNTTpwfh6Xo/RegdxVgc71ZohuQlcUYqlyikVHT8JHW6s6'
    'ivs2cpdU7RnDb5QgtOI36jaNMYKec+Pu+M7byIGkvTByV6UrM8lFIIUt53TAdNmS0gSr0g/OP15S371y0Bc6F7k58o7v9t4+L+a8'
    'FRTvAocfoAYsY/LOu2Iya+8CKl8gCtnVl4OTva36k9WyuI4aPwstn+QCogpqhlkRu6b2z7k9D9vQwqaku35MlnRZ8vircs7yC8ZV'
    '+vMJlkuYO5FPva4Cn0ghRBKO8JI09CRlt1ZJkBVKbaGA0LeEqB28fl4DULlAM64GJdsc4RswiQYaak+lJixE+uA1yFe6crH/8Nr1'
    'C2/xqnY2F99YARYHREojNgHokt/SPeHCVqHT8/zq70IrG07QJtKtRjFHLdtygNUM7od7WYO2Z4TGHEU58LGsisDxpG1MtdW95qXV'
    'zK0Ezj7X7raXNONOnMBfa+kEM0RRKwlRwL8ZiXhJIHk+jFBV2XojVZRfgVEXSIKu9eI8zy6t3Zesr7Vq5SVXvrkRWWDKcpBd5WWZ'
    'rU/PczlfkIKJvMGiNltjuHyjJ/I4HLO160NP9WwL+4k4PparRQmXNdoYARolvUvEY3ha+gbJa+2SV5Qu+Z4VfhIv4dxi1dHNBdyc'
    'lkK7ypGw5IcZe0ipNGyYecqO1eXUi2bNqmCxSwUVrFwYINoRzgjJhYkn9cwaL/eDJrB9j/RG8Cvp4jWBY5bUq1X1fTSlN5XbCkku'
    '3KnR+oxuRtLKz9gv4r8KS5/l1XO/gOCRt2FaxyBOiK4u/ldcTDJ3wFcsFFVKhdnVdLWIKt/B1oqRR+6Ya1evncXbvjbeTO2nMq+A'
    'ft7R6HhyFP78W+fOb8+hIZj0G/SikkyqU2Ht1oHS0195jTEa12kckQVy9M6i/PsUoD/H/J9JNDMnSczl93547s/b7b+bV6/2d5v5'
    'P/3N/nv7709p/8WpX6FTbxqA8fnHig0r2DIeLYK797zGFDMNKJcyUg63WnRkTjtgdkisaZwuvik4OJYTT0iXmETcnheN74pY3Mt4'
    'Q6R4iMQoDGkmlpPfjLj9lBTdom/0EbTQmHF90jHbjtH1pBhyreeAtvBmIubhEdp0DV3UU7J4zRKEjcBHhBBsKDnCwu4jJpjUB+Gw'
    '82C4+CaMEXJ5N5uBHWBiw23B7Oa9fKRe8DlKjb9A4JiIEFgVeqsi0EVqt97k1bN9mZYFoGi+bJHJ2CgHFr+Q8kA36HfY2fIauNXD'
    'pY/RrjVQNHOxhYAYPQUVlw+RDiZUviVxrNDlJl45dPSkLwh6tiTbZPFmn/OoiclncuYcSxQRF2pxArYhBTBnWMJ+PlFkgKlfz3Fk'
    '4URS6dQHuepMPKAbILWwp6dDhp8PJv8VjZiWVCNJFiHBjIuGm+rAo7DUOtPe2TfYYb+3WeeGEix3jjjsXbWti1+1X3OnI/ayjyLU'
    'ay3MgMWPwrSU47gTNzGBa69k6ThiJVwzg3ggTX5jk20u7eSqPB366WDIUONWUwWtgpdyyuW6Ra2XNiNdAJ9wEXYs7f9HgieiYDBR'
    'CpECdUvjGSKqU45oopXTGajjxD4RnfHC7Z3hsKNJ5BIGC+kMi4WHVbFbyLiyWiIbXTE+5TSEJHJxrDR86GNjE/MS2Qut7Ewys3EF'
    'Ofh+EiVt1tsGVAWwGs9v5TcLWEb0XDwKABod2BJNmGaFJMkjgKA6E//Vm3TnkIRGd5JuZTwmwVYTltgX4QDSdHxVkLhMlN2+OMre'
    'Z2pCGwa9kPgWObB4GBXnV4bEBFppUdWfpdpBk1jQlBcLPYaDtOnWe+dgPfmBbj4gdXXMrXfnSus4uW9WJRz6k6ZZY96uvqT90mMp'
    'eQMjdaR6DVeTiHI6xJmK0VxLmEvfRdLyiUDBNnKN1qercoZeMijcVLjYc5OUoISWu4iePyKeBNqNRtPTqvOmV7NRdK2nXAEZ4aR+'
    'M1jrruTMPFyDjDP3GGIozodGwhy0J01lObkkEfNYKFGzhYEnEhmodKSkzrQV3+fDJorNnAJLKxOu2y81bwYExdJ1KcHxSZQdHQiP'
    'I4eZSZRr6wni+UepXxB5lJRazCOeTNn7KdTM2Ltv3CVq7KwqWKwZunHu6IJqY5d5c3aWbs4QrlldYMOetvhmYrkt2h8gpErQlGPP'
    '00zJP4hgi2WDq/5r+VJNHeFYpKF8gU3S0Z9CDc8kpui7f3ZcnIhyHEbmu39FKnYEq1aBrsfq0epqUWwmR4LDNVqKMKWxOCSi53we'
    'uYsP45xpeWzf5imlVasbe6VdDpNH2X2ynnndVbn3Dgch+6JOKMYblVhI9JnSZcppPxxamvE6+Cp3K2w4NdxLV1B9wlFMLoObY5R5'
    'JFjNOM8jJxEnQjdkmuq0jKTOqjClaWLKIh4gBvxRKpCVTX/3r4GxGVxndJmqXMKyhtLChHArOMILUc6RFMNdKqGjaOtxBRvEL6jl'
    'LgFbQghaTETcjXBHxHVluYe03E1vQMU58eRkL8SOCNrOme4RV4mVgCwF3qBEjjpzPQu4qIIXA0vcBcwL4WKpQCaSpgVAdpn3b/d7'
    '3L/finAzFG82AacsSpirAqboORL8l6/dr+iRUOiKQF3bMDB9g3yheXl2rInG83ET18T1fJpw7lSR+cX6VGCbRBNp9j6NCygcQzHc'
    'QloFGTdacIed0YiNsWW7+GThY0QVd10aW4nowkzETc9yGJvgUtWnvDhFmNdtXQHeyZwJLlEmMA3MIKa4PEfzDtDnW4eu9gFB1vYi'
    'KBvTZ4Mk5qhLBxNuZRwRSltNQovvOysoOGCUCaGeMhJCAG2JOGxMFXqA9wDL6uoyWB1b8g6SzX4S4HcBaLvuFxcFdyUKvjPSv7v9'
    'J5+WRW/2fPaj2BjOt//s9q/u9hv2n82rO9vv7T8/xX+fhtnwF7eh/bIzRiv2cji++HFV+i5RcrXpZfOrJIj/Ky5I+izkL79djLas'
    'r+rOStac9Z8oEZNsLPFooDRfKspmJFaeqvAxX9cXJvday7gcE43zBv0cceA1hxFz8LSMIjVNQJIqwQSr8yoCVivoVo9wBTyQDYlx'
    'ohXnnMc1R/UXLo43yMWFOGThDW7uRJrU8GVPYsQqguVIcZvUBVSLt6q3Zs9CQOq7AyS3m1MrSF9IVXf0bf2ccmfrw9o4N03n0ZYm'
    '2TCauM49yMU64yrtc8trmE17kdUqgGcFqybS7bNWMONpaRtOHmcJejZxzpkEW3OMFxNwZXqcBXsaI3qH9X48mkRPWRLwH5XIj3EE'
    '6emQA9bhAH8q1VVNgYZ/aYZ9JFoEMw1Qb0YDOCB0CRyPqnSm1KlVUTGCs9NqaLaDJDGhyAqz7WU7AKW0oG2f2q/gWmyUABnH4FcM'
    'QDBO4YlzVTPyF0bqZCzejAjNA+Jq5VP1DrGvX5uS5ZKk6kdS+DW7LTrknH0F4yZ97VXEtYH+XNlfmuywWplU5Ynqt/gixTy8Rojx'
    'RE7cFaMpYlGXwlqFlK5XImUJsVvgJLgGyWIowhIigBXrYTsqLJ566Q4hiNVcXH3DU6CLun15/HomByFofQbcFxkzsvnGkmo9GZCq'
    'oCmFHL9BlzityFZRBtFTkAVdhyQS2JxGzsCk7/W+6MthJscmuyIM+Jod9KFEUMJ9ButWyshem08CcyRuys2xLxsQWgk6Jt9bGUxo'
    'ji2/jG0q/I858xeJo/Gkylhewgc/DxBETnqv3faLaLB3nLNaCtvF2kZBENRwsUKPEteq3UryXgzKM5GEP641sX/RuiI0n62EEjRr'
    'n4CiaNwDUzy27i6+ToRvSNxmtU+t4KKRG37dYy2bhfGNC0I1rPQVhts26X3Hi5brJCLoWU0IaJGjWip9RSgvkuYkZguzlFYuo2TG'
    'AaSgoZFX1mHOroc3IITI9DXDclImrIHZgqrLdF9vWGMjaeZaV4GDn0lVa+3cxqeGJOWh8hhQkuEqziKFiAea28T8Zak2sy1JfN8Q'
    '+SZSRTckUmhB8eRT0F329CEu0ozaIdlkKvn+CejaRK1lLEXk2YuMAz0KBHBxvRMBGqdy5YHWtC9im0IFQwGp+lloznAzUOsJaQKp'
    'dlxAsFnBVG7xxjZaqCiCahmMLZnm/iWgGwmnCIUyHeMXtHLJ/i+JNDPzhcXrzLcOS0pURqwDuq0a7DhHlrQEvv/6GYpeGVAQSywL'
    'VyBMDhJXkahZqjGfKmqxXKNFolVP59TYxAyNvdxsSH5hvNRc0pJoMhFM+FM19Oc5OJOwIvojRnoBqX0msZxZnxAL5YhoZV6ZByxL'
    'K+yBR89n2CZfJFdigIOwGHZy3qfZ3PFPnCMWXuvQKwi9+NYObxs+0My2QIOxtE9NK1gKHauNfQmjYmykv7OXd+dKhC15IBASUSZF'
    '5povS7qx8HD/rvHddy02zzggqdZR4QExlrFEvColEalFcvvZAvrUpK48mn/kc+GpZ0QmzriGPEiQ8nNODpSWq8D7GfdBCmP1iY4W'
    'XxOBTEQUnSr/twl83GvXhoehXvNZnHClI77t4DtMI5gEsgAoBfzk+X3GVJKAp0PI9PqQLcfIpmiaa4qujRP2VwoAUZEsHudxoZWs'
    '/rEkiW5mtQQO0/cGO0QJjMQOGKdhbapb7LRkZknqQjyxHiCAkVvUcUEAI/2BDU1qvXuG98A1eFlgcIy1YPNTJvY6JzoKF1WxJ1P0'
    'UklWsIzRWhFOHhJbHWS5bM5GdlsdmX7lYLsCcdH8x5zbdcxkEJlLYyeJEkIgYDUHBZ8Kob8unhqtFtnYmRvbe5wlAXhyh2q/juqV'
    'KmQZQRVkK2IeZv1EjblsgZoT9+IiDJKM7A9RvWY3fyNL4UGOc3c/2BMpgrZr48gGIUtOSbYqgWVKMAqth6nJGMv5rhUlrNQ8j6XJ'
    'vat1xvRqLXBQEzSIs2iofblIyhfsxqWAkIrFOPdI0VVFhD7mO6bOy0xh4cTixFNsK7WF/h8axjNneK8UJY1StUKXjSV1tQlIYpxM'
    'rYwCwzoJRKIHwc00p/MQEjFk0V7yxIUtQ3riPJuu2LEUdYDi7BSTiAFlqLxY6KoQYvJ6k9TC8gpwVuEAfgVv8KmCqH+WJCwJGVa6'
    'YNpMB8z8EYgZv3CqkVwYsYR1ZeIX0OJ41ikpd7Ya5fHiTZhGCJi7EyWDrMxTNfVZtKLfbZdJf8mKHaixYqNuWaKTDCMbEaf9RfgF'
]


In [ ]:
# PAQUETE (3/12)
PAQUETE += [
    'RMIuXksteZtcLPodog0ssWJ9SnCqaoGBzsU1jLAOi/twmnF9K0a5PNHwcr+jLar4DzInzcSDUsM90uCzJIsmIxjB49S1Bla93zsd'
    'xoN4VmrROS6gNKzqiHkLG5XS8wHP2MbCx5AccjgDRZnlBvZZoZHU9jY1OSCcTkSLYscGneVDzbrs00v5ZNg9Q8cU2jt7KCbf3Lcq'
    'FXF+pijItm5kq1i3435lkUlY3JEwDo5rAakw6gLhdaWjJB4KjC3xq9X48ncFSmtzwtjFwkULZFlS0I9dNCwVwf3I6TMsHzMhx6Hi'
    '7W+SKOs6ZsgAAD2XFUj4JF+QiLRfaY6TcSk+8Q5XvUleaJsJdc2KqsDVU6ciJhPvZJ2/8NxOU7q/eZZ6slqlLYq3zIeZeou0gwpp'
    'qIXVXMU4TqgwiGe2lMsNHjcxXS/Dv/Keq3NIvBiqRBL7ya2fRzU/5rtM/F0CkzapTbSBScDWNIPrLwJvFLGbQDbuEilDuhtP6bQH'
    'JWvj+7ZQrkldLSqgpRmqTUuMHGzQs24sNb3x/cy0jKNowhOJd/otnBgSioFVMLOKuC+LVPaX0ockXZWzzHF1LfCccFIjCCQQLx4w'
    'MZ4hy0+Ac9cG1XNFMZqDJLEC1ZgA115w14SxNcRVZkzT3p9dNDGCzZk19JhabrWH8SnnQ5C+NiYaYVMiVK9lEdX5MSH+ut7G0JgR'
    '8uQQkD0TiXBGqCv7DKZc4oMWX7e0cmLtTrJfXdWProRS2N7znt2ga/FmaFRvcmKoqzhWKJ203+j6GU/RhIYZjbh9kPUQe02H8sJ2'
    'U9CDBXmM2Z2TSSwT/U0kprBi3u8Cy3Kr1ds984aFyuGDihIl7AWr6pa7KKQ0q2ysXbF2u5bN6oL1ikn3SJI9E9epjcEouNnpAKNb'
    'RX9oC5NKsuncBVlJ1ZBMem67mAvxvKlRXRBX6o6TuuP7c5E65QUFKOnXAqPSGCs2tshUmToD8ox3o8YlRIqEJeQIJlMK0/vIGm1k'
    'M57TvtzSTlqgyleOQRFLpSnpOsvx2X5CKqqIPB2Dp+nMtsX9kVeNrGbKnEbErVLaqgnigSsJUG8W75fixuePUqaenFkOk4bttIN+'
    'M+eV3tY11TvXMynUsB+VDg7LGZuzeT7/aXZ8sEF+aAVtEQi16gpCiuKlt/xKYJ6Qqro122MYlPNqx/QwBIXEzOwRcgNz50n44R3M'
    'a4kytY6YnLmFo/d6lesi/t4UpBeOZzTkTbp8rooU9AkVHSaIEqzUF5robomood9l+bN973dnuV5GwbmHJBxpyuqvylUeWoyI3uQC'
    'LG9hgon9q9c3ddFWGGZXiBVsiTo3JFtrMGClyUqayt2q0kKeQlPU5WxPLSC+h3rrnngPKVmX3bUtRxuSnJ07j8+MmglGrNlGXAiO'
    'e91tXrMorJpKladJi30gPQeD3xBnjnOOGDpe/PeM5M0yyapNCimBobdlR04FKhB0acoxx57y5HubXUjecHiExld9ICLdXnxLIq5W'
    'qpgH9xBqQvCOJhH019UqjvG0mUpBlE3em83MV4bvHO3uYQaRjXd1B4amyCTVngawJDWa4dJuZIRKMWje0nsg9XG2VJIfd+a3Jd3z'
    'oRWwKiVFBDFRSKx40BWGow4+r4bDvtNF6iXRfK3laYQWG1DImspJYsmx02CADAREeXxe01Tc0xxN97QMOf1JSqu2XDJazm+yU5MS'
    'KSDBbqz4IUJfpX+yvupir22BbYuENc0oRDQaa0AN6t3Uk7yOdN0KHvVlY3OiKa2E29ymLI/pKBD/ltVKiMp4Drju9NWusFp00I5P'
    'Hgxc8mBNfFAdns3S2tPPWV0UQIs/TRA7EppJAyQ2N60qbl4BiWWlpgrZ1GTVcV/EWrNO+CJdar67XPaqSvsgQJJCLsU8WGLmzoEs'
    '8eke4flBkxfVE9Nn3OjrTomY57GNlxU5Azn4k0GZeMF+kFCQozHJfNpSJy26Jfd25bqs2Tr617e2u54pCjxWYt9zV0anV42C2B31'
    'LibO/6HYXkmPXr0SW6TO2jnCum+mW2972VXnRqjWc85bVZ0UY362+KYg8cEqGu76VtWEDIkhhDo0342bd/6hq9KfrFid32pjZpZe'
    '1Hl6UbuyxSquvrWi+y/q1cNP4Nr1sCtoYIKbiLEiKST4XDgoF3Ey6E4O86wsmc2zNz93LRWs0yabW0H9TlYMF99a8nxskGxL2O6T'
    'k1EZ1c84wbGA2qP9baYeo/71/jXCNh6ucaTtEqprXu+V8FjVZjjyJF0rDUfPSORcqu9vgs+i+ClJEaTIE2Ecn6KzSVR8P0F5SV5p'
    'InplzL5hYm1cuspe6ppWzj0UJkQh1adrXbyVEZ4e0bgPWP4lf3oac5ZVKGD5VYl0kYow0LwZrPr0Q4ha16uPSXtqETMd6R/Dn9Ct'
    '9xTQT51sKMEpkjnb29tx0jmwLZmxWqzHU9W2VkVbq5InTkwWBVlKD/r3plLRnZZQK6fR9bUzy1PmMDjMnC07nhhPQ3H2VoDSc4BH'
    'ymLVnhSWWo2QrU8ke03MkMOMMk4ryKv6hhINnQ+dEbjZ6tuTeY5Rn9Lg/vh6if3UK2PtllumUoTaiZGroClMo/GlEZHN6VBc8qh5'
    'SvVt+SEcja2J4yiB6mONLyzw34yS07hW9cNSoFqtWa3JOWHjHjAOn/O7MMNOBmZ+7n2svJNyI62aYK+kWXElnXbrsDvK6YO0F3yR'
    'Jex3qQnwIg1f35SyTMyayjQeWZPcJ85OkPideN1BdxvSBf3llYj73Oo0BOjPsjTV2i6ZtuLKuaDnuJQiC1JlPo05QJXzJmX9KG3b'
    'Cz5PMg1VaiHDeSSeyuAWfRJbdti/trdLwOtfu9rHmT8suURXkvEIWt4p5Mf6BBM83ZOywFNTWDAlmYRdTKZJNOMkkwS4VQHIsg/h'
    'kfvBEtvsVTUG/G+9wgO2eN5+8JCktmE8NslUO5Nq/bx9Zkj2r15VTW8/uENkjKNnqrJ68vRvMk6U0Ap7/tT6UY9r++2LQDUANWyv'
    '7sfDEXLzJ8g+9QrX6VQk4fRcHah9Szl7VYnBfXQZRrhAzwZs7SvRksoPtAomWyqcWtF77hwMTCf2HenoVUUu9hs2Da0kuM96FfH+'
    'MOJIEKlGgJ7KZaoh81quCjAUs87vlosPdr2KjXgwIU1Gj7xKJatbHKobLZVrbec3iJ/ZCnuHON1bvugGbCpBjizxSbHFJbEY/N5q'
    'CZFRa/TZ0SfYbpGUPGMRRe5dbku0WX5OB8AEy3Vmryp/CUceEtPSu8CTnxnxo4mZb7l/nxce4WzEYl6x3hTJkSus7lMWhk3q8MFJ'
    'amUq1mIFtvXirVpIpZNoFtbcRqw1DzHRHDcbhlIWEefpVUXNODlG3KuV2OZ1ScJsMd1iV+Cz4tpVf92uyuJONLd9pF2Eyurm7Ozx'
    'rQdEaSphZbMnUY/+NE8lgNhbG9t5u1Vrkfo0bMVhd06G2+oa3x+lS03Do0IkUoTsVFBjY3lcVOO713Roa0/wCiapglbvTLCz092r'
    'VySqAk+hS4W5Rv7Vu6VjKduu8PJ5nQeIdkA54CDsqmjzXvWud2G0ErM14bijql14tYDZbe2vKKzUtg3b6EBiH1zsOY7YDQgipJGJ'
    'zvACs8kVz1fiV9qvtbvpXl2q7tSzR18v7LRitMw7pZ40IZIw6fNaqjtRV264CBz1YFkLb1tZqylF2rPi6qzaElLlIans4/TF3rUV'
    'tSqrmGC/6bjYnL2eLKNcLI1u0n0VjiUclsuVo3/JF8Rgs2d+Xg88pleuoaH1d//jhghS17a7jQ7bMgmCsKQfolGXq8zR1tUClljr'
    'KK3VsWvLUNboAyYJp7GlXxzNJT1dOCRbo1L15Gq4KIUEo8LFUN/PPIeX8ErPyOW5STjbbxzZbjO91ZQLRKUZySnbKoUUCrG5Xfdk'
    '2wa9bDqJn1btCyZsviFOL9ZKq7U84jVF9X5qsD4uXpdsJJrbL+9GY1VWWfdyYu1plKMWiNM0xnkcobmIYdODBIpwrKdJxUxYizuu'
    'AhN8wrQv0BDaVrszXacRi9bP7W+rLxWaco93/W+aFJn3RQIWmt6L72jPf7zWXE0f8K9MxYN0lbteRbp6c3jXsMK7roRX8J6zW5Yp'
    'GMvzUwQmY5zlKmYIKFDnM2ujUHFVyUBnVjr99C0cX4u2ViygrVBr6+1/lFYBZ5yOQvpUpGkYEr0mDRwkqC51sUZDkyAJY14LWDJV'
    '/0yYTwe2jwcbU6GdCIA0pO948fqULZFH7nePUoOKLBciFBLtVnxj8afbt27e8w0rUmbNLcS3Iu28fUDfQgOaMaADLJ0p4X3/7J+s'
    'f/a56P7O7aSdde59W+e/hLbO7jj/XBoJW0/CX3PrW3do77u8/t/U5bXKsHjfMvSHtQxVBP/r61yp5un3nRUvq7Oik9j/zJsBOun5'
    'fSO6v4pGdHLe7ztK/UV1lJJDfd+k6LKaFF3EoHPxLjw1r+v77jc/rPuNH4X719x+pQ6Hv5oeIVZLf9+p4q+wU4US0vcl0v9yS6Tr'
    'Eb+v1n0Z1bqXw0bel5H+cygjrUzufdHiv9SixXLA78vj/h8sj7t2Xv3XYjS7nE4/71D/dWdna3O32f99c3f7ff/3n6j/zyyeZqgi'
    'jDz3el1hr/Po/pkU9J5Fz2f89KdlEeW/+L6NRT+NsuIXnw6y2S8+Hc6ev1dre59+/B4ODAcgRrMM+/fFrlpB9pNs8DTiQtzv29ei'
    'fS1K4l/kqvdXXPUhN5P3DBhqJ/VabIrNVK2iagFo3Pj3hvi/KEO8kK/3h/qXdahttPgdr/9qkvweaf6ikKbGX9LsZJnFcG3zkzA6'
    'cW2gWriM7XwUFcM8Hkjay0izwmBdKvMiPmN17/58dpqlv6z4y82o3iGwsrlJaQnWs+LQxpa7wreRNaDDWjVCQjUiikm5nmgmIn+n'
    '2i0+VDtmVfVpMo0T035tvvd+Vl+Y/9P7u4AEMTTTK5v9nfZztR3i60mpHE3zy4aU8GeY8C+c8M9x4SvQ9rzjWo2kf4YAuCheb22f'
    'Ixm7endttfxczT52gLmUshVK8ft05b+idGUlG++P/K/oyFeJ1z+EhpxDkd/j1l8Pbl1YRrt6Hi9bZvtDhWDTfvuXUZxHjbB/IZtZ'
    'SV5WH+s5htO/DKBc4F6MzrV+OnHYJjRXuY1VEGLjdrxP0fzJUjTlBr8H+E8H8DYqc6FLsprW/P/svfl/JEd2H+g/pXb23sVAeUZm'
    '9niX20T3iDPi0cMmR5q1vNpEIRtdzTowVajWgNrDI/m25fu+afmibdoa0Sctez8f4h/beO/7It6rAqpQxe4hG43kD+yIQmbGi+vd'
    'R799X9v2vTRt3Q/EdaydUgqIcUjKIoHEzih2Hi5m1t9m3eVNna3g5sbUaX6+ya/NJisyjn2LVt3GnnWxyuBGL0NKfkKHKPqubFKH'
    '3DTPbaf6Vsx7ZwqZb9fuaSU7az9dV/DdojhpIS23COAbFHobdmjLCb49c99Zl7eDjnpDJlYmYuvnuc839/Xlm8OF7Ff8a1zxHUwE'
    'W2/LNqNWv41f1za+PBPt9dqEX2o/bp+NeJIrfF8rkS1srDwcPOZohODWpO5dFKd9xIl/WbfFscoXNjtGK7lk1qtzWGds0SM87dQe'
    'Ck7q+exiQAHyo+fkVL3FXHvz3Lbxeq/mXHcli0m1z37Dqe6Na306X9uCIdZh8/Wd5F53Q87BTd6Xr+1q7examW1RLhLRQwGMVcOO'
    'rfGydtP6+PeXFf+OS92v50tbz03awV1O+WY80m/Qy9qgnbU9zY3aHjA1K2mJN8vJtz9/EFDFazCPm7VE23d28zW9/YuzC7/sLyUx'
    'zFujJQKy43zmnaRI0eCRtcvRJyd+GcmJcUH7tXwpa7k5EuCGo70ZO/Q78zJ2ZmeJPt1Hopc4snWq/U0U9cU1/kZG3kv4DUu2+cB/'
    'E3PYmb/bRL9shfe1mgyLbrjF5+XupRoVA+EdnPh1F2Xng7NFi3r3lnJnv81yH3TOSWGvIPNvtCaloPVvFoa9EDwWcQt6/0bnsruk'
    'spkRYNcQU5vW35CLzQbuvhbX116La93i3W/B178Fm3HGtddnF4t3v41f+zbuTGf3EpseyUq86RdhsY45b11pdmCb2wf2XlR9dcs2'
    '39Zbtwy7MwRug+j3SPLerd1XMsZaLEcFdkYoX6euLd0p5UNY5zj7onVfsWidcMv9+n3V9bsOJ7z4+d4iDPRb9RW3aneF5k0RWtfq'
    '53qN5sZgnWvX69apMyV52XYhd7/cRW/0ucv63GX9pt7+3GVfHQP06cvuYvqybYqBZk8F/PhK7FvIFHEALk8CWsOvxqzxMDh6L6cw'
    'Skjdic1x9BAq1/7YYsUOQoQ+O9Ctx+fTaaLs7eyTvlJ8TFKJkb9sONwE3jRyNK/RfPY1DIy3hc29Puuyu04huymtyEryXOvNve4L'
    'tVq4sSgO3Gqw4chLTjI7wjpky/PYo111LqHh8lg6b1so4uHg/tPLz+DNohVLnb5rPDok9jd4Gt0GSLfmBNm4J1uceG7BpHdG6fl+'
    'EqUQx7UDeyW7kp+tp7ee6PmZHD1451dAjkMaO6nPIblNOGZtsRq0tliRpVZKndmotSzZELV2n+XuNqZLYLPksSe7ZPn2/Mrgu/7q'
    's9RP2IFWl8qxAGSusv3guzGO9zk5P/lvzC6CTuruTHcP2VmWYvO1uTOrtqv/UZpsuHusoAueulzTc9J+7G83eORQmoWeGE0olXt7'
    '1Z22r/F6V2q8xuw//X7fjf2+Did/FYyxNRdRf5juxGHaWbdc7pAazoTMSOGQawqXbCiCc2dqmq7WvLk7074pZdpOh+fmGjd3ZkF3'
    'TvaT7pfstB13k3Z6NdlpNx9ffjY9HPxwNn7u8c+AsMk8vshR503CgeNw7VhOJauHx1XfiTEcY2s0jRaygzUYfM+oSoULH9CKvTmb'
    'TqWQ0Yx1OQsqxQ3vZXJT7kg/hFQkHak/u5iz4PbCv0dK07B5W+7JrV2HnaM63C4aDdUexqSG7ca4jrc1VatfoMd0P4OnYcQ31WGS'
    'UBpESjM3m89HHTnlIQOhXOx7K0YDrc4e7AbrlanDjTdF29++CjmlghUU+taIrA4XiBS4z44RP2JkpzOM7oLGer2AjyUhHlE+LWSV'
    'PFgLysXSLWZtSEzmvw626YS8M8mhggxIC8Kgx5OWCnyh7K/EoJeyLCF0uF/JF1/JG5Uy2w/3ltjlfndeeHd2psvZfnSZ/r1ClSny'
    '+3Dw3fFM8qNeU11k3o1YwBh8jz8g8ldau9IveVpXKbEtHyw5HYMXGugLkjzhhB9LPYWgpw8RZn7mZSIhGmOO96ECOWMvDVJN1jEZ'
    'X5RcyJ1/1YHcg75iEzZfoFd8qju7mWX72HnFwHyljJFk748GPhKWbQKDs27qJ7lY+uboOHoArH5erz7e+XDKHAe7CXRq7aN0x2Jw'
    'gaqB4uRGzzss+SRqyV4xmPayuIZl3lLs4pWa3e5203oH3UIMiFyVEDfwazYunSt6x9c5dfaTkXhgUr1oysgFjROpmtg1lf7Eqp0x'
    '0asLpJGRzFetJl+loMfjTjJ5nFGPlhDaGcll/fjyk6ftXKqGoG1oaXOQXSOjrk346PKnb3/vwXt24t+hBxQQm5a2uPmDdgXJeeZ4'
    'MRsvYyGIfu2+0trdpN7Yfn63ppPpt2Pv7djZ+l3vx4SdEeu6jmy4Qk7Eucx/L87Bb5vM0fiTh7ybHM88V7mc4xF2QH//8rNJO1x1'
    'TwmW/jXXlHY1G+tqZsZXBJY9OCpe0S3H/5WY0O6Bc/s5U6zEYfWRc69O5Nwu+3Unw+bqHVzcelatZ9VuG6s27Fm1V5xVu9kCvS4u'
    'W/f+Nazzy+3i6Wh6eu4nEKq4cdLF+dyv9WRJnMKEMDQSy0skwbt+p9rBj2bzj+6Z9pa0JhdGAL+SzMRA+yQYRhYDAxgCwdKqSeS6'
    '3zag9yCsq5u1+YrdsiXY2bvCbSgvdfmpxORw2Aw7i7B1THLcsv47mMiD3+SJCet544qfLpdCQuUFfpo/xBmeZqFS+vDp0qKL+lrv'
    'SS7avqbIFw0/1i+mqYpqfXLzIVU/4xE/pznl4bCFxIwlXbZuOAssEw0nwTGS0ip4KJHj5RXefTyTHMtkp2w9CgXy8tLECio0iNAP'
    'IBMJZnx+ZerR57I9H6+6lKbWucC/KQ6t32EGbRH3zMPXymQWSwppfk6gLOyrpPyVtzl8J1affWgm7OE49hze9PTy90eLySxu77fD'
    'KsphJg+m9kCKMLDjgNXD95t/Nzd/U023F0Yu21y9+6N2B4/a7rbPfexLMJS90Zs+vx7T50378DpbP8GNFdvCMmIZo9b6Ji6Wc7oO'
    'q3UdNri6stcTP0ie2MtTqtosqdk3lDsVselCOFT4C1v/xFBZJmDuENX9i8uLNrgqSiGE75AfpceSE/w67fzS+a/zX1i95HEDvYRc'
    'GN2iPTA1nE7YpVuUUaQ9HZ/PW8qLorkzxN38nGvTY5pP6C9U1GHV9/UOr8NGF/49TtfNvrB3d4FfOALrB30EVh9E0Udg9fu9cwTW'
    'D/oIrP4w/XwisG6otnPUV9vpq+28ntV2jvpqO7ej2g4wVPEVEuCIpHFNBhyPxikor+XguNX9knKDUjPtIBQDRjVoRDSt0qeQLCWE'
    '+627+YREK49p1bjowaAbfI9SqYhGLSaTOKyTDfkkNCPOrYN83ww5smnbU+TctlXYOcAs38Ffeebl0HY625Lk6eGKB5va0t11cav8'
    'mZBr5IZ0QYNntIU2aNX/7X6Ia4FP0eVv+7WR9v0lVYPxIEf0MpSMt5c/5U4QcG4RwDe5416zPVuY8dsz8d1zxW5lJvfOFdkni+2T'
    'xfabevuTxX7F699niu0zxa7Z0DZ7NEUbx3p2j6s2jjd6E1pvQtvdhPYVDlhvRXtxK9pNZXZDIE6Mz7eV1K9azUgnHHaVCpEMl3Co'
    '/kALe8SUjSfdkJQl4vf87mhsGXR3SFGqH004gGh5wh4xBAL5THcQPI1a/Jj3m+q88+8LoXzy6fuT9mNoTRZmHiuDUd6Uj/w+LUYE'
    '2Tn5J3DYxdlSlt+jWU8cPLsPadcfxvZkMiLvJZokcql0i+tG3HDQju5/+OD+2+89fu8eKceHnPkA8btEm1mJzxVcKDkDCRhCSEnq'
    'iAms+wX/+hb8Bnebrfdkq62o38GvZwd3DlZweybdgmB/Jb/HUTs9RtnBd0eDx5efnUy7i8F0NHinGx/PlvNpJ05/M+Q3ISYVDu1h'
    'ya0JztAmz8SGT9PCcxTafHlBhZJ+4s/RQnLicr1ekx7qALw/OVnNPVe6mMVc7Rh7PnqOwkoX7LnlOW7WeHBSqaQObsiv15z2yc4V'
    'dnmL4+xrtTi7Ww7yGzmIKyrozi+ExzzrDITW9wRSGdJwbZhH+Ma9DUkCr8vcfjh4PIpu0bCrUOYfKkoWP0hLdT9sr+z8aDFuv70B'
    '7JWMSgfVFT1iLImGyS/obJ3MNi6CXK07OvebyerGw7P5Jt7NtdzZg3vLhe0WET8JGgE8NsclXOvjYmyS90XBcm9wJXf8oSpP7F+N'
    'Y3vQldwbfOBnPRydtuMzCYkUdck9qkcSe4eqPLk3eMeLVL5xaLQoePr7s8Xlp4dBoWKHlp8OWZVzD9HXxy1lM7tWmcOf80wO/3Lq'
    '52NUUjLUsp0eRtXSvcGbs9PZOQ0QNUr3yNTcDimXveiU7g1+sByd09dI+eSh4PwJolbyn10O5ySYeuAetR8fQiF1L5ZlOVT1lR9u'
    'yRGm90dzoh6iOPIrQ7lLn49OOhpElFBkWF+i4GPUItEaIh3Rj67qmg6Mgo4eHLfHAYf1W36XtnwL6n4RNLIlR1Z/vO7O8XrhNECb'
    '8oD0aYC+ahqgbSv6uqQBSpr9pP/F8uTyEz6568Xy4h/E18vLXgYbHlC+5Fkw1MBKx8qie4NH847vRfRIlhQWIbHMoHvWkZLoORmv'
    'AsIhz7XjGZZwJe0MiXWeCcXjF4M3x7Nu8oRu7Wgan56deLHx2fKE7YHI4HpNaL0H5/uzp+3Us6jHS9YGiVpH50UrMqVnW0koOG2h'
    'wQoV9/pFWVmUPRQQetC2FO7rl9cu7+4qjBs9amx99aftxdpdf8Cmbkk04ycmLnGet/Ff4TDdYJ69GBwvKScxZzNm+zREvAU5xy5I'
    'KJz69sizKdEEfj/mcVYP0oMB6tjF2usq8Z7N5jT2xqrsk9Fisl6WnUG0vFk7perV7TMWRy1sw9FU/KiQonl1GBawQ+n3e1ri/eH0'
    'irMtr+Px6HxJBv+4aswQjhb6/fiafDocXmORA2bpt+Cb3IKt7ihrd2cz+ur38Bvcwz1Kpe6t74Ue+42+UurXXSl1p225G8VSs3Sf'
    'vBX3pRjLG30pnVe5lM7N+3enq+ncnKpFamO0K6njN7n3jAcPn3d+Mc+BE+rDumi2J5IPYn2smgUT5LvdmZ89O79/cPnZcXfO/P+k'
    'I2GAljEk9PGjPINbhL4jASJSHTZLDilRKpG0iaeIHyPrYrdo2b4D689kxDzFAttAYVP+Mxw2Sj9dA/J48Hw2HlIm1ndaIsO/1LFZ'
    '9K321y8/GXnhZnDKnl7CIoToFFuZbLGktVzGCNRjzwNwcqHTZUu+XwYwPj/+bLKnQ7uW9P8DxpUHPCBHpz0dTY5n8/ZjFt+iL6pA'
    'YoITMGw7fNqyXywyV5JYNVbnoX47X5PtvBkzbr3mW4Nn+iPyOhyR3Zn78iuEgT4hHe323NAP+cXx8rSdm3cs73dPDg4dQgnRX3rh'
    '6oezxfnsI1q7OSmVPDX2B2vw7ZoS4X75u0cg03XO9Pa8m1DMIu8CD+I3AYeMYm6X02ErY5iCkRTZQLkJOnKyCeH/q3PEyg6Xx0iG'
    'SyuL1AdnfEyfjoKgSjEKRLZp2dnnDANJEOUKt0zHcHHWLWJR0X69dl6vfQNeeel2Tv3cr/zGld85gCTfueCwf/6snc7gTL89rDwG'
    '7GgQZ74pKpm3MZaPi6PF9Ixrwy4GU1ZWL2wqEUpJfIYQZgbND2AK4kVFlKl8d7SkACkIvQdrkUeikL9oI/UorgB/BZZoFI9R1khM'
    '4mny+YiTWkKJH1yDrl9T8YEiFmDernpq9Iv5gou5W2Xkbaf8RneHfo9ebI92diTI91EQyeZcTdK95plyMGg9m/x8RAaqowfv/ApE'
    '/ZCSt4P7vyRf8jSoo2wfhHg930ZbG1Lwij1sZb9sstlsQ14EIm0jz4LGCILxuJ20x+3gAVGokScSoovgsM6WtG+UEBcgM3l68N0Y'
    'Ef+cNsB/Y3bRxbTUd2W6e2mfwuHYlmT5jizci0seR73k0User67kcdRLHq+E5AGNd7OD+gJMw4rv/LYSV8bd5YctFZeeCk5NDooG'
    'uRovpKIPhQSMxutsGRU5QXqClaQI/mmj9xlGgsDJsUV55b/5qB2e49Nv+2UhFRVl026y5iCWfxVvHWp+73TcsQm5Pb/8nGwNeslv'
    '+zxuEvu3buvWS3jLV+aFq5ZvzRnfp+7/elL3b92E179qudvvbN7vg25fp6Dbm3a5D7pdD3rYR1fBDvpv9DEPXyXm4YZFfQ3CHqCy'
    'z7YwzprFdzVUy7qQr52uDzmveXKTxu8iKiZWUunaqF1KSUGaxAXRJklSwXcMOsBWspkxH8ackv1VGUAWSaaXX0w6MGRXYDkQ3Sdx'
    'YFCfDJ8S3Zx4bNAybVzXjo5Dnh+Or5oR1KxznMwW5vPnM/G9D3PmGYsC0zx4MNAPDp5TSpARwSo3pF/Pl7aem+SIXU755uveb9DL'
    '2qCdWcZsv0gt7Oc6wxjCEiOi7tYcMrw059disaRg/uOYwXD169arhn7/cMpomL26O6rKIDlzKP34NueZYJx7tWDaI0hJ1niLTeuV'
    'mtrupYY2Z0Tcs2zIG32lob7SUL/fd7XS0FdEGn2xof487V6QfFsJzJuqBrfbqgb3Fcn7SsF9RfK+Ivm15U9fCLP05cj7c/aVypGn'
    'xX6Go/eXiytGo3dmi+HlF4eDdzmtBCVbetSd+0NjMk5cY1YbkSOFZyfmywXtHOl20yatDwf4HH0qnPTx9cktYtz6iovnBg/PoETm'
    '7wRFwS0EfA/zD+/VFl3C7Zv9yqmezn7t6sH2rXn7ayfdrw1nnn0mN53NqgfPKE7F1c8D8mQ8Gp7PPL+LZADTwXtzeCG8052MrKPq'
    'A8JTM7pyflE9/8mfmfNcnpAqkXO4BL8Gutvs8sefJe5b0Mpy+JTq01ECBmLbiT/37CeSyJAnMSWM5gIe4vhz4Vnn0bMO5jJCgPOl'
    'lzf8snfPPP49Xi6IMnjUNR4tRryG0MUCq56TRDQdjiZQcLJD0nS2VcTaa3G25aR4XRZr96jhHTDqtoDCPmy4Dxvut/N2hg3vesf7'
    'mOE+Zjh67m9SOKkyM6Q3ojxGLLbZnFA0sRFcgUcAiarhnlIs0RopeRTciikiCBlGmY5efi52ouGS13PMAoWHcHz5+Sm5SG7O40Qp'
    'ltbSOAlwSySGQuqlt83pPNfCHLQ/T0bPyCsImZ/ozvgpfTiHCTQ4cnzIMOHokA/12QhFIY8uP1kO24UUgqQ/vtudzgM/YTOLPPXn'
    '9US4Txro1PMvpzNKMUVxG7NFyJQ1oR9QWnjOHifLib9ygrv79fvK63cdsnzx870ZjfZb9VW36qXJWEeXn7PXBVXdGbcwIXjpkfZq'
    'JpSM8wIJ6fHEpV36/ZnOJmvSFokFC794nmsno4bfrwXpy/w/Sw6wINadac3Eiw6e8j2fQdvl5c3RKSez5GyXZHShtHanVJExOrxz'
    'IbEFZYALvl+osb6ccu5Nck2fz/0Ctf6kfkYuJuyE5s8mrf1brHCCFOsPEUkkG93gXmA1topXt2t1Xryg+lFfUP32FVQ/6guqb9C/'
    'pvt5YYXUmMTrrrN3lz+dUJKzk3ay9l7g9OV241J6onq+5FkHS6ymKz4YrCYkvgBuWnhWmjlo8M1PWj/1UQvzrBBmT9w4V/8FHOy4'
    'ohmTLcYcMg1PqiheTg757QN7D4+ulf3afMRv3RrsrA2rNpzvBx76kxD4uBq3ueiGmyMKHzJ18lIlIiXJ2nTeevLVHog7XLRCseUJ'
    'gisCPvmSekm6m80DuxfKaCByk+1tMQiUXAvGSzZzXcSgy+ftmKis/99zjwhOLj8dwpQ2Ao94OjtHVQyqucx6dXJI9dvRDUcnHO1D'
    'IeucdffH0K2/e//xfUJZFIjK0RBn89mxXwny45jPnnt8EhRHd3Di1120nQ/OFh3L3VvKnb2C94qMOWqn7cnlJ1fq0J2ft7/eBsvS'
    'BzMSh2YcVTQjzVQ71qCiY7IgrWXUXRwO8IWwyVNx7Hk+CijsPcpNMppd0XwRs/mDZXfcDUNZuFcClL0iY8KibqnP9ipM6oWzWQ37'
    'bFZ9NqvXPpvVsM9mdRuzWSXFPnTw++2Z50rWyeAHs49Gs5CXhJMOreYcWneLuDbjUF5d74y1UpmTJdnFkl5czo9XYvm/aSD2In2y'
    'jptvwTc8m52l+j1jq86Xc1KgrTtW3Z9+RGd28A75P4iPwpDVxh5AumCT4+U42qUW7Cfhr8BoInXUn1BoWbsamC0SY3wbtYBWwFkg'
    'f0Z+QEUJ6GKR6tov4+XnkCVDHvZD/cqq1w9Xx4jOP+qHb6Lrgsd8MFaerBY6OVhVwx94EIixJTaZkBJ7PvrXhk/B7795+fniySxG'
    'e/brtve67aHSCId1i8KuX/891/+lGV0esZfVSYcoVIrLGA2ee8FxOgsAsCuUIVUPF7OQHy9sPhUgYzcAXuTz2QX9hecxg3wZNnNO'
    'bqtadig6pPqD0EZbgV+40XB0NiJdFLuTkU6JAvKpZtrh4G2yVFwwDHQcOJRjTIxAy0aQ4xlZ+jbRkV2nuy0A6DZMf2d1W31jcSGT'
    '5wk3a3PFCnYgtRxndZBeCX46QCgOKVQ82YVOQkTY6NStTNohvFIHs+HyjGc4G563z2dWHb+lwA0pUY5bSuBA+RwGT5cLioiezdnH'
    'OtQYFwLORcavy8LGipZxe8rh5GRz9bcWWv4WzC65kK+xr2A0KV9ENx/AaKeq19n8pJsGays9LaHYbIyDZS0kVuoX9KUt6M31mrad'
    '9C0Zofo9ell7tDO7vGf2qoeno7Pza/xkj9qReESsuZ7H7Q/17AQXg6Z7OcLDhmxxpA8jbzjRswanvXdHYzGbn3nJ8xPyKVugQvbo'
    '49YYUPy4s7k/K/4fHK0Dk8TOSxPXiCzqF3pLwd8jIgEbt9X58XYuws5qhWq/k47g4fUiavwj2cQ/aD2POmTm4X3kOvUi9LTzK0ja'
    '4MeXvzMbPGqX45nqgw1/fTVjV/gwIRSPGpanyzn8ldLGJQfkiEXlF09C8sUhoQJis96+/ILyKKG6+8XgvcWQ3JNGHpldEOZo4Qm1'
    'IJl8bifYjuFJOhLDvM1p9bpPco9bA4i3VGJ7vZfqhQMvhn3gRR940W/nax14MewDL/rzsW/gRbF3JWYx073Re5i+ah6mu+3bnXMy'
    'BWe0S1VCUg5cryy5yhfZ2ISoKMmr9DDxoJJ+hDMDjaUMCj9Od9TjjXPfp8zyF4xEKOPioh0Tco3pgBC24SfBpRT8cs47VsoTqqCS'
    '86TIWHB0MJmKKYSD8MpsePlpu5LI6Pzy0yGp86WGNgcY0xzZtYonS5HGnPNxOR6uOMgIsFGLggXiszGaAeu/Sx5iISXHdyQog0/a'
    'fPaEJVKVTfH6m+3oIw7tmA4ej46pFjHWLj10RRbozIWp2x1zJxkgx5ef/Xi5XvU78jf9trxS23KTO8jmG7eVS+l3+VXa5ZdmuXvH'
    'M3bDkMOz9Zwe3FwI5OGc1Gaere3IkNiS7tdwIO/OBufd9JRFfM9ywdVUjJrMBx4jB8IJVcohj9KQd85L4UtPwObwJKQkch8P/beW'
    'JuCHvjVtT/w2UPoGSu3m/z8kqIjBo9Ui7faDbkE5IiQtD6ffHZDV9PJTPk/xJLM+fE6a5m6x0aS35zpsviq3dV12purF1pzj/kDL'
    'vaSUTxdXcs9NhzMTVfDe3DPho4+FQ3lrdDKfnc4vP6FKFJ4r8ULHtJU4xihXMCrJPNK5ZxOy3z8f0+jcvvxtz89w46fzc6Q7kpIE'
    'B+qexljKX9RQyonSCj7wq8G825Is5eejbnLG1vxuyhonKs/Elg4vIfibayUE+T6HwVE9J06NSBsgQhcjpJiC7U6vweZE3mtHZ1ty'
    'sbu8grsb5ferqnUD/90bKHur/K2yyu94zHuT/Kthkt+uMjjqVQa9yqDflq9RZXDUqwxeY5XBVgFvo33iCo/+Ri/f9fLdzfLdptPT'
    'i3gvKOK5XQJ+PIPmmdYulI7mrKWbSyf7rblPfjhAQcuTgL0MCq8EgTP8c0rJSlEPo+48Jp64N/hhN+0+XnqG8sAG8AXb6npCbw54'
    'X43re/sq5GxTB5p8a0TJqy8QOg8j2o/YWKwz5Ng7gky3UrO5Ut0G5q8XskoerAUVKeoWszYcEf91FLA4IZWg2PcpRGSxPPbEinjm'
    'EzVVu1KWJZRH7FfyxVfyxgCh7Yd7S6HHfndeeHd2drRO9nQ/xXKt+59K0nzE9HpB857ftuG8W3tfE9UwOehM/r2DUJGbosBOvCj7'
    'bHnCWJf06n4fHrUfr1bu1vw2IYpmNj3158mjdCTfiOlA8L3uWTdc0gmLyQ0uPxmec+zXEWLDKBkH6eNj0A87kVw3+wPJoCO6+jPK'
    'w7HgYzGhlDqzKTsxssQdfFf7Bdq4QPv4veLzWxxf+3XetM67O0GVeztBPSFl1ZV6yG1kzE7aEKFtcvPwS1YMuicrgrw6XOaJ8tf8'
    'cLY4n30Engw5dIiJ+3bdHGSDL3/3CN7Hdc5rcd5RMQIWKjHIcfsMjnYkbC4hkfLmXaN+owLVoYTU6iSxhcPlMdIDnYVUou0Zo9+n'
    'o068f6gqi+T0OVsiQ6nxHxp8NBpffi6CIrIZ+HMSCtn167Xzeu3v7oUTuoXn6Bd/x8XfXQC6OV/Gdg7xjV7+6eWf2yX/7He+exHo'
    'GxaBtpd3/MEuRdju9+Ud+8prfXnHfvP3Ke/44pilL+/Yn7O9yztCvK+22BBDbqbno+csH3C+GpUtrkQ5nc1nP/ELjqpt3T1UyOBl'
]


In [ ]:
# PAQUETE (4/12)
PAQUETE += [
    'Xw2WOYgh+nDXOSxW/igpksDWlPYv8RK83cr5oqgwv8THnZwoZx9fMerKAzZW5mLw3rCj1QxQ6h4zr6MyIK1wKIYRj5y/GX6Lkc6Q'
    'rUjLKedxopS//jtXrw8F3/kjTleSrVykixEDFUXdeZ5qGlOU9Wv5MtZym43zptO9JRas35yXsDk758Ro9km16aWjy094zdfRk/5F'
    'Zugx8SK8SqvYjk9nyPE2E0c6FszuDR5RKi6qJblJ1Ru1uF5qQkqSE96vY8bLJ6sZ98ekChq3ePxi8OZ41k2eUIAwu/tdo2uG3HVN'
    'pVQPzvdnT9up30ZTlBWZ78ySTOnZVuJ8g/ld8Ey/KKuLslc6UnPWNiOLfoVXVnh3ziS9KXzFFgG7GsGieff8gvqJiZrTM9X+K8xt'
    'aZbi4yUpE1gNoeWHjZ8IFT4aLe4FbK74+cCgzgNJ6RgzPAacG91KNpYOm3gmb712GIPIUetSIqSdnvpu+4wxuoVtSC44B6pbWR1m'
    'OJvMYn2ye1qH7OH0CpXhdTweoYZvXDUK6Pez1+/H1+TT4fDqfghy6bfgm9yCrfE7a3dnWwWxfg+/sT184QR5G1L0dn2CvFc8Qd7W'
    'jbujCfLAFuxSLuRKeo/O443Ln62feMUsK6nd+T7qJ+6JVsxqgCimyBhESQ1G6X9jbA9UUSwrkTWCMFL8IDmPiHdp5O2IM/v2BqhX'
    'rDwH1ZXApliSdCWAaeMahMI3d3PuN8WfbXpzS7GdO7mQL35fj/r72t/Xr3Rfj/r7+nO5r8xJ5vtxkhNYMq6YtkUbMYhlidrjefd8'
    'RCqBowfv/ApUBsGUJUFibNNZgD1YrPIHixUNxGJT5ZZsQyIsEi9Gp0uJHVtIHNtxO3hATlkjz+h8tzvpOJyEgsieXn5G0xSQ2SPr'
    'wXcRhDY7aZ+TGtV/Y3ahgUR3Zbp7sKpyMraZJu/Iqr20FEBsKD4ZPenmqNX6lCyFbFDkakdc/nE4ns09aCdz2BcJNyzHTy6/4H7Q'
    'xk1m4264HLfztTofFy3qTFx+NiH54AlMD34ai3aMChYi/V9+ejh4NDsRU+xwRBI/6zFh+SQ1JxXi8hK11BfnchgDj4385lPyGxPY'
    'OemecX5Gvytz1oBSYOfoGOJChIXDsPxD4eRRAYzgGg2FxNnGpP8/h3XbXjDktq/j7lXQ9qMWz7h+V18GbbcyaFsX8TbXQNue9eao'
    'z3rTZ715/bPeHPVZb25dIZpdohgUUUd570ryhXZ6DNvEu1R14bOTaXfB5Zm78fFsOZ92Wn9B4rVEkdvJIeFgdHaUXi/JED4tNU09'
    'AMsLqsnwkzNKCw7WlgssUEZigfUgCL+Ts/lsMqKCZ8wchbHno+ewVF8MnnBwfHeKAneDtEnqIAO9XnPayw9AN3qLtPNarc+uFybb'
    'qv3rFnG+0R8i3NRgroPrZ9SubPClFwny3uCKUHmoLpb2r8bv8i0/Alnz7g0+8Hs9HJ2247OWs8o8bsfPyb/1nm9NY492cAhHzXuD'
    'dzwu9Y1DKWbyvv8Lnv7+DMy9/7vfLTu0/EQBnsf+4bfbwVsebVJ4xfuxBOXggd8UynYzxee88Mm/ULqbR0uO5Hqf58tDLdvpYYzo'
    'uDd4c3Y6O6cBYrzHPfIda4ck5IrL7r3BD5Yjqi/Ijq8eitHEAyD+sCFMNoS4HsKNF5CMOOdO9PH1w/ltpwCV0ZwO44fzJXmw3uNa'
    '814yPuloEPFrJU+55RR8YrAb0xp6qeuEgkV+cTmaenaVYGTPF6TXEXMVPThuj2erRcn7Lb8TW76Rg3oBHHJjMfX+bN2Fs7UzLUv2'
    'U3aMqFjDGp16d9k997vRjZ+OIvvt2eyToGE8seVPOvLu71h4v/ycpHf6nd+lpZ0ckxbripVf5Xr6CwpGcHK2ABkpo/yadRQWwTzy'
    'lUrJfqjLzzgHATDtbQN6D10K79GWFOG3a+a7G2VvLgPsOUtEjO9UCXgtYP7Dqb6/oNf8LDopwzPgCnScSH3sl7fj6BUpxjxsxxSJ'
    'yurXWOmtjfm8KLaVPOAnjCE4vwUtAZzUJW/X48tPnnJx8YexbWRkCry/KiKvzfjo8qdvf+/Be3bmCCaKgFipu7j5g3YJKcz+2Eu/'
    'y/PZ9Zke+rXbbe1uzvpwwxHeOf1DvyM77cjOSpRsz2q+IOzrEp9Q+YfIp8MaoJXiYWfd9PIzKpbeDkbH4ml5sfZ1WwCOfv9wyovP'
    'pnUqCBFiaSg2Z1uhtyCSvFow7VN4F2u8hSN+paa2s41hF2dBSXRpz/VVz1iTmFIviyuvuS38mWAGvkljyvX9Dgdvk9rzdIlltYkv'
    'yYITcl+apJQ2ySXwiyTGVF/YWwPwTR5512zPVv/X2zLxnT1ysn10zh8s52TqvaJyvj/9iOnIO8vh0xD4SLwgZwojdfrkeDmO6U8X'
    'nDDVC8+jycyWAF5VPQpDGt++wpkiw2vaZPkBeUtxRpJuQOmULz9HrrIgoB7qV8hFX1w+/CPsEx9Z5tHEQ8gl9oznsv9Ki9hFqDBP'
    'VsMbDlYd+Q/Ywj1nBxGh02x3oPyw9M03Lz9fPJnNYxxuv277rtteqvN4XrdE2/ZbsOcW7C4HZl/BOfdaEnl/1RZZFAfuZsdJT/FX'
    'I6lpuFwKY27Hx4eD++SAxbHbmmLd6bsmfJnWehQjXm8FpPu6vd5EF2/DpHcmiOV+8gNbj9eO66PuI8+tXkke3g7e7EbPRtPTw8Hj'
    'p+309ClVPN6mtAmme6hswA9Hjc1sg8ZGDuI3DMMe0gGv4ObD9c1OZOfEXjcnMr3ODUPD9E64ZsR8nbN6kwIBxzZ9zkG5yQ3DFskO'
    'TKdxgjhhr8NjEwBo+M2oIsdQ2UFtEkBchNQ8XIHiIDfpHQDf4Gk77LQQOT1GySIXK3aSBWWW62iyNA9OJ3M+i3ngRNUfchjfuWnf'
    'rHDa7fxsyVd819Z09/jWfWSghxwk+UYf3vrqhrfetHd3NMKV1aPlXmd9xRNu/cz/crvwpPv03F/RB4eDI16zpZ/BfO7Fh8nygvx/'
    'qG4LatOK/ASj149m84/umfYWyn1hlIFX6LUB9wnCJ+hLBjCRtKomkZN+24De73yv7tfmc37LVmFnTF7tc7qDE8IVngsuCQcrxXiC'
    'o0JcL2qLSjk7dEVUKlP0zpgS/p/IXTY5bcVzRLLtSpgGPzKlks/nHQIrYhyS+pxE5fbCUn1CDH57Lz/lX6ViwIV4S4RqEK/HZPa6'
    'B3Fnt1VveC3W5YVjaTas4Pf7WJo9Ymm2LuJtjqURnmHP/C9ChZZMhXqm4dVgGnbarLvHMex5toe9um9Xdd+2FbzF6j6Y/tMdTP9c'
    'xiWWJ5htrk+N2PNYgulktKDEi+Qc9MG8pXSJlNswBo77G0YBveLDQ0KlTXp9SP4NH02YeVmecHZogoD8f8TqA68HLuZyzCoXSuHA'
    'vy84hWR0D7o/aT9GztWFmcbKYE1Dg4VkF+eziZdZaaW5xgvtthdkT7upl3pnIY1jezIZkZmKJrlgFX23uG5EGsVEVYeKuUf3P3xw'
    '/+33Hr93j677kOs+gG+iurfMapFnB5emIIUUZg1xPFRI6xf861vwmxwwNl6SbTHw/fZ9Tdu3Mz5s9gu3DpljNtefbW3S2R+256zr'
    'xVomB0XDymqyurN7IyUEYdXx8YiN4IvInjB3Q47YnGiSJ7fqVz2MoQ7v026AhaAledQOz/Hpt8nn+hOO8kybrDmInmeSM5d9q0+9'
    'AEeJHNvzy8/H6t3+Gsxjj/Diq9u61Q32lq/M7nVP99FKSVjLG33Z077s6c+z7OmNB7CvfPpzqnzKOpVqP7kTnMe1zgGUJ/iDdsKh'
    'Av6991E8cvD9dtpRqp4pBYT/zswv6XI80zBw47x2NeY7fJiYFC8zLU+Xcwr5pvBslxwQL0JJ5E4CMh2SLEXL9/blF349JLzwYvAe'
    '8Rue4+om3QXlTqAo8XP/aRLRVta6NSEMUXRbcQV4fSe5h+iMQ3CThf+1XaqdfXHyHX1x/NNnnudAGayY1JBVCFcM/DGLnLp956up'
    '3tZ4k6jAiMNFre3auIvBlAsnrOSUI5XaGaYvbp22qmJMim7KJx4tKRMKfOkO1pOYAFVftFFtXFwB/gosMZRZrRicA/J5Nz8fcVIx'
    'OKqH9JDXL6rooSkv+bxdja7vF/MFF3NXp6GtB/3GQPV+m15sm3Z3mL7eEenRjBTql5+u1QwgrG0LLVDc8AillUfBm+SsO22v8dx4'
    'FOo008QQ4m6YMz/AcDmmPJFjDjj0AI4vPz+FN/qm6ghUuGCtOIJAt0S5BRQ08Lxd9O06V2UNu7+PnoGokGw2YRf8wYdz5IkJiqIP'
    'GSbEaVFR6rMRAmKOLj9ZDtuFBMHQH9/tTsWNh02tno+D8uRpN+9OQkz0jGjcqKN0UC2n55wtQv2JCf3QSSTm0BOw5eR4HlKy9uv3'
    'ldfvepT1Eo74FpNCv1tfcbdecm7Z2WLEOHZ0vDybz550FMy87DwW9ezfuBuQz6nfmGeW93rQcZH6IcpMcVa1hf/UBTVOOj8koeTZ'
    'mScGc17uMIaWlaFK8iwzchYFvxis3ZEUp3M/95ZChBiQjp1PPf1pocOiWnL+hUOqlXV6+QXl2Sa/D//Jk1Hr93PADn5PWn8shpef'
    'LhGqJ6bPH7NwSQu2PW/szmuyrYjO7V+jnXVpe+aEnVPGwXVWfrYYXn5xOHiX5SNKSvOo81fUlht7Ii478UNjCrEigYS+SMVgSaJJ'
    'm7Q+HOBza+FZG7JwAKGsMFQb+KngV8TfCWzzLQR8D/GW92oLN3r7Zr+zNijd71Q/mZMd+4ofQjsnMXmFEd/LPh5yvRCV8VO9/HyC'
    'uOJAH0W8Zw/9xYgKXQV+6JsYeY+TJeu1hUX4Biawc2aonVz3NkB6nW1Ni66tOonpZzwlOhtJVJ4nLE8Ji1/vokYJxufX/eFgwN5t'
    '/j4M/dQ+5q+PR4vzdhfnNXzVRPEZgw6FglC28XO+uZR4jGLhWX47GAT/fb/ynBspMokqVcIDfzj2HBrVjiWOzH/pOR2AdsrhvRSM'
    'S6SMfZHmI9ZDmUzo8OOcduLqtBh1E38EBzh6i6CJXi5Iy3ylVGG/4j/3Fb/RQXPbVdmlVGG/hz/vPdzZ7eB6f/8HfvLWJDRennoZ'
    'iVdv4bHz5kRjD6cruYjICfu89Xx1eyBqn+lMqk9yrNnh4K0WRRNkPfxedrN5kB9Dasjp0u8mEkQO5lQnl0TBGaV0YuPSBZuiiKt4'
    '3o7JDdz/77lf1JPLT4ccjU85q0msO52dI1fTYkklHzuiVVPSondcYwKk7h2ujyku5+/ef3yfFFh+WaFt90f02C/FYrkYzGeec48e'
    'UXdw4tdLZbufnS0OSndvNXeW35q9w+vH69kgQmLPA2hkxNMr/GqWNaaTWk6xKDEN9aZ4Db+KV//YwoAVM0iVyTVxILQBwxlXUplT'
    'zZO5F5VPGH/S6JQ9fhQx8ZDAm0bu+TWaz35h/+MtOSVen0XZmZht4/NZsSPJqq6rR35E5YUDBfaTeW9+2k5HHwsj8NboZD47nV9+'
    'QrR78D1SSIaS6SKYUMo+UuhkSZLcs1mubBaqmIJKskyRYhSZqK74a9vMVIeDB14EYrsT8wnnRPtnpjry3PM2XLKBUumyc7ak29FM'
    'V1N+ft5xnnrWZ5mwrVgo7U6vwaa7d+XobCuadpdXcPco0z1pmNgw+zDT2xlmesO23q0Y0xexAgk2GlDmg0H3k3Z4LtIypXuazT3R'
    'fjh68sTf16ezCyOavesvezc9JVbVM83tORvkxQ0P2dkfX37KK+BRAOUqIdu/3xH24J8twnpehEVqWSUnwisRbnaVGnKefz8SKYxR'
    'cVERiP8EzHZ+A9rz1gSkjZ5QkiqeC6EXTCZacLa5aOy8ElsyiN/ildnZmyvbcpZCxr7V0gsiZV0r4n8YmbutniAXUdni7LNW8zOF'
    '/YrwwmwgkSBc9ga+IX61/dYs25BKZPVXkzmEtCzTyy+oatS1sByIHoZc1OEuQyF0M7LZntBGTQfrXjPkPXc69peZKybMCGr2RZkw'
    '/Q+fP5+dw9wc5ixCLju2mAcPBvpBfzRChStB/P16vrT13IYqdjjom9FEv0cva49euJrHBl7ie301j1enmse2PbpT1Ty2Kuw2h3f3'
    'CrvbrbDbtrO9ws54bdR76gFQGXXtcrx3ft7+ehvcWj6Ykb8gR2lQTap5R1qREKNxTO4ra1l5Pb3DF4LJQsKdZlTWe8RL9R65IbPr'
    'IP9tJbfiD5bdcTeUc/1qgLKPCI4V3XwoX4kZ7Xye8v1QbRv84NcVS6bG2eptlIDb+8tzLxwiGHatItq7cBkk8Ckd+WgOP/Yr014d'
    'Q0v9UqXdEZVDiyqJ+eykg5PthULvHyYlA0mMQYF0y4DeA3HGfdqiKbpds3/hE72p1G1/ol+tE33TPt2lE/1kxzzfVF58t7p2D1c9'
    '+6N3aV6lh0mScAaEw8FjP4MOBhF+nORez/R70ZZtLSwBcDKPRTumVCWhODliAUhuH5LAcUJqRS4ZQdEKJxSCPSLn6zEl15+28FJs'
    'Y0Uc47pzfvnpcCru5UGsGXNhzYcyW0JyLOAvx8MVAirAxsTRWKHHS/bJnTPRfJd8IVrRQ39HQhpY3JnPnrCyQNUGeP3NdvSRVNt8'
    'PDr2XxLP3PTQFZnqNLUyQIbKAGMLpJeDfrxcrysQkyH32/JKbcvNcY5bLt3WbM/9Rr9KG/3Cktcm0bWXvL6q5LV9RW+/5CXVO3ZI'
    '53dt/YXW1l9YLOd9+Y6+fMd1ae52Ojx97Y69a3cke/qTPl56rMXVG9Z11PEPkjeA7OTqN3FA3rUzVF2bWbPbPQp85PrvGxMWxVxE'
    'Bzbj2VF7zHZyxBPNx4je9KzG6WgxbvH4xeDN8aybPKEFHU3j0ysZkzik8zozgQfn+7On7dRvkAmRg0nArMiUnm2lBGrw5wqK7n5R'
    'VhZlDxFZl26LyrxfXru8u9cs3e/OhyqQfdHSvmjpq1O09IbD2lcs/XorlsJrY59EnWyAf6N32ng1nDZu2KY757eRpXuaXsbdpJ1e'
    'dVLv5n7k6eHgh7Px85G/68RszFWFTikAm4Sz+AENLKcSIHr5+fQ7gxNx2R/bpDQxgv5gDQbfez/ikcF3Q40HL0q8OZtOsTtPZ2zR'
    '90A8bxHxRqFtHXkJIKqV9FGeeQqS6O2Ffx97jGzeFgHy1q7DC6dJGPZpEvo0Cf2K75ImYdinSXjN0iSIG8Y+fG3UAb7Re2G8Gl4Y'
    'O2zVnXMtSpM9E6gjl3tfVaGvqvDzrKqw/fT1JRW+WkmFP9D/95r9d+Jx/S+czz7qpqOPu/nhM8+ZvfQxEv+fKwr+1/+3+m9WuLzK'
    'w2/4PU1dkv2BQfJ1LMDSn/u5H/6O7v9vfIvSBvmL7pFf6rHipPOszMJ3/tAfSpPmYJBm+R8+GPh27dtJgzb9ntbcTh39XqJtfs8K'
    '/T3L1p4J36zi71mS+7Yz388yvFvqN/n5NDcwOPPNTMeVZ/jdALN/N3O1mYvAk6/D2Sg8dYrfk9XnM8CZOZp7VuAZv3hZneD7pYGN'
    'xkoBQ5YrPKmdi3kmrcxYBHOVRxiyuuJxKw9n1qDd1HGsrHG6tqmueeYUHsyrqsx6Njr31M69MnOpda1W9ldgprlXic4xrH+h38cc'
    'cz0zsj4Y1+5XaZ6Rdx2tOebramrzmtf2iCVmKoUuG4OfmSWsGPw8cWZa6dq2yBFo/FTylIfKs5raTQQtq/hoVJVZWZpJY1cQq1+b'
    'm5PyLuY6rD0ZWYmh3OqqyWz5XVmprCoVhDK+isOGFclzhjjRFZHP8KbLUIX/Pc/LOJPM7o9dtEL3U15NUvo8FrPIdMHTbBXKcAbr'
    'chWXCPRyP/B8zWe21hVsCnzHw5Nj3KwyOAabmxk8ZFZTzrKjKeIZ3Fc8k5e5uaM8F+xiTWOVzhwGpysu8NT0vMsUH8j6pGv3KSw5'
    'DqRZfoEh5V3MIsxyMrBWsm5NE7+T4wTw3EsPZqkYJi9KM1ISb1NeKkaNv6d6YDLFfryyOW5ZnunuAnPi2vBOxBvKmKGKM88rWWU9'
    'SbVBtDxBZ5AHNiWv+XdzBnUmeTjK+SouCJ9kvFAX5jbl5ho0Bm/KPui5ruhuB3Td6EQyc6To1dyQBr9C+plEj1cgB5VuT17wtXFx'
    'AeV4AQ3KFHENal3AXMA0qDJPDBp3OlatiCFr+MqlONYZzYufLwghCfyVYu6qWF9Mc2Dr0qxCbmZb6kUUIsQQ1AKBrlqR84I7QzAa'
    'Q0Qthk7NpjSKQmsXL5n8nhuCnZdFhC3nXcSq5QRDUchKKZIA+pXvlIZKgPhl5oibg2SpSqZtIUICfx3hDxfX6dryacixK1gfORmM'
    '2AR+wwDhkIc5KkGl7SqAawgt+Nf16BuUElezVHTKI+VVRGvyRUbdgtYYmrwyqwyIC1w//n7hlMUoaGULzNxg94pRDT/dGOySrFOy'
    'QPmSOCh4B+eUsDrFpUWlODlrZCmVCubE1xQyEToKRQbsRWuWY2vzho81jmbhYSiaJAIvqEAvASGgIjNMXKKMkj9IBkhnzoQzaC9T'
    'Bg2D5nSvilT2uIm4rta1oSXLzUhhj8EsJIac1IaaO8U/cmMqF5/JwBvp3gdsy8gfe5nzd2rsfUYzT3A+ijrimbxK47YVSaHnIGFy'
    'iGVN9PmicBHvFaX/vcRRKJh8AnMUTRWxSJkW9Ax/v/Q/y9oo0mMsE845nTJnGSPFnZEDovMZ+DTfLgMHREdOdhBr6SJVCutNdyeu'
    'q95YYL1wq4lIFrliqCxXbIg7mNM65VUZb7iwDTnYZOxDmunvdCoKaRfUdtgHZ+hJlSgbwCw/9rbI/XfKEvtD7FOOZ0qwbSnadaQJ'
    'ZV7GE1jmDBuP5fyrLg2UosChAAY2TICQUdAtc0YrIx4EFjJf/d2jwLgneaC19nwbkSkzuI1qAFvqE29YE9cSe5IUkXXyxDfy7xHT'
    'NhGG3MDJdLosk7j2YX+IUuRg5hnNCMx53cTzktP3S7kPuV+rMpG2UoGSaTDObEkYPsfeMj3GVHjBDQ+OBTcsYG24ccKouUH7Hpcb'
    'FqYw0rMsPRMqlagiBqYjkTRmuVM99oFb9u2yEokzN9chV8JfWYJHS1Omse2Ec2auXq4JLVlRZvGaCNvFWxJFrUS3J2PWKTcoCu2c'
    '2F/ZHjrGuWxhw8udxu1x+H7BGgLMtygV5qJsFL3V/jslGIgySeJcSubScU3KtI5orGQWHMe3bPw3Hda8JGKfAyU7movLU7TpjgFN'
    'OSavQEcuz+LxZRRYBPpU4iyyGFrWuv9GkMusxiTw5cm6ZG4k6sycBcPMxKtXrHLLmWgKmKkT0sLnIsmVyQksqSPU5SKV9psUGZuA'
    'GgvekzySGVeWEdXJHHPnDNqjPRG2hUgaRB6+hXIlCj7KwJ4FrV8hWw5EhhtJipEiw5aTQFXgiJcsxGLZShb8hBI1SaTfLlVi5FjJ'
    '08h2MtZGOy8jVXCMUeT7dYC4Uq0IXZmyyJQJQbOJm8P3CPtRs1ilwq6MjatfGJWCiHbULoWnZaHWWaG8MTx5Gtu5UbjkiQq+lrpG'
    '7jJbFR0jlub1VY7EKp5ycBUiHbnIHwXeuCoNJncGzdD6FoVBIVnkbLLAGBoqDcYwVQ43a5TfDrw3cx6pUgQ5WhAe8TsEceHD+Yji'
    'd4LNOeF+CL0VaLPyAwSz5PXEsS9ZhC6BKkgX44QCM9OKtiv0Gcfbjnm5Wqm3IypcFHiG5l5hfx1xHJUcqVQ5MMJo2HWi6Xr6hPjV'
    'KnnVKn7WevoaFXka1Ys1eoAYxWLfm3xdF6OssywDeP3E6g8rwyCoDqUKR7Iy8gBNMVE9VFGkRhjP9Rg2iWHmam3XitVkaYHVnLbl'
    'eRauBY0wKy/wsyiZC3WsDCWrVNOSVdUqFSxSZUwqYRa57eQI01xSpY4iq/DRLoRpZxTkVBEpTBuLknmjx1b0YtyWccFcivgBLY3q'
    'm4pSGH5lBJmyBoY/d1FfxpS1zPF7oVec22XRRIWDE7GV1qGqqqjWyeXdhq9REa9OEBCYMcUZ4HYpzxAzJe8yg1uByjJjJYe3KBVt'
    'MHqvQLXKhNcZV43210H0KnEH8Azr70AVXa7UzxVEfnGVHQtBjVxT5mbwO1MqUOuKOY9axHRFwxU9U+HKVtA/irKY9xeSZl1FFMUE'
    '1QWCYNS6IhRyE8eOmoUISWncNKaMcqlVmGWpGdA1yYqwKZSoYclU9eSuVJV/lSSR4YjPVMo0sAzQqDo3T1WnVQWK1ShKAJUqlTKJ'
    'rouPpFzrlBmIwrRVuShMI6OEqHlJDGUy32dZL9V2WasOLw8mHTXjiKrRqQxYFUrJMkUJgdK4RlXBTNXyxKCHzKis89h2wXTTqIzZ'
    'JKs6vHjF86jQFc1aoyp3Y78IjDGtuQOlRLupI1NdpIlRNaeq3E2K2HY4ZqKzzOIVr/MyMmoBJZSGmlZGQ0ftEuuW12qGgvqlMXIr'
    '1qpgZjS0VRhhNCD7WJSp/l7p9Soqld8LQjk1zlJRWWae5iX6CeL4HFRDTJVrkYvzLK5tSYypCBdlwWgDHBzpKmow9i5ReuhYbyaU'
    'uKC7g/viCJ3XTtpOUUijeuIqUd1axfqKgE6IOS5FKHRRzqxIP1DnUKgzapdbT/tbV9ImzgBCUJ0yPqnQprng+zWPBbJW830RlqBW'
    'FFgzJwCSyMIIOIkkIChq4ZQyHgTOdXlsVnqtqFknkeU1mkLhLCuVL8Dm4wE6wK5U+17EcHLUaj3JaFZqaWgiX1NHZaTwck0aEVmj'
    '4DSKWsFgCwpJyrgt3K6zymg0jQo2yGJO0QzQpJHRXRJRY5EoasytTbBSG19AUanRpKaZkekIbVeWi1Jm3glKpiPQFJaBL0zbWpfN'
    'M8JF8XUMlhSLwpt4RcTcmhtuzKom66i6EVqVEgVtQN1ZLd24NKLYolB9gxUcCqOHKPJcTSBii3KKNqDwD6g3iVRZLOWlCh1O22V4'
    'viDYVNUXn6drJ/oMOn+NvFur9RrWwiaJ6FzQBrcb0Z3QyXMBtWdGRib1EbgECCxBSEkjJ8rtuhA0nxmDSGXUjaoO5LYIfSAFUA3m'
    'Bm1wW4SOHOcHXF1aGMskcVSVqolrITvEKTaivyly5TKh9s2VUxStPGGJJhetPLMVZRSmArngy18kUbAPwhetoew763ICKUhUQmET'
    'VyNkAZ4KwmVWqvVPldsumCTiDhY03wKcWZGWau5IGZ4ikqagagXpEBUf3a9GFQ1pIloHYntLoU1oy++8uGBKCiKEokEpiRgUQK4l'
    'GwGTRtuijcqdEioSJRsRN10e8HJJl6FwubK6wgJXddQaMNsrh7uss8jLlbToMuGyqaOu09Hi1uDNXKXGZG4XosiqefLYYdcokXZ0'
    '6hvsQEXYrg58chZ3u2L3A+FxC15RsVCqMr2im10Bc1Q1YzXg+0xlzLrMI0NQl4rVaqIEDkxGzRMDpm8YO+L7TconUVjjRs1YhOFq'
    'iPgN3YAmF+mXsZQIyMStNJhMA0stWKaGJJhG3s5VQdAQHiyxjE3JMwYUTnWAjdM72cASIjRjxZxGMIl6y3dc3HFPiop4WnzH0YVt'
    'Aj2XHUyVdmcrtFt0izCgVdE6JpQ3WREzwNFCjikC4Q3CSU2rpwoFIbyJkmNaXmwNE+kqj80mUaNYHRWgRSTd4qPQGPIFW50ziohG'
    'SbqQU5ZkGiXLZaXOYNGqWqsEwjoe63pj1Kt5UFzQiohOjtqNq42Ek0QVfCTjxu6dOkN+ndreWQ9nfhfdD6SgTPVzhVOFiQjpaJvn'
    'XapONtGwahznssSQ6yRytNyuSqtsqYzOz0hQ9vda9YJ5parlKImVESVDCROUNsbhLWtUEsuNdYjaQZpyxar3RVCeuNI4ibGOrYjt'
    'KkkN+6DOQGLBlLZKetHBj8+J2ndroxTyZ7pUEdDwHjLJzOh80Taao7qo1H6SWHeplXZjtKLaboLxxbhOVc2qbUz4fLgqCaBoV9ou'
    '9HchhGiX2i4T0xbGpTY2Z24LA8RqiEw1s3mwP/NuqFpL5HpmjJpKmaGoyVXVYGaMH2gbNViAgZl4p5rf6OfgVu2wZfB5oJvhssgk'
    'Zcb/If7OOxy4LVyZxDph5VHr1oBEwqwY+Ce174HHCibgXEV4ajdFprxX0NKRKCd+WGwyFFE6V280tMPvtN3Cz4G6NpE/k6Vmnsxj'
    'qDTK82GwgplcF1V5gQl16rKWs89HkPlJnhRjG9y1RAlRq3MSjKRBGdCoyp03XzYHzpWlcmuBi2PgYBZi1k3QfcFiT6ntNNjz0Cld'
    'ZOqq8BQvamI7tXi3JFGUYbYuDA7vCIEqU0ejIlPvvyJ3UTZhdwfZT/YuqYVvLFNjlNKrXDj18isqVeBxW/aZVReBd+XriOvO5v1a'
    'AK3VXM7cpPDJzE0GA4TRApfwTCwaY8S0KlLwfmzDriPH6XCVWR3ijw4MncQvFGIlM4bdktW9YHjQdlls+2eU0ZQZu0SlNwfLpGhZ'
    'QJ+aqJwV7YWjY16KPqJU52ZXKrfrnHrGOBgy8B02OwujmVTRUYgZ07wQvwc1O9eJLl1Nq9UIQwn7FxijOlduvc6bSIjqQidQ03JV'
    '4G1qp77CNV8psDR1pbJKzXbTwHVWUW3VlKwLEIaKzRPyB5YxhHEUCxjeZjONsF2NWrCbRpVknqNUbyDfIT2H6AwSy4UkhVI434Gx'
    'Q14Cqx58G+EQEzx0oA4NPiN8/8LnIUOXQTHC2ofwkVr5kKRW1w5/3Vn/Flg1mKSCS4r4eoQe1DSis2kY4sC9QRUeeLYyGhlTUbiE'
    'P8EvMRFGL1Oa6meRRpuH77AgEhyn2YAfOnVEGr5jPHDTyvBaaZVGBzE/H6cTTg2x9B3SrJeBVyuUUc1S67kJFCZfyOAJHTpq4vJL'
    '5dQHLysY1YUeHCTywIYZAx6MpiI2VlH4rOLNZIt3FVrOqA/rqD5MYisNwogoF0mYT9PQTMWPh6XcKrTSaGeP0kwRdZIkp4lSlg5z'
    'Gj2amlSt8E5Vkk1oBpWkUj2YYvLoFSjYQ9WMaKbaDvZ99pDMgsAkX65VecMWv6QKzQosAF1kUeLXygfW5YreU1Q7tYoOfJFyaGSg'
    '+ExCM4yhKiG4wooArWA2bK5yKnVFWSx4oLAsrzZhQZONYuFGdYCNnkYwcklousaFpjDkjZGHoL6x7cKoZRPjA2G8AQtn1LWmbVS6'
    'UUguo2Mbt0szrktV7VsnJnIkSVTvmxuv86gzduqIwQJmrs+Ik4e455bGFdy0c6MnLm27Nvpj4/RRa7surSNzbX3KVeFcByCaaA/h'
    'dl0Z3/csV3E2iqqJ0QgnUZoXUThtjNN9Zhx5K+N2pGrs+KHU+JsYr9dUlfNoO30myKHAcolpq2o8eNczj5ibwC45Q2jr73lm2gE2'
    'VksXKrMHdXth/GjUbYtV7FWi7TRvNBIl6uddJM/cji9U0Y+M27WzESS5kdor21bnvbgSddTFoR0GrqMQIUr82ijxTdhcseLqnhm1'
    'QGGcNFXTL4yHGFCtWkDjWMQtKkSX5MbKatqpWlxjlEUZ/YK4LZ6w3G4qExBY5Jnx4Ki07TRoITfOSdHs26wFG1Q28KAwcVwmKrEx'
    'rtZBeaCaS7RLtWeIeARFhZxkeEKJKwjc1hNtV9quCtVaNEHb4KKNlttNVUaNRNRykOuFOFy62mhF6sjHid1alRZFUEhkxusqM7bt'
    'TGHmtnGDEXcOeTczBvCisSoPVYUImswM5c0M6eV20FoYjwdY0gttl0F1Ah4vVw+c8FHiQYzXTVGlJqBE2zGUJo2eQlB/pOreLUZJ'
    'blelM21VozQr7uDBSy2Lrk7cDmvN7Vy92oQiiU5FfffDnsFukJlOluobqQgymTG5c7uyrv/BwMYdURUiLKZo5MOsQijUjJWYYIss'
    'RJdmRuXCNkh1XSiDpxKfgKQ2Oh3V0YR4BG6XopRqVP+eNdWqIqcoS2MFq6MVzDral0FhU8bd4ragcgR6mnAbUXoh9CYV74ikiVgU'
    'yp48uFDQnhaVGtGEinBb5ObcGGG5LSYHKIGKwljXimBSS6KrMbwx5A3jlcJmN0EruYnBgrOWKFHYg18crlgcddoWaxDHgabxBUb5'
    'WRL9PURDzG2PrJqoRRK6AI9dAY/EBDntsPKlVrtUaASDWIq4nQaf7TJRoMpE/Xqp7QoNfBI0w4FeVXhebe05xA1xzdegQrTFfcUp'
    'u8mqLGHfc6MkzmEzVvcVjz8ajSqTGBnCRE1dx3YazhsLBaL5g/+fzJk7gnNY4yVoHO4vTrROkKpcbLtg3FRdJrvKpWXdqCosccZy'
    'WRi9lni64JCJ+RFRvHUdjZohQi/VmBTWhKXBZZroe1Vm0QfPb2kdnfBSOSoFB6rJ6Kx7BEIuChMeVKgNinVkwdUHyoNK2xJLxk75'
    '1rXPL2emHdlujijyr5RRl+bRiIv+QWJGLSqN4ENb0FlhHLRgtRWcx9FJwimz1dZ/tjKdRjzHQUAq7WRB7ZYwkKkJ+cAfTLAYq9dc'
    '3sR2BR6G216qraNPklgcStxJMArsoCTCO0ceiPaodBpwVTr1WeR2U8ozTZQWWAcnB4zd3EPkQWOUeYyHgXu5nZYCBB9hCali87CQ'
    'KXaHipq9PEIHZV4pMj0jHxHZXKlmH/aplJvLDlRNcKDKTZuJl6gR2AxYiGeVikVVoiEuFV8SfJTbweeeOWAnftFpUC1UpXrYoC1B'
    'lxUOYJZEa3QeZH51XaxYyJbIQqasYuRl7aHnVpOoPhQJoGZ1owMKrHN40pVRrxhcs3I18XBbzBN1bvSHfExS2G/QKWtRE6gmmdup'
    'KJNqdnZ2aVRFBo8yviBi8mG9pPiJ1HBDlD9wx9WiGkiioqWB9JJm2inEOM6KUwcYmwxMSaUdCdNpmD13waBeRP1qkzPTX4gRXZVf'
    'DRQkuNONYTi4XdTBcg60I/KyjYZICvWG8R0yzbkg/5dW3k6tUxcjlTJEd8coDy9xw4wVZGsbbizoP3jFE08fYuJwSJIg51b89eAy'
    'nxlVYaE6bY8g6bkqptRg1B/ecjYDiYkb94tQ2pBs3pY8GIP5ijUhMDIxMeC0/gHcjF2+Q7uOBzFllrQKopxBNKnEA4WQvlot0zl8'
    'XiTKKFO9ewpXUyfWbIR9F6JbzkuTlYSZg1SMlimzB2kQgzmCIw06jbxEmJyE1rMOspEvulTVqzmsHPIS05mqDD2Co8rCW3wu09DT'
    'PBUpsxfCOaQcvhn8/cBsCPfje4QUBCKWjcqwAPC5ETUs30Y5BzmLGsEZgF3t/fzlGywXlEl4ku+qqETYSBcuRc6eG07iYxFSJfq+'
    'FMH0Inzn7GkhIn3eGD12YeLXU7ArQZMGjyo5WUWmnry+k2sIaAEpHwcDPIccaHb2ryVQo8hZry4fyI1epCg1Z4jv0EmoRd+CaLAi'
    'dJoYX+DRYMpbmkkvj7xDWhgLfgqGQaIqUib5YhTzHeP1WLBHaRq+3hjEwbEJQZUEZy8BjyMVgus7ooXkdHPArseWGBb8gfhxp+z/'
    'lQa/lLIoVCFasp+2XPqygBiDxSz59khog2dU1LKZMqPgHxQ42BApt5YdxvxZr6VXqZanRPCQdNhOIccKHmRy1R2ztnLfHFveZJ3Z'
    '/ueXHi85VuqUgoZdrmFRKWKkBG3C1VpUmq4wOXw4RiNELjo2l5UBCIssHFveZO5VYhRXVWYUlRUjYXFaSCtQVTll7LYWvIcqZ7R9'
    'laRHka+LkSsP+VqcxngmIVRLLR9i4sgiY8h6JXXN0laTawixiSY2v8acHxLGwe+LzZORRa5RiXUR80yk4uHBnpoakFLXGpASmyIU'
    'g5zU0bc7FU1mxSE5oVWrK5m4gWssaqWiHat3xHvcxTgSfrZM9T2h0Gw3heAjyYuyYIsROGt1PavBFyUhckZssqoS4qY4KdYa5lir'
    'uoHVMtg5xrG18GisbNCY7WisCV9Qj/faRO+UUcuCD2jUt+CR2q1EgEscHF3CJo2Od4n64EncuMp59VoenrRRy1AZTEDBXqQemo2G'
    'DzcqSLFnnhqGZEJNthIhGGw9WbSiIVjQhaZrUnXAb2IztR74rjSmHm3nJlAptyadYJcuI1svpp7CBDap+ahKrVd/aUw9hf4hXBB0'
]


In [ ]:
# PAQUETE (5/12)
PAQUETE += [
    'SmNlCtcFkQAmz0nMLeeMtcetJmzJrTXJ2Yh+pxYkySuFtn6nWEkmpJYlZ/Lb+Vtikky5sjEOjqVpO9M2TpCl+T3wvXxwwnRYYeI0'
    'oVStyQeCWSBpVpNOFYn+XqT6bsyy0kTfM9ioCpPer8hMIFlR2TRozrRtajXjo2myqNhY6WjLMRlB4CJu2sGHkiyjVWJsXbWxdRUm'
    'eZixaRkbWPQTTZX5oHZt2gErIcbDxnuUxmKV5okJ+HAmr0Zq2mrtykv9PXqE5mo05G86ZwYQZYF0XGMMYeq8GhfbJJRLTTKk1ITS'
    'pYUx8BXGMGduS+qMWctF9z7Y0DIbjG7iV8J6VWbDq6gr5HZTqhks2Gq57RJjEtM0MXFJGzUqc7tRp9no19HEQHBu12GTG1W0Sjyi'
    '8bKtKuNla9ouNW01ueXm3cI8UzrNpiU2e8msleQmRN+2C+PGa9rBFTdbs+8lzrQ1NVhh8jWGneF2pd+pEptUx7gAu8oEVyambX5v'
    'zO9NZtqFMQ3q87l5PgqgZbQlcFuCLiWdWWIyZJWpyX1S18ZOqO1wSfmZROS/rFKejq2JMZa80fXNDVrKTZAquzEnlXFpFmGJ5pnI'
    '8TVRlWx+zJKqMvZHtTMGSw5z801tEriE1IelJmph+2OluY3KVD2l60x/r3NN8uGFDn05Ew9A6dgMqCY9XW7MlMFV1zkTy+XMcLzG'
    'GtfVFMYZOyk1mWBwqHbVqpd2HoLIKqUV4ppt3LSD23gTLxi3q0qfCW7dLPuZwN4QdVYlMQYFbRkYnt/iFV1BYa4m0vgHYzeDedWY'
    'VJ0+b82xceQs+pSKqTW1NluTbkxNoSyzig2aGX+xW3JYkFi8YHltbBKfSjuZ+E0gN4fTdgRXrw23i/BVFyO1xVardtvKhNg1Np2k'
    'JGbJhM930aAbvPmh+jDtMHIVOUr4wtdJFjv+7iQmYagmZ0jN79FVX7kWSYOpbbELcLsJ+UiSaCCD+diYjPNMzcSl+b0x5mYvWRqb'
    'tAu2Xha3E80xmorPQ+g00WwcMk/UismRSSLXZ4rEmJkLNT+LHo/b4eSz1r3Ub3oA5aOqRhW7dKmpKgqTtqLQeIDKtIOJFZ2qMIZs'
    'iZ+D6Tsp9LFMvC4RRBAs9mWMnJK0L5WmHXJFbkIzg7XcKROcGRGJ235Azc7rwhh0JFKndvDchHNKvpeQZ0YfSsvS/CUGNCSMM3PT'
    'CaELGobH7Wh4T3Vv0K7Vpp5o/Gg4wI3GjGXG2xft3KQwDs9osh9JbWxCLFLNz2XfDb4TRpsucav6vDPf96RRXlADILdFaYO4jUaz'
    'YooxCb+nhUbANrmx9zubysdpkj3J94iEe+oTEGIMuF1X0SegCr4FpaJI7mRJZkJGsjS2syzRgNvctD261ZRCQe5CWK5wB3lSr8bo'
    'SlAUt6OnAtuKKs3d4Dlv83YiGRnRcU47cXg+ssHHIDGRvZqcEIkHm9S0NSGhqFjxe1Ko20MmjqKSqlBT58XMMibjZqoqVLhAJOoC'
    'EXaS2yFuJldPVHTKWp0pUsGzSF4TJlEawEt1UEg1PJzbtZj3JfGNybSciKMA61pNPszC5MUoQ7K5VI8Lu1QLduROlkiIMju4ucw4'
    'bySZBk03wZODCZt130gSDaf2CMdpJ5MdMCxknqvSk9t+m6qYqyfm7VGCCfePVH8PXhTUDrvB7VLdSPzd04f8qdcMQFkq65ODxmrU'
    'UYg0zxuTmY9tzcbbRPhGdMo8004jg3DGtOBuwpgyUT8UUfZxO42ZGk0YVYEcw4XGpKcSTM7BOZIdmFMWVZq+SJh7cV0pNYpdx2A2'
    'tNBXshDcVZhsxIUKmGiLxw57wcglL9TnEu1Mfw/B/ay+Djer1FxX8I4R75USi5OajuAaVnjLoSpTTftSqscDMjUJomJ7gMgA3PFS'
    'Rh5TngmviwSkMlGXG88ZvtXm95A7DZ3K+ONE5xwT4AYnycr8LnmrYMPM9KlU4qNzY8/ktpgF2eImEl9uKgOw8S0TjR5y2ApJQJpg'
    'TRmc5XIfjCsiG97CCiLYXVPAxYQ3LmoZ5PesiB2/ScZLqNTsWGnZ6B+yNGRAUDkFJjo5INyWHFocce30mTSQEI66Fg8lMDCZRuFl'
    'EjPEjkgxMWai+AQ8i6wmsylpogF6fgnz6LvkD5smYBCbHZIxiH6VO5loCAvIrpLFSwRZTc7gbKKGPFeHp6LSPGEuZGpIVrM5BOcl'
    'VtWV+nsVggDZccY4TnmEncdYQQ9Urt5SmTOdpmhiOogieE65iDTZcyp4Z7F/g+QxYTcqv1hNjCIUVQjbN8Mbmeav5TxHQaWJTljE'
    'DEtdxGTqwmazdTQT/gpOWCEHcB6PDHLPhuhEIhYSDcpRi3J+OGVhiEgs1I7J7eBExW1JJF9kSubYbStsAKfwBAuAkMc0i20ROeDZ'
    'JfGy6GTAz2ytzUzmzjQNf2ASJCekVEsjt4UqF4ycJOF7gRREtbabVNNGhXhLkwmcTVWN+HVBgks1+FLwETohRJMj7eQ4s8Qjw5Ew'
    'IsoEhGVKpExRa8bzolFGhtNfi0oWmUDAF7ITWUjiiJskLl4c05yG/HbUEcVMaZxhyySPzkVsZxYlWmkqoJRJHSOu2fws+1WaAi6l'
    'KWhRwoNVKk6kmp+nTDW8G23xd+N7IGF4bMYWPMNGbHETgkG7Dqkxs1hpAu1K26mkmuGORx9JTKMc0itznKo4p/EJLyrNmBLHpnNW'
    'ZOpUJ/QUka1CBUvQe/Gw404uwbNOOU9ui2CG30XLVTqlDaWDgJlFR7z4hzoKLaUzoXROuYWSCVFIGW3SrpS1ap4QVluFeFtVwnFb'
    'eBu4+oU8ZizUmyQvHnVJRi9FqJzhLMsbF/37BClx2+NN/UNWhL+YhDGppmngmF4xeXBbaCW3M0mTgo6Eu0hHEidzRwxM7EUY0/Fq'
    '8j1u5y61noZNzNMb4gSF+3YxaW9IeQ5WOhPfRDWWwk8xJHErNMcmex8E9yFOFunPZBqzRRYhOFkjp9gRwY9RaKRyJi6MSNyfxbbk'
    'P3Wc/0y8Krk0SlJKHlg1dztYY8V3oHFRv8htOZFoCxyNqtA4/ZyEBqEdsvRoPBqCpHN1pBSbC7fFWZ3T1WUhTU+OzRPHAz5fdROz'
    '13lOB2MUGhjGGX8kvRC3M7kZVQF/rlJzASWlSQwkVtGK11bSD1SlMzkywVNIRQd1Y6sY64tmq6pUUVyZTPdVnZl2HcO34dYpuiUE'
    'iOMUcluE0TrTfBDczmK+PQ0grK08yB1/KHL1A5V0PTXovfEQzeSEsZOnyG41ewBliXTU4FwXlckPqm6raAfXhCSeKW57BqiMWZRk'
    'OeFFKnMl95RMyrShk4mHKMJM1UFUbPHIu1SLf2itCk4Ob6dS99HX029GFjuZJCppci3KwI6ffsPFcaCM3ljcDui+KeGsYwLnGwk8'
    '5YMhXkYNkLR4ixpkjDYWpTF575tKbZ5NpUS1qRQjcQh+JjlQOAQ/2Fg5Bj8kMeJ05LnTdqltcdLitmcQmxi072dRatS+uraWJgu8'
    'iWMRN9fEdKL7AUcGxKyuhTFDJoVxDUtwhEqN9A9aL/9ZWxuiTG1Ybqo5yNmHNkubxHrHxuywRncK99gsD44aTWNzIuZMbyrTcyGO'
    'Ekq6kFOa0FMdymIx3ipDLGVa2XpfNaOukBzdlOeSbPixB21N6EHwD1Z23M9c/XTTJpiOcxtAyo7MwYCdK+tIITPMzOXq0JtpOvY0'
    'Ki/RcSGfI1/IGMTJtzCGnha2IlqpfhniBBxmUll3AY53iZnfc/W9S2FIiYmmeIuCtRvB7cG1oK6NhwOHdZchqy8fXpcV2suCuyE7'
    'IItbt/gfBzMz52gLlQhsYaCMqX6wQCOQqLadRjtpGazfpPwow18qY9ouq4ji0cnSAA9LHjE1lVEvpGLaCMGmdWPs1OwMGYJkgf3E'
    '4ZbuoKB9dAKWQi8Tt/0UsWJZrg7Sacg2wR7SYXtyUxwFneDqwB2R/NLc5AOUjsAKgT9Y5NGTyBffY2QrHjjoBR8krgQU1oI7dfCE'
    'TTTDFjqiC/ToRpXQ0imCwyxPKg+dQnFYIdU85eO5ZmlEpwhewMxNRQdcBylTXIxrk1O1gEugXG8IeOKKUtRssJR2rdHN6GTBzbk2'
    'qVsLeL6W2vHMTfD1ZY5IZg+pKoTNoxe8FkqUHU1DD/Emmfay4LVSZlrFC53gk19ycZT4QWa5ysz0RLpIy9xge+5IcJ9fvNT6EjN+'
    'L4IzscEQ6MjRL4tS5Qr0spAzoeQ7J5SgNNrHFJKTOKCURoeXlpXJ3lDWms7HI5BK/+IYE2fiwMO9LGwxWPlG/DUcNNjhvVxL/KAT'
    'jpNzqpRCJ87esQQXsuk6BPvJAeWeF2fkPRvm4CwicM2K23FpsEeVaDAOOsHBubKZhDk/UfxCYarWcifc/KpULQI6oTRk1ajtBp2Q'
    'WgbcbEgEULOzmuxonaoBEZ0sOLZwBFO+0pFFqXPgr0R6ZuM4cCmEiHAnrkNdmjKs3MlCHoGaz4swQylHL3k2MJMeM3WCf2px/5C/'
    'sconUNuamblGEtvXVRHTkPiOwSw1nIwlMKSGZ4EQ4rpWzZbvZDEjrO/kat9IazF3h79pOq2UU2yLmSmtgYEF59YN+H+BFzg3fKMx'
    'DqR1Y8rE1SgBLNStYRle8hD7XhYVJL6TM3daS6/QcmoNsqyEv7D2Q1BLI9WJsTiI96qkzRyQuMo2TMNCxbwGaqQq9AqlBJx+NWiV'
    '0kYK/Qq87NQXIi44IWsWDhKnZBXTru+wCBwgZttb2chowtDJ1KSYZXiSrWyCVDiBq++E9yynwglds+CcyRld/XjyHiukyrDkLAkH'
    'V0HIMiK+pohcS8NaiipKYMlNgErDQnMStooRVaBALMVk4SI3Rm+aNk6DuH2n0EoJDV+aTJwuGwefGHkQ6R/kwQo6TwGqAk8qI7MH'
    'UJGHXhWrpfkOY125JpypNsvCJ9kvJZF73rAdIywWe2I0wnE1LGHLLeSYPf+J0GOaLk4TSWoyNrA0k4VCpEmqNMN3Ks3UkKC4tKS2'
    'TEz5bd8xCSSTTDfCr1EWMzH5Tq3cV5YUaTSmen7AeIslTmOk/CkpVTXte42m2pQUaZl0iogWMk6JlsXqx/BXEWcKzomWpZIqgvOg'
    'xSnCcUJcBDnxWagsw3nPQk0JyFfBGxDyVcjticzx4qAhqeDlL2wSlzrWkJPCmsEJU9JPpHwDJLtulkKAl8oZEHKEZ83Y1zHETXse'
    'hY+XeNykJZZAxi6hIKilp2xXBglIcreyABQ/7+DKLB+E/1r4G7zNxHOHhZSQYyRtoPwNPRbEwkca1TJ6LlCJlO9UJvlJJj7bWJ4s'
    'Uz9932G9VRZ6jeZAzaDNFt+ijEPbMnHEysCtyMhQnQnAWaGJBTNO35bHv4ADl8874yAm2fblOfhxSsoY9u4Lye7hGpeJExY7u4Vz'
    'zq5Xwf2RvZSE3cng/hP8H9nnJwtZVTixRybiYYaS1hKtk3F+OeH3M8n64aSTqZoig6ON6KYz+GSI40zG1dnDeYAgJD5J7EcRcqnk'
    'bF8QLyZ4GuTiR5cXqs/IJLpURuJEdqnAx/UCI4AcRhguKQeNZsHHli3TYa1hmpZdgOm3DqCb4hsZ23LDUnNu+yDAZoVJQZ2x+BQw'
    'XoGs5djvAnVOnXRANuQLfLYkVW9RoM4NJlyU6r6TobJSfNBphrOsMGov36nVHRD57eWAcyXFgCZhykrEkbFEveFSOoXZ0tLkFsxK'
    'ttoGvIu6anLrWJ4Jzpqc616YVd/RqghZaUqGeM6AOVc57FyhUaoQZmzzqELKJhh45S+5WWVUaQwJmXKbd5otqFlwleYssNFXODde'
    'toyB6+CDmDh1psm4NEuWRofNxiRYJpYhelSW6k2eIdV9EmqKcSIdvxK+963ns2F7/K17g9/4VuL//60/eNae/K/f8r+n3OtmC+5l'
    '3DuZDbmXc2+56ObcLbh7PDvnXsm94flPuOe49wuhW3F3Oltc/taXn/IvNf3yq0vPzSTUbWI3ZSiS2M+4n8Z+zv0s9gvu57Ffcr+I'
    'fcf9MvYr7jN8v3rMbQbuV8+5DbCm3FaY+LkMMD3hNuCZc1th6bivsODZCEvKU80iLCnPNXOxz3PNqtjnuWZxqVKeaxbhSnmueVyr'
    'lOeax7VKea55hM/PjvoRPj9D6it8LfcVPp53rvANua/wnXBf4eP55wofz79g+AbcZND+K24Cqm9xmyH6r7nJwPw33GQ4/ltuMgj/'
    'HTd59P+emzzw/8BNHvN/5HPIw/1P3OTh/mdu8nAH3OTRvs1NHu2QmzzaL3CTR+O9Knk03qaSR+MdKnk03hzHo/G+OB6Nt8TxaLwb'
    'jkfjjXA8Gu+B49F4+R2Pdo+bPNp3uMmj/UFu8mj/C18jHg03ikd7g5s82v/GTR7tPjd5tDe5yaMdcZNHe8BNHu0hN3m073KTR/tF'
    'vqI82lvc5NG+x00e7fvc5NF+iZs82tvc5NHe4SaP9i43ebT3uMmjPeImj/YDxgA82vvc5NEec5NH+4CbPNqH3OTRfshNHu2Xucmj'
    '/Qo3ebQfcZNH+9+5yaP9ISAY3JpfRYcH/MNo84j/B9o85K+hzWP+n2jzoC3aPCowScLDDtHmcU/Q5oE7YDIe+AnaPO4p2jzuU7R5'
    '3BHaPO4ztHncj9Dmccdo87gTtHlcYDCgsBkwJo97hjaP+2O0edw52jzuAm0eF1gRKGuJNo/7HG0e99fR5nF/gjaPewHMzON+jDaP'
    '+xto87j/F9o87v+NNo/7/6DN4/6/aPO4X/4RdHjgL3+KDo/85W+iw0N/+VugAjz2l38UHR78yz+GDo/+5R9Hh4f/8k+gw+N/+SfR'
    'YQC+/FPoAII/jQ4g+DPoAII/iw4g+G3QHUDw59ABBH8eHUDwF9ABBH8RHUDwl9ABBH8ZHUDwV9ABBH8VHUDw19ABBH8dlA4Q/A10'
    'AMHfRAcQ/C10AMHfRgcQ/B10AMHfRQcQ/D10AMHfRwcQ/AN0AMEnoK+A4B+iAwh+Bx1A8I/QAQT/GB1A8E/QAQT/FB1A8M/QAQSf'
    'ogMI/jk6gOBfgKoDgn+JDiD4DB1A8K/QAQT/Gh1A8LvoAIKfoQMIfg8dQPA5OoDg36ADCP4teAlA8O/QAQT/Hh1A8B/QAQT/ER1A'
    '8AU6gOA/oQMIfh8dQPCf0QEE/wUdQPD/gXthCC7/CDoMweVP0WEILn8THYbg8rfQYQgu/yg6DMHlH0OHIbj84+gwBJd/Ah2G4PJP'
    'osMQXP4p8EyA4E+jAwj+DDqA4M+iAwh+Gx1A8OfQAQR/Hh1A8BfQAQR/ER1A8JfQAQR/GZwaIPgr6ACCv4oOIPhr6ACCv44OIPgb'
    '6ACCv4kOIPhb6ACCv40OIPg76ACCvwveEBD8PXQAwd9HBxD8A3QAwSfoAIJ/iA4g+B10AME/QgcQ/GN0AME/QQcQ/FNwo4Dgn6ED'
    'CD5FBxD8c3QAwb9ABxD8S3QAwWfoAIJ/hQ4g+NfoAILfRQcQ/Az8LyD4PXQAwefoAIJ/gw4g+LfoAIJ/hw4g+PfoAIL/gA4g+I/o'
    'AIIv0AEE/wkcNyD4fXQAwX9GBxD8F3QAAe4PcGK3QIchOAG/D5zYTdFhCMYtOgzBTP7CEJzjL8CJc+kwBDN8GjhxKH+R2whMmgEr'
    'dhA7gBWX+Diw4nCEDsPQjdEBDPIOYMBjwIpLTAJYcSiwAi2eAAigxZF0GIS5vFQKeDgMQIxn8iADMZL5AjN203P0GIwfy9DAjS3A'
    'BW6cztBhMBbyGIMxAezAjSfyGIMxwUjAjZO4ZkCPZwEQIMhhqz80+sMY8lQSv7DADwzTU0hHwJYnnTwbNuhTdAusIjoM1lw6Th78'
    'HF2Gqp2iwzCNpIMtgigFlHmKc5IDZy5G6DAc5/g6cOaJ/IWBGANA4My5/IWBWMrXcFLlAwzC8Vj+BL5xitOeA22+jb8BbZ7hNaDN'
    '87DYOTBnK68VeHKOXjgrn2OewJ6nmCew50Q6NTZaZgD8Sd0IELDoeCY9hughnhY0+pugenke9udn6BZyDDESkOkTHO0c2PQUMwM2'
    'XSzRqQGr/KnBCcDowKdPsMDAp8upQAaMeoSPAKNOMRYwqv9i2Ftg1fPwUVxnebHSZwOswK4P5WkGaAwQgF3PgCRyoNdzfAjo9Wkb'
    'thkY9mgZtw9Y9ql8SPCLPBvO72cYE5j2RCYKVDuVF7E+LbQDgmt/C3xMDmw7HOlBAMo9l+MPnNsCeuDcMTYLOHeBPRCcu9CjB8Tb'
    'TeXhWqeGgwH0eyqHH/j3rNUJAQlPoPIAEl5OgWByoGGPHAOUQMQjWWRg4ok864Ag5asVtlUeBPIBiMDF8xFQfQ5svJD9BTom6KSf'
    'GSwl3wZe9ocCMAEzQ52WAzNPw3kCZj6XswbUvAw7JLg5/FFwM1Q5AAp/AWo+H4U1BnY+m4Xj1MgqYa7A0EOZD1D0RxN0gKLj8jZC'
    'MwErMPQjea/Bpk+hSUowDXTSMH3Q4QKY+eESnRyPztArAhqQPoNz3KIDCX4uX8FtG6GDcy1/YVjO5Dmg5ud4Dpj5vuxeAdzcTdBh'
    'SJYAGrj5TMAS5HwuH4FkLZ+v5IDLF2vwC1gH4OcnI9D7Avh5iKNdAEGfYBmAoE+wJwXQ8yTgvwIY+ifyFQbmY/mI8DQyfFYFvLlA'
    'vwZhPkWviXglPgEU/XApMwWOvg/wgaPflL8wTMcACPh5HjiKAgh6KZ/E+gAi4OfhKJziIg8chuwPsPQEmKQoIjhgTwpg6hFgAJ4+'
    'w4vA0+fAbAUQ9XN5rhTi8nvoOkEL8ldwPFMZoAYulc80ikHi8gNVfzSJHxRsLbsMdO2fxueBrT1ily7u11N0SiAWTBa4+hRwAFGf'
    '4TgAT5+N5U9gNeRwODnQ+BvwdAd0WwBHz1vwHgWQ9ETgdOFyfYq5AlG3sg4uEI6fSR8YcTmXLkP0pn6aYXoncFQFEPWxzKwSLhUD'
    'CZbGi5FV/im6uGitDAMU7Y8oegGmTwP6LSrZPJxoIOqfjMDJFUDVDOQYmuckQhkfAso+wueAr49ASgtg6sfYA+DpY3kJgoQsOtD0'
    'kwhULZQf6ww8Ldi/AJ6erEyiEaomBxIIuwVjUAgzPQuIrslXjyQ+2wTKJqcOiJuR/IkM4uRg6zgM5pEMw1A+kDEYyIeCmkrg73aI'
    'DvZSZIIS2HuMuZaJHHbpgQ/AXS6BvIcCTwn0zYyAYNISONxzaKAMJdD4uzN0GKbHOLwl0PgwnLcSmPwJRJMSiHw+k2eBGQSoVARC'
    'IJQSuHw4Cs8KG4C5ApufyQSAzM9mY/QaYZNFHiqBzicd6FwJfP5QKEEJjD45Hkk3MrjyOeD043YhHwMj0IW3I8oC7SiB2T1G0Edq'
    'uSqj8EOD7Q2otgRup5mbcYHhYVIqgeHfwvvA8I9whUug+FYYsRIY/kRWLRfOAB0GbRbIXAkMP9O9Aoo/E86lBI6fy/YAwS/iw8Dx'
    'y/BXCEyhV4S9wy0ohRsXXF5GRC9/DdRwJn2wdTNwgCVwfSvcTVkKY4c7WgqOn4W/iji5iFeuBKIfjwQVl2XAr18AWiB74V0FYKD8'
    'R3K1gPPfnJlv1IKPwEyXQP10xmQOwP7Y9YAMShABvzlyj0AGeEcgK5egBK5EhwG9v4KTSlCD7wF0EIPhXD6HaxpY+BK04AnoUQlK'
    'APpZggyM5E8gAyNI/CXIwAySbAk6cCZ3CWRgCWBBBJ7LQQQRuJBvCDshf6qx/TKFoDUBoi/rINF8BqmvBOafCKhA/eedQFeLaC5/'
    'LMC5xeMAAjCOXSdHWljeEiRgEWYHGvDjZTzWtTAW84BqQAPuT1cQIigB81X4ThOwbbiHTbikQVIrQQnuY0VABu6DEytBAh4CQmD/'
    'd+QzwLQyKsN2LGKSA/YfQo/igP67Z+iA1fkJOgzKM2i2nOD+kbwV5Br0wBHiCLuA9WU02N862VMnqmi/KPhuGnbxE5xGB8wv7JUT'
    '9coIorMT1D8KYwH5n7TSgwA4CryaSwVhTNADYKLQcED/b4d35SZCeeMU+eNLwP6ns7iKwP+L5Vno55FAG6naZaKJmsdhszIIDZiv'
    'kIKnMk4lvKAsdCYqj2lAgg5koB3KioAIEM4EKnC5KIbGQjcciIDHSXpVHGjB/bn9MCiCP1sinzrQBF4D+zkXJ66TEDEA0+zC+/XK'
    'wLi2DtQCPhAOtOJIrpUDsfgQCgwHUnEMVOlAKoYCBUhFJx0sqTwHrluOLWjEGGsFAjEey1tAJ1i1UkRafAPE4Xknf2M4PgYmcKWQ'
    'LGGKHMjCPFBSZ8jCz+QXB/4PKi0nynEi2Th6orVp9ViDLDxc2K+oAkeWJ4gG53qZRIfTzQVDO9AFce5xIAw/Xsr7pciyohxwoAoe'
    'ZQuP41yA9KeRF3auNmzniqzpXOQyRdB1IBfEW8o1AsUAXpWLB7KhiBLspFOV+1x2GTQEV3AqnyvDXRDu2oGamDMrhxuEhQ5aYNkc'
    'qEuGpQFtuY+PgLK8KUyKE4niKToM7i/iQIGo/KLsPIjKe9AruVrEL0AKgjIUKEFNnsg9ATF5IuytAyk5BZPqQEcEO4B+jE7RYUhG'
    'srIgHOOg2XYgGxP5pqjm5XiBbswxB9CNhaAxEI7nQePgQDs6qJ4qkI6TTpXyFejHHMinSkQ/BimyAglZTofyuvB3U/TKgMvRdUDl'
    'c/Qq4dvlQ3Wg9fJuI1cKPjjCNF2gF3R2n+JZkJB2GrV4VRAg5tItwBnKx7Bvi6D8qEBEpiNBB1UqoC30plQiSsw7wTuVSBOdCBMV'
    '6ImXWYAEKpUmRHdbgaCArZZTU4GofG8qgmkFauLxY3wJ/MBI9JwViAmd3PBAJQznVL5Qyxm16wGSQgcwrgnICiYkcwRhySt04FqV'
    'oANqMpHn4IMkAICIPAJCqEA8frBEh2F7XzoM2GOsFmjEB1gE0IgPhPBWgUagg709Qwc6a1nhQlg8ebAEq4oOQyEYrRJ9kR4YkIlJ'
    'Jz2GZRoPsSj2obKqysjRfYI+2AK5DqAVC4EBlCKQ1wqE4jxsXSlqR3m2ghZIeqJUE0xdgT6Mw1RBGs5baM0r0e0Dv1agCd1cRgVF'
    '6MbyZiG6ZCyMkysZP+zkz/FtqKw7aEcrEIOT9lwehyq9g9a2qsSgKOdHNPsjCDEV8P6kFdRSyVJF1qmq5FoKbMD1XmCT54HDZBuA'
    '3ucjUaJVVR2tXGG4IGnhdaD401k0XFTA8szXyCPAZcuACytge0/eRKSsgPADEcRPYj+aKyxA/VCQY0uA/h+1kWmo6oA/BN+BBizo'
    '9slYYgLoTqMAWjWizA2fbUSfa2cFsgCJQ54qREcp5xmkYY2g4y9ONDR6IJpK9RbKZ1egGh4fBmDFVriAKF6DboC1HOOXFHyZkLla'
    '1E7jFkikFrvBOPI2daLit7CSNSjI+0vR99cgIR/E41qDitDF0OtdJ3U482fymSYcNDEq1CAoskEX+IkBbhJ0oFbEitYgJ0fQZNWg'
    'Jkft/8/e2/XGcaVpgn3NXxFNY+CkK5ViBL8yWaPqpiW67Grro0TZOzMqLSuYGSRDysxIR2TSolQadM8Cix1g52a2r/bOmCtfFLAN'
    'XwxQO8ACzX9Sv2Tfj+d9z4lkkpar7ZpZbAmoMuNkxPk+5/1+Xvy2g9+0Q0pLPpvjSbr3OQTovhKRR+iCdOqRko++ko+n2l+lHUel'
    'Pqhnpto1+jD8BuayryQjj2l2X6nGiX0D1kQ7oQRjNEHtesBrPA1MUWDHqg8d1EwfoLTTiVAScWZq9r5SiVd6YvtKJcauru4roRib'
    'RroPHZTtsb5Si6keqv4WdHf4VqktGt4G7dJOK72YmWjfV5JRO/XrK9kAJe8r2ZgbH9VXwrFQ4tPfNnunPsFghmZ1X+V6BvpKN+r8'
    'jTrjKieM3b4D6ys2iRKNYQmGsL+zZRY+fdQlhMa5r3SjKbACSjcm5dx+Vhs+tu0OtIhDvGzS93cq5/aVdJRT5bH7SjrKqTOAfSUf'
    'o/IC72/h2tXGduFbgAO5Cy01hP++0o9Tm3qlHsNFWORd2M/9wuorDVGLsQ5fychENfV9sw+rLqmvVISuFe3PXpDCsQxKRZ5gGUz1'
    'hKddSEDYoXt7fuvqM1bUfx/ACILew3PH1qmPyTuLdrUSETVZ4Jbq47oA595XKsJik4v9fSUjzB76SJWMfIp17UPJYpJVv9833agp'
    'S/pKRB7mdnyVhBQvdTBKPV4u7Hwp9TgtoBjqDyDb+N4A4RjbdWUyhe2UAXyfptgM8PhhFtgtvn2lFMQqQpzuK6n4tWvSBkornlW4'
    'KAZKKeiu1WYGSinoRvV1GGxiRpuZfbNtt65rNQdKLOjqxToMlFrQDXuBZ2VHqwke+61lGWthmNKRushLbzXCY6A0YlsjaZRGbA/0'
    'YUt/wU/SvT17Uh4ew1MKcRBRvoGSiY/xu2qQK/w0UCuP9k6pxCE0qwOlE58ECX+gxOLzQh+2lE/XB+nSQz3WAyUQD61aVWrjK+Vd'
    'ajwp126y0kApxJFSloEShy+1bSUO/5Me1IESh1wP2mArIltaAKqlZGigtOEED+onUKCiPVUS4Y4ZbMG3A78OzKz0Bw1YUAFWPQkG'
    'ShqKOZ5wALTvShVOnUwM4Cq00AdlmvAhLHFqtBxAnsBUKlUoXTAfKF14FZXsQNeoDSttmGBmlDRMKvwGg6+ui9KFqd7kAyUL1Uwf'
    'pEsVLocBjM94UX0YrEoYn7U5JQeXYFcG8FP/T9YG3DL/l6hkCzYonQqlB6MCn8M9E+qggZKDUXGCl2G/sV/hpYnlMD9N/dEcNcHq'
    'DWCBBrs4MG9NtZMP4CiUm65nYB6bNuo9WAntc8g5RpEHkCeK0HelBTNMvVKCGYSXQd8kHVy9gz5MvnENSgpotOE8KiloCtfyDZQY'
    'KGutXenDZK4LpFSA6LnVsKfPdjv2oW5X+W7Qhw0MdQ1A7Yf6BHqFKYY+aeqqzcEALhioWwkAm8xQG4zmKrMM9P4fq7p2oLf/IoeE'
    'NoCIUNVOTgeQEgrsboZUUEXoCCE9ME27Y1y6Cf3SAsZzzqVkUoFfAenmJnRfplti7DXobJSYppuwVtOO0LFy5mE36fweJepdk59Y'
    'W2bjmaMghdJ4gccUzi22SxnMTa3AL/GormKlDVAJwUV5YRUqs1lP7XkXbkoja2IPnkpDDEzJAeuhgtk03UwHptpFLJVeewtndahI'
    'XSQbXyEqstP9D1EDGXwmhjbjSimOFqOwKpAnqtBxJRkPbCdRAVjl+gxDz3CefDKzgRdgFEo+JotLPKYtBZm9pIqonU08bsF2Ca8Y'
    'Ktk22RQNbe2YVXGKMULUIHEVndvaM9vh2Er6wb4Xz7QSmEflGFUpgSlez8JuVCoDJTVs95zNOip9Y6+aPjluWgnPeVXbs6qyzuKb'
    'hErNyGDsKIMCqnLMt7jSolM68Vagtt1JfCzTTaNHPsdQchVnpSmvqCyLFR2lv7oF0fIME7Kz7d6KaHQHbuKXdux3dmPWNZiy6Je9'
    'wC3m1kIf7JJd/lQ0iHmmsHeVmhF3pP4/VJAagwQRjooyMD7OJFDZFtifeEsraRNWZ+RlO8rgxJtJiRzxOcYQUNEemB1og6mkHzEB'
    'wfxPP6jL98xu5XRzbzOiuTayvdTI7hBDUwqoJAhbbG8r3HH2XXx43ahD5Tt+Xu2YKE208eIMKl3E4MA0cF51YYK38CRD2NnEiVTa'
    'uOuPqrQaoxXIR3Z8lCA+sCfp7aelrbOSwkeP8SQdfIx1NJ3a1N5V4yMEd3qWXj3FbCHKNdz1SgufqiKDHjMbPK44JYVHtjGVFB6V'
    'mAClhF9Y20oJvyxsmyAM1va7ksIcsapKBVWsS1NzzcLUIiT2xL1SUgTGDs0rIkV07MgEqBQhsiMXQlMEyhaw/KUIlj21CvagthlZ'
    'EzBIuZ9CitDZU+POUsTPnsH5P0UM7csFnqSTL/1HNU5hAhBLO/bZR0DteGh1KycBm0VqcbWmk0gRXDuBW2SKANuJqW7T1Mz3xkWl'
    'iLWdwK8gRbyt3eMIua0Qx2shE1gEpW8zW4EMVNl3BwJw68I+V8bBm4YyF78qWWvC6JWwNfBOTxGSO6+xWErSFuAZEJh7kVtt0rev'
    '7Ud1pjHRIkWE7hvcO4jRfYO7FFG6RYNfQbwaG6hSLmImUfs2nKRO7HeYwmvrDKzyU9uaSqqKqTcAd2bbaKZAs3WFgT6/tGfYOVwL'
    'mCKSd57biEGbcFsinLc2FhIRvbxSmE334voGc6DkqKihxksR3VvUzqYhxHcxtSr6iC7B4wAeGIgHh0NGxMOnCPitbEkt1gKd3jWv'
    'FrS3C6eRKcJMUgT/MonEAds1mmncEqKAqWRkr5hGaG4j2zXvZ/P2TBEUzEXg7xAZXBczNK2EpS5sjZWqEBvkzospwoRFVkZTex6S'
    'BrfZFBHDuSv/UoQNT6sgeKWIHp6U9hEsCbajlJrQz2jZfb4w+31Ylaz3fUjMkU9Omprdxvrh8XPfTawa7WvLeytFkDEzSuZgkSLW'
    'uJz6OwM8R+8MYPyNLigEcRRL+0TpzlmdL/V4AOtYYdcvAu9Ke4QoZg2qoF3aORvA88kuWKVAo/LUrhwlQuOxYRRoYNfYDH2pRSqz'
    'nApcgk04QJ3gUXu4sF+hCGhmgCtQEnTfjWcpYpenue8HBDCTdBfIHeKYDxt7RIyV67vTzKz/J3aCEdQ8LjA6xDWfkajjb2ATN/D4'
    'SRHhLMxSVAgTrV1QiHYmOupVR1KjveNio73jvoEloB6UdpY2bqVN0yJcQgiEvrAgnxTB0LSFMJ9KoopJIMmIii4m0dWVGa2qYEpO'
    'ESFNJYHgI1CayvwgIVyatZ+gmIiZHvlqbSHAxcg2Qqdfl1gXpVtvcmvECBf0ISliqOUesRrMp93tLyniqedB1MlMxecdQ4jJ1F7Y'
    'xqXW2LJtW7yCbV5EBBqZQJj1/RyUDZHW98/LiZdYhMAYz5BwoRzItl3AtQ+gZpt6GybgxvQMUdizKnBDiMXW+3VkPVbCxheEPuq9'
    'toDfYIrIbBGioI9AfDbchD3aKEWoNouYVjeuh2ZoX+J+aOa2axBMSCXRQoDUiVBsUw1fZfb3tPaU2g2rSWuDIXilGJVWEPkfjr2J'
    'HfUWVNfmFEHeLA7iec+98UbWXN/OKPquZO+8ugRYyqZ7Sp3hQBrNG5YegZFmJlGdWRxpijhw84qxwm03XdtSK/V7VMxstyjte7ao'
    'WzeF0T+fKSgZF7479syIiGclf2yNizwlUosaV8MamkToixrSjMBkpnFkxwL0Qqlh8bKw1Vdy+KQu3B6VIqacYwX9BPRdfTtB9/v9'
    'SJ2JMJE0M4o4dIY+Mz2krhsWQUniyJ3aU8Sdi4LQ+oFomRBQmiIGXXR7c79sEC/DmjO7XJQoQjWDvg2CisSpJWLTRU0ytL4NILWG'
    'Wwbx6SS6xpcwgtRZZnU2FrHqj+fOtSJenaVVj5BPEbXOIiuWH6HrT03sR/Q6SalthSfC2EVcda4G4ewksoYrF1HtLJmaSIDYdhJO'
    'X+E5NQl0mTtBrDuLkdCrIuD95SJatS2LreTeBDEVEfAkBV1Ywa5JFz55iLac6hZCWd/5WrtkttJBYG2xaoiOB3Nrc5Clgb21SQXS'
    'UsGqxhJFW212z+Yb4EvKuDmfghh6YZ/cJytFJL2wPEH0Rki98h5+xSO2HhR/FL8+MDL/EpBMm4GAe1NKeo1OuhoHUfdKtvzSQPA9'
    'SEOQDRGFHzYw7nqE42OTjK1qGZ1C56UIy09xKyMwP0vxNFDHWqh8EJq/neFJjaTAC1RCvI16lAzv2HfSv91te9zRR6tHOrSHNpX8'
    '7vmn0qH+Fp6kQ/0BwKykO/1te1THngG+VDp7cIQn9WzEzlYa+zE2q5LXj6EkQGT+x2D6EJh/P5xupamHIDmIzv8EdSkl/cQ19ojP'
    '/8RUJojO/yUUOAjP/9TaVtL5K6y30sxf5fbjrj7O8Cgd+7sC66rU8qHVq/ZtVKSU8iEuS4TmP7IuKX18ZFOltPGJXSvmd4FOKEV8'
    '4lWpSdklfwTo/9rGp4TwqW9spYNHruNCmP4RAgdSBOofnVt7Sv2OoNBBjP4z1KYU75kNROnds3A6ldh9YcddCd0XYKsQp/+FEVpE'
]


In [ ]:
# PAQUETE (6/12)
PAQUETE += [
    '6n9plyRgVM7wpLcpDrMSNL+hlZgZzUKofg4yjFj93ExdCNfPAyOHgP086HERtJ+3Qr9SRO/npkdF/H4eHPhSRPHnkdtBikD+PDKJ'
    'IZz/pLavLMgRjzuwxeNRDfUL+1Ut9fCqTxHUP1zYo97pkJEQ1z/0wSK2f3huvytTObVHveNioRlB/sPAcCHQfxj8n1JE+w8j4R8x'
    '/6PaPoIPXFyP3tKlzYPSnsKEBcT+FyYKIfz/dIEntVFh/yD6/9Q01QAAOPW7ACAAZ+d4UoKCexkQAGeVtYTgAdS9BeAa9EvpxhkO'
    'CQAAzhZoWOnEmanQgQJwFvScwAE4xxkHEsB52B5KFs4XVoE6n9kGULpQvgS+n/KAr/CUmhO2rZ5ShrJBXUoaSrgWpAAEeImeKG14'
    'iesFiAAvTdsCSIBXHj+QAhbgVe5vQK8Akw0wAcbGxAISYBzVAM8MhEWlwAWYmC9cCmSAiet1gQ5gwgTgAaZhdpVkTG12ERb0xupX'
    'P3TvoZKMqatoABVQvcaThhPY7QGsgMoPJzz30DU47kH0AVjADOwGsAKMzQdWQGM7UGlGg8sGSAFNbg0p1WiCSRB4AXO/zIEZMLdt'
    'g2BR27QA2GqsPgVPurDOKeVYmO9HCtCAhW9rpR0XtumVclzYwirhuDCFDhADLqLDrrTjErUr7bj6D4A0BHIAPf9feN7F8/+NZ4uh'
    '+m94hsHepqtvATbo3sB08LgsBhblZQupNIRNf1isgcVQo0aoBhG3ngI6gL1urACMKmIjUgAH8BtWJeINRv7JwAIQQEOAITA2jQlg'
    'BBidoEZBBncee0E34BTiGMAE5rlXiHC0QN8AKTDE1ABOYGj0EnACQ7uvgCgwRBhEapgCwc0+BazAMI8sQwAXoGptBgEwILIACtxo'
    'gF0NmIECiwKYgQIhXCmABor6wmqEySC3NmEzsBYhvETCFvAGBFkFBbrwsJwCcaCqETyfAnGAhRerwfRrZ3h2m4HVaCYDk5cAOkAl'
    'ODwAHaACOBWmQB2QEjS8ZUKW6RiBO8AGjHCOdlwMCVpNoBCM8gked/AY9sCWhYU5lQEgwWiJywEuQV2YbQO4BHVhO86ACYLTCdAJ'
    '2JhhylkAFNSFUVhgFMxs6bYN0cRM70ApmOVGq4BTIK9YOw7c42cKcAVfLQoFj0kNsGCswdgpEAvycVA3A7WgMeMMYAv42VYQIUcF'
    'WA+gFjSF8VmALZgogkoKxIKJIhakO+Y93loDpUSiqXDZGNAFDIQVM10OYDC0Lu8Ctc91M4AvYB819HEXOyNCNEiBYkByr1cEHD9j'
    'AXYc0uY7q8ftM1ag7PD0LDQFWD9QVQAazIvlzQSqVRpRArrBvHqFxy2LRkLFezDIWEN77hzn7iyAOhiVzpgA72A8dnIGzIPP87nf'
    'DfA1rHwM8DY07TZwD9iHZ4ECRF7Y79C0WqNKwpoFDNLAPqBnO5V9hHibD9WOAeBUsdZ0x3wNG2d8AYNwWppuHjAIp6XRO0Ag3F+c'
    '2LPa4fN4npSiAWkvukHgge5hlynwEDhGy+uH74A5WgEXYVyeeJf6iEH24Q0MwMQsdsBI4BD3AgUA8ANtAVLCeQlHy3TXHRENCyQF'
    'ZMKoakorUInOqARwE3JEYqVATshrSJrAThgjni4FdkJT4WYCdoKFKKWATpic+HMWoIUW9o1uhmrkPU+3Qwk6mu54HDkKzF1y6CWG'
    'qBdxdrvB9GUVDVTsb80LYmFr10QBXUFKIHwCX+GkdbkYyEJdhB0GhIWLMqrNdL+vMW9uBLtmWN11a5gxu0BcuCjcpAvEBd502B0G'
    'vONXGkAXOPIdhwyoC6OqRixoapgLUxc6gLjwcRVWDI7uxRyPuJxruwwAtHBRWZ3a2co+N2pdvQGAOBTtILAAWKBnZ4+BsjA8t7UH'
    'zgIHV4DzANgCFfm9CciF2Tg3vhy4C3SmYloN/AViPc/sw76Z4vA8MFudn60dhNWggztmev6HIEvtOoZmJJYDmuFoscwaAKPhaBEf'
    'caWDJKYEvyJHaijdagm0hnwYETAANowL89EDXgOTe2wrpYLpQGEuUmA1cA44PMOaBw0lcBo+NYsBkBpqaiFS0uyaP70rggHZ8CQf'
    'GtsIzIahcye7juYzsW8GXoBdBFSfJtKL7BrEmzq8RidmGa8B/AAAGxTKBn1RmvgwX9juVaL4MHexAXgNj8qYSwdYQ/Ea3BOQGl4u'
    'fJ2UJErsEpqCX0hBBDdm7oHdkPahJwZ8g8ZHorL+VhRYamXbqm61SIgUcA6/pLs2FBmmQ5szArrDJPjm7JphLN6sfcCzmmkLSA9N'
    'eQYlKNAeuMC2FaA+LaTKPtzSwHwMUakkXTuYPaWR9OzKAABAcOh5LHIACoL9fN3BGYAQwc8XpQOPV8Y0AhtCipaZQoBEPCpMEQWc'
    'iGeLOihUABZRzFr7EJgR7KuA+wC4EVPf0ECOmFWNfbKH58gHBwgSs7Bl9gxHtHSnQABJaBEqT+OIyJEVZlGh6UUAK8HRjq3BIyAY'
    'hlirwTxLmiAyA2kCUd/oZeqbKY+8bwA44cW2mYE7ke7BnAHciY9zc8UE8MTHORRvexYSFhNf4E48KSKNGbAnnhRtkg8EiqNF7UQG'
    'IBQcvAUBAigUJ6Xr14BDcRZZyvYMvnRqew8gFBx/MkZB6gX+ThYccNxNDIgUJzRsPG+3vO19dqHQrOrcXdj3jOheVtFxBViFelx7'
    '2/1QFhMN4Fewp6cm0VA/9Mrd1gBi8TQ42BqSxYlzg4CzOM0n1jElw+z12tLTA9uiLsaBIgHiYs6HFgUGZAteASgXi2l5bcM6SOpp'
    'mZv6EbAX4oGIxXA3lbnz44C/gNsfWoZEujiJ+BhgYajzV2DWgYnBPkYY804ASix8IpAOiqMN9TlyV/HjrTRZ3Bo8DmjPwp3FF6E1'
    'aIt8hkuCC7/Az4A3gN8UiHuDpTq3SrYj07JdFkq1D8aBCQKuxscFdOGA1fi4mlo9fdgSI4Uz0DXEohi0nQDZODwrzT0ZOBufFMEv'
    'F1gbYl4s7KZQes0WxsBPAHPjlwuLSAHqxqdsffc1UZr9qyo+qkqyf5VP3SgCDI5f5bNolvdgX3TpA0AcD10lAxyOR+wUs8TGAZHj'
    '0cHRAZ7Ni8JEXSByPFEOFF1Tyv3rhR8FpdtsK4RNBYAcz/KgvQcix7Oqdg0mQDnETmgcOmA52BhoM6FU+4t64XIZgDm+JIIE9QFg'
    'Odi2Z4I/cDnUmGdHBybAcmIiAzA52IhmBeD3yiCtAY4DZi67qKHCfU2npEA2nk3YobAWAOUQy1O4lwHNcZbj7gI0h5qFLLQ/BTzH'
    '+SJXQLUU4BznC3ODBzoHG0xGKNhzA0j8Xd/sGtYrN3w49EkKhI7qdeuqAUgHGzlw0oDU0cgdg5Ittz6Y9bBvGLLEEZlDHZA7RM8O'
    'HqJvHitKKzGs1JXnEbkAlofo2DHb6cD03R7mnALXQwJVrSSofYHrkgLjQ7SybYEYeB+qeg22UoB+iCp0SYYA9oeqL0dWtuuaSLDg'
    'gAFhXWTub/Vdkxi4VmCCiJoOvVWqrUo3GL6BDCLqtFb/tzJXXcW8ALBCgsLJdtnWdqRy8sId1zHZKQR6iCp5wrxs7blax3yJgCTC'
    'ShfbM0q9We1iNi6HFKlel23dKOBFVPI3tgIYIxCzXTEBpBEVmgFrmAJwJMjNmJntnUguru3d3ZYobHpzAJGYLGs1BMFvHHd40Gaz'
    'bRZ3No0xBNUASolybW7AAliJMjNuewFkScybzEPN20btjP0DhomRkMBOAs1E7+YQtQFQk80+njTebBtPmgZyE0nBNNwsxZMMYXfX'
    'fpS+9zftUTUyn+NJ/U+/wJNS7RmeFJUBy6vk+mCBJw1IjBF2UsCYHADAIAWIyUGwlQPI5GDVnlJy/XHQbAPX5P4DPKlEYYYOoJrc'
    'h/UFsCb3DYM9Ba7J/cjHAdAm9220SpcffIL0acqvPsaTdPQBFlNp8AOMHfF6duqU/D4o7VGjCiO8oBQwJg/McgYgkwe2CH24+9uv'
    'CilpjhSAMPnERjqAsyWegHZt3CggTD47wJN07jPsQCWtn9kgla7+CsNSovorO85KTz/3AFGAlnxuXwOz5OG/wVOq4aN40gDTBZ40'
    'tBQaYACVPLRjBJCSh7AAAaLkoXuXAqPkYdhogCl5ZN8DNcrU4cAneXyEp1TR//CkbqnoitLFxzWeFMsSjaQA8cOTKp1qe4TrFrqo'
    'lO/X7n4NjJKn+Fkp3hG6BygrzXOXAp/kyGZHqdsz+1K9tmr7VL22oBAESMkzoOmlwCl5pqCOKYBKvrSq4LI1Rra/TWBk4VF9oaBe'
    'MKgSKCQMq8RMpg5VAiYBWCUBuDUFYImpvIBYkptBH4gleWNdVfKTA54rBWZJ7gHggC3Jw10yMBjEYHEEfkm+4noBkskJduQ2gvcx'
    '59tgMO1RJXf31QCeyUluvyskEsROQJmc+CqAhFhaiRR4Jie+4REv3uLzAWwyBMsCZJNhkHgBbzKMDPKAOBkGHSxwTka2TXcCeBmS'
    'N8KXHq8jcCEIDkA7GQWuDmgnozisDZAnxQWelGu0+F1AnhRQERnkicm7wDwxAw8wT8yjB6AnBdI2pAA9KRDgB9AT52SAelKYL5iB'
    'npgjDUBPTrHplUqc2m97DneDgr4VYOGVRJzafPchPGDhlUicmtoMYCenFnAFpJNTk/+Bc3Jq4JApYE7O7HVkJcNIYH+srW0DxcY8'
    'ATHX45mAcxLJC8A6OXMGEWgnZ04sAHZyjrlVYnFuZApYJ+UbPAFAFP0ZIGmJta4hgd4ZdXtT42wGkJOgds0Ac1IGrWcGnJMyqPoz'
    'IJ2UraOSAeqkxJbJAHTy0n5VP7gKT3vqFWfNqlYdKQmBbvLSzkcGdJOXuoLZpgV442slHuPKfnX44W9QoFExDdpGhLeZWjJAm0ys'
    'M1BzIhUhYE0mntJw0+DZx6gPAd4zzBci6HLUhuBuI9gZUExU9ZUBwgTRqxngS6ohFkgpSeUTr5SkGlrPlZRUAf03A3ZJZTB0GaBL'
    'YnN5BvySmbUK0QfpIQFcMrPxAbUkEpYyAJfMxpgjiDi1TaEJNzblgEgs7Wf4a9jOgUjTiiTJAFZS4xv4wszxhKBJbBglI/XIvkSG'
    'Q3vcNZhFbKdtQNeg9wiPs30LtMSwP5SS1AvMFsK8oSrPNg1nF1OzYylhUN0OXAitMoA4Yq53kH/RHtWDsLFvVTGKpJYAF1nYwijV'
    'WDSoWanGws+y0oyFGYMyYIksTq0rSi4WQZefAUVkYd5gGSBELixNJ/BDLgqvow8IAFt6pRsXPkAlHF+7AJUBMOQS2VGBFnJpS6mU'
    '4w26rHTjjQ1pDwnJ7FONwIv2JbI//8cwSKSA/i9WoaFORq/0YUO3VvqOaOJvwNHxNJTA0xHV9qEMQBZUwIPws9UJfc3Mdo2hJgLs'
    'NNs0F5aZdwNJJYPXeAawkFHhVyEi1Yow4XDKtAUcQDvsHwAXwY7mYMdwENAxxKZNvWMDcxgrbcnMJ/PUFs18Mu0QpOaSiROZuktm'
    'hWe9smMNTAb0EIaSblAAv8wxHtVQlduvu/i1wDOic6F7zIAdMg8hrhmwQ2rvRwostxPUkQLCwcCFMsCH1PnQmkW8dm6BnxkwRKom'
    'Hgvc+/PCnuE/OrNv4ECK5LWAERm2JyQdBBdRdNC0dNZh6Ohyj1LOACcyzBGwkwFQZJhDQ54BU2SYL2x54J6Zm5N5BmARPrUe3p4B'
    'W0SM7aEDSBVpMwoPlTogMGSAGSnq9hRtITOq1b6FDJutKYCnZmkXEDBHhgCBzQA6MizDGm8B/HNkc2+JAQMHA/CRYTmzvQV3lfLM'
    'CrbNDRYuUhkgSArk7MgAQVJAZZ0BgqSwnCOZQZCMVZrKAEFS1aa/zgBCUtUt6gwokrltY4D5KsOXAYjEbDqZoZCY4SoDDMm8hA9D'
    'BiCSeRmUwllquL4KIpkBi2RRnNjv5rALNjB169cZFn7HPXZtnncCygd20c4g+NmiyAK1p+5DkQGShK18blvPUgvVnppUkaXmnDkF'
    '658BnmSU25TDLcXvHcA8mv46AzTJKCi+szQgx9s7cIeYWIHSsFF8kSgR4yJ0dw/5sS+iSd5DmmwjjgAnqYuJ1QJz6QxzapliLYFk'
    'lhr6o3FFACVhn2DvHfiV6/bTDBgl9LnV1zdIlTazBbCSuli6kvvBlRAz3nd8lYAVlgXYEjd8Z6nl/vbl6yPlXeNf9YNzsjUICcWu'
    'DoRnz03LnwGuhG/EGmMawO5ehH0Ao9bYlgsWrTHglDOglNCF53w9kEryMRyQM0CV5ONTq6QfvvkGRfCTDhwTAEumwO7NAFcyrdS1'
    'OgNYCf2OWQBcybRyoOsMkCVNYY+IV36DR/hmgK8DVkmDfFEZcEoYldtaMOQafJACuaa2ZyDXzO0Z8URWIRw1S68Qfprm359llu5q'
    '7G/s2idTFOzhDawjsEmoE7gsgE0ygmQEZJJRBetEBmiSCSySGXBJJrln/8uATDLJ7bIENAm7gKNfiJcuxjbVTui+HaMA99rEmrVU'
    'fid4xp2GGJYssyiEMxuqEra6MpYBaCR0yxt5BSBJXZnUCEQSPnkzDG8rIAH5wm0ZFhD6Bv3d9I231PcEM1BHZcAk4dQJuOwzc8W0'
    '1TNXzLkNadudiqxgCy4JpX+zjZJIA5UBmqScRjoFwJOohxJKoKQI9BcQJfNiiRYCqeTMDVpZZskT7RHmPNsVkMag8sgyy3ti5uEM'
    '8CRNaQIfwEma0iiuoZL4ETZQkpE9w8nDEToyByU5wyum0QOEeWZgJKXvREOzjyB3MyCRjEqLRs4ARUKvGYMPHJJReYLV3EX6EasX'
    '+UdsApS81eWJVYgk42DggEFSlxEWfmYoJKUxN0Agqcsg6RkCScjwkQF/hD0X7B2g8BuPAPSRWTgbe/CQM0VtZsgjVZCegD3CXLmz'
    'uZmFHrh2JAvBB3bMLPogCuXKMotBMHUTUEcmRWATgDtCFO7ahlSqRsQXE4P8WWXu9xRyaJXmR5IBduSssjWFe+VibM/I92J5TTPA'
    'jfCaoxmgbi18+8C1cmEMCGBGmsWlfbFroRTWipKJRWBaAC8i0Q2xC0UGkBGaDszklqXbaoDBlAFdZMThg/ZO5u8sULJl78TxV9mW'
    'Q3K50jMDwAh3YWQlSNkQFIFAF7m/MBe5DNAi9xdRbFIGbJEpUk1kABaZ5gApy4Aswn5Xzt8DUkSS7NhbQHi5RIcQxb3wMwAkkeGi'
    '9k5D0Gv3B9JeMMBnhiWycFkUSCKHUOcAROQQOFHZlqfpwtwq+RuX4CUBHMLBJeh9hjSZb6xC5BX1Gc8Qn9I+H4AJmZkPTwaEECoY'
    '2Xgyj1KJNMWGD8K++VB+AR0EMLAZQEHmiwv7Ysvx0lDzVshjNLKRwzOytBjGbMvAuaAmAgbI1DGBM8CAMJNrBXqF1DhzQALJ6xN0'
    'BZasOpL9gQhCZYHbBi4IldmFCmyQMVRBwAYZFyHxTQaAkHHx0gv24NcB+geUECmwvbVtITb6CLpXjbH+O3HORVfyATREXXy9KiWC'
    'jPxi2xQR4Sc2VJf3NPW3le6G0mgulR5GRwdReZWtK3IJAys8A6ZIbfpgQIpw3jp8gGzyVGBV7hosXmx5ALoIB//YVty1+KA4bisD'
    '0ohg6NkBMcJopAd4I7TadkIQkceROagdMXnBJzcD9AgVxQF1GSBIijqsn9JHEktiXQjASOhuiHkkYJI0LeELyCSNUxYgk1CBc7xA'
    'J2lyV7UBn0SiJTCIPlJImeIYCCVCTL9FB5CiuEaoSwaYEnUvCyog4JVwSkujFcAsoSJT0wK05Kz0EwyjWekXgScZs2jRDNglM05T'
    'MLROKKG8MMYLECYXxtsAxOQiKH+AY8Iw9pjEAWKyLKw9A5gJVWInFhrPiVk8AGbCEHEGCZcBzoTKTmxvKI2s5y70AMzkohh6QRog'
    '8cPCAs1EgrbcxpsB0IQoRzA9ANVktDBjA3BN3izVB5Xt3N/y7EGg1tsmE57ZAgPkRN5BY0olD+kb6CMAdHIwDcokgJ2caGrpDGAn'
    'JwjGygB0IjteMUYyIJ0sGsPHygB0chEOFXBOZoW9AIoDDm3bUliCuwbIyaxAMsoMKCezImIQgXRCcrk1qxSSoaoMxC0D4gkjzttL'
    'O16AvllW5EgLAuST4bnLUAZ+EhobeAZblIA8LswQCQiU2TjHSQIIihSg1i1s5/ylLe2W+eiZmhFIKJNcFPRYOEtM4+NUCrmwk2R4'
    'KJzcHcukBPJ+bkNEPuQ8qL6BjEJFJjwAGeXIMxVmAEc5qfxQbEOAcb38tuNXnmH/GnlsrLfbeyF3ljeujF50awIp5YHfF4BKcY0T'
    'oFKIDXSlLLBS2EF1SYcF1BSBbIu37w7Iu680sioXU3eRzACgQlwMLHcAUBmG7iuJfBQBDGfAUTkKm0/pJOPvoLe7qRkTMUJLqzyN'
    'QIAyIKoIOULBthegh4Bt5hKryvMTR2oHAKzIxnDCA5iVFMhkGXBWxONLnpVMfspcGwrUG84YcOCs0Au59XAPYYaRZRBoK09MsQe4'
    'FUYWtY8A/Ti2Ue3tOfYo5hSgzYv2eTUdqUEeZMBe4SASZysBwCJlZh8DCAsQ62yf9S0Zrr2k/nwhV10GMJbPpsucCVBZhrXddTAE'
    'aggjisxd1l7xMElMJ7SijU030ucgx2UGYBYS6AMrA2yWC3bVHaHEVbmnzhQZRAuz0qYPBEoLUmWjaC8q8rM+iFKT+MeDEIypGEkZ'
    'EFselXBGygDZQuuMGQVkS/EyUGWgthSv4a2WAbaleO1+BwBuKV77DQDcFmEpvJ69KJ23fWhKPvfTyQzDxXV0BuJCBTgXhuFSTdyK'
    'ZRAu9FLoBOTIahLpJgDkwhzGyF7b8RJ01TxTmNUM9H7HkhBw8dQ+7kcpya1sgHhq4NZkAHfR4DOUpNeylGMk5upfj8ox359WvAUY'
    '3BTPlgg66iBww0LcTwbEl/HC7P5AfBGsWisBP174N+pLUIzNPwaIL1FgFDqlpPQrp287FosXkq1mQH0Rhx19RoRR7r4cQH05pcFi'
    'hZWQMvuDPikhrRcza0jVawvcF4b0Yl4ZQHqhS8a2H7z0S4fLyoDzwukKPcNvtmMuLkMPAMsA+MJlJ1ay44G5KNhFgUmsAHw54JC3'
    'sXei76kOg8UGqC9HIdVvBuCXZ1XLsAXsly+MJAL7pZx4qqEM4C/lJOgRgP8iMUX2HXb8dOHtWarViFIBCQaiiOnoAAVDV67tTCWn'
    'F5yvBhcjIGDYtQ/vgKAujWc3s4zMWOXdrZCBGbtndzvkYLainRB6HG//3d1QbkPdDdvdxqpUVYpOrCO6AgaunwEU5qgMrBzwYIbE'
    'bKAeZJyjgkiNA1gYKo1034CGYTOr88mAh+GyyMAJgJjTxTRwFYCI0ZhklPQ9Ra1bvIASo5oJFVOBE8PwungHdkj2orD1hAkyj/x5'
    'gBczdL4KgDFUAC/hDJAxk/zSlri/G4dJYzigsiXLkmZ5M9QYLvSLoz9oZ4CKOqNUN8Q/ozQNiaGiLTVAXubYpg00mSd1lDo7A5SM'
    'xCj7eztWFOm0gCdzYm44gJM5WwSvBQDKnFvoWAY8GSnQQQNPZsJJtzE5QJQhImb6Z2DKzMqlMwdoGQXatiJIUHbmds1Nx9Zs17DT'
    'OAXiFEV7ngTR6+lbHkQr8MSG1s/UQ2adjOymrWDZoCoC4kxEIoE3Q7TswgoAl25yJcBmpKBGya5/Yw3uBSTwNyjqR0jgNh4kvlMs'
    'cC0yXGgHL8kAN3OSv7Rny9JnzRnODKOHx7sRYDPAEG//4pHk6Aqi4AVaHGuS7Xko8Rt/rR/nVrPKBnGhrcTWZlxqekBgz0i8emSy'
    'BwCNxGmhYMtjzs3aDgSa/MQ1nECgyc01EBA0AlEdBCvA0Ghp2DxKlk/zYHAGIs1YSZ5xDoClkdLAPgCcRkLYxijJLKostngBomZW'
    'mYAGgJoZ7gKM18Pi3TsaGDVEHkI+ugwQNQIW4Uw1YGqIHUckRAagmgvzfANKDT17QeoFqGQns3RIrqsBRg3cRmzkSqRp/cymC3Sa'
    'irkHewduS2VQOQGchm4wO/XuDcSWMBT53boA47/r7kASue93yG4aZ/axXWdaYi11Cra7u9V+G2PehZfC1H1iAVtDRZGiFsg1ete2'
    'Zn53zxLqhOsaIDbEfcYsHJBsiL+6iPqF9A5caEMAmNvixFXtqGAvi1AIULTlKV8i7Pts11XIbiAFmo1AB55ZhbtRNg+LgMx2zd5q'
    'xdav/lKxVQxJP9vSZ8v60AwX1vN+6kUxFwR8G33VDxb8ZhUfwQ5r3wdkzAAwbgwywT7ejSATbEb7e9fQEqyOfgsuwSoZhMwBYdsD'
    '69RKUTd8jVDqWxvYNyGtABoctJEY/FoctLAYbGrhecv4+taJ3VBiL+2Foql3yxIK2m2uVJ4L9A0g4ORnbbM+wG+o2MQXoN/kxNbW'
    '/u0WioKeZc/RtTG3QL45YfOtbeE9B9keWuV7XlCjpO96UxRARC2Dgg3QN6fBEAvkG9aL4e4C7I2W2GeImjG/KKDdSIG9Aobd/Rz3'
    'LK1t7IwCoJuLPFKxAeXGyjAt6cAQe7HUALlhzN0eClID2cXdBpgbLjGPLoDcsInCRpwZDMDCdNyAt+GiVoMGTBs8QPciP10PUwHM'
    'jRQG7SyQbgqOMMdR2HNU1TMAvGZ7Dqt65ja8PQdWnZsJZ89hVefOYwPuRoFd5164E6BcW36pQLzRX9rmMMDeKNpr+4d+/INr2vYc'
    'dJWvRdvdhrwqhZb+LwMKDjGbfnsDBof5T2xeQ18tG1Nk7RkCqxRhcBDJxbTlywRyL4XGugAIR2BEbfK3+w4k6pczUHDM3hhkPCDh'
    'sJsIiDWQcBREMH4xC0iCIZFWBkAcLY/tgQDFuajGIdYxAyQOF7oveYSK88aOvXsGU5G103coOB89uAHVcNg6QAPOSAEndsp2U8Ng'
    'A7BMBiQcKWptBcjtpaQetsHvbrcK0XwsvMdTpdyAYHo55QEqDuN6tZvrW7GpygGN44hf1tpewI+K2lJuQHCV0JCyAVpin24FuKO5'
    'vbYdlXkbOxGSi20opDrUwkAJlA1gT/S2LRFQOYpbErJLZ4DLMawSW1dlB7w0nkfknQf2RqhI2QIrj1q2kBoAaoy8r8oeGFRG3MRO'
    'hI5hyEoZQHVCsVWz18aQsEr67WLXogNk52AWmQ0AsnPQtMMUgLRzsGhrQoC2cxCJ9oDbUUAF1/ADcef+A43bz4C2c1/SB+E2U+bg'
    'PhEq+8ahFP7Bwv8z4O3cP/gCMAYZwHYe+OULrJ0HIQN7BrSdB8t+WsDbeRAbA4C5w6gJpmQA4M6h4PlMUbQDdIQgXwJ159MScdAZ'
    'UHc+OwuWP0Du/GqRWz0DwB+4Lgl4OwKBAAjYDGg7D2ONOPB2Hrr1CHA7DGHgKmbA7TDqgVm4gbbz0P0DgbUj0AfBawBgO4/cpww4'
    'O4+PHuNxoAAHIRAFEDuPpyFGCRg7j+vrJxFQO4+deQLCzpMqNv0BX+fJdUDZDPg6vyZmB9Z6oOs8LYKzMtB1jixFZAZsHc5v81+s'
    'jwo2VcSaJkDrPAtuMOFcAmLnmXjpoA7lFp6VZ+anClydL4u2eAVYnS+rcSRIAVYn1wRdGDh8oSexvhCoOjmr9IIW2HB16CxFfiDA'
    '1mG0BOM8ga2j2j1T2gFWJz+p3LgPUB3GTbiwki1DTpja1Qs4HUZPcPYDWDqCQtByEQOeDnAE/G1oLac+j8ofFBeuqDUQnaFoHIKv'
    'OEB06BK0MF+A6HAMvwX7AUGnGC3aKiRg6BTiCmL1qep47EoSIOecqp11aIW7HqwfucYANec0r2N/faDncGj+S/t8gGj7uXVxF3lj'
    'SkMTyAClw0H3djsCTud84eEBQNThsHkPBwKuzsuFuUACWmdMjK37JgFgR+LDoz29a1Hh5lkEsB3xi5v4+ij5ry7xlZL9amjyFqB2'
    'OILbds6eBV8X9soWgqWdewTKDsmfY/toByHJtlP2di0u18avFP4NTy1mcq8fx74a29A3dT3bOvAmpPz4joeULzJsZCcACI/EsYbP'
    'tzyU1SN5+hYiy5GqPof9EH0aYpwAzCMhqNCv9PtREOrIykIYauy8DaQeiUYNFmXg9WjcpnFrQO3hMEybtkEWBVS6OQv4PRIy6b7f'
    'gPHRsMmJ1RjiEduWdeD6cOSi++QC3YdDBINIB4yfCjTB+z/wiLzIrwN4Pyqt2buA/dHYvNILsygYD9saIEAeetdY8XYUfGcaC0AC'
    'afyd2fcBDMQxbPPo+71IIxeoFSCCJA4rt0oHHnblUcrACuLQK3dRBWIQYq3i2wroQRxihebTLYuICgDM2cCilkZhrgEnxNE+Nh44'
    'cFPDQckEYCEJ+nEeGvBCEsTjAgwghiwCx8JUgDWkQTjWnyyE3USi/yALwTexPg0QRCHixiYw2w72rrbMP7DskFBI4RoEQJHGlPjs'
    'ZHsWVvLSvu578Ib7LAGvyEIxQhibIRdJPIZNxlZ6PYzCzgPAjDTqwe9ZYBpJoEM8+K0Q7hD8SgFwhJCGyLAMnCOJX4jCroB3pFEM'
    'OILAPNI4hjMrG4RIhpbVBQhIGoAQkQcAIR2KvsY6AQWC+M3bwYAOwT3nQa0BhATf+aG9vGPu8qa9BCYSu63HTsnARjLvdetrv+3T'
    'PUbxwJy1AxhjBrgkdtkOXmzATBI/bbvcAZukrtrBN3lgsBficO2bamfbXaxt8yHaSp2YA0kDkBI8i9HVnb3ITdg21U6/7T0btNUA'
    'VlIPVjSn7ANcWKMF203NpdSvt93MvUrN5gqYJXMsNSd/IC2p+6R1dje4Nbb1N0BeijwLbZ9BrQBXQg8IAhZTcM1DKB9AmcQhDn1W'
    'BiO4u0F2BT4TPNKwcntZ5Gnm867shvhimaXN4JrY1cor3AnOVvE87u26l5T1fm8v+EkFMzBgnNwLyiuInHxMgwlIJ3deiBQRgHcC'
    'VrqVZcErAP3tt03tDhMBuKdgW2e9gVWzE+zgs9Iq2o3svTaVwNUPhs5oRpQtEWvnmdUxCLZNO3DwJggGxRBOC4goNfm5OQo4UaaY'
    't+OgnIlK/CZ8AjNKRO2l0C7AR0Eu9UtBmROX8Wb28l5LyAqV9INw4Z0etFl8R57ZAtCUsT5hNbcAOIW/s+jvrejv7ejvnejv3ehv'
    '6ej6O3oABvrxvHpFPCAVP1//17N89At+8V8XVaN/jKqh/rFoilr/Oqnm+sdw/lr/uOt/EWW5+vYX6y/e/dVf/v3VX43yeX5X5rd8'
    'U9TH7DjVe9lU0x+zDdo0m7vb2/Jf+rf83810O/O/pTxlx6u/Sjb/HBPAEmRNzf//dP3friVsBxjmJ8cN7QA6Y3QC+91QSmd/rKWp'
    'lE6PmXEr+DSyIUfKiIjVzXEbF5o+YXeGtb8cs//hz39TzHuzy5+ujdvPPyek2m2f/829vd3Nv5z/P8e/9fX1tfsaEsHi0XfTZFQk'
    'YxIUG/6jdaR7a2sPqiZhPuz3E/bv6CbE/321KBKiFwnx3kl18rIQJizxtBn7a3RBEO92p1VV0mkYJiAZVvVs0fTmr+cb9FqSfCku'
    'ubm0rUQ/kRgOzbbR9JKjIuG4wnJI35Icoi8leSL+GLX2vZGqWApXs11ymTx7/ODgKCEeNFHDH3MtCRyA+ducfktmV9/W7D3aSx4U'
    'CSco/33C+Y6SYiwVliQOTvL9ZHj13aRKmiJBJMllkRBrl4inOU1YQuJUfvVP1bjHIz9gDWWRnJZTH3NzOhcKO9YhP8m5izSoM0ZR'
    'usvSmGBAclViPKXZomFxqrBe8m/RL0l2SkNNSmXBaaz7Uls0DOliPh4Sh5kcPf78caLN0/zY1PJs5Ym32EsOx4lm7EumlVTHCQgS'
    'eBvWNM2qqK2lGfSY3hkni2bBImz3lg+kvqitozJpfIPxXFKvRlIDqy9ZM0al+SwXUYLGyerHUZXk3phUeJnkDJH+TdMauQAI0Y6c'
    'sN4n512azEkEqvgHGlpCi2q6GlmlhCaWppunBnEqDSuhjjnn4Hprjmh1ZbWnJG2yPZLmrKm4kNeZKtZecTMcLczJZeRBZyTnjSru'
    'n7RVhwsZD/VG3AYTUfFRH/e5PdYNLobzHIuQwKU4Kc+mFSfSKPNo8rpcCT3SXqiri5LHxKf6tK4myfHx6WK+qIvjY2wVOhfTak59'
    'p827toYy3o76/iyfn4/LE3v5CT3qD/PLGQlBVv4ZTR5tzLqbPJ7NxWbkdU0XkxmtSpNMZ2v41LhL+/rjJ4fPrKybfPz4WTe5/+zf'
    'HD9+cvhI/7pP4s9hNzl8fNRNnhw86CZfHB0+XVs7+PUXnyX3pFMdGlg5pmFt9NiPZXxRdDZ6M4m3XVtbY2mzST7nW+zJUr4KPSY0'
    'P9FV07Aow+4EOCK0ofVi6iYndDe8ZDcP3h9y41x9y5AnPZ5hrM4pzTJJoPPj405TjE+7NmC2zewvDbZezPNjrXxfBtKVSlr/SFLZ'
    '5/1JQyU+uJtwas5ZYUVbGZUUE3bhtKLNjX2vRfc5JilqjecpHx3zz51iOqxGtJr31hfz0zv99Q3/mvvfK0cNVTCd9VgJddmJRtOT'
    'L4uONNKlXTk6hojW3PuE/q/YoPNLO6W4R19T33a3l6qmoVHV9P/tYgYSo/JJ/rqTdvnn5A5GvfR9TVtQulbz4Zn0aPLzxXh+TOUd'
    'TMrSF9PjC1tobWCzm3TGxbRjQ92gtlL6H7W6kdy9G3XoZ0kaKitPr9dHM7/fWr46L+kcfpmPF8UhO8R3ri3u6fqhba6ENtPbVk/e'
    '4WLu0roGZarSp/UVVV1E1PIt9f9dL+E4zUQAsBO5FXU7VMmEffL8junRkkeblzqBvUuT8QveVGFYdUHXx3R57OHrgk8D9v1JPh+e'
    'ixghO1Mqs5vi+XwxGxfPaek4JQntq24S/n7xIjTIB8v+vg9fMx6g0ELiAO4j1XciobREqvh2LxrOVVQZP1IIORjmiyb3uqZXYtuj'
    '21uqQ8ACzThzKUq3+Y+TxXBcKNcS/tGi1uyWmsiVy7dA5+Nu8vmGEKCEthVrnFF4J91of4xfn5/QxfCCNmGasGlZlGEcRwN+JvBe'
    'OXgUfp6DSWlVuMyvrHepF0TKWjXR0OgaU/YInZYe/Cx90WvVFmi+BqMlfPxpJM82hChT71/JI42rmzD8WkL8Mp9Ceu/5fjfdf9Gq'
    '7pK5EomZrIWMzbmv1Vk555f371DryUFDHAzzYmas5b5Ohxwo0Vu5DSQdPbVoV0CPM/AslIh1lnZmOLG0YWgXlvQ/OpfTs4JPPh83'
    'qW0j3q0b7VNcjviSkteeUwX7XMnPotfbA/7cOhbfavzPtorcV6eL8bgj9w7VvsF7B5Ttpusy2jlawxvWkbaq4IsrvnBPx1U+38ra'
    'dcgsEMXgaSimbHTJ5wXXkHwUrrqlCRA6wiAgNjRe6ma/oVn4/MW1N213USv73Dv5coN3uvx17f0P+LDNrkkEutGZCYwPgPKFvInd'
    '6e9ahXbAuAO44UM/ZJLk2PU226t8ah/2msWks7HiOhdirHmu261elsV4ZAPvWj1txuPok2fOa9zK6PeSz/WylrXmUBC6IoRhB0sa'
    'c+ttLnpfbroqueIkszyFl8Zx6+w5D2ocec979KfyLiS9vD/jcjObItuK2uJdElp8L2bhh/MAxtMTa01i6XO6I+Uifv5i9WvHzB1J'
    'II1/osRL/qSxvOgm/ucL1NS6dsbltMj5xAUujObtVhas18zG5Zw/bDpLx1Fru6f/7bED36yzsbyXianXF95zCxdUIbP9Pbo0Rk1H'
    'Pl2qdNR0oytIpsinpu4Uz9eFl1h/ca0vTEHKJnlEUvZ79qY1/718xvEfnWLj5nfiNbL3O3GPN2IOJ/Qam1wFLZpK4VF0dU2SCcv8'
    'Iog3UeGNvIq2QR2ohyLaja44QVVSsFPBm7wtaof98vmSdDlWKfUiTx4cHj354uo/HomEffWH8bycVImYNkQPIsEYTuS9PmaHmgWt'
    'gYiFLvp2lW2AAHye003BUZxnV/+N/W3odlWpB24ma63VYm4rH59ViSX4W02mVW6wk/39skLMW5MQKFuYzgxVs8RVKwvK26kr/x/1'
]


In [ ]:
# PAQUETE (7/12)
PAQUETE += [
    'rp7zRlai6Ky8cCbCazDT9rpDlW+A9LSmMe6AVkUsf/KLUN/39WNp6NKuX11U14trPZ2U04783W11OszFB07OnFlcZgk7vIimBeM2'
    'manbYP716huhGdGOWrvOSzzf7G2+IOpv7bdurnlgl1Z1c38lZztfpq8sLBFNtaO4SlK6cU3xS3yWf6Cs4jLVqssC18LfclxFUc8v'
    'vWouqGpWDR5D8xNaEc4qtCO8KI0vlSljXVL7cuLxF72zYt5Zn5ezal0mwPVJG8v9nSZ3RTK91m9i7tKNn1DUupWvvn0WfwQ2m5XM'
    'vB1vbOR5+UJr57pvYcfbLCl7CpoywfeuVMM76pjr4pav0brjk2VOXV7raoXvya5bJRGzvlTLe3HrbUrWZt2lvutkVUbgPDgP+oXe'
    'SKvY5ehNa+SFTJket+usrta+T9/QrdpFFXgktrfkk0mcPKsZeasfH0/yks7puvZT+byYmRSuoyP6vLsC2J+v8x+uKRQFOU4KR/TN'
    'O6fhx/1ELJPs3VSKLC9aFPp5491vpqbZoPuR2lypAuRXu8m1xoMtYl10UPfSrN/uwTWhZV+bpnLR3+x3TYPDx9F+2niXxIoaTrOh'
    '1fqmFBm7FKGop4d8O1phax0n5i190mvO81nxLkgNb7kKK41MHBdX7IJODS+rj07X33Kt+hmLPxvv7qISmuJ30SUVmldHHdOQ01jC'
    'WaWJYJIvRSxdNs83Xzzf3918sfHX9bterxfVd0JM8CtdI+KJbY1IWrphWdxcglUhyaK9Kr+Z5sHIghVhbvtdpMtfi0feeZtubn7E'
    '9V6/8/d7m6fvkn8lFi0norJuH7LdoEqaq28/3LhpAbnOH7SAv4uEu8AFXluaFcvH2z6sV6JjkoJJkU9JhNCBbEQzX7IMFnEk4tRV'
    'gW8pXxDTxkxuZ9XS37nj6oz2UlstG8/303TzBa12rE2uxtUxD4uv+bnyFzRZFzxVb0iIsY91AqkLGyKVXyS/SDZfrO5FmCRB9GFr'
    'xyhvd8lbXdkn3X5/sfu7/Z8DGH5S6//32v+3d7b2lu3/29nWX+z/f45/H/z13UVT3z0pp3eL6UUyu5yfV9MtMSA+4K2R/PHv/zGZ'
    'LaaqhMclQOLrFw1MafpFYvsoWfEvigEnAUYsr/mQracrK7gzY0dpV5TRuReLK9uDvHD1h3cEdyJumVHjvhVl+CX9ffVtwnf8DR+z'
    'XiVvKzab4uqfcnUboFYXcHfUEYgV5H7FKdbZ34FnpoL+vjVenaW7oWvsdsv8gPyqP84LugDRYerkJBfwksYsw4JJr4zc3TFbwgv+'
    '85wRG+jHq29Y2cADo4Ybe6/Jx2W9tvagSCaVRE4k6udNPXzw2dPD+88OHh4+enbI1k/tNCNCoKkuCZ3sFtHktSgQRKAcrU05iGya'
    'j3sksJkdnN1gdVnV9izKEvmIbufiDbtL0CwvpmZjoLb2Hq7N8vrqGwVsNSM2D0GMr1mPrgA3hHF+vHrMxFFMPCdlzZVxhqfEsCnX'
    '+MKnBSpU11JUNX1R9VSxAi2tWdtF7wHz27KFjaTqtVwdmlXb0RQwoglU9MlYbPqJRBzMCrShyho2aKj4TdUb0Bq3VzRrvGtF/0O8'
    '0tXvh+W4i2qYgSpH4gmgS6jRR03xg833eX024w/tublsbrbmr9GvPf6BRJiGJGCW12hIne+xqrM+TeoUX2P4ZEitjx4//PjpYTdx'
    'YNXj8EpL7T+tvqL9cLi9mWlNr6bV1+NidFa4XwBtwfsC/AObwOp/12siQbpmPCb3MHhqBV2xG7KocBz5Tqyuau3hwdP7B8SrvFUt'
    'OUC39xO6gtZV3b1uW4FL/72XjhYjjiSjsr/xMmIoiQ9p5M1//q9c/M4NBHKzunXgk5xj26gXkkctTHBXz+RIfEPCpFzyXq/F8Sby'
    'B7nVIWHBoVfjY9rdl8Qhi+TJvGAv3VlWyUuL966tRE8iaurONU8BVh+sWPiOV7b0hSot74UFCm+2u3kvfliqhLUmVSPc7Jqu4p1/'
    '8b/kA588uyhNW2xERxTG3UToxH5yUlVj6oKqMlneox9Xa4bZ4i9hic5z77OtVRApa/5y1XqKXppWINIUP0DAJYz6s3E+1WtupKiV'
    'aj8+Ovj48GlS0bHEnw270k3yRD1cRkHTZr5Zgbrm45wm3S+oYgwVkV21+6yNlC3JIQdeEbtsEd2i65q+aBg6gJq7FBelc/ZjyycV'
    'x/xw75pFoogmsS581aS5e9k9//Oa6QNmD1+fVRpGOxVSZLou2YU9w+GoO1bD6n32s3tJurbcaN0TcbFtochZ2KZGVtw5nTpUXtCW'
    'ueFLO1Y9n55josXHSj07dU/SZ3WpdUl61BM0yPDuRqubulFXtfMzM+dIiFMl8U1xB83pQ96ObpX4dT0b9cqtD7JI1JYdAvOXtHGu'
    '/jAvx0poRVznfScnnL2uiNuwQP9evAnEKsUH/XlrFOvrbeMjy4Z//Mf/PZHJufcWs/QuSfQmoQK9+XpbJBIvydKnHNp6nr+R1yYV'
    'u5keo4Q2f/Ihsb8fvovae9FSfA5F7dlje08paSef72+9WGG9cxvVCi8e7vt/SN4OsZjlaD/dfUdzMrlHZfSfcsw+Etr5VV5Aw0JA'
    'KO95DSjY72X0xdsPkw97Lys2O2B4zUYsD/t2DmO4rf/obWfKECt/YJc4fKVuOFPGbRSKVdRVrHtYUcs/XteEr/9muq6d1fc3fpL7'
    'XVjxoNxvbd7Y3YmPYWDWq2kn0sULx35DHbS5524JY8rWkwhH9oge0kcbK7b36fpbb1KQhY4V5/Vdov9VxrziG3a6v/4iNvYdc1f2'
    'EzZqPxf6xEZKZmLetXbquVspuEf7yybb8960vKDb+6/vKXjy+nUNs7X1/LzH/+E2rEhsHVrcTTY32PTVtioxQeD22aQ06thnG7ft'
    'ND4Yb2VsW9u0jb11aXv/F1vvbtxdrBK8r2cgYWT1HG7hie7u5K0s0PMPcU6O5Z0PX9xWIV1oIgkQ6/RS77Z5MT2rVtRVTts13b6z'
    'eSupRHfDXqIfWzvJpqEjqwRpsLl+numHleTwdL3FYOu2gihJyx92oVj/3vXWlwfS+aE1/GZKQ7+2mX7GPXy++aJFu9Z+jIPu/9Z4'
    'coMoTud3tK+c9808XHBZUK3jqKeXxUascqa1PBRhVB3/nT0xIR/ivInqKor/pm3OWP/jP/79evJRsruLnfA14yYmz+pFdDbn9WV7'
    'ESOmqJzOFtKV+dUfkj/+/R8SdiFZ4o8UZy3pHD7+RNxRu8nfFZcnVV6PJJlFvZjNN5Yb4M61bVKqMX0vnst9O4KWeTKKObhx9XVR'
    'tzk4foOuhs66zhOHaPI8Sv4ML6Hb8+q7Bn9VtN/3b+9iqFSWxCq9+na6vnLEfGW8VUH2nczmW1/5+DCv9F9Bc2z30pVff98WlIj8'
    'sBZ0U713E7hcbm5j7fvrCPKQrSMO0D35f6l7ySLDgxPm+Z2vvQbv5MP85Oo78T+W88l2ws7S0cvZXmA6jd5BfbZgBdYTfoo8qkei'
    'EBL3nHtEPtFnupGPj0fV8Pg4Zto0joa2/LHI3/e88qf51w9CPZ8W49kn9uoG+tJjz5Ucneis35nxZrpzx9WT3YR1mdSJdaUW66y0'
    'Eo+0e+LdsLZKhXFOLd1b9+Ms6B15QjQAeixgsBgHdb0Xd2xfW+OMslQckxjOoeta/Url58016t69tUZXTEIleXNl2Ka31nbKbLaq'
    'NxhC4OaO8S67oapbZtfUty3V7WXgWE2PW90yDBVRqG2x2oviJCwv609u64AIHcQ3/H7KHkcq+jDOGEOIJOocLUjZ1jzv+llPNiZ3'
    'onF+k34QutVpKUjynj7iLbokcr2zli2PgYj9HFQ8fCCLfv0D3EwrPtCFvf6FXTThk/DNdWLBTOFXfEOv+rFl9Ad1e/vV8g1mb9x8'
    'W31l1xQN0y+q26hd3PE247Bc0/f7POjF9j++odFVdj+hDfB2+x/nR0+X4v83s+29v9j//lzxv5+5vp6tQ2IRW1tTG8+waiZX37Lg'
    '2lM9Zn31zZBuTdbtnZYvc/bXuPpPLJayXUYsMZJ9nUNSRhZFO+aIy2K6RuKB6T1HzI3Ss+og6YkDR3K9kU/zNxYeyuYfdXGdsgEL'
    'qOssd66x+nPOzt4NyKcE8WgmSuIehues6uhyvKjmmE0ka4mYgBJBSS4SgeVisrDmKPaw4E3KhiTWzw6oV6zpEA1poe6vfHWP5UWW'
    'CGWi6BouSLR4fP/ZgYREq1XRlLlzEkA7j8phNWYJkuq/v2iIg0q3tzc3WPdJ02okp7KfeepZymSz1Ryyp1j4GG+N7vKuFml6NAGs'
    'URsB84ws6Y6vNDGGSr5qo6LPfy6cQKFewVLzOI91zp53QajTmggcplrWmCrhpLRxCTJbMk5oOOwU5sFEETlpo1Cfrr47pTngOiWR'
    'ZnBH5rzmRcK5cDScoWTzsGKQcouFeA0rdJD4v14pPrGYoxsJB4HKWXThkRkzUQxPEdIgnzPZ5V9Eq9IVx1rTvVM90UxwohxWdvNu'
    'EahYjbl1k6GEXI1V45ggMwx3WDXjGjIum/ZajLXgsyo/pmG7sq5qPeS0W8VEkx2J+zbAGEdqBvi33ORUthtMwzw0cS57yXLolaYw'
    'kGlUxYcEpfcYwogz2On55GpL2SkcHP3zRAGikyDI1r3ki6ns/uHisrIGGcuMMSCTkKdGd8VC3l3TqGdzdpdZbtv98R1bOmX0BR+e'
    '+3ah7CdHjz9++tljPhGfcK8bYQtZbKADohpGBYriu2GaSDKKsTBLqnpkyOKyVtN1dcoHlcNJ8svFKI/Cvs853l/S8nWhdhbM2B6d'
    'OgvHtnPG+0OnSm83djqgc8WxGJcSjVTIhmOrdSk+9HrZTd02PuQVN7MM/ymXo6wqVXXOF4/eugpOJ+lC9uUahQlFIsdHokSItqaY'
    '23iz6RGZ4kxRG+y+Wwp0I+0rmpLTRUOXoKM08GGSImmUj13OoaJ022jwOl2VjDu5JhHn1I1193hbf3+TtLzDznsiabGzhL7kRV26'
    'DorxyF8kOWdSRG8VPygQPYSfKxtIrDI2OwqO7z9++OTzw2eP/ZfkM7+X19c+Pjj67Ih/Wr6yu8nqK3t9be3pF88OqNpHj+9/9vCz'
    'w0dS9fdYz5ccTcOFuQ45WGSh41FxLLKQhAxpmBXbSpfba6sJzWBy3zxB5P67+k6OhcY78CXxJiYEsCqSQE6E2dSjYpiLiKfQU/pO'
    'dq86X4jgwsC8ZgnkdElJDo53yoeqSBjwmZ1ILtn3opoOlXLQdSJnSbAyWpKfbkKzOkpdQFOgzT6GCVxuAM8EZ54YHAdRXbJpUi8A'
    'uHLwGcujYDdze7YUjGL8VlfIlo6NxRIefSt0a6N3Nq5OOusfwQ31ehBEXC+7ncdFXamwx2riDmvojie831XmgH7u8ZFo5yKZio7J'
    'WohRjKqXfl+zVgjkJQj3+lr0Ax+nHp8abpO6MJm1+rbRK5tKdSMsa679rR9S+Ck4Y6hNTsHp0Najker5kh/ASNkv6yQZr6+5r9Pc'
    'y4nHutQf2GtZr338JNdohR95V9gvclt0LMjvlL6q6st77fOygRVW7QRi9ujzFzd+z29swDk9mFs8UvFHtzt9kNw32BgzKf0Upq2W'
    'uWq1RYHOw30mrD264CLgGjpizNUJ+kggafLDhOlY7TAj+XThDifhEEAxRCvZTWDpEEuA/bLhCAr+LpvBOdeeZDVKDAP8msFhyVyr'
    'Bgbdiu+IFyBO4q1V+S5hbi9RUxZfL+26u9dsv5cSW1cLQ8rcW1lTdcYtCOnrLVsv3sv+obaPn+NplUVvRb0PK9qa+EQPzruuHi3m'
    'IbQcJy02zESmJJ2VY2IZz6rbDZyuN3H6+E48Pd8KQXy3/hMcgAfOLzNBcZ46V+mOJQcEQJrDnAhAilu9zKlzfW2WiOfeMlPvgwmj'
    'FWRBjjib05IuuVpqHBFDWIN9B6OFCtWFkqjoWLn2RjCtxKeSWcc5w778FIeWh3/MdAs+FTJJITp1+fR+tgTWJWKGkL1Ynu4lX7I4'
    'fSmo3wImRb/OGOF63jq52A/PiL04EjMuPEyk4HnwYnsRbbOVvinLfb8WZXujf9RT9+mn8yer3XJKXfZGLQJIVcEsc+QhdQgeoFKn'
    'Jk5AK56pLhitcEZNVjqjRjEMf5JTahKcUoM5LnJO/VTuC+FifDjsb2pp2FTEPuMQ90LkkNNSPFCLJnJRqi/Kuo1OBcGXrhtWHYxp'
    'ly+a3Bx4tU/whJ1XXy2KVjAIJ75MKhfaaj6RMo0t9zLivYtpWdRXv1eCIPNcgCSsDgs+xbFnLan5VK6yTct+Wf1ZcLtcbdR+K9++'
    'Y1tzp2X63+f5lZUpBAWhel1y1ubexg39M0/O72/mSTWsEnhUzom254zDzBPJfkOlO3jLJEkp9yJutzXqH9v2/QGJpmym4fxgHC6n'
    'qiDV7SWcA1yxrohm/tzkAs3qK0o/liTmwLhjUDGGfe/9yD2UCyb2WwG/tux8G+j/+lPzYDc4IL8wLnHFA5KtrTDoemg7u02GBVg3'
    'v0WWZdTzW+R+VTBMFuMC+sUp/XzB+b9BdTeuOwOv6GWX5fFK4QDspF/fiUmVIK8mVaSKF2HIYl5lHefMdB5L3XDv41VTJd6blmhi'
    '3EselElIL26Onlx5vGlZ7VmJCiCaLO46T3Ih/v98uymzKZoaZm1LYZkWM4a4X56qyBs66iV1JmK2iPw20CS75upQlUZ6S7L3Gv9e'
    'sUe/Nhd1UBWgjfSuLsZjRqjk7yRrpNz8CgY4zheS5G0JPIU7EtUm+jWRic/pgqcbw8fz7ic6rO1ww8izOxF+vRGtE98XfKZ/7NP4'
    '2S8fPX568Oj+Zwe3HEki9s63RVfk+iNnljUyxHAI4AVq8Icyi901qwyOjccx1tkNtX7K+1jBpAQC8qxi9fWZdGSMcyYyv/JBbgKk'
    'W8sbNA8tSdAdNfOMHblEWc2G4sIgABvxtOcdwPe8Ul8Z3lw06qPqUtiMk0UxjUYl9OC2EXG9tI58udHCHszE/J8nb2fEuOTvwlBC'
    'K4F2eCsQyjlmdbg44RQwNy2HDk4mTFQ/rO8xM4YgRTTm9N1EbXOaYt6D0XHAWk6K4M8hJp4rzT/AzAhxBMGmjWmUk8ocdOi8Mjvf'
    'v+pdvYQuaApyU/ww66SBFF+qRk/vkvUHhboKvuMvqlnJ4sQX4MgmOV+7vG3Or34PRSnDmNVUevVPwgQSv8MGUIxXMw1onoEkV2/R'
    'hNN90hVATPCzw4cHR+FQTCxR3X70dxfJI9RVOuSSCJkkFFu/avyz3NLbaLWct36saTgmIn6cWfIAS+XUzup0WnJaWskeM1EDi1fl'
    'meGXssQjLU01jXLUKJq/UzckotNXwvOVplDh8O+zSnOdfSd/4st8yKk0JAWEJPOZl5Z0xv9ulcuVGsmv4uAoGaOOEWq3QoowFvPD'
    's4L7RSf07jCflbxqzYf8pv3A+/KP/+t/TsKvkBie6DlgasMogLqDL0GNhGEaCybtvvHMqj4WLeZ6jqO7ppEU6EK7oXXchcSD8+eq'
    'zrwoLH5QcX1+P1U0YyWhGldOkm7u0qxbY5RhwxZuxAY7zMOxta19QuQ9ASCQWvc1Rr9WHvAkUqg2bRWpTn/jm1t+87FpojabUtkA'
    'PtD9+KHboslIjd60EtjoVqZNy2koWvUm9H+agi7Ucl42s3xaydCsqllO71Ffr/0Wfcc8OfOEQ91w8nj1jWf5pk/nxPVIB/zv6HO1'
    'REqSqP3Wk3Sdc4dhVvAgsxI+L+YYnNrPZcJK0M7wZ9f8rVUFyoLf71GLOnCzzVvWpHUgzJNSYcM663fh2mIaYdmI0MNhWUW6n9F+'
    'UTiUmSBOSOXmjLcqkIu3iziyKcRMWyHMukQODPP40euxanRHEU8uKh/ipY0n0JMkCpeRhE+6ToKkManpGn7Y+4UMhjj/7wtkW1vS'
    'W8I3Pez8816dl2/0MtoIPvTauxUe87pmG2sWV6hniXPXEuvQiBzI9FaWMYohFi9vvPTeJ+/Wnd06OCt2+G07O3x74w5/750dbeJo'
    'np/bYGU/TrrJRCd3wpNr7wFPD7vZd13nmsL5nv0Reh5rW+8FmKbbFw9yyns4dt0c8xj7d7MDfEsb2/Ygv2e+3+0v2tEm3/fF+m+m'
    'n1cu5LMasNmPkElUqSFe0BCmu5HI2nW5sRtJZytQS34zTZLnbzWi6sVvJIT9rYapmNZSfnPnutC3W2SauJ+vuokgggQRpEcbctKC'
    'QQwYIG9f7WccD3Lxbn0ZBoZ+/FDs5hcF2wg/1BeZU+R3/wL1sfKf36j/Hf3/0s3da/5/e3/x//uz+f99vIJ0i+K0aIhYa0Cfekxf'
    'cgCvkFBOBsLJKEWCGpcnhSr+VdMWGIBHj5MLumkNynZWNFXIOzFKDFKiuyZviQeNqICiEGejjiyOj4jFFEH8/rIiP91tafLXFOSS'
    'nRKYlx4VkH9nuWqBoE4XsEzO81GNq5+rGYnV6EImmCtZk9woBWMTs6gi4r7eZSJCUtM5vcb31EeSmkRY8HzB3axJvBcOoBibR0XC'
    'nodN2w+j9aVYD87KWhQ9jlpWL7/FtYsz3qJQnzN1t1IVqIhn+sV9d3VLxuPiQj3h7h8+fXb47w48yBu9IXlNHM9alrZYCyU1Ho49'
    'J4moRuo5cx1j9gapLwrTvkhov1gfpMDizHULwSDSBn8eXTfq/MkpLN7D+ahLy8Zarn9xzov3cAV6Dwega34XT4tTEu2FULYEtAeF'
    'gcpKYhoYnSDOKszArEp+S6I8ZL7fIow/pInhCAhD0BcnsdezMcOSzA1/hX3zFJ3NUOp7fIpZDVKrf2MtKTtlB6jmhLi+9s/UMc3f'
    'neQksl59Z6lpsGEvYVY4ufpuSTHO0jLradZ7yVE1he3VnXO72nnNbzwG71y03WD51mJvGgSgD/XgR051OEBU7dW3cH8T9kh84HUm'
    'v5ji1OgxtLVoItGbhO7TIvK9VatfL3nCwviSZVly4swqToijQzlb8CRJBCmmZF3UbZMCui/32KvgeUVX1CVzagZXSrM8FjtFMhU3'
    '1gZjkkRMBXUnXANh98zL+WKsBll5XtRjd7+BjxRdsFUoW4G88nX5qpS0qsnvEjozNNjfJV8XJ/T/5ZRvhoL+6vV6FkIhfrGjG6r8'
    'AH5gnx0J1DkUlL41zLPIIQbjaM1Qxz//P/CJxl67YZ9hj/0NFB1A89Chu2GbJ1E8ivIbvCfch4Uniu6hEYAidGZvMBqGLzg8Lfrg'
    '3frKuHP9kdEH5JGWif9ex6AagUg2tU6N3v+tXBxE+86rkQ+HL7Fjvuc6wzGbrVW1L6NaD1dMpH2V+0N1DU3v+NivpWO5NZvj4+X+'
    '0nudjz4inpxY+JiXHxkLzzMm0Iha9buVbmaf8mHzm+4Lcdm2cISWd7krCUZhF2v4O6/lrf8+EPj65GVBLAKdLfEGWF+hOnS/tQD5'
    'clulcSKsYT69ksSqspXNCLGM2H9bhR8krfRftHXZL5J91WDZkJO/piFm6rD2nq5uUrkxCBIJl7PRmhUGelaBGRGBAw1WdnCz10vF'
    'U8iMxIyQQjPIWQboz0qYLzljLGD7ISOOijXfq2rUn+je0EP1uyS/YB31SIvgJ6eb36tTr6tVFxT8sX7XCuCiR3Z3rqLD78DumDtW'
    'Ni1VlY8uJIk3exh11oflKQfwmeGWtR7FfNgztZsTCSxJOF/v44T4fdfc0m0nPhdWP8gmrjrq2d+Y8xaOMy5WtYnmU/a4Ui2Yhh0q'
    'b7p6RtYcDCH6HQATAQFdfIJcV/b9wBoSrBEUl883X9wEyi1W2GNd/FAxz9C1mrlQXQ6jxbgR7VurPPZ5z7+n9nx62al7YZnktqsd'
    'U+N6kwoSQkLMsYE+3eyP+awaVS34/URw60VCQZSLnveW55Yrhp+bxyBis9EjXCTdZL3lkKkXx8aL65mtqmtBmNKCoWD4S9fBLawB'
    'VycH1zalPz50fgqNcGSrcuL6Rvjh+Xo0peuSYwMv1hs3z/2L5Z59D2W6YDUzwxnwXn73J1LST9Uc9ycQ0Vdfc5gvK33fn4BGIzzl'
    'T7WO3qyadVpTpqj6G8w5RHlO9O0Vk9tmXSSJBzEa7Is2LDok8oTrbEP9dkNBL0xOfR3NXNaJ+7oCQanFRGjXNhwjb1lzHywOckoK'
    'jhkcFY6PQ5c8TGkkpV99wyRFBFTiFZJLpMGY3I6SZyS1nfNDlveFXXhLUAUOFXbcosf6jaRsaeYddFFWYglh7JjxXMpR7HPh30YA'
    'PuHdZbCfpS5e+6Yu9CqpY/CiqNQP5qoxaee42nOGpUrOl5CEzKX5hn7eBkJk83U9w4t/3muKOSimYwvRBNp1dL7x3t9GdpzlKn4K'
    'f9786p9yElDjXbWvy/o9u8f7Je++10r+oHwYaKb1Pdt+4go8iYRuqmuV8fs313ZWzPne8iPFmcq41ojDX2YGUM1zevVFqEq3ntdy'
    'HBwA/HTe0MF464qJib7eaFfsbEo3SA9Sd3Sars8iH+WlXcbV+8a8GR3MefTrnYXRUff3Dcfrx9+lK2kcLKNC4DT+jJnt390YhXYN'
    '+kKA1xVQYukO5bG3btD15dt9vR2gIkZl9r1oRG2skVeiJVSPz0qSDc+rcVGra4XEZTbi7CMKyhF7IMZu7gtBTQpBXcHMu/Lijmgm'
    'MkW2Ra0445fHy7kRmVtYjhoTSi4/lM0xXw8gpM+57MXKEDSreP8aVZ12Q46xkBpEIs3eN8dY91o6n/fLNbacbyyxfGcc41bPm69L'
    'muL1u3fXV9R+Y/6vlWhXMZd4e6oyX0t6U5Yw4klGN7zaw+zSDd2zUcsEsum3m0xXfsW8su9y0f+0vQSQwio6CWs3yfo3zkPYlDeQ'
    'uih08FD+wxIux1Pt346X/PHnh5ub6cr2sMcD4N1bn4p3+2+nrLdiBJxOsdEz2ziXFQ79FLlcMDtnpCH+qYdGjuWioRctBest6hBu'
    'dF+tEMXzfD6v79BNRYs1usZEcgs/CTVvYTfeIMeItWBYkXwtEEVud7hfsUtpfQ2ukQaPnzo3eJm0KWz8vTouxBVoyXt8Cg1i/C2K'
    'vudjaIdE2Dw3fNHVH8Upz6bEtJy2cmVde12dMWIxP7Qq0vb7fH5NmL8mob5t48fGLiLr+9cZpO7S63ySj2GG8C++h2gv1WFLv67c'
    'u2Ng9iZVM2cdwURcPlZ8JcsbfyYF3/8dLAvRh1py+5ctUEz6uKatQnwYLYFtgo0oXZmXdSUV7daNtZXsQirJ7/BFBKN1rfPwSD6W'
    'ZRJ3P26f2+Mpb+lUzAFu46az0677g4Q9ZkZQirnrr/h0TpEE0QxFhsTWUH9GisvfHpvub9ei24ZfGk14Jdqg9LY8LE9XdULTs6hz'
    'LBUPne5iTu2Dyu+2EsX5IFPL9xN5cP0UcQqCe8kd3BfDd2KxHxrfv440s4nwdLSC6o26nsCLnQ1ClQBQNwv2HGh+AsDPs0VOK1rr'
    'zHRi4ev9uFq53flH1zccsDgnJkQ1CjKLWXKWi1Fk2TcGlf0jXNFwM+MpfODkFbOBah9u7jH0Z1dRDI6rV/K4EXnlUh2qApTa62I2'
    'zocFq0+7yfrxOkPuroPTdK0ufSKt30UVijNK7FlSEZUXbpEBNVmRvcwpMjtxGtG33tc1o7sJFzZaTGag8D3T721wHQ0b6fNmWJZI'
    '8MmdctxTU6RSq9/vZPdeaQjM80ukBT7TkvpLkg44brJGkr69tszvvF+x1m0FWPP3+KKl/dgXjUMJiznzKR2NXF1v8TzrIicu18Rw'
    'sk+fPn56eJQ8OEzuHzz95cH+eju/YhE6F9e3CjJPclUkxZIv3jPhRaMRz71GQP2tGl9yh9g+zi1m8xrgh+cbPscbf3Gu+//wP/G6'
    '+e+Z/ytNOQHY9fxf23/x//tz+f8dmvOVoE89q/NpI74ctajVibbnbDGfFwJRJXHERTE7KopX4qpHXF0p3uO9tTW1ZAmTUAEpCG5F'
    'jxaTJ5cSQMdueflizpb3kTgMnuRDVv6P3JFN4B/W1Kyp8QMjSXuDACRuwryW1AFHQtUlYp64Ds43gagYRZOQAM98LbgCNgLBFrg/'
    'wJFxcwwzILm7JZ8XOxzlCvQndm6DSlvjRjnAMOpvhJbEPkqlBOEoDhijs02LM5mhJ6Vyv2qTkJalDn4x+D/tsyj79OHRI1oGuZT5'
    'j3xcvsnF/2Mo7FWdSK5Dnm/imDd6yUPu50ku4XvM336eXxa1VHG5JPUXk9nV72tEnbGb30L4GfVv7EXr+/k451wuYl3Or/6JA3Ro'
    'RjmgqbdUJYM7a6zuwaNnh0e+mM3iZJjP8qQzlYhtzqGqyd/EQ2kj8kNcqrDlwGWukOw1qmG57AXGHk+FzJt4hcqkVU8Or6VkjzKi'
    'npXQGFbsvcaaRA2LokbqSxroq+JSYN3O2IUqX562xvzVqAOjhcBOyeTX2kGRLJDRk3dSwtttpjtFlmq0rO6hyXzw2dEzcXtHHcV4'
    'XDVA/5N8OZ76Oz/hdGscwf9LBbTh/bDcx8hbVfYthx8YriUNuWyAlNlywJSp++WvD7yWL3PFeaaT8iaPJkj2ATtFSG0Ck9nwnN29'
    'yMcAxBvmw/Pi+khz8aQQFIlFnQPcUV1DHUow5GOl7agwm9aDv/tyWd8sAghwwDi6s6rpdE8W7FE2KSYc6kIfM0oZxznRQZfTOnY3'
    'v6Ovy19+/oVW9cknj+Sqm/NmmOTzWsyIlwrvr86wExJVS9nfvHeX1Xqj6utph1oaLzpnrIl9TfLoR8liRn8kG3I70Z7hVuSIcyru'
    '5Ly6lH48rHy/Jod+hdA9zK8bAAgPi7YWF82Kr1R4SjrrdA1S5xjAWjq71CkS3edFjR1bnjFSzOzOK7WF8h5lH8uxojzk4+FinPMa'
    'UL2vuj63y/uffRQMYUXqgH9p6LF1j674aRvehP0dlzcrj4vj/dh7KPLelhmDY/K40ltA7jU5E18cHTyVqfuYTt50WKifJh2ahhp9'
    'XY7LvF5q5lAOLF0REw7BVOuwiG88+eFQY8KawnJWieM73/try6pq8TrVyWfk2srwYPnYQgqeJoyKkgMk0tIkXpQCaVsvzwSMhok6'
    'YF99WzNIOU/qnP6W5GEC3Eod41OvYD90E9Jsl+yHWQpa5VKdAgEzkrgvScwoCb1OXhaCo1AXAgeBm/7Ol1s6zR6xvbzFJ7mY3QXq'
    'Zv2EHQeAuymz0BUQratv6oaXOlcQ2uqsnGsYrc7s8gm+9FRQNJDkRC4c9euXWP9qLMDArUtBMQq7ySdfHD49WD6EOiVEHBvZJ8xg'
    'lMoeHIyto3opEgFTpMQxb6yXPOlLlRlqvKJhUMtEKav2h0TZCgEbCYtAhLWoz8RJc/nokHg8LNVv35ZXMK8UK+G3xye6mY8XM0bt'
    '+y1t8F8zc/DosTpBdQWgtZpdfdeU6gzxUfKQ76Q75wWxMJ/z9TpPDub8H7YHdB5+frDR1WykE44rMwCfv/sy5uDufJnd/XJLaTng'
    'fqrZnMTtN7kDTNltKh6I5dT9zoSyaNQEEnML1o3AyKjmgul/dcJARbkGcIjeDngezvaFS6AXDUtyxSdPag6O5xH1kqv/k1OI5SC5'
    'rM5hVykl/yjkCn69KEa5ZHZdAErXAUrET1hAIpvqT4swwN8TjxF4z+CB+5zbk+YgDh7AT9PFZHbJCpfpDPCpwj/6pw/5+rwvRaLR'
    'u/fj/aPalB81CO0fuXpxDJw0zPl1Xu/TAHtM+ur8spt83X4sZk3wPU2LO7ut4yNKufliNi6exx+Fv1+4tu6SUSFeJ3e5XZDhr3tR'
    '6kTenZwVg37u2r3vO9Ewjl17N2GTBzUjOdVfU12v6Ry+Lpt7d9IusT3FbFROmkhXV9sHzVf1vEMPP+ORtRRg1jnpWZf/QvS1cA+t'
    'WZJhh8f9uBqupZP2NqkFeoMut86d1xsbXtXZpCpH718bV3RbfdXpfJK/PuY07cWo0wzVIB6vBf90nI9G39siTe2RVgdQOoWopavz'
    'pUhOckEPJVnGqGReK+ncoUuHzjNrlgFByVfiqNjwdRLbu/SK+m9dCb8kd5Kmx5rzG9YuNhBAmOBjSde0iUgaaqJry5PTXtOCJq/o'
    'sZHips2BAHvibZpjES1nlx2lj+1pJGJ4VsxXzG1cssKEG50QZClZdTgU9sKYC0E4x5V4Bxzhb9GB33L+To0SenB49OTzg3938ODx'
    'vnXv+Ul3/sKw6AQl4xUCODTWhYU0DgTSEcrbPRnFvRTrqH7zdLUrlWGRRbIOgFtz0o1YQZ0yQZipF2+ALW7bUiUHarBqIh+j0ed3'
    'R+BA7gn3x7uARMDzat7RYQCEVwiHJhUQ6H/bfpcJp6Rk1CIXzJLp1R8YUYx7FcmUF1ffsFyxhKbBidu0A7dtPXn1jb9KfZwAG3h2'
    '/Cbstzf6Im0xemLbOf98644T6bI68y+oInrs6DOr5idr9k45P9YZ0dfm+aviOB9X07Njrrzzxrfl816vp46lL/we1KqE3I/ZLt0J'
    'jd5pVb6hX2+6l9e0mgAwWbYsQxO/kiFtsDmxt2ljaHgF9ZUOt/GRbKUNfZVOntTkKah4oTE/9Bt6couLQefj7rPul9rW8ob5KJFO'
    'eRvR+Nc8XmMusQ/pP/9XfvWuDkvk40hyFygZmYX9pKN1//F/+8/Yjhu+Bw1e9Aiaj3J6R+w8qjwwdRLJEpNp7kjXDkUg7P20uENV'
    '8o5l5DvUpxEAgm/DNYiE+8ZGrhyxRNYlX0LiJBm9GMPgWUxv3R2Ys9v3iO6PWW+2mL/vtzc4zoTe3Ll5cZYaNg++igNA0eZPwEqx'
    '6ulHrhThoNWseMaW1CZygm5EHmEssWml958K8SMWn48U22UZ11RgN253gc5Hx3SJCBA2E57X7GCKp/l5MY8ihNLN483Nzd7mipUa'
    'SbosWm95dyuLrEsfJJ/UQUMlVf7PnTtZeXe0sa94ZALaKGGfRHkVkorkxzmx9DSeWRlhs9LhDUeMbbIbkkWmMdR7Rl+ZzlW0iF89'
    'owuyyTcAcMqBMdXLqFJR9ZiQoXraomHBZjbPkTbFad4FkR0D+GS1IEnAdX5aBeVoqaeGzur0rOhshinuJll3aaJ2t6PMiNOL49O6'
    '+Io+l0lKPvqIGSHa7VZBeJeq1lZEyu6E5rB+K9rpegNLfsbDChws/dGhSjZ6eSOuX1JFi1MK1VuX7mZLtbFiRvnhcrqitmgfzmV/'
    'YxtOwxZ8hv+ej+SPOF/9eT5jjqyTdpmX7vBHdClkTJA6z/iL5O7dJNtY6SA85FTNzyT9KFfTkf/f6HqvV/wYOqt6bnPwXhJsRP2p'
    'TjddBfsJ/sXLvDAO8y/LGhlTZqzUz0raG+XPUtqgvxRvhkTC4kk04eRGDvVsUkvLz1XHfS953ZNOP7+T0WXoD2nsndUosLKlAscC'
    'vO5NZXNKRWHuXhOD8TrjikHF9/elZn1K+cnfpV0IxmUym18ej8tXrBCNf47q4CpTWj7ed3e4iY9ayhN/OY1f5n7/TF8eVk0UeUNT'
    '/1zmnF/t2BivbQGqNUazpisSazm6/Bcv5mHboCWxetU86LjH8bPC7rWx9KOViQcUGNvLlDvKizG6bK2GPbaXY/T62mqMLqNQq9ft'
    '1aDqsRw/k1ba6zF63V6OO/q6Loi+Hq+I5aF4/ROQW7da/iQ096EaRFsc/SdVLct6GVbYHNEayC6ftxXZAn4yRXY03kWANHlrqTRk'
    '/7zTtFnC8hFndlHmYwugF8xL2g9waxoLyf/j3//jGIrMAOzAMbmKa2mRoLrFXI41k4zLVo2DD+b2V7DGmdmU6PGiOHEElVhbIgpf'
    'ddcpxRIgZt8gBN3EZAxPz/ZjrRbfuhNGWrb0G2m2tf3DuIr4/MhrNjE6SyELmmKf0nzD+qVZblChwS7I+APUWDnXUHToQWFbphni'
    '1AlPH398cP+zq//jUUvjzHZJXc8inB3Q3v24xVjZjzheTvlWiozbYW/EfDwPSUSFjwsj5Q1G7DSrPerCwMMESK4mKTtnRdrehtmx'
    'xLPQdgXMNHkyDezUmFFGJUMaa3Br5n6CjS2WyW1LVQpDsqlAL4WaatciY1F7q/C95/jzhrJJ5ath45VMnzJnQ//fLtYlllttFLMT'
]


In [ ]:
# PAQUETE (8/12)
PAQUETE += [
    'fu0YQyS23knPAq+pvIO9ttEKQqUWeiPNZNBuaRYH20UqlRWxdme3vhrxl4fCWiqYfkkCYS4ZSMoRW2sYd4iRU5Lp8Zit9w2AyVXT'
    'sMIw90FwHxDdS1mb2iJXdCNWRGYfWXUbIe8edYGzJXKsfbwLPhDnimrOkECcr5bRCj2FQGyYi8BVFMzpXPYcD5otf2FV6WI4zufz'
    'KeZZ7oJmPhI5fn6uqlJVQLIuQEgIvWY9Zs/iKMhAYGIXxUXVibIRdTXoZ181YIxbPYpCkOSstcKPVpFucY5kroE2CYey8AsMscx+'
    'Fx2pXrhM7jhHgfB/QuywhvLEo1sRdeLMb9jDK2IYZ891YNzhr6//fBb9TMN4Q1dRo0T9641V+CNft6duQcLizTO3Ij4Rx5LB/GQu'
    '5GajQTLyNt1SQttSvk+Tf7/JqYHGOasZRP0h6ZfHmtiT9keryrmQM1ZIBOsPVyW7h3jrFke0cm5Y3OFAJul/N7l1TldN2qoP/Uvd'
    'YevF5KT39XqXOEla2YtqmJ8cN+Ub2mCjDdlk90Yslt3Z7O20UtrzruAvSIo8piqKEfvWLoUCoQXiRH9wC7KG66dT/XDUjY+HjKZo'
    '6Crj+/E517poCnnkaKVOZ0x8WrqR/CvpIBXrKYOoeS/Z3LgpYIm9RscSzC2iZXxEN6JguNteW/ZX5S6ero/fjt+111oGeLr+dsYZ'
    'nvJ4nDe+tXo2wkTbe19/pW/5/NrltHHbR6/wEY/m1YUI3z/g64t/0dfVjR1eW45MC2v/fPzieuTZOW2r8wa3MC/96en0+LwcjViE'
    '5zKSEumcRsXXqmj3Ta360eimx2plb9DXzd5men1DuUdzvEXsw9UBc+12i7c8MWdo+LzY2HjvjxZ/ykcj/uhcDuXq0TTt0WAef8iQ'
    'mrdNPKTmvXqnHy3+lI90SM31ITEhW7F1sGvee2uEoazo1NKri/d/VbutvY44FnEdEWxzdQjaZ7eJxr0GxZ8kORjlky6xPPpUF2dj'
    '9aiI8xapKmTR5OaOQeMWYJnli5Sf88XrY9Yp8+VZbFzDU4mruR7B+3535Hue7IhA6n3aM2+SE0P4cLLXOm43080l+vkvqbY9u4zE'
    'w4E3rtOWT4NWVC6n/PVxU3ylykUu4I+ORQ/aXbsd5wtS4qph6arcVzZ9FVgG0qXV8Rs/euj9NSgiZsgnahWGiv5W5ADDC1J/JFYI'
    'LO9+FeLFa2gqDqSyyXXHr8r99vyVBsF47NzMmJhXvWI6QkR3a+k3IqiG6bGO4HYQiv+XvfdrcuO68gT7GZ8iG4pZAxIIkkVJdkMu'
    '2yWyZLMtFWn+cY+H5kIJIKsqyUQmlAkUWaTZ0TP7Mv3QD7PtmJmYmBfFPCli9TCjh45wTIQjVK8bM9/Bn2TP73fOvXkzkSiSbrl3'
    'YrfVbhaQyLx5/5x77vn7O3DWGYk+fTyGjNN+6dY0eNyDf7xRxxch/OTOvb86uHfru26YGEFqHfKYFV0BEmAXgWbS1BI94NZbOvZp'
    'KqKi7HRmf7kmdiP++pfxPEtqMBH89/EoeiB3Ss/VXlz7F6pKKFZ+/OH+9vY87vuIYxDhywevXAT0jxAWFr1sPfGqUXnNz0qoTU1a'
    'OZsH8nrnRz2GT+gsbkbDsRgUUtZUQ0H1xxxR9/loK4gVLtszNWMc/OLhxd/5koDSbvzEwQTj5ECISWxVaDQdoB3U/E4zoMM1pfij'
    '2JBlES/mMSE/5daLr1co/9IMnLe59fMwNrfGPtyzMuVXmFg72BY7+mGsAlV/Dc/Uwmh8GM55YWJ1Q6+chYs2g35Ho7WTTMMzktrV'
    '5WNFPlI0uXk6A9zmy1bXXwW5azcf9YWa+o9H+PSx/X3Ag0Ouj0hxNf09d2S6emSa1+NHctvjDrV5y4G/GH63KsjNR07meC7HAnv8'
    'vNda/C0OX4e4b6Wmhk8+zxn2NUVeqI+OG9UHeKD3PB7u6lWus+lFcbSn3XStN7v7BX+KftJ+jShEjxs3Pt1149PWjWe7bjzrP26+'
    '+1SEu1yktvzpmcmRzeNeZEr5XrkvoiPxe7P/EEG/8C65j+mhQqOni+F4jZQlIV06WPdG0XWkotfkIbc9GJ22TDJP0eDTdoNPzy5p'
    'sTl8PH/2ds83R1Q6eodEMHZOxS9kTDfBVp2OtUq+6Le63v3k0+0nn/Z3EdAXp00Cetr6fnZKakJv0O7Z6a52ylY7pT5XynNlr2WR'
    'YhzSKdyyMlGtIU1LM9kmqyReD56WCClfWSTHdUXPkQZ+FF1XY9vTsrUcrQbOTl/TwM4hoS/tyWhfkZY4Ttw74sv5tuaARVV3C0W2'
    'NSYbc6ZOZ/o8batg9VukAb5F/jaNWrAAwFYpNPQT9qFJcKA2ITtGv6ITIa/kZnjQosV5LMpKZgE+ZbpBDMHxJssGcKQ/kAPoyvXk'
    'LxpSvSzv/vVd/dbm2HX92Dzy2Pd2TCmGNLLbH0EKsFieXe+Q2/kC+dsay/q5XpapIVW82X+7OUUwrvVzHdT6efuV0+OMUTHysXvz'
    'b7EuWZtL38QW3ev4pTnOIvilgwsXLXbt1VKv5Wwd/2ixiN5r3UeUs2DW82mxWYsefm187ZK10bvQ+2ZWCtbmubyj2HncLtPFG563'
    'yDe6yhye15y3e3okHl964B53HLjbk7aTek5EOzplZGhY629rjqUzPDf3Lp9nueHyKcYNWxLAsUkAbrxvbSI89poC7YPPFgNpe6Qt'
    '3nwTQ1HdAExFlzbwTvTTzimbwFW5TJGmYS5L5hytkdZTat0sRYomtkOrSS2dRx+Xc4OXrVDupi8TkZnH2eZJ4X1w41aThznwMlyW'
    'ojqEos/9sn1uzs2zeH7xtdYMJSq2+seQAjN+232ISTy+nD4wu6/dhu4mIlB37sKAQkQyF17iZV0vux0bLTEgZXv39NvbRto5PTZR'
    '3x5kk0FD9UufTTVYqCX2Y5K2HSh6bLtb1XNS8zgfunt6LNyQDY8fhL3SG9gXFwIa6AELDUL1Pz6ajKIJwqonj6nM4/v1yeOduuu2'
    '3lr/7J1V6qjDe0aX2KnCB2vl6hLfoQzO32czj3dwqP4HHV8vmC9Gw04xdpwkjdSEekICw0XgMveT3XRB6uWt+4K5RIARXxl2fhEs'
    'jQvPbaNvsrfhuu04DxbM/us8DxiK4tiSWmlW3g3aiuK7GSA8d3mL34lohGPu7Fy0FcQExjnLzEaDo1H07T8MmRroIyBQmaPOIbbE'
    'gUXox/d5xwqYoEYU2B+WyQuU7I5u3KKFYe+WK9ld1/9xUHqNBhvmB02in6dLmE7OEEyIihI50yrzfrTQoAxNb6hYF8VVByW/4KHl'
    'RRisZRBUWC849o/cuy2NnAQ7dbPrpk1wU0V7FhOqg0BXXDtBZnVvm9udNM/DTfNr1fpZlWUh742wMvkTt8IX447+LfqPOyIJ/azY'
    'jOyEa90tqZAy3Yn7j6TMHZZAlTqD0NAgLImCa3N1F8M3E5mPRoEIe8Q8jXhtlHHtcccqeaG2KdCWmV3annbzMj6+rBOHQexPNo0X'
    'T3BWZUj7arbV8l90uWr0lCmzjr5r0+y9fgxcG6nSq+bd6a/DjibkDj5fpQErfmpWkXWxmj6tTxgXuClcGJgLKYzGgyvatqhfsOh5'
    '7ZYs9unjYPeHT7EsdimsMa4tmZnCAXxkP1gGfmwG0DhoaZFA5EpzliE5j+zJ64TeqNIE2cVRUMn4Ws02zoSNdGeNuFHIIP0Y6h2I'
    'HvmxA1N3cAVNbd/p56jdPttlO9sP1ashd5kxMlA2TqZVknW3WiFYsbPLCNoCP8GzdVrWdqLhe4jJ+4vgXRCn3IOarxR38Tb83GRg'
    '+mj7mjyvjA0PjKz9kTa7HaCtrjucV4thtyswsEU8X5mF8VETrfhJbWp92rKvJjq3IM4nTW009nqC5eWT9fl3AcJsRH5Ak9aTnQYS'
    '7ZTDrW3BAmOc7+3bJLATkydYgcdygsRNe/HbePCrFY3GtTu9v6WX69CqZ+lJtlGmvvLD6exj0XsL//GmKnT9Zmk+B5QryH2MSL0M'
    'SHyyVVE7fH263wyteF34WXB2VgVXGi/SqLwjmbKnCMXrXXYC2iET5nWEc+AOtuM/7mTr9/uQ8Ay45STJNQh1ZKAMzN8sAlwOw6pJ'
    'mRH1+XOmtDLGtZYnmkLbYthwj1KYaZ9Hx13CTNdNf5wwA8lbJql/0h9yZ7vvG/lu8krH3VX79lhvbwo2DoGxq7eLfuDRbezJrVVL'
    'mqsGqm5c6XbQ33RZFE80j+eSRfafXby9oss4AI9SxAHmSvr7DkIICHimUI5vad7xAKZFhpLMKfHix2rjsZrUcReEfDu/GmI/khiQ'
    'WIdhvVFLknQx3gzgPbv4qoqA/nOq2GLadIi/7Loy9Kmbi6QCMlFKYBCYCj5CrGR28c0JMTIYhk+Mpe4Q6SYfVxUMy9ECj0wXYG5y'
    '2yZP5XQfJMOtshuwMcPnKPc27QSViWWoMdTk4Nwd1TYtvZQ2WnvE75Pdt29at3fumB27xuUJoYtdkrt/yaL1Er9fAHgptwxx0jBz'
    '6DlSLmrdgGfj7iyi7zRC4eODmz//U4UoOG10V9kXIa57WiXM4n4UAFA2S5DZoBn5lmLpox6arHNXOEJHIAJI9MxHgZyMCSTWACSl'
    'JIm0o8esS3ctCNM99S63kY/j3+FNHHU4Htsh/xYB0XRW+1+ne3hFFi9nizh6MYleNDSmF4E+vI29f/j9G9d7HfaT0PoR2Nz0F9jR'
    '/hSmMiaUWXPWzmXN2PNhwB3VWYVmU0BFojo3LO8WOObf9RhijkyhS/oejh/IRsWF07DwF8fu5sfseFtvdsiIisPceC2E88UzM1da'
    '5ZxlNZ3JSbaACbI2czYtlHJo0pI5bA/ADJzovbbbMQ8rxTMP94XieGkuc7OH7wTPI7gg+sNv/4b/+7/ch7f/37//3eC9of8c0Yny'
    'R7W71RCmv+6v/feH3/6brjP+D//+v7UubN0nt3Q191t9Ja3r1u+IKSP48Kz4w2//Y7vhrieOj/HAf+yc6cPM1MpFmJfkssTOFckc'
    'qJpyFmyAO1MQuC13uJu0/gftHd0xu7+C2MxjYrQ5H8LEmeiAV2rVvrWYny7NeaR4RKG+TiWtZOyRzvs4OrBsnxFByhSXFHjW1HoB'
    'cDLgjWF6fWN2r4/1LnMEIBqI+4M1dNQbMnIhQXYPhJr7Dz87kP6wq+NGg3vSJ0sJYuqcghqeAioTuUIKJFHMEImE37WL+d6w2cqN'
    'ZiuZTXOjnQndQu55Tl8e3bpzX2tCj9pEBOi3Z7YFtbFzN0EyiKEl8et6NDvz/rhxI2sOcJ6igVtNqO2Z2kyAHBbdGO8iseM4A9Lh'
    'xdcrZMcllWVk3ohiHp2Y5+voWVxmFBsJCRm6o5qpXY5o/KA8xmzmUVEtzWtjTHjMOvHMxrO806A9LTzKx91miK5fFR747X8bRQUN'
    'OnPigVaKjjtPaIAOkxRBiEGLRI6duEBr1l9iqFgGWzPqYTNZvSoU2D7xeHQuPTVoKkxUDeSNcUdcV4KsadRZ2g7wujTCq+XZa0hJ'
    's6zAlNY+5UZS8Fv6UTWJIjiwad2dOXcoNuHNEQ4/F6VU72i3V9/A2br9FnxvvqUhi6BwjdtSzclYTLEz9OjsODndi0bbnvrXBmR3'
    'WhjMW11HK7UO30boeeifd+dw3ur+8yUrFS7g1dTBdNqKfaLle8qaF8rC3oQy6ui+nZSxKOjfYl8utZtzFUxqEmp/VrRd463gjmdF'
    'hMWTkYc/UIJaFK1H7/zlHbAWbvBxdBPIJPLU/duHn929R7TT+SZelMwvNvkXmp9JzdG7rdacwEwuY1BDcwfdrMXp0xIQ4V4Hhl4b'
    '+yDYtn+eEboT3JF5ICKNl9Gy1+eelwR8xE4GFJdutVfBvo2TuiJyKSGEz1XJl5EOnn1x9dnTq8/Orj47ufpsc/XZYkgEOSBW2iRU'
    'rQZrvBdgkkL5LmwswOWOcCBbAHHLr9+iVsTfOIm3K6ynFn8XrcVfBNFEQlCdkT3jB72tZ3C7p4y3iJNsx2aNorOS+kcj0mo7HK7V'
    'A43qYkd+Ii10B6R1hVs9aA2fb5fWdrTwE33HjritNnlokJnQ230UYUEC7+Iu3EKbpXx4924r8o53YSC4D3+v2N93cXV4Gd5aR0Py'
    'z7vtYL5W/8IYbovBWyGdGOF2E6U4zVKaxbOU2NXYwH99jW6YVlthTMuMeF9y6MZOzouuEVKccEdgga0wdHY4X42fnSZlMtiO5Asj'
    '8qIfon8/YOiJnBj3t+JaRy6c87IwUZJQowsMiJWOaERj8zdrcHF/J1V80Rlt2orYbJ/jFhA6ecMIwQV2FQQ6QBozzBTc+6e/ONg+'
    'mhmjy26He5HGCkalKr5OTVF72+lXC4YYy374o1rYITIgrtYa1thVab+1kxlqXYcXKz7OAsv6mrjkxdPOJ5+WnXHJrZeiR1+c/vGx'
    'k4unHNMbtvD0rKuJM8732zbRCjZkmGGT7SNa/1LZBlH4OC3wdH0wfHH5M0+7nnl6+TNnXc+ctRfjec7V6M4WkOfew2x3Zgjoj2fd'
    'WQFBPJa+ZprmFDnjDpETeQztLIyOjIe3ED8bmRWXTlMcippx3p4dyplThTxqCXuBhElDg4qOi0B63DIfeXNby5Al17es+y3jXz0I'
    'uMoXCzmqB3hu5DNvaq8gV/r5cNu+5QyJGK183ja2JfTAVMnJpsQJNIly6ezF7/LgxNFaeWECklVLDOtT8LizmhYiYAGlEFfWGxGn'
    'StE9ZL6kD6fxjLgQC/jlGiFMjTPvnWYBCwdvOYsr1KgBHpXrpUpvZ2mVzrKkqU1eln046dT5Tpi9yBStP/e2WHdpm92WcSpS5AFz'
    'vEQkPURXBr1u4mxg3r58+oq2d53Ml613v8LLMYf2S92FRkFWW8B2figMG3XyM8UC/+5hkwhf74vWn5vg9YObu2OgQmw0Z8bdHRPl'
    'uEE3OlvLfyw3v8YRqSgg02c1iNsbgIq3vW6KMH7VQXgrTpfH/lbgb8KFf/vfhg7/u97Vcv9+9D/+y/TlLF7PT0drxM8Ur2Ro3/6D'
    'NRoCtyHaInr27T8szokX+/z//jff/gMb57Vv/0EuDKPHCijuHhvIRTjbtLFeUAMDWxhGRF9SQk2JWsIDfEoBnw3qC01a9RoaLgFU'
    'FGL6sOzN8xpRHZZL88AuaTYCfaKMxhpGIgLCmIVKy6HAlNYLi/0QqktaKbIz6MVkoSj+VNKS2rTEWnGpwAnsId4H6gIA7hWKJxTz'
    '2OMzfYEExijLVPOfpxiZNAb2PN8sgV1T8wfaXivH2jbAQFppquVqg9pLrMMEgDaZybhsOcm6/bU8Vg3+PaQIoSdZ4Hfl90CcI2Wa'
    'hWtxPjZ0y+vDgINT4cKDDXCc6bPA04O219Y0CecyuGh/ug2g+Uhv343mw1Z3PaJfRzQJpr0ryORNeIFFdexmBRZIMusKSVgUm/Wl'
    'rMFFH3SyXqz9xFdoeKPIhJsxg+kM/S+2CAUdw6hhx4YFJpfuMFISTuXNKMaX06Ay7bPFZHAqP8N2LgPhrZPeVpiQurafBdKGmoRg'
    '94rNFiS3RREbe0O5aHEQ0c6woZlpIc1E7F3L6R7V7nj43uu3b/j4ATEgt731l7z3xD+4sYe/137aXZQb/+f/IX+FRV4XjnqCv8IO'
    'ea2leLuYnWcn4VWdJ/6m83QSfeQEuRPe3RJNXVTPs812M5u6mY01A9Fpw9tDw8znmFWkh6i5D0bdkYpALGkXs7YwSpCps/D+wae3'
    'bx0EYiOq5YXVtFHxLNWafGrFHU5oytLMmCryBaphOkP9miK8RjdCAJUKAg6iy1nCiLiy6sNwprfKgCadT0Mp3e3c8DwIav51sz+W'
    'PW+GKL1hYt7RKKqRZ4IIKtm4gzCGaRTsg+FlDYbIR2YEDVjgQb0jugKlsEfeqp8ukqQx/q4Ir1E7pqvu08ZvswALMQg5DrbO4oQn'
    'CrcWditx7LW0x4n7eCUK0YJaEwnsHrdRdt60qW/abIXGcFd1xc2pfrjZETDndo/P1Jh1xMNvM/s3CBwMMDNlXd2W9BZ4OgGsFuG8'
    'SHOKBvVGgwcS0nJ8YoGGdRaNCSZBiRn/SFDa0MFKMEw4wHsM5JdPzDIuT+oJEPGwqE7ceRGcE5RcWWoT6SC7hI29hv2LCYONiBUX'
    'nuPCYIYBGQfnB2Zsz3bEGzzf2g+LYWv/hwkY0VswgNM2kNBJIzEDPLje/Hs7Wzptjs1t9b3udI8dW72jS409/uYpKa/Z4NM33OHT'
    '7i0+HV4yaSfOFLTnzsSdYyU4186GNq2GNm1uMNhmB5dyA8cMapMbaa8Nw+59qlsc4lfd0p86XN8ub+ZpHVjG9IzuwLGF89f8qpVF'
    '086y2UqEuTSzZoSQZR+kDNjgZ4tW7kMrmWE7s2A7Y2Ar02Erz6CV1tJKX2knPWzlH1ySENQAI+WJIicw9KbhpBO2k5SF33tbeOZb'
    'Qa9bhrWOQPCmeeM7jvxXSbAR+z+z2P+FBv/7jAYM/ZIeW4xzq7sLv6jN4evl4ZtmZcTTJ41VsySKR60UjUWQN8GgWCXyd/F8oKe2'
    'QCCYeuXu3M686L1hfsgbBS5rhDKyXIT5TsF95Z94OoocWTcSShpBvtuOkSeQphZPtgOdGUYSrHvCaOJt8CQvHzlRdKrs8EnV4do5'
    'mPJtrYBod+hs37/h/QdTng/Trd+bB8W044UnUz0y0MJmunVoTFunxmWDo3hY2aExvfzeTXDvpqtfzzVsGxqUdKI9G/UZsf3bpukG'
    'ty2E8ru1rUi4prB7mxh8tBKvzYBMo1NuC3JA+XQ1+h//hdqdu4DZc/d9+w/yIwxy9iO/269DPvptHeYIzyo4Jag8SIn0gWb62CVJ'
    'Y60HF45tu/7I0t3HPeqcvWJs/V19g/8B/RoE9737bhQGAS2U1bfcDelJp4uBeXBBnZmjoW+X+XGwBLEXgcjCDE++5d1I/3X0hpd0'
    'aRkBwGwd7VJmDVsV6EZa3pk0Guyl2nLlpYpftTJ2v9Ng+7ZRXKWUUFfZCr6vA9F3VjaOBkHJ3mGgP9wtrBq9y5PRU89SXaLjMkTp'
    'Q2IX1JugILBvaHD9apAzBidNhQcuvmENhIsvkYePGNGPb3925z5R/lMtaK9FkPLQS2PZAQyDREc+Z1IuHvgcSYkIHUpcE0Et3kb1'
    '3cBqsSovvnmO6Oqz5MU4+lUddORKOe2sG6zWbdHOCBASSCFB8OWEoUtBueRGcQZffn2HdWN9sg7kR59prMXKan2x7LoJw59m5S4E'
    'O3//Dhg7Q+t8XVAiJifNW3iImkt408BFCPKK7L8tGBzc1wlzsbPlWZ2s0Ike27pZdrJMjnAFvOkKprMFi70az7N0NZiNCIeO6BP5'
    'R7b5/izQCeRhl70WAJJ21mVoJhZubY8gYas75cWCW8/SkxSeEK0m3oWv+tLCTycu92Vrsl+/4DtRgbtaC6A0X/1JWFsVnzl+JpJq'
    'TLUr+g2LH3O+8SFAgJVbEH0lFwf4PGz8Ater7L7x8ukiLQf6RQ++kQYNT4unLR8F6olJD15MrbQ1Q4HRFJBqRQU5Pk6fD/rjfPWi'
    'LyfRu+8qFbZe27wXk40Cz/3h+FmZrmXfJs/XTU8tfh4vNstVVa/PupiCtgasp4bU+n2YfYSOCjj09/ub9fGVH/S3zGPogC7MT1ht'
    'aJmsT4uFn96siBeDOfLd27M72i6Cg/nua7Wi/pvMueq5QfmdMWpO6zC2x4uuVIPLJ6xEiCjna2vggSQJtUXGBCXLjaKVgyynpCs+'
    'xxnYtaTDbR8+cJIHS1njLcbIn9nyNtta0nEOsQS/w4duzqt6etGEVZthx9xlJka1q87o4i7/BLWu7lhNdjnX/iTVroCN/lfNYle4'
    'pCbQpIrnxSpzkRY4M58l6ckpgt/msK80zZybKs5pnWT+rVbF+vQzouHwfhZw1irwYSIKnCDAoEgq1Ls6uhMiVSwL3GLlTOsKVYvU'
    '16J3AO9CYiKhwFFxbgnh6stOTuDdTTVd7SwpF/EiqNi1FSGxUG5On/bzebY5RxHMHYjabOVBvFwV8lYdGHNfONaYuMRWOSVPbZBB'
    'ht1Aen7mQoPUk7K++BJn1CTodeACh5DCnqDyJ2waw9dX3NK0ZSu6VYj8VtZlPG8kV94fRbNkHVf7g2vjv8DB+hcfDDtcpe0a7T8Y'
    'GR1MOdb6t2vj62GdRjBLVvrVam2N61lpduSZMyjP9uyDvM4+0cOJO99lN1ktvvnuZqPsQ/OSusYr34uOgJ+typXaNBCU2wDp1tol'
    'rk5wngA8HQFnrBugk6EBO3TpC1u5/mrXq38j77b4rNY9siOus2eTlsrmhicvGHb3erulvT+6pVoiWCcrIzbI01OIagGxjK/xnOL3'
    'SXuxRPK73gsD6OZFKfzgJCtmjIculfCJ8r2BOoQQKZS8CgLQEtyCyk8uCKUZKCajEgZ0lq6NyaRz9Yh68Z7xJ/Ga9YyCsLIgkEM3'
    '334A3AqN3cpS27IiQksVa6tj3T1vwQlmETP7AO2APhxMH/R1famCxoS2/Pl1nVWHcDS7jtfafAZWZCIhzD3amNxRnRSNQJ75Xqul'
    've6WViW8Yqu04AQaQ66MW+0IqrPhTrat2+gH7ZsaKoF49lOh9zZYu7fIu6l189UY4nIF7PYyK7YtyNgmesT7aWr+8J6ou/Ukwix2'
    '0tnKXtjKXtDK3lYre6wTBgNHK5xhOT2NvY7oOnBV1rKZ6d66a8/u2mvmtcWVx5mh2qSNX6VVhtTJdoYORZJRaF36g/BW8KWn1Bjc'
    'wnkOtC0z8cXvhW92zbwbMreOSVRh68p+M/DLhCYSes1OsnJazU+TxSZzegYecnWoYdeyz6KILTcrVzjyg2vdET7SnOyvaRmv06J5'
    'THVwJTlI/4qtilSZJ0wVw1ykJv7MWXnbBbG5XtbHdkMDTI91wn7oOro9L7VFQHo5i4WBCfXoLKNQ11V7src7rl+kEfoD+dAVu39o'
    'wD3Xbb7qH5rIxZS4RZWmPU5eSfaG2s/8sEpBWo414UXtgJvdYwgn3e2Q8NpQa8V212fOLClhu1yKvUGV+0m77NppXMXrtSvN3J/a'
    '3f0u95L7se57Z1fcbb0/+//if8jRSJFPO16d/6necU3++/D99/lX/mv9/fD96+/vuWt6/fq1Gzc++LPo2j/FBGyqdVzK6//s/5//'
    'gVvdUxogq7v78PDeg4Po1mF0+6dHd+4dHN28fSCa0iGSt5cX3wivK4KAYV8DFPG8zLxkhdflCtG+1JOWyLo34QFcWyuP9m4fWAXY'
    'KlnF1BZhQeaPsMGexM06sVU8S6QXtyi7JS9iUeZ718fRQ2SzzTdAZ4bFN7v46jlUv8GDT67cvvWJjGZW5JvKUlc1qseZ+I5L2dJq'
    '36b4coauQ0MQJS1Qz+SbysEAkhIxMi+gAxluKhJgo/Liy1WK7qs+gOKpogGKQuEqx/gIagA8cYgEhz5XBAWLm5N2kd1aafo+Qo57'
    'dLBX63S9QU3XpFptkF2LscwufldJO4xcEsmHTgLEOVteWmVKeoWMe8Ad9PaIZSHNlhqrmp7IeRtjKsbRg1KhL4BCRTERhiz0WSOZ'
    'R6x2jcxcqNgmoCFEMIkeHh00qsnKmgozzVPRWqHNI7pRmk1KVxcHVhbHhuNh9O3vIfIlZ7YoP/YxllD4kycyTmlhk63hxKu4xvhl'
    'IyIlI8NleMs0k9lxEREzNKli7nEalz8OwzYBOyCDB1zNfMPcZ0zQaQL48Xbb1txcmwPEh7dE+Eiui69+bD3FDLPgkAMDAYAXkTLz'
    'RER5nmKfarR5RXeGYtYQPiTqw3DO9UI9r2iOyMv+qPkL3TTYfVAMKDchXH1VnBd9uEDMgUEo4gzIDxUDzE6QiqNuELdC8ZNNtbZ6'
    'Pnkx0U73iARtec7xcSpSWMm7FxvIMr0e3EpfCOUprHmjqrbte8NEyxOZWNngUMv8yzFlSX6yQWUlpWqXMk6DBpSItIwd/hpGIi2v'
    'YRqSNyXcRtyhWsFa6ZfUhug8Wc+eg+SouUWDqIBdWtJlRZNJpfqjrOVyg2yiYk4EZY3RFq6kb/A3x9mJj9yDmsGh08tlBbUreIlO'
    'sWuQDQ5WCltqNJ0eb0R+SKZTmzARsUU4gdCTV72eXYOA5T6XiT45L2RjzXmfe/QmAB6TUn+XLRTTYpf43/2lkZBWki30xvX5CilT'
    'ds+DX909nN782eHNn98++unI1x8bCbP5Al6PRDp13LxrUqdQy+KSiGKMQZnER44SSuKimEWNO0fHkeqdrgO/TMr02HCvVWtLylLr'
    '/8Jrl84zJXT3nA7iaV48y5LFiavmHn0sPFuWqpibMD6KfoYtbEMG4F76QviN3e2jAEoYZO+K3jgDQoBxYK2rnStXpq/Ps9Vx7/6D'
    'O3f/6s69W/DdS+MvhM0m6wEWmLuLhCH/SN/5b2x/Kv0rfD2mRZOgL7G0mem/wEnQv3ZZdFb7ywtpFFfy/2h2k2ssaq4Ep9hEIPOM'
    '/yiMQ4aEltIRPyvHF72FFjXg2/RVaD/SlyRm4IjI8krYO8D/+T8AteBu2lDAHFS/j3v4Z8b69DF5EQ4a+bdSvkTGgH8qOFfpYJVe'
    'n8YqHKiCfxrjCrgitqfstEz9vZmajWlSkCtLIC6AGbEqdKH/VtysXCY5jXOelf68FMZZKPRDT/EfcDXSn0i0skvhy5urQ8+wjvT/'
    'kQvEfxOmHRHOOEFiDw9U+f+ip1Ds3AKItAfKDlAjkfIkD2M61lZp0VmHZS5EmqFIE1capKu28PVG/gcG1AO9kFg2TMiX38CYF/qv'
    '9OUMhdmXRKw8j6PzIlx1Lnjc4xWjATOGzzXnNLJ8Ux0eaWeBU9hXfiOtgFeNK5FU1gOYld6JHiQs7c3hYjXqI52W+FkSFSvmORlh'
    'CtPYKCFxLxBwKAUo5zs47U5KrrohHhmBUDS6+HoOcedc2lnIXp+hWrvCEjgGa2p2skzKEzLn3qe37z84mB4d/vTeQWM7vlS7N4qB'
    'zKVvcnD2WUekYBkRfD0WjTcWHROfAfya8R6heCEr/p7CnNRXo0EfJ3MusiV+SXNADAknwhfovUCLmdt3OLlO2EA8X28YQ46Ar77w'
    'R0B5SgO9VzKrvZ943mxulpsy92ktBlH+mBgP03BLE2nMSNFTkFo5653VosOU5p6JLv6DF3IgnCBjJMWAVORgW8sCoR4iwvrCophR'
    'HBsD0a1jkbamihF1vo87VDeHQ3Y63wjJy/JUztZyLTCvYgI8MuQKbovcDuo4gKJyElfdaG3J8RZ7Xhf5UvbWm/dyh6lA+iyHsO9f'
    'HQ8AWNdmRdUSp3yq8sMMx1I9Kt01szItiZGllSbZtG7yLpe/2tz8EFlZPUKwqb/u5xO/tW7vIp17iSHkeGXNu+vueTFZkxb4reSJ'
    'JuflSDE8QxHU1knFKVGLsg3SMt0wXAv0OytZivY14aR1hip/+3swCDAsJ0CCJ/z4TYj32nh8fRRQ6sLL/nO3VZRucS0onOo3khoa'
    '/c2Oslu/K9lPse/jF0XQjtGWr7z6TiCp1DNVurl3xzBOVidijvm0EQ+EcGE0F99Y2vs7tRZEi6HpD67A0c3bDw7uTXjcKJP0ShuS'
    'uhYmH4VNOZAknCGqZlDdkXGJ/MezLqe2zEcptyPyOWkPpOdku0oPLwaOsRl1eJzZLCzCyeqHUlzfz9uOzUetpN5yMtONHXfz4hsc'
    'TThZTlEGuvQapCbtZ0o6WspYVIi4bVf1EU+kzhb0Czeh8ONpVkyrpL+Vz65t/wiWz+9/0P0w8Q6KSx/d8aTC9WTJFlfoi1Ilckvg'
    'p6WW93xdOD9t/HxKFum57F7n5D3AM14PdCqKyFNWK6qhAe6cuGDXdI4jiG2TM5eG4hb0/9x7DOq2Hk3qUbSC0LQZB9Z/3L8SvZyP'
    'eefYU+ar7dAzdw/Uj22HRLvRKBrEZymcBS/DB18Nt8Nw+r/O++MnRZoPtJEhIzeufHf/SWsXf6dVMcw29R23P31w5+eHR9N7hyhJ'
    'kowhQ6XIGu8/iq+8uHblLy7+68V/f/xef9hQgOQoXsbnouzMN5me0iLjQbytzzyT8lTzS17McRgHR3nvnbAF1RARLic6+wvdyTSv'
    'RXoZQrW6Amt7gPwE6U4kF8BESnv3lQcaVxNh6tvf/0LZ18XX1Xg8/nFfDoG1gnz08UvfxZuyz1Exy9ITob8yjWl4kybncaUqJg5A'
    '1R82+Vw43NGd6eHRg9u3Dm4dbit4ZJp65l98qWI3hGkgHJhMTc6lkrWCB6iAjfudGsY/8nXttTL9qmZN/uGvlf2sdj3l1yqnk5fr'
    'oXIuk7WJDBCBR6QoUPiX2Km0aahKUVSqV1Dwz1WLoUQ/i58UpvepFqYwslCQRLmTKf6vcqBtkOAYxQ6EMz7WPxWA9+ZJnGMm+QG5'
    '7fn64ktk1PAjc2tcSwUUEv0Q+09QVeTFUEdlQGl+ksGWib9QO49hi8QFfqDBWDSBzQkMnj37mNCulSxl2vg3lhGLApGlbFA/yLSe'
    'AtbiibDFHO3xA2FfEdEp/8Q96XaMMyfNFxe/i04L0UTOhT6XmAM5aZt6kSOTjg3269ng0cGVf3Xxry/+9uLvLv7+4j9d/LuL//wY'
    '2+7iy4uvLr6++Obid9h9sgkfvzf89Qx7sAeWX4uj5Pw1FIkQIOWRZhjWkQWuMORIgxZofDIhexTmOlQW0nRXTX5CTFVanonohk3q'
    '1QTlRAobOyuyiiLEyjEHC4GWpaF1j+3df/iXhw/utMT7sd+i6SItlrFWQ5hlDJX+eANvqOzZyoHRmkYfO0Bas716hVMb4fgOahzY'
    'LLq/KT1EseVxSft9vb0fqbzR54udPbQ2GyehImsW1dhe+rGMNs3cbs3Qee2ZG6CVFNGhRH/4m99yU6k8fwyROXlBQYx2bezX4zqW'
    'TeMuTHjV98mN5GVqf0LcCmwky0RdCQvvUdb0expYV4nwtBIp+OQIzitgOEs0vpi46KdIZVCb8thXo5+0rqoA6A9do5vD5+lJWtqU'
    'VbsVu9gdCqFWF9GEpw4GbV1G+wLSscPCRvS81Za3yYoGm9yrWoa0YsdMqVRZCdu2WDRyRETLQW/ifG4hFRDvuGhiPXm1ptwQnlV2'
    'cbWZyQ7+3x99+/sff/vln/+6khNSDhz5f+7IsWzCdDUwP/f2Y7++j9utDMByEi3Hskab1eDacJwVz5ISwcB8CoqNzJfLkLNkc2h7'
    'wl16ToZ6BhkqYDRj0dYXMrAB2wj81qyI5U2ag2fDhlDHKI7cRXE0Tjj8IlM0yIfEOmw6whWSJ14sBgZY4DRY/mA8a17MWIo1DpTp'
    'hpo4atgyAmQlBmnwH6/LtxjcTT0YX0duLRuCrm2N+YPIY6hmC+hB3KNwXWVm7lV7wrBJENC3Ufdka0Q6DyYp465JW3C8NsL/TBjW'
    '07+xNtpFUtPUMGIKR1DaF8jSiWY5Yr3YF3lj4haQjT4OFwQriNuGCKOQzzamUf3DyNq2RdMCNO1Txi+DX4W7Net3J0nMajkQH9bF'
    '6llRonoGranwDT4PYlzc7gjGzvc1SOnRmgPluLzA2iJzDH7tBl9b3x3prkm6j21ks5TmuDcYmzsOlaj8IYeVTuabdXpGLGKRK1Fg'
    'V2tqUnBlyCCMx5X6uhyDpJX7TJbv5p2jenIAka5H8+fOgcGshmq6SD5X7R7G95F6hMkNTQMUARpW0+hATuuv9bTOzW7NDKhSgbLU'
    'uGoS3IbAQ0qqPIDq83RkxNifxyvIRObohpmiXw+GW8Lf4n7/SMSqzHdCOS7zW0zp1+lRCw+5uHtZpqZ2Gq5RArlShQDWf2Q6cHy3'
    'LBDcPOdJ5QalixG58NwAwqYyQxBDylns1NC3XdyBbb4K8NEQcCsGEULSkFFtHAiNIwLu9XBLNEn0uP8yfjV9OXvVJ63KgToDJb6Q'
    'g2ANjlI9uj55PHz8J1AQb4H6vLVpTTv8WbJmdOThUfTpQXT33p27t/Hn8KcPjx4ciLYkjx1alWn1aULPz7ViReB8pgOqTKGCFXTy'
    'lnTjmNPae8B77wRRFA64bZE8ofgQn2yeyMFrwbcBV0YyoFneA+GCuqHfZ2xBpQNg+icwox+LHjFSMa2O3Fg4Rt66Bx3tvVP713Vi'
]


In [ ]:
# PAQUETE (9/12)
PAQUETE += [
    'XE0CNWA6CxvsPVrfzdX/VnlPhIV+XThBWusvEl+kG16NgtgRMDvh2NBOiwi6gBOBU1ob3qr0hIbiRJ3N0thC9yhkwJEoS1qvwxZm'
    'Hsu5ti4vvjlmmToKMOqsqVtMRKk8K2B/m+hAUfEh1aJim0o4QFo4RztXLQnFwjzF7VUiugtci05PTUulkXuEeWR5ddwjiyityBRS'
    'RjT8O5PAY09eI3NbYn3TMjat9DQ+771j4Q2wKsu8FGHcB3Sq3Ox5aQ7xuOTd3jiFsBB0Ud1I4+94G90/vPh3B58e3p/+UhSVSZDv'
    'p1IIP6KAFIyWL834eRMWAZ07J6HePfj04ON7B8LdP7v76eGDg0g9/Z/cO7h/SDe3LAcw5o2+x9bSEcwKKuJuZoqsJZ8SCkd9OWP6'
    'aL/+CaQK4i8r/AbXlNyRuWgMwkxHVuE+1XJ1QgSg+dggCbELiAaCKs1aI062hYpNld+h1r3aPzaJ6gSvvrCXlJ6u5Qn/TTMequor'
    'w/6Zy0uWBT1m6Wwj6udxkqvPq1qlJf1tdWtCMulMeJhzvOn3wn1lF6gatb/a64TrZ1mjRXeNNxyL1g5HXuDTw0dwnDVnUCZGGlvq'
    '59Q+1o25n6vg98r7Cy++8Q7Dlv9QThbnkAybc8zBDUU/8WLsL5pHMsmPk1IuqIew/pZUYYu02/IQUOejlTDAFzlUN9qUfmK/k+VM'
    'luhFodQjwots2bC9WTK7+Ao/ygdgX/SFMyxlPYTdrtJYV+jiSzkqUk0LiPOTMtGuRwuh3EZjapYS5Qyu7GVc0gVq1q3WZfMmiKZg'
    'DZjuZt7XJgXCiLPQ52Rf4HjXL082MAqyk7MCaQL1x1jdr5D0m0uCtC9hl5nekMX+c+r9vsGHc/vTXAS5Ia2y2C+D/64LIVrhTOY/'
    '0y4mLzhdEMJIVHFuzwVUF+fu2VPLU2M74Pt+LyyE+5c23PlGTqWF3mWz05pGc1g35jFd8ihhC+4ziQShF0muc5aWq2P+VY4Tb9bS'
    '2WLZmEO5WOg1zPEa0rz2BPiALYdYvrZeLqwaSNTwpWtn4R0vY920/FhtDcc87s0Bwfe+TsvAD+83RuCV918SjnYuSrvQoXxdNPZ+'
    'eJ03ztI1gkHVfT/3z5+mK9kfuoFX5cVXWKTG9CDTWK+B88gEWEUGYxSIF2WbqwJCB0ctYonqBeEwAj6SUGaCKOPQREvbXvoxqjsY'
    'zts7pi9oXI4WaPQSdB1xV0WD1fn6tBBJ/Ul8FiObuFrLKThkkKlwBOlmkVuLZj+CGwDRLScb+KD63/5ereAqtWA3qAZhBVDlf38p'
    'Lf+4bweoayuQEWuBrmUig5v0a+bTM+53AdGU4l0M2cIdhfm2+bMeXg09HKoQcsfmBPrIiB2uzEb4jgvY0xhbU/p81WBZn5kWiUVA'
    'eM4wCVm2BBMt8pnLBGvGgjTIdn7xjQsLqQNE3O1lM47E6LkZWBI0RTNz7CgB+AK62wg9rB9nG5Jblsqo4eWov6TNozk7Kcp0rXTr'
    'v1R2ftZCiPuih6xTtpJGv1KYMrB82gk5S8j7uMX0hJKXI/RZc/e42zbzjIaO1tYPImcas3hakDnDKI+/hrlAQAOLrYn6m45rJvqe'
    'yLn2TfPwgs9GLnNcrgCwbvklAFW1mysiMmoeMnvW2vkoQuR+cNzRc0Trs2oIdE0EPdJg+UajMgevzJThbA9TiOVte0YjTKFp0/CW'
    'L6cXqGak4Sn1BpzHiRlFEbsCd73Leo711bbK6n8y8fc+Q7K0qpzJrAqw0ky5dtYIVezTDIF3F78ToY1GCRX8Fcg2bjOSwDzxiZof'
    'hIUoB9KoEQ2qNOFc7X+sCeADsQsWvlM5XMs/aUB/h5Suxlf1HpSbZObN1pzt0H+ieqX3sVTwYFkrLvLDPCLy7clGk+FrbVdF/qpp'
    'd3CAbZ0GMv+oGoW3jWMElLL4JtrQaOvXeWGoftTQfMap3NConesR7twzLfPvdLt/24mCffk/NMGbJ51QJNpOqol8k3aYD9dmUpOT'
    '6cJNk5xCaXcnVZuZhuaAZpGbxstda5Pm6+3ypJkEosWReq97lX3XUBcfjGVJU5NdGfczDngrXnl7cJpzMG1H/ml64t7uBxqBVbz5'
    'g903W1hgePv7XbcTkBrciEakKY1IFvi1HxGSpQMVgE6j2jHwuvu9PXHqjSBv/MyiKPexEB3oWZp7q9l8zNZr/NCeZbmpfanzAY2v'
    '2W9MeueNLvByvzXlzZu75pe+pe3LYV3R2i3nnW3mN0wcq3Q+tLaPNswmqotYCeGzrkKjoME5plmtPc40x+gJjU6jzUqoXk+MsP4q'
    't7K2uIgDHBARFgxFWc1wnX5X53RtCo4N1+u4OYdteoMLp3UpaPGX3no9d4jrOyP5asMX6rOag3wj7GrcLPJq6QU0V8tpCMu8rxo4'
    'qV0An0c5CqgCc+Dz2htwTr9t0KBLqqtv/mVN7Z8HNsVNbilIlVq9qWo368XKOE9YQlimG4ck6MV6i2KGm6RcWAnWLG5Na9e2lKnt'
    'uhz6HYOd6eDB6LTp3OVNhM/L8kQWRbP+W/ia/fCuwbB7GHpj8K2Fg2wBasAVAXQQOEarpemimE/VYWBBpbUHsxkJV98uetOb3Xs8'
    '3eTpxOX2AIBNPw2G23fO3uhGK82Z4pf6UKqv+kDMJtuEfHDqo/hc8GGrsuCnDYlSBOyc8SNgCotCxP2JiXXHrkaxRztPkJJmj5bN'
    'ioMyAbVH5nTs7kJW+F70Xv2Dp59hfVUEEo3bO9UIv6qVlD5D095LGDb+Xni5brojObxefhdbKD3edaM06u6add/EFR97KMg1W9tx'
    '56xx4yxtF+2a5sQheT6AazRcuOEouh4uf7o4dqWTjj0RjaJ1rWJsYx808qRC1zuSaMB65hvD/lik0PxQ/iZXJ2lHIDxRBLLixCBS'
    'prlDNRgsjglctx5F14a8yH/H1+rumwvURtCIN6jdvXWkdz2KL6YN4mr69fUGUojOrpGDv2vYDpZlc50hso8CPOMvpoz2CLYonwtf'
    'arJ2E0FmMHfIGX65HLnISsmsAE/mdUVVqBuMNB7XOuJ0AWlBfvQTa2KeBkhth8tvx/hKL041REHWvXRApp7iJq06zO1JaG+mR2mr'
    'EuLiTzgt7alZvHZqzDVEjI/I0shVbulW83xwzmapML/st3tPTeCXjeWyoVy2sK1aOVTjpBuyt75wuMALQwZv4tBoElBzqZ0iJ838'
    'KLrWgZWnD9Vx1l7XeinPTMY3jl/1t8rvfuzT7y2sgirXxKwUPoiB8KGQZkQQFxLLaCBsNdUV0jCyPEXEljrvNBT8cXtU2O9d5dM9'
    'F2hw8i0SVVziVKHZ95V7/G98vEsXdndOOpdUEQmggd0A5go4uHsAbBHf0X53GSVdZI/Hgq+GcdB9/9aiqSL+3ks+07loRgM/3O8i'
    'Ao/z2sSQ2cwYczWqY5suCxhr1JKwV8rTFBulqTcgvVrXfCkPvLr6Uh5v5w3QMiDX36q5a9pUZMFV0cvvjaLvqaRRId184aKuhkwn'
    'aKLqm6qI2Xs3Oh23FcAm43Vv95xXY8b2T0d1pO6+fBppu/v8981rndoY9+3vqJXDt48le+PG6sS0fa5ykKK37+aj63AZY84GT5Pz'
    'fQuWnE+iuebPoIQxPCJJC9nV4aP6NnbVjnv7/wL4Wp8j1yVagJRX06cuAecGRY1d2XcmM328qebwYxisZbYDayOUj6hJMhbjDLEX'
    'tz87vHcnOm8EZlhOfBZGe0MZ17AMF4jRq3UkKs77LZuyl2t8BlOnFYKApj4Pzmt3aLJT9NmekWCDfxJnSDxkfXAtHP7ojQv1NnP1'
    '9o/7sMRNrScv9W+DayH1qPJY0yYz+lE/mnA5A0HNpDo+9v/q0BCJMPXWh3BIGvm2r31slE15JxoEwCQ07Yf2/x97qmK2u1PazmNv'
    'AC9mF98ErcVlmc7ijyyY3OqDzk8BC8GTtUwWcknRDOiEu/im0gRuzYBtVOSdA4lDXZ2FQ/qpO4H8+jQv5lowGEYjIfA6CEy6nZ/E'
    'DTuVH9zC4IZyi1zy6egou3VMDBwoKGXBEnnCy0ctw0hs/tIIYRPrQEC4bE+0iCNOX+DwxdK4pDW5NEUHBz70e6vAPJ6SPRvksE92'
    'mZzfmO7Yh5FSx1vUv37N1kJXW+WDB9swPSLjHqdzYgP9uL6zydbu3Lt1eORsVUZMDq8kDqNgkQclAzYf1PriK0C/TJpEpZ12JoUy'
    'WVmOwSxhYhkj2gA+QojP+yZYktC9g66JBWpmwzNn7zOIAulYI77RB8bViE3R0cGtgyjJmxWtaXSO5/BfLzgzQst9pKwWTWwEn57L'
    'qKKIQYhhgME7DtlYw7rkNFkkWhIXNshIO8JAO14N0IgAjhXdFVWcOWON4VbEWJgzPxkjQI8+0rPFQUrhUCHoB3urKTzaeG2ObWwY'
    '3QK1Lf2H3Tb2P5q3avsmJ/wxtP7GfFYovIUaRTPWpkndfsi6652p/4ddDoD/dQZtHZqipEZrzAuOOYt+dnjzZ3caK15nEp/GwfLv'
    '2OmK/oIt6E4MUqZtdKtknmmSI/y2gFyyKgruVSGPvviv8UJ2rM9lWsereGdGVABwxjiVL+pMtF6IM9uRk+bOnDo4+bVpaEGT9zel'
    'ZVe2E+riTLnbJelmRJoKGrM8slEIkeaTx0b2KFMAkDnmY4ULplX5qOtwEnenfIXpXl3Leb8ZOqAaeTv+5rx2QeXeCxVHR7ejh0d3'
    'Ql4W3uYZXoOncJ0tLbCmO0st2z6ht/w+0Bd11wRAG1u2CwVkmqq0si8PnQ/m4xYwSJ3Lzh037Cquom9qPbh9lL8T3WIKQ52qYEvA'
    'aKk1cfPODdQQvlJhvUe3j3568Z+OOprySfX1RGfe1TfyZ0/lw3ISDw7f0RozoI2iKsY6GOQyNQ4AecHVBIlKvoUnWEdbl59p4ABU'
    'hLiLSUdoR1MLOlrjgSeHNzUe9QvC6YXu1bgCHodFg9BtFjpaG2icvQMSyYthEy/f4U0g1xCRbhf/PWE1gG6GKjTjDARTpSOq0H1a'
    'ekLaogLVOmr+MZJex4HwFqKeHh3tQ0G/hjVBIU7sh+8CssWWB7y3NSd09GFrogEdOu+cpnktIPYbp04yRMWmwMXaTITw3tZRI1jH'
    'AYYGnMWLosQeUASTFsClqjVOlb5/W5tUhkxlp2goGrzrFw8v/jZASyn8AwquaNmphnBzbnCYcuiouNBozyq/32f1DKDuKP66Jjcx'
    'uQ5dp7blwCMhPjrimrioKm1P+ShxKwvrhmYAN5i6OzbixlniS3adh6dimnvXqDIqjm+GPVsa+FHC89ph8lW+yAdAJIvmgsBJlHpg'
    '1TI9sawtlw/CzoZ2ijKE3HE8HoTkKyS83n2srbR9vvWDg+39HGpurlVR3w0lpHUIPG6YtF6/a9H/f6QU17lZR4ETe18+/ikwSlB6'
    'epN5dSnTTCow4a/6mkmEADuUodWUDFZSo0+QIZPgjePVOaMGqWiLMkIqdZncQoxlsUgURhLAHCoOWmihpr09Xxc1gJEdjhAshSC/'
    'nK81HdxjzC6S4zS3yEZgrPwuB+xvXM5PwZ3OFXaPKg6lRiAMJgQS1SgQF3Jhx8LGxJu4Qu6aH+UIKVjyZoxAKKDQvEPZE3K6w1pR'
    'Vr6sAlNwqiW7x9MlJxgXhsqblzGweAIO8F3nPzGYwi+Gi6aoEZxH0YPDzxDYiSjBkUl1NF1YHOqxkkA5rc2VA2ySSQfVNyGJgOXk'
    'k/IJB8sjQpT5OVdloal+IGldMJdja/UHGyt8qxAG5u2k/rRBT8bN7QHPXfu0dXnbeueYuX9WgCWwdIRY8LQu7fsHCPzRn/ThRn90'
    '/TFjHyeMfbSu6EkHGafNHBh+yQCI9QCt7nPC1QPIoE386+zknsjk3QHMNm52u76+/Kg1THNOrZjnuR+u5UDnKeBzfCntUP4XG0M/'
    'DG70/XEjYOP7+gpMdHx68bVHMPGQZjsphPrMlAdHI7Rvi3JcLHJ9ihGrcORURVJQlaKCgCYb1/ZxheGpND5LYUpSAz8TRkwU8bnu'
    'eb3i9bFz1TRVJFcAH6Yry31gskxHuPjKJUCrTK2nLoQNnmXQvsAKjDvM42UKROXmeRyesqVTde2UjdaG6OnSbeuwuaSBtI4CeArU'
    'A7nHB9G4qesMosJ6xNmCThYskyLgBroMiYEHroJsO8rYqu9o5LGLORg1axz1frRFfU24OT2xeVN9pPXqODIfP0ahfR2QUG8rNlde'
    '6DxDqrY2xRmQgYO2o2p6YDyntnI45cnBgQuxjHzOSSAP+cqisG2XsZxPf/ib31YyEXyda4rQeXgQmFP0J6WmN72jJwPC703JPY1D'
    'q7fXk2vlHlAzCltrIpk5guoEmxr5j7InziBCPFWKj6VudlUIGefO49ZeTwTZpEFNOv/jfCra6H7D8WsTPjjuHzmJQI0fAMfjuR89'
    'oJ5Its5J0GPSTEpMs+ZYtnWhY0gZNlUkC6FZxl0SEAAARLmzxU2il+gg+KDdWAyGj649tjPh1dgZuDzGSk35AxWadOJ+5cFojQw8'
    'dnUD9cijMC4Sn8pgKGd5EqzDLJUF8vlNKFaVKsuhKdVnrbvBYUvCznqWCpEc3Lt3++MDnMW9oEwVaQKoDGCDKHAQc3tHxBtGkgqS'
    'b2SEmn0QezkfXhdaVeoULOoEZbJILU8NzeptxTjkBpjZDMG6gfGSQIUMAH0UKyBCpNystQbUgsfuO3awYz7jozt1cAkB+VCi5rj/'
    '6/zX+Z0nQOdDqIO9Z/gKM80reIcDmYB4JT9ZZkfeRURCRVp5p57kuAor/6kZQ5Un7GXeKxrcPHbYE3/4z/+hu2FTIs+dIoO9VWls'
    'cFWrq9U4iDzw9McRjzODEoreizjwPsoxKbBOF9ROijSCXDbudIp92J9Ol7FM99Tqelp0ezupYKxRv+Z7Ao/16QlBXCtKcKBMrMMj'
    'dJGhC2f95Ozz/lfKkSrtb3+/H70bff+aE12olZMyag/4t7+/+ZrsnjC5rf45FmWyBHRb+HuQKgTR8eJrs/3KAXRi5SDhpfzx9jMK'
    'pAdTaVKZHO4R6Fq3U3NnDnxkQHiW7078BaDCYWvlqTyKG25AZiiqrUbqnLPI+pVaiNLF79azgh/3ru192O4sZRdvtiP+9cWXJR4o'
    'iyfFFbgfKS3fkwn68Y7Jac70z+MX8ZO0UhN344Gl4/m++4/rhKIV02Z0TUO2X63ARMd1pERQN4q7B3axP/znv4/uH3x8GEV9L2MG'
    'BiFsrKM7vKMflPBWOvx1/pLtvIoeveSDxIud/OgHrx5H0ctVGM7jHmnF2uzrc5oIY6FMoefV9wZWhVo2adfOarftT5wtcSZdvOpi'
    'FMIpBs4ntP2Q/fJqFAYTBXdZhM6wgU/a3TORnV/ulrEeTa5/cO3xq/E45EddxcLazXpU5JcdatarN+rVTgERj//ZP//3v/B/mrxb'
    'XZ0LOT2dYhN+94XALq//tXfj/WvXW/W/9q59eO2f63/9U9X/utmIl8g3teOzrrBauSJgidU9BNl0+HvrOmCiGjTNcKpEw5tL70vv'
    'WKsiG1pjFQ3gfnQFWUXmgud6VCfqVgBnkmNKRLmhmQrl2JReP30Wl4vekjA+Wk2r8SY5Ru/f/vTw6ObtO0RC9a9A+1F6srGA45GK'
    '+KwEK93tIT4CQmhUWBGTTW6miFUCuy7lwrXDiqscuHDv0xiWOGbCqWcOFbRQyzwba9q1yPapzx2HlmKJc5b5cHx18Sz6w9/+2+hR'
    'dDx4JhLY6TD6w7/9P/kFf+UrCoIO9k5FJDtw2WErxtvOWDeERk4U0iboaAGkB5qjFyl/Vfxi+L7icD7oui56CrKq7RQewYkyY46D'
    'piz8nKtF1eXUcqncT6r59KC2jNUwM6LFxHmnEeoYa9GAhraae9D6iy+XIsaKqtJ7WLE3QOXSUlPLNFfs6aIRlDbRcuVT3rV/Y0/e'
    'sQcRidL6Au/qucigMnpw59ad+w67zt4FxAithqC1y134wvsoXaLR37RLlcUGeTQ9nxRjgXMVh6oluGAHH+lnzTZXyIwJE8yX8Tqx'
    'QldVUCQu72H/yFxA2JSB3yxT4rdRk+aslUkW0xA5QIUfV3JU5b9yFa9Pz6WHi0QN4E5SNnuntrAf/SYW1TIFxBHJiWkd8uU3VjC0'
    '/vk3o+g3/lcVA36ISsQfKBjrsQe45dUbDJJPVgw44uUf4fKefPjZwa+ih0fRxw9/+o+pkVWdV+7jOl1awSyMOUtnro27uN/dJX1f'
    'ATQryle93r2D2/9KBo8bBvLuNJM3DyE9FdlZMhgSrzFf25+evGuMpsdpXsnCDxBsvy4HaAQ2XKvVlR+nvsQWS6vf1EtRZ5mMvPgi'
    'nkSH71/b0+dJpo3HgRgrVKhVyQvp+67nUbBHqcQl/NruJ1uBC38J3G7652X34j66HGNgf2iUnTlZ6Vt9J4B1+Ajc2h42lBd9VPbN'
    '89SwS1xlXIIEEnEhS4jphjpqTCn78H2356JNRezI6BTo3ORFuouUHkGsJTinq/tTRRdfyywuiQo/OLT2ZDefqm+SEefRqTBIUJcP'
    'FgoZu1pdiKxONFN6kuT4WAOwbynNp0sHpomd7l4govo8ZvGu6EP0468/SK58f2Re0NN9ed37W3MNzL7UUAdjY6oalBhnIqMTe3TO'
    'QNbZ5oTHo+eQdRjIeNj7Gapwy9bq3UIVNpj1V2ObSIfBe3yC2rmyBINlkWyb1QMCbJrXlbRkieXpjTPHYUjCAg/uK2Qca2paWSwh'
    'BGX+xio9Ygj8V8Kmze1lZpcEKFkwpdw6/PT2x4f3Dm4dfHZ49OAw8HbrTMtBZWgJnkGPouPjfHqaLoT17e8hsLVIpsGl63LH07Op'
    'vHX/+ofkrSguJko9D+3EnKqBEGDx8mSwwrUSYoPmPp15oRTkM1W9x5zB0htLuheebMg05KVnYBB8Tg4dRFhagGZFA7/GX0JDjmmz'
    'ZO0SEDZsTcp2m8ZYkIFimcv8fRM53OLSeU4VS5Aaudbl9GPRzm3iBVGO1RDCso3+wAhX3WxLAVXUHnGYmWfTKn2R7L9/fdRYkHya'
    'AXqn2t/TaiRV8sU0w1J8WNsUZH2SeFHtv4/bZX30mzywzfOELnUBQd4fjjy+K20qdc5biwx2//dOowXreBCdskU+lXCNZDFtveAd'
    'rGwNNevoMxihHvgcI/MH9uu23E/X6xnZVIl0JdmX/5cxCusWAWEfodoK92M7WB4DexosJ57dp4tqgq2OU7yMzzHn1dPwSjtJV5aX'
    'YVJCwCB4TQiZx6K9CQkFBWaB+bKGzSfx3lSRZETAmcLdObbHBymDhOSdDTRc3OmRv40LvFWv29EOoCTZY1MKli6z5kMYw5fwOvoK'
    'YHVaL3E7d2EePRqo03OEo4KemwhCC+DEgLPqFBjU5MVWnDqpafjYeNkti8SrNZeJUUGDuQPWofHd1cIjXpkWQXSx0YesuTFHoIAJ'
    'whlqWRRcIExoFG+ep1lKaS2TqTw0TCg5oShkzlJuapz8lqjk2RoQ5T0MhKYyEOpW9j2rbgU3mrCpqFt+ItjggOpN7EhnODLYar5Y'
    '/fEnCUymYU3fBuqy6ipLA2fzeqLyNqR3lInpUnVERKIMGecsPMMbM+u3mFV+okceBI5iOXZV3uT6wGhECRR9nVpHF0zGfCmEdzae'
    'i7Qkg4dF8+koOqPHfrxymbIwBz4di4RtwQC2NlO01h++6r3p9liO64n2FlT3vlbXPBRTzbrokgHUyiRcMNa+LM7SIC1Huv7o6eNH'
    'AKwFiEWv18yrqxN3jwtfWUdHjP2l4mOSQw4PbLrPOLLVI72/9s2sWJJlP3omk3+WZNKjXaz3TCvzuCocolAuWW4Hh9Qz3x4sBnzV'
    'iXuVa7hRyYCvHeMMapYx2N9KPPVJp42s9MVzDFkIZDw/LYQFDHLwQjnRkB3b4DfCEIZwhcsb58k+oytbWetobB88SBb8edPeKQM8'
    'YcFm6/EjueNxM/Gzvi43+fvfi37WvGu6jF9HX5e3eGW7xSQv/lFt9to57NT14Khjd+WV9hIiNuxF70Y/a7Zaq5L7ekoNsPx8TSvL'
    'X1pZGmpFPKsG/kGwNvnu3g0Ei+TK9b1WMq+pro0npXf+Keke39DMcwdOoPKVTU4anLJ+AVZ5JPQu5/kqGb4mQdcfNWxtVI945N9u'
    'x4wL7unIYeVRCl/eoAXBouZ0c7G9H3rq+sCgvnfn44Obty/+/ig6evjZxd/eu33zQOTs6KciZd+GkH2/P+xsxjOHZC0aV4JdK6IJ'
    'NsyjQR9Hz0BNGEKl3rLxntvYsH4M+yPVLC6LWxz0P/nkiCXoga2oGytA4ag9Pci+fem68gqBb4FXQfQaZDo2tJvA1yS/qdgxkFu8'
    'xLAvetlifb5K9qkvhbGalxwlN+p2P5bxIbtXyK0WQ1PmkYKnQHg6ETEYir+8d1xLzMZlBmhgOBzHFXoxkPfJIx++HzrJqqfaERx3'
    'A94vFHtdCNypdjf2mrc/Eln8w8ljggNc62bBGmZhMRIi1ohoDtj5wAjW63CbBRauSfRyOc6VMeJ0Gr2KfiNHkHDzlx+/ev7ywStV'
    'OS6+rOasQhGXcyBkBhO8vgZkFVEEx/hnEOZuV8z89wIjBcU2L1qsm4/LnKyvbSXnlu3cCqPwKPrzP7f4HJ/NSrG35abaBirgFmUP'
    'H5WP3n/M3VHyCEmq4DBM7CZL9yeKbJDBLkf3FTw+fDT54HHXVNOLXiYa0aCW3VqVXmhttpeL9WR8/fhV1enjbBn6aLZFabOXORx/'
    '+DywsQwn473kVbTlkzzuy/o9h43jJdht4+5wJf2cQkD7wMY+CTpVixc1+4PON9KiNfZAk+c63/Cdn6tTOFHr4J76hAf9v47q6x/4'
    '6/3PDj69xNPoXMUvtTevXrI3r6JA7N9/GU/eG3+Y7HLRell4/2U+tTt1rvdfJltTg7Ic1YmIZqybLfS6MrRKb5b2EMrR/Tuf3mGO'
    'Vpkcp0+0CG+8CgPkB/3s2rg/1PR+1p4pz2KrVLbWkAYdGPOBmOWqlbQ0dD+LQ0S7XL0mK9bIRcG+eVxDmqpB4KTcrHyUG+/XeCLa'
    '3YKm0hx2kKCABGISsxURfINcs0C0k3arsLwCdTUyMy2t8KqTdqbuf4ltuJaAl8nZ7CNZt6p/4tdGLK+wIwULwS+Prj8ep9VC5Jn1'
    'YNiBWGJtWysa4Dt2Ab69NnfVoL6MXgk9F0UxvHY1uX412RNxHId38i97WwA9HFYsoj8OqbG3Hry2PyaUCpWPk5frV+iXfPqX49ZW'
    '0HkfV8naTrMBHwf0wNDJKUmDeg+dk8Irm4Y1jox6YOczaQM+r3P6rxAsV5eg1PI8rZwW4PAiqk701ScJLarrYnXlqUvkRvVf05Jv'
    '3b5/897hg4NxdJB5Bbps5rSsmIKmfjoFN1S0LjWhMi7PVgA5HgraHFR+MUtvo8ET9dDBzGuONvOs1Q7NGIM/Q2VIBRdYGYZvr2Ff'
    'ssODFkCcNT7YMyc4+V1FeqwdQer5o3XxS/C6HwTNaXTv0R2r0zPbnDAvsw42NPO1t4ezvovZ7Acs6fkvhs32Mvao8I43RAmaddnH'
    'MsbW2l9fF0misAcKrQ1Q72cs1J2jB7ePHt5BHdSXfR1P/1XHGdHNBsPTQo+DKWwAiSIcXWsWCybpsyYpz1al6tYe4ZP6yyM+8LiT'
    'H6CZsPddiZhqKFGjR4aiIcxzD+bZvHvq5djOS1xS6+MJWqH82LXxBx332HGH84upB/BC80Qrnr67fRIlz9dEfEWV4nd1/9QEV0y0'
    'XBZi1hgcPOz77FPfbL+FctSO7+noOT0Le7v77o5q9wAP7PfbB3azG9vndWN4/faqLUPEnG6KeU/6+XoJgMs/uf7hqyjfp7AFyeZG'
    'hxRkkpBKT/uh9ARR6DqOf4hJ+y9t2JQBXnIELVFAISZQ00UrJyWs3e5zs6HoLQNQiFVcVeHjoRfXtzMGi1ZKvHf46cGD278kiwAE'
    'CgINfR5s47z2wRge82wOX85Aluv7DtCWnxd1GngtSSaQBJJWGqAB9RbMc2CtzdrSKEN9snlxEjsUogD7/eBjkXoePjhoyyVaPVDj'
    'b+GHm8cOWqAKQkm0hrEoHTUrslsDA4Fa81RuiFWAMOnBo/a16EgeY8AynoZxAFYBa7fRVF439bj70Dx3YQ/zTZKdcAQXX2UsMxFs'
    'VxXC3EEY4OluZP/OWXDu0YKvhhEhfLvM8sKVl6OKUY+gG8jN9p80E5gxjVsPaxZp3f+u3su3Xv7Gd2rX5tbh4ymPpxArJobn0Efc'
    'PGR4jRPY2KIVtNYAi3/hssXc0acICcGc4/q0gMoN3Pw4ejfKwfmugVa6KEJjFnqXgEZ20UvAFxiRr/TqFnzoIiGYx2o96rCHRNhK'
    'x2P0q8g2KufDt471uqJzTIslpw9SS72XtnW9RpBW9FLvI4ObqP7ne+e0nDBIli/bFSFqv7J72pZ1ThXPQbAXGlthtEsHCxKpVTor'
    'W16EYadK/MvDe4e3bt9EZbeX37t55969Q/n8PfSfi8AT6XsIMjm8d+/OvcP73yMDf104vdrjev8c/6nxnw4peiqba7Wpvsso0Mvj'
    'P9+/fuP9D1vxnzfef//6P8d//tPFf3Lt53UCuNIAtTD1JVmqSa/3UyRAxhGC5+ZMEfJxHqOwhlSjLqt3tU/APo2+1s/Xbp8XFiRn'
    'qRpWa1i6cffe4ZVmB6JbzgPKcNAs7mI0CqozcSmiPtnlPKnTj0eNKrh9wypPy472cPQIm+2PfC3aoHznkrEzMkQ9kFjuPKdBhYlZ'
    'nYn/7gYFG6JPujpej5/IxGVeiFTUHQXLuRq8UNMxDdthfhqv6+mKAc6QdM1ImndMHQa22BCQ16f0evwV5syeKzRMR4sOjQ/5FfZI'
    'rkaxEyUQoaJ0Bi3G3MM4/pMnMl1Z0VXiJRf13aBKHRWlqrudJTX2rlogEujZQX3XXhdQrrypTFqoFajTmjBssRnq5HIhe0eFzHAQ'
    'zuwx4OxUJSl8cMPyoyyrTAGwdMOganEdnHreYz48w694jqvH9EVcKlaCzRXL7iBeiNFvqvzNGDasgWya+wfohDPGJflSIKsUeFh5'
    '4soqx4w3U9+9a3oe5jiyPDID6HqKu8S4bANfOCcInIV3pycb1TRWyHvNY2D5Zs3w7reN5QR5u8/qnAkjO3dHcx784uHtN47cVEd7'
    'fPH1i0ZnOd/uqEP45M8M+miZQGAGEA2fUSsquhEdwL9GAFelhzJ6mhfPsmRxwhTP3DJJM2nMguFJrm4pzJKbob4hoPascCLfAir6'
    'XEvzRVeWn2PqGauOH6S5z4rFJkuEFj8pNvnikLI0FO8nKFlu2cDC11wLl5zgn6Om8F29jamrMj4X2qr5YyVQLxVMjc1worBPlJ+V'
    'vuzZeFdMLJbHx8T6KXJL2E5W3BHQek90yalIdncfwhDFFb8qPDcG0NJV1FtzR0Zfb73/yYOu+zwX7X/3ICl39ZhihgH3MuIXoqP0'
    'jFwlQj15RD+lsWc6MOvzZ1ZCA0TwEkegIXHJM0+S8gT6hX2Faj3PihXMFMirfm58irCUWrvQtGztxixFWOJ3DSVy996d+we04fSR'
    'yhDUb7ScQ4P6MayCzcKAmJDQTuAnC0l+kCKbeWQh9mmFio36DWnOKWnLkJCg6QTlRStfqdFpfSzoaAgQxJ5x+HcKO2ZwljV7dYmR'
    'CV6ADFKAuWxy7TKD55aJmrh5ZNXgZVum8IxMeq2vDXowqm9BTV4ZwomGxUL4KRnXf84iddBgZ9S1z00ewWmRad1LWo8ypOryLE7J'
    'OfIWfI2mlOh0GmQNNWND0VmmILA0T7UoB9zjDpaGpiQCGNRgGogDwZbXGFTA8S5Z1QMDQ4LGmWWCYJ4c1q4eg+yVIglC8pvY6RSr'
    'aQhFM0EWOGmVodwvMuyPCEDqDK1WtuWjyyCwnGzi6DSFO2yEOGrcmiVPNPo4uNVHp8l6HbAYFUoAPFFPjZZYR+Fa0ifD4vLoBXbL'
    'aTxLmY+A5KXMj72Gn02qYwgrrjKpJYhHWXoil5dWDlJkLFeRisRcZJoIkqPA76n7CZZlROciNchEmJO0dIh7sKmpc+7cIEZLOReA'
    'hSzkeSzkDUly84STuUTSiKYcMd5ZvRlB4UXvIiBmrZICK9IC+ghy49M0E1KzlBaFwyOQSMN54Zwfrd1LkUC2zpN4oa7JuS+66He1'
    'zSC5gcEzH8K0Wa0ZkMqa5Lh48zRdzgoUqgZqfHSo89Mg6I45Uho7S15oItNZIrpDatR+Kq1Vjj7ldBf5lZS0AIM6V3/n/BRkYkln'
    'bD/OymSRLIrSEWbzDdgnTv6h6xU1vRjoNkvhdXIGUCQHyQXPqqoNDuWR7wdYFyRSC3GkifdJYvvEHl4UumQ6qjUKDdN+Q5Ji1Xcn'
    'XbTeRwjimeJ7+HdMdADKH8G79HfnCFY+45KBLKvofpogU+/iqzmLqJINVtt0QPHJNgyrAsteZWXeqJhffMU8CVxWq3LlhFVlXmTc'
    '2FCLgPPSY13IibF0YKuLFNyGEniuSFkTEfPmsiEYVXWwzpDTz88KW4CLChkg77v41yV+4zT7pyKWBc5d6orHjDT7rcORDcZJkIjC'
    'ZXypPUttacTY0GUko2LEPcN+oUGZnZtednnQnTKZGidLrzvIT8WcaGJC7ug7APwpmyskIKrJFIAroHaXudkFPYpMCG0N+TUy//MN'
    'cqyJLCbCOFjTNq+XndUaBB2YVIZ4WgOni+p5vjDClLP0Z/HczhRwj/muk0QVTwQbYPfwPDFmtSDMJ7Yzua/MzF14w1FyWrQf5xGA'
    'dxYrYCMc200YoPA5uRW6NMaXgZcp99UQZdn5L5hmJ//DoYAh6IQx86bUQ2hVVJZXaSIrElDjM9YwZPU7ERwmoVUZnO3ia+WTIScw'
    '+Y2UQuxVPVlElBM5lkgqDhd8rtWxsMobYcc8bBOG89bpTOoiEvG7olZn3hAIQXL0ak6hXTsrsnnM4Ogesm2WsoyAqtJFxBYUDpYX'
    '6j4xkYryC9cld3o4SBlhBbHb1IglgY2Cr6wQ5S6qn7xIBRBeVcusUChOIj16FP1kpLk27hS2O7D+cXaMeI3YvTZzR1jlFhu+NBki'
    'N9DPhNdkMeoQpCc2b7rWp8V5I4cLHXdYf0DTMGK++J1rPld+DF/XQQVB+VfYdJYJA0rXrsiSaqYagc5PYoYu166tDdCaN2VcL5AG'
    'a/rAAECwa0qAC1/XM9vtMe53rgoBEQ3I0LPMcXRPDpIT1SeNe6h0IlswNuxeGYxXDcPlPtfzU7Mdcg4PBoK7KmUo2hhyRgvUiwQY'
    'E/OyQErHF18JU8xU3LSSlOopFBYXV3yP+buF4Z6lGctYc5fntLLIeAJoIYKv+MRV8OmTdDWH3G43ZSLgISOEQezyrlUhHGWZzn1W'
    'jfT3QXpSppY5e/G3m2PUEDNNYH6a5mFjBzJOcB39NZX1D38VenCHo6gEDkMcOYi6CCSqRcyIKZlzkcI3usYxx+BcKMFBWhHCuGD+'
    'uYqIemLmDiJKycqkVaUukrMRWo0jxURfoWWtbZWB14wUc39O6KiRA+A/Mzumf5cSoIgQPPypxmxoXSLPHXlPJHFVaBsqjUh56ju8'
    'ZvYxVxNXRXOZdYO8ziNy2Vs/UsOeJpScwwKSMQWh3UT9mBs8cskAc1P6faFmYArUvkCpi7YkCxU5agMKc6ZFhr/MNwYXwIAfFVWY'
    'EV0Y96tVuOAI0/0mpLsBG1YBT0/6hUNJh4ZwhlqqdNWLJK9Ujc0AQTSnWRWIkylPZVU0ilxLSZwUQODNC5sHL/pmgdJaqyWaBmQO'
    'OpVca0VIs7C9gDU2BPjUrbgZQ22KMjnTJqGpWtZCWQNd/Emuqg6PYUhKAKmJKVR7ssk0QAniKDCH7QBlZ6GHQmApm6BBlTsfcJIq'
    '10dGcs5whZLnUiUcv1AYBSHwF7Qx3ozzGQ97gNmlL7z6o5vFAVjyxS+gpZWaNZslrmb8/YuvF3mCVPXPkmxWbMrcLLWOpGCZlf2Z'
    's8JH3WWhDNjtHMKTRxfPfUUMKqd2c3R/s7j48piBX2WiKSuqvy2gqlhfqC8pPYUQzZMmNbhqO3eR30UbD8mtzGLFb5KfbipNUu2J'
]


In [ ]:
# PAQUETE (10/12)
PAQUETE += [
    'Z4WXXNKZdCrj+6KPsyJZHsNunYrY9XGRCQt2On2wMqSBdL2hOSAjgMVmDmu2Qzb3HfPWRrnnU0hdLxR1WlYtOTFlVeNjQeo4B9xO'
    'ap94GqO8TP2x5y0adeEd2SZcFQiEcxQTxl49MJt8aCmq0vLMSI9IeMvNuc+Om9RWloyijfQuheR0U9mDlRzVPuXHGaNutNCRmYEc'
    'rkFrROCuBr2tTpr4BQQAdkvogFiOD2m2LQm4CKpVOVhRGhCrCEtClhQjf/hx8ODh2gOtw8CNkYhmO+fEF8IjRuZZKggGgg3HQBgN'
    '0z/JnIoLDAsE7TLhuFhTn7cZ1rxf2bel2rxMLqu1wRm9VuGcRa46K95dbVA/2jRTc1DE5SxFcnEKNeYm283ikRcICm9OAmYkIfsV'
    'QECURDluSgePapodz1gy+zV4NqDPahQRzaimdSx+Epcm1CYAS31og3ZmF9lls80TWenZhpq2Kx+sKMn+zFghb1rPNUVB1ZjQTKt5'
    'qymN+7IoT+IcgUc15MU4+gX8F+qfQi94OCUsM0k7nwNmZkq8P8VjriUg4ObqbQPRpTMyYCCi00UUeFHQvkhcFRw8mM9xdIRIDDOs'
    '1SbJWOfOn2iuLIrqCHlx5ow3cQPCLqD0nCinoo+dsOTIItRZKYpCNVfRVkTcOQGZc2oxDipe1wDsSlgKT8KYpZsrrrQDXqlVR4ei'
    'QO2NZF87xoSEgEJn5s4ssAmMHM3MY9OLvLhZlAuR7QiVcx4GXrr+kz5R+Ciuo0QArJKWZnNkYTGl0JFbVLBEuQDbgAIqIMkahQVl'
    'qX4VZVsVadx4OVjlalo+yXGfjI6vugwK+ZexQG8vHTVLF3gcAa4Za2mMRVrlqpYeolPLONGZ5xR4JUKHHqN8hTWZ8AFKb2xu2uMU'
    'fiTFA3WuABLsQmMRN0GZQcVPVY8dtVVj9VHg8pP2oIs674IZgxUf2QxGiA5cbBZx6YCN4BCgH0RnpjLpu+nmZaSzAkPkvkShsLDy'
    'zHyYzvldKvCIc0GDogPUeT9D3qttPuzA8eY8q+YU7mn9rPA0CIdVWgkS55sJUBmyHe7gTx4e3juY3jqc3rzz8eG9Bw/vHXjUy0H/'
    '29//ogEhKSwN9ecSA4suiBMKjrIUuT99UsNV9m8llNk04qlQ8VZaasCM+vJ2sPzaKwgRpHIut4nJfAlrwZz7hI2ycm5+3ULgVsLJ'
    'HViMeoLHXMaL34EZJASuhdMJOrWizyJcWnb0Ro1vNmdfeJY67ltioM7DThTMBvRlHw5uv8Tj6LNUN5QzldRn0TGwBxnewLjUc9uo'
    'db48zEiBNg8gbUJc8ngy2Fc5ZZ4k3rYckw7cu+CvmVvEnGEmOwDduDG2my6CRLZ1ZlEWwkwYv1hHB6jtXeWoeLNWZ0RjwQEmjF0H'
    'DTTxS46Z8MVSdJhL6arIJ2dMYRC2OhMFxmnoetgSfJkwI5kXRyBfyLlQW6bJtTdqRfSRwEoVdrItYIMsYPubZb5kyAnGVTZm4K4S'
    's/SfSFOkZ1GvYIEwsZ8hnM1VNu+KZ6Iw0mWu7t+6OMcvYAYLU6b8Js1i6LOO0n36EsMg3bCEvmTKVqlugUyRuZ10yWOw2pwXkUmu'
    'hmCKelFcnllxgpymcIifaWoMlZRcKaFcpzwT6ROXxcwTaHIxvA/1SI8KQxeWwxmIb6OgZl+4l0ULETIGp1JDuLxIDbEx3ZjYBC/k'
    'GFhvgkUkvBxlCd3FKzgwoLHDR7GEN23BiB4EYFLkXiS2P9W238aiB143WLtwjg4CV3eyoXg7xAeAlJcVZh6KD4MIGjQt7E5O8GXt'
    'nrDJoB1VU4UUclnLeKqVUfVqWkyQLkcoIsL0gfB9OUbbjLg4x+EeHsHLVZq1NumbI+IKdVIOxRKgt3KgyFwZf/PyPYq/VXKQ3oQm'
    'BIx+dXIgfy91cM1mlTOdLHhzqC7ayXKa1BOiu+CMCMMxHCA5qb7JUHGw1P4+1t901rSLL1l3a54V5amo0aUKPojf2GTHF7/j99ji'
    'DpZFJmsok9vcneexbo6Lr5fQjZEaqKmJjAsLCgGCS98t1KdE6lsntTwEcSZRSTOfGwPTg7ZOHQx4tFYYoEmt9PocRWtSrvWFvDao'
    'Eodd60QyNUGvttceh8cyXSTNQx1ijNBcdJgeHwNkvzjv2LnYTVZzLze7iSKoObIQZWwRmyYfuEbUZXDucizNlnym+FIbhQV0XkPh'
    '/bDa8Y4622sJCBq1hWUxTDmejtLj0kH4G8Kk1pNWmWKbUigqVBa26RRmPSsxz3dKPV4+g52osYETPY+1aRU42GfY91Q4qXev6nZ2'
    'AlPiYJ64SJS1C/Gc/OM0rWgiodIPzG4DluRcnRf+WFbeyMNDZAFSB2NkhA3Idk5NrNLgC4s1QWpi7j3OTkHGhDx2EEtwsh0X1ZRB'
    'pYMaT0ik6RpN6NPYgk4V9t6qaiPBSMPvkPIOi91dlVcZf2rR7U8T0TRFqW3BbD1ajR3Uu0fVZiCNu+xST4kCT0Aa/4TvO7o0XSRT'
    'Vd6JxT7ZCqDaMST+revw8KDUegyLutaF1khWf50Q+5V2bC9xjKq1Vi6Azq1OdeWsZk6x1B4qHyzYWTalOIpq5ZmyDMUJ4oFgLKnz'
    'zdHn7di1zymgzQytjZ0/d8G+qunMFNXP4i4rpIH7umXyEk7+KHLhyAiQkSN+JScx6cMhxQp/sVyAXG3ytVksqWNPVgXDC9x4fa2L'
    'EaMFrOfUmkZGFI5okDN8vt3P27cu/vbowe2bd0bR7JyltfFHnQKNydyIJKiFLeA45XE0i53gFShGI2c/0l92Le6n6DjsNEt6PmUK'
    'DbFTmZZP63Sn7UaEfRFS7mAS5LOyZ2a9u4JyKlM6J0gp6vFCz9IKgYqhZ08xcdU85qou+3mZs+hnz0VbW3usBoXIWekGvYU2jJsB'
    'dWsJjGK+8YZY29pZjFKCKnK4iOnteGqDRs23p2xbw6Twz5BtM2BoXUrVUzRkxCfYt2K0ceaGDeVbiu1nMgEZ5qxyQdpkkL0gXrzM'
    'ksDhEqtL+gUECiuWQrTdBkeQiWnBRZ0SqyoO00aNfcttpzWcfM2dwmLXtD16Aw4Ec4IsOnMgtlpIwB/B65Ub4RpGQVgvnos/CmZa'
    'qZi+8CS0AWytTpjEdDpGp1pJTOp0seT4476L91VrGCR7elB47aU24EYcpki50kb9qK/I/augaoJOG1h5I6eKzzR7g2VwfeHPDbg9'
    '/Orh9pp88LJDgBB5jxAWDdjidessOKRIr7LDTp4/sngKDRuwPaRRlz+U7z9ybCNLzuKA9MjRinKRuJJTwkhWVn7cv68ZKeuMCAq1'
    'pvN+nD4xFIyTtNL60BldtQaNPAoNYsE4LK1/4dM/Zj6fQ48qHk5aJ95vU4IRiGKNtoJ4fRHNXqjUNrfCzOQrLLqerd1Ym3zAxVih'
    'cr0GniYVtXs82Nx/QVyzi1NfPpUzZKBfqn2ALI30sJ0WT/l1GO7cLYHmvU45YdgoxCVShpHrcZ/r+HLxqk/CZfYnmh5ude8Z0pSn'
    'aEJJVGvW9EeEKF0Ih9nvb9bHV37Qb9Bu0MKo0ZywkDXEnjVRk7bIuzpe76DtLVTI9/e2yB0p1LvoHYe75ZlQZK2ZbEfujBN6Mv+D'
    'UNDhvzy4+cBQbBMf+oITTbW6NHeq2cgXMYQB47mxpaC6l4MTJ3itbb3KWVvcK8/VX6bX1ZlkbNLoDiJICrv2WMWz2kNWl5R26u9G'
    't7rFELB0+hbYIvD5DB7rHv80QRbd9E1UzuRk6yFixaWA58XgEyTsTGozz3t+HsI5eC+wQyDA/JKTSJk5BUUkbEJH6r8GDNDRfLPe'
    'q9D9Rg7QHwlrd7179cOkqH7Ub982K9Y/+uF8/Ry3+n6++uFVXGrf3Lyl2VxQc8WmzzH8l41W+gDR6E+iPuyo/WZ2WF9rz8ivp+N0'
    '0fqNA5WfNFun+ZsbI590X1r3+J5Pi9mTBEvHu+tKuP72Vw30g18ip2G+XerUvabTJk9xlcJxMW+Vb66NgoBFoFlQk18SjWDTdjbV'
    'Bm5L51ncKIwAAuqDptRhWHP+IBTH3AAiu6LWGEO0gnFY1MAXmwYcgvqaRELQyDQrF/6R4ZPD9nkkjWfxjFH9FvOrBSlaTZmWWvn0'
    'SQ0zqQzAOjqDhruIw/R1FN87yQnuWS+hBwgSXR/YQP1hfeXHdsGklhrcORhnWKSr+ZpxVjxLSgDLXvqfgkRYz/JUZvxck9viakfD'
    'ozerTx42/JoujksdYh+afldLCmMV15JlLbMCslRhYgNG0EClIXZrMGetKtOh7HdmSvxweNm4PA5GZPVDql4HksNZ5N+tXYRajfJ3'
    'xBDY6wCOajO+enjjeLGQB9uvOSMMwVn05yFFbTd8Obt6E7bVzb4gP0xkYnfd7PiZZ9VnylIv5cgdHHhH8wFLPNtxy5tzxDZn1CPQ'
    '56wGR6Dub6qEMyJf4VS0GxEgmG8dg55dB8ekkMW283HytidMXkwvPWT67CwkSd/Z/o5Dxy9S8zTlWr28fEXClXibo+mSg0kkmHF1'
    'ujk+zpKBm4tAoL3/yYO3FLaBbgIEs3zgGhDe+qxL9kU5keNJg4MQjMrLTE2RQaXqAXISx4vNclUN0IG8QrJsXM3T1PCITdh2VUxF'
    'dibTqsv4rQtkqU/II/yQgzJ/XOiJaC3LwfXtXhFw8ZESxmOKVrw/bMARy1u04R6xZl5t6QWcRw7mOyhuqa1viEB9GpeKNdqhLTfL'
    'XdZABtsGHtbTkwZf9ZvPRNFLvgHgqEBBnRNIPxro1avXr+29T/zO6OcfD/uNjrmF29Zzmm/4db5LRWn3KUQYZ+seV7wLfPTp5Pq1'
    'V5Fw0r6v1kn4NoXgXbJ8kxrPne6BRNoaj7feBzqeNyH/dKTVYEgrhESOheKP2xDbx1EKPJ6Ok21WJvHTJqqZTCG3TFbEi2rA5rvB'
    'QR+9TB59DyT5vcevHkf8Ao71vcePJtd/oNUQua3+uc7fm+O/PI1PTrLknwz/5fqNve/vfdDGf7nx4d4/47/8E+O/nKupNQZMd+Lg'
    'BogFm0c/V6Lo9ehyNIyP8zoJGn5eS7ItC3lmKXqhPuMaYj3dYp3MiuJpZDGry4sv4ZlxRUsU/mKtDrlePJ/DQ8l0R4Qj0+KAAHwL'
    'jVnN64zrNUFlhUVcR9j3LNVEGQsdoO51K14LG15rYAD7hWAN5JYQUYGOe+3cOPrEoqrteKRnViNxotgsyC4An9ZNMypnsDTyJdL9'
    'eVlYTKqaiLQp6xIjRSzoTF3bKdOQCZxNzRmxBHDQVgTg0CRuhLJrsAXhTHpmWoZDIQ+jUqFW7o2ju8yMI7bzWHZVlF18LXxU/RU2'
    'M1C5k2yB6OvbqOPExaZNlykxHpfNjdHSTS1lw4XYaYUkoJDjjawxQn+aPHsDjo11Yk4pxeJwXjpGnMsxz5iIRE1BH74f3To8enDv'
    'jg/mR/dG6mnvGVKgXbXULlHJEqSUBQgZjP68j9A3K09FO8IN+tfavRgsZS42GRRXiiCLpOHBGDlL8XnPO3+Hrutw3P319RvXRBBo'
    'DAEU+dfXv4/rE0vcyevRYFYROhZXbY8jfe3jdHWez6KDhw/uAN708Oj2rTveEalBsSTw2Oh4FBU9N3CNOZnHmbq0jRwrI3SMkeWC'
    'ZNs8t/o2/I7Yolyd3uicq2OFc9Xt+CP6AGnOPFfU4Grt8p8a9T/r6P7YoB4D4BJu0XeZRMx6sytRq6V7g70fRJ99PNRoKJmuXINk'
    'QNG5kvTECMhA7hlwWZrv1rGZKgLlHB4d3NPE2ZEWyUnOGBLJZAhuKBMCWa7x02BN1NMWCogMrk6U2Y3Zc8zo1Rqr6VwvBIhW9STl'
    'IRyWg/SttKQQgnbz8IUlFyL0JlsqD0iUq577npSaJaSawm7sF2wBn4IcLrFQnSOXFku2mczBgLRkuOU/IA4tS3qIK4t15R1eM/NP'
    'hSB5dBjpamIR0E/iswTRdsgGOFcYg4O7t2sW3KOZMAin14yRzbqAZ/GFJg6B2hcOQkcjyZYAM1foakbwMAIQJqkGJRNIBKA3ymPd'
    'MURrPaIZFZOg2pCu0vowYugdLH69RZ3+VR8ODx1S9Ypjs1AmP42uJCIyl6qEMYnm91wW3kVtzm//kJUzICAJ6p2a+3AnwI8X0QyO'
    'TW/Xq1cX8Jq7W3RFWKTzSuDM4Io5DrL9cCovQoZ7UY61CqU87LIhR6FkECAZff62eFCi2K1AzO67Mk/3LS26cKPCGqBxCSyoS4Cj'
    '3q7k5/2DT2/fQoQ3H7sa9XVGGPNuqRVMbSHFJ8+5xyOe1botFbErq/NlAw43Yom2i6+qUQ/G4bWmcZ+l5XpjIDVY+mPRNF08QQDI'
    'N/7OoYzI0TkYSgCa4UGpbxTKdGeMNdY43Up6gYlAXjgCQOQI3ZSxi20JQlZckHIcodZoCP6n4S0meQC7Rw4qZABo4oUVBwaUrMah'
    'MmDQQBfoVPho613WiIMJeUEh4R0LG3GxFhr2r4AhBGGrGCcOCpcDisGCAFvKM5dqjOWo46cA17SB+9sLgCGXPq+FHz2V965RGlhs'
    'Ba6euuqgiGLBVN53M8v4h+MMGVEA9EgW6Vqjzf7y/p2jT0eI2EfJTcSa6jtcwYmFF2k1kSIQWs5Rii1n9gYc1Y1TzbPnmkB5ViPn'
    '0wnt0qCLZHavd650lShAJMI6vRjswjyZEssYwItvGIuq0Z1fASUtnjlwEnUSOdozxvhd07mS68G9Xzy8/eDwJi2oflFWG8S+39Ts'
    'NQRrjnt3D37x8PAB7K23bv/0jvco9t8VFmgmzb7jxOE1BK77K0O8VchcJLc7N29/hiJNdyYuAMFex+irjSODwyCqkPTJ4J9zSzcK'
    '1BilYW6NhYtT5S3rMpajw0I1sXDl/BSoy4vEECiREmaihCsv4t1S2iUgnKPCEWKFGVePdoGYwhwdn9pH3/pylixgE0IJSBMUOwgp'
    'slKNxJg78egFlSHmyfmbHqeUJLLEkuThfjMS1eIUYItnqWKDBo4nxzG076Caz1IGGFWWJ7IuvqCdN4s+vnsYMfLfBx8y3O48yBhR'
    '/zED7eQ9YM+GQmph0jCAIXcU7F1lNC+yMbDxD3/z2wbGpAv5xpD0mO+9Uz9k44N6mFgqHuomMg61QYM1+dSUaLJmvc+F6hTtbhTc'
    'UMeo4rfun6bwrG//HsDshZdrVL3w6mcHR7c/uX14/8Ed1xCJ/wFyx8hHa1zSJZHDRKxjeECOeIje7aObnz68fQ/nc3PjvRd1zULv'
    'uwf0O1zqoSc8+DtuG6EuMKBW00UyWAG3KgdqvQvchUBSR68c6HatGniVBvlQxbrFk0rTQrWlkcZeGWaUEuUqYd22Oi55wyT2+n3N'
    'KEBtSqPYtNHaLstHx7It4FoK4t0gF41PsmJmQ7Io5rSiaDUYNuKCrMaHnBU0JVfD4VYIkJxyfnL2jRyCSB/s1GoUjKAd8OOWzyTw'
    'MXL2ThRck6/0AOqYpVtCUPc+u310+/6Dg/t1FVjqEoBsiEzfqWExDMXK5DALf/DAWo6QhcV6647+5NhvI7BXQySoMNfWFc1EFZbM'
    '2Fu1upGpEfuBTRpuqwVUuB8zS5EbadoGK0BRENHABSeLMI2zz5M61WBdKEp9BQI+lVMGw1mrGYMoExrVuUkXV0/ShRxGc2LcLBwm'
    'HMCcW6FEnOf9DmL/f9h7295GrixNsD/zV0QxUSsyzWRKynTapktlKJW0K6vTklpSeqZK1jJDZEgZTjKCHUHKVspq9H5ZYIBtLLDd'
    'wAKDXuz4o9FbwMzUh1nUAFuA9Qd6/0JhsV93f8Ke55xz34JBSenKrHnZMqpsioy4ceO+nHten0dG+Hh+AmLNvPsYk/l0R9MU7uhj'
    'QZfVqxtoPgtCkmtzcrvUjUi0G9PissHvmBzU2qHncpbKoN+5Ydi7LlaiRkiXYyb4lB9/tUFv3WEdZ6P5de/0dVMoU8nuKMcJHUMb'
    'H3EYhW4NAym8yWTfhqGT7AQJTry8kxn9B1+0pgAUGyKetgEU1mlXV3symOUtbNV2u73QTFfGfSNaXfyJBpxJUOnjKX+suwbPs1fp'
    'HxWyFx6XqYxIszheiBnZxC16n3g0YtmB9jrRSSBBaBTxvuAMTpCnwiPzLo6BsagoI+FsnyWqEmxubT7eRB6RgltJgqvzFznzPaBM'
    'YQ1CUhH5+EOdeum50Nkr5f8ivO9v/bXebmsHQjFvazAZkE0xd9RwZG3eBkvFmFcVV2rk1eIyaA4KEsYZikwc7fxC7Cky4832B3Rd'
    '0lm4HiQvqc21Dx7CzhInbhn6V5Ui7DtQ3EI+Ho+hEEJG0kPi4yItLNRyM0BZ7kWfxrQcRxG8NN8kDFG0gprws2TrZToerUR0tqxs'
    '085eaQrSu+g3Mx0dRC3usLBkfyU99zhKjw3qDQkB5FgyJsyUIbrsshEEspjdwIzuVAKE5hsugYL+CJsQ6rPWf0AfZtlOc5EYJAAw'
    'm6Vwjq91P3hovf1NUeYhXLU07eq74jhHoxw2ULQd9ODJzufGJaHQ7lLEU3IFOc+JloQalgDAUL6C/QF8lAKmQvAiBkva+s7U7lV0'
    'Gqk6ZunP8QTpCJwO1gyXwoYcq2kmxS3GHwEYZThkBf6S39V439VI0rHZf/r57rO+CSfIlaZaBv4GXhsyQ9xkVDNPPmStN0+AcM9Q'
    'dwhD4GPtkJn3UgGGMVkKXyhIKoldLgJKlI4U/5IDMR6dgnbHPtxUecgiZ7NBRwa7SmGwHG4uFsRpnAnyga4gmYG1RxwP8narBCwe'
    'uRgRY3XiOQAKAHNpzita16GLoY1zMebg9IQiaoJ/UgnjOTgZ9U+FacUSONgkU2JncLC382sYO2uPBtS7RmNze+sXO4NnT7f7cMqt'
    '4StT4jcj7VEmuzUs5ugbK4ue8mgL5BijfKFMTkw6+Lpkiryop52tcapKd8FArAXHia3mqOZtC1dxChUnKZTaLVFb+Xtprd2VoiVd'
    'qahW5Yb8l3R1PBy0EPNSbMlzA7cNxs/CSjtZ4VKaLaaZZ4q6Olj2bwMmwnh/uc5SRJQJWI0N/sLmvV9H8b3X0eq9j7jF96L70YYn'
    'qjJx1dlNkqXUCJBNTd38KLYONH1biVopjURFFpakkzGOTGq94rrIAC4WC6cjtVNRPAOXMRSMwfGjh1o+xmriBn/Tpf9zDksiC6Xd'
    'Jf0efzU5+UqzXmTqYCDJzYdpL6X39uZG6IJTR/O52uFMLLmelBPvWq2YAv28nJLCEbXWiYKFfv9+cJPUb8jKo464Cg7pnHbJNnpU'
    'peuq75/c3O64G7V3qmPJ7x198DtQrrZUaxecBqNl7XuyccR1BjHH/YZ83scKHVNVuKg5s0vPHXmuXGuu8QLCHa2/Lq1kVX+YBc17'
    '20oXKYz9rf6egud/Sa1vvMV/4Nje2TrYjH74D3YcAkkaRa21+z/96fZPf9p++49+5saRC6WV1jo3zhqRWdkt4vzUmGxbQRT9KHIB'
    'ePz9vgl3yZ/rEftX1TT9yBh4bjY74vuWPIcHpnqLmVlica2bAH5YwM0n4fZO9IvNrX706eYzGthn/f7es839aPtpdLCztYnPNlbq'
    '4afqMAD4DW4Yaqa1R0Lq5xH+TdpsWxRQWZJP+rvPnvY/e94XeNfK+ERdCbhJb/wTAoEL2NsVbSZQYwLedw5MshxXlYY3jFOTaxRQ'
    'oxjXney9BZ20ccdppYCIZHCsjOlowg6CkKBMZgKwgzFTneLcKJlsJAFi1CTJdKMDVA7HJhNE2rKHA/RIDT+JY8yATInt1FMo2Zp8'
    'gv3EhucyEwXhwK8WI96pwco04BdJJBBqsc19esubynh7uUqDwbWe9PfNatmIipWVlXcoQpygfPsPYYNPCtKufjeeoRTGWc0qu7rR'
    'L6UKyMJ7y4YfAdJGs2t4c+kW6TBVT3X/wAgxfDbnImnmyXFsQd3l0Hn7U3dDyDovf1TAendv51egNoTLv8lvjRofhn87xScrvfCH'
    'pdrha1JhBMBnOtsLUIpUKy+a6YgrmuIRrtJIHj5yOK5pHLQDxgrKW5M0mRwXUmpMOvw41KF/+P2+SUdiylykeJUMSc93WaCcohTI'
    'MuDZpVn+iejPe3CjvY7Vf2hoMMsgMPOi231htd5xIpUVHQZJ1GpzwySWZKiIKh3aw5hxHD1MSkOX48ozHPmR4Pur33AsWXUiMiYM'
    'zlbmHo2GIeASUHRJfzJkpD3LQ8H2uKDWqLbFDQaGhtZ6CpdKJJkmHcmtUewoAarlCHdhQm2S9JPOUsSQDE20GiX7GnWS7DvsDoTE'
    'JZsAT35Bq48O5A2hS3qBX17omnxhgWlKsUs8KAjJ4VISqQfdtXU9i18lRZaMS88Y5IHga1bv07/WcCQJAogxA2WCxiFu7djz8VhI'
    'X8ZHmM/Eyrb0ZgvOZ5PeocuuC5ekqMaphGAQntD1lbSYOJi/HRXpmVd3quow12L4+nGz221yuZZ6ZpErMjO1+ixBkfHY0sW9sZ1n'
    'Sdvukye+JPPE1rmFdmdYFbyFod+R8DngyQx+g7RsXtP8Sa+Bv7vDr0etNjNgsbDwklv8u29bBXNMfcGj1HdHtpOxlqw1okdWu32D'
    'C9xz+XOrbeMQL+AQr/d/TzDK6uWeIMxcBGUOXvGcFVHtRbdyEackckgVg6ebvXqtkyYPsoCDTlKIQEv0xpsRRLm8dC6bC1V0E1pB'
    '5flE1g7+GGevWrd9rrffzaaWhEQjs5Y+WRe3mfD7kVxW1zvMbV2HsF5vOfPX08mblf8GBVULAQHMKx8Uwxl7/Ce8BvIiJckovn2N'
    'GHytEQNBWevVVu7Jb1pbJW3Qyo9HlRikDp5uVzgpUMa7fLeyPvQjFA3RMRa37IKAsJIJERvznZEvpnn3yksI98yNJpZWJiaNIw54'
    '+4J4ssdUo/bVArmrNqfRC0ZrqcZRW56WfR92jnELjRQ0lj23Iqbzsjt8icUSvPwdWy3e0bitnCO+ZSWAQdlrJppBLJNxhUqO30Ga'
    'omvamIKQZMEZDLhVG4TRAYkE8F7wHItoLZqYnBHeo6PChP6O5ycnScEgo/NSXGfwAqLeSClnFM9INFrNRiUlIvOTU2lS3ewg9dFM'
    'AMddy9kon8/g6bCGnHz4GGPyy/n0fCbJQHDDipFW2kFNsrOUzKbD5u6vDn6xs/18+/HzTz8FQTaq8qLmWlNdSefhQpJndjl5jdTJ'
    'eZGwd2ig75uders3+QaoUFFrc8bw9TORZ53oC8Ti+LNffhaXZcN3ymXzyRQrI8qm/tfyXMlAuBVVZON2dYF+OZg8+iKbdgcDzVsb'
    'DKrlfXZPmP0pSbwXuk6rl1/AkcYlfJdV3lf1aJUL9YMC+24BHA20Dtn3JPAZEhllLuzqH8Z0yuHIa63QXQN310onWm37ZYBZzsRZ'
    'tH1aTerouVgE/J9yEo9Jgw+ANPRylS4y+t3dvf5+/2D/BkyNqZQw4gaMySSeJQPYxpOypV9TfwfysSXPadfxtiPmncxoPviS3qPL'
    '6GJ6uMKlrEm5cnR/LXnU+5ArKT/3NM6yhuS+hRvp7IDrZ8DOTJhC2oS2oD/zpuGfDZn8nWiLyQ2OHT6uIdDVw/mlEkmNLNA7rXUG'
    'QubNLOgqZkeLSWu9BJoNXY8sXQebJ/FPEFFpg14Riz1OLOxdpR5F0/i4yyPh6AO0u7jQ8BQjxS1PEPRnxtcQ6aWQjdYnKvBKkkw/'
    'y7OhkpMIYDAoAmNtEalqbHg4eM+u282yh8mi7c5n6dhsaRq+0QCAinIhnEkZ3B0Tp/BZm5eWrL0c6kFaRjihj8yZKXd7OnudwpXp'
    '2GTe5qYtJ/deNmvVA/hZGlube59tPtlBPpp14zJC2YROYDIdBUl9KPhmmsDksuQNewk1jden06PxpP857Hhq/Mt34Y2F7V0I6LSH'
    '/H4ux2iKTIl34dXxGY9Ip0rZPSbp1pyxmoF3hyvX/BzJnjP3xFOrLmD10rKb7kzRAKWUyOSS5/D/0Yc//O0/GLcc85KbLswl2p8r'
    '9ruc9br3lJ2cbrXMqxyUcigLEmYr5zA/OSBo0+3frrOIN4j1ypi9sWe+6Dj29cZtT8YCdQqmgdbSw7ENn1L/s+fbB5v7FgxmOWLy'
    'p7KMDEhs0/spFtrF1CBEUCNDrxHDXWbYQ5UClskrPqm5Bc5l0Kvw5QCJzZdeHzBiMB9VZnN0zaUSn2AqH8tcXi7hIni2jLlB6AeM'
    '2gYSgnNLh8SocQbVeainSeFR4yhitQRtGZ8rNfLxjRCqDaiv4kyyMDsWcKTH88qFk0SzFks3Dr9SZlPRNRRw/OX8mOtLNR1cesyA'
    'v5qT4Hr61wGpAQ79IbU1yccRg0+eC4p6lmoy0AOhpMLzj2ihO+hbXXQ9FbglVIqiq2DsSPxsK+4qeIFJbd0noYsesN+GLu8ycwvs'
    '0ahJ84ifmz6iwUYzuht98DBUvA4vuL3Lo6jFoCRdJla9bEekd4RqXfPe4v12F/Kntv9TWw6Jg/7+wf47E+pOSUEO5LsJ4UFrLVG+'
    'g6AvmKbmAkKtSkxmC0gFsE7A5iQD36SvJZI7ZFWWl7EV7zhrYWE+fCThGAMYtuysokvXHiB+hxJnJadwgphVHxxl78x970xLWbs8'
    'Mqpgc1UH/j2Q3nN9R+R/b3zu+EVV7+WLk5tuL73I9aRbzLPWISw2yUBD4UFHenbUkWFVzJjF9fnZ3uaTd7Y8vwjmJs5iIRYTQU+r'
    '9hVs5bf/2E1bRRyUXViGI0Pnw/oD9nvqkRcxIK4oLlJfqTlqprsGBtr4zJVXdOyhjJJJefV9wWiknlM89v5ABPrcx5+T+l6l7SlM'
    'vWeZSlri98UIDshjrr4rfUR4ropCAY9WrgmfkfzulcdpaQDtXFF0RoJLP0psfp0tdc2ik3Eez5Ai44WKe/aHB+tO2QelFAPdOeZa'
    'ST6UzGrxtKia65ETqBWk6fPIqyg0/h3YXBleTHLbTpl0yJsr47yR6HbuuXP+FDv/pq1ni7547w3Qbez46mbkDWgL0N/VJuwHDru3'
    '33wN0rJPGlQxAZBmJpl0pPx4kAQ9idwqEgF0/vVoa/c54ntr0WePO4roDN8AHBlwDgtdTUkKhaFk42D8w9Xo6vsp+LOjc+yMl8KH'
    'LRUPLlJvOFKglHWjTS6hw45h9ZKeHJ13tPSNnQSxHGKk2ow5gEYq+Ge7z2X7bP7y+f4BzeDmNp320ZN+1P9lf+s5fSMZp+JAiaTf'
    'CI2+nKMkWjvdheeNlCT6XsrNZOA6SK4MfRxdbq2/u7NFmrkW0DOnUGp0OmHQMeJCHBH8PpAc0tedryzJhOGZC96Svqf3Yp8oHW9f'
    '0yjfh+TDB6PNqvzj+RLv2bmxnATDPQ7TlN2Ycfqr/KUQ/5zILqQJUogLWFv4HRgwWSv8Yd4VIutQWx5Hv9z8lxHY84rhS+tD4TTn'
    'r+DsyERG/ElUAJ3bjUjcaw2dHfr74WqzcXtBYcH/bZFoEIa/d0/WPqkT8sQOvkt4ldN38tRa8bL1tL+3139nwoWpoIQChG36t/+E'
    'X6kHoAaLw0bkAPljq5ENqvNs7iAWE3UhMGinVN2bA8+45YNsoZK9C7KgpA5Iwvt6MCXzgllWZ8oVb9goPSOR3RSaZ1Z1F6AEa64+'
    'FzYu/cZlk7FxPDJp+7Lu4cZTXWQ+FNYd5gkQTrHFweEuncBpYg3cIMAD3ivDgIfKLuFuNP4TKWUNoVrgU/FLiUc+V55hZeqw2FUG'
    '+3g8yn16IHfDf6U+E2z9pca7svd+QpKhxr41TbQbzjhdgRW7UmPhrqiFu1JnuZqWLr9kJLsltirLh3dQGuXhmmgd735/d5PsjB0p'
    '3LOAAO8FKCjMafHW02v3PtvcGzzZPNhEDEPF4DuQg5/nXDUxvgnr4p24WBM3iLQqxEXpB3IlU4hp7sZJYqryg6mQw5fTaEplCRWc'
    'MxZgzc3RKHqaTeezZo/FpLINzOJ7mjuv/1Tev+cl6poQ8rlk51Ya8aIVjn+ht5Caa9g/ypMZGjK5bdynL3zokUqea78WcSQAAQkh'
    'QKDqjU/zjxdm1NxZRSAxbCwjYc22ZDMMBEKteVAgy1A/kCkrBgxdf1/BdVKMuiZUMYjHM8GlCAoI6eWARsTh9Nje+nVevCLTCW4Z'
    'aGpMEVDB9WBjUcjabL5bJTP2nalRXsLjS0RA/PTH5fmO4pOSJAObRtaxIS7FvWJObA6g2RNf3AFA0ZEyPIN8McJXYjmIMdDjRETF'
    'KpuZoH3mmrfgZC41kJuSrISOcprJbR4RumdvSOTQ2j62DIrZW2VrAkODiRiUmYTdwwkK8BmoTFRuAf5o1WTphbl3bLqINT5KDBmh'
    'aCZGP20HzwhTnAWFo8um6ibjHXFmmUss4Fh08y83P/vsWX/wdHv3+QH8XsESbtKBQ3c/3vzlzg33/4udvb98uv2Z34KuZGrDJa4J'
    'xlIZleP5qXBaJ7TDYAhc/Y4Dj7zLeCeqlGP/JHLYSYRwOSlnaWrQW4oCyNSi3e0IlRiDrKHHh4N5afqyr+n9HkBwNBeFW1MTi9jM'
    'BVpZC93vIUenI45t/swJtLs2scfCF2SMnSPepr3Nq7/7teTJKqaPIvaRZDidxxpKTrhEfqZ0A+YXzf7U1eenT+PVX6dTjX4U8dVv'
    'XvvPcJAw9B2u48OGqcCUiUsSZG1yKoskSfbhGABTTDpkgOgxysqkGkD8+jYFVdAHyvkkLWzOI7MsM3wb7w19jV/5rCbC3wooIeP7'
    'QjEXBq3oCWE8niL5oiI85ekAn5Y84/6zzYOnX2xy9o956ywXVYXTIxDa0txfZvzJa2uPFcygZLdDs4KwwpBS7KFAYmc+Zu67Fy+h'
    'P9PeP1w90ty5F2reWrwIh98UHmi0ql8wxMqLjnKV8tyBIX1Cf86TU0a7U8GpGau0BoGOI3ghL3wwgCJOX7dfGKcLMBdM8yzKkmIk'
    'NH+WUF5myxTasK/lmBqmVxvycTwdg+ldDbYg4cwY7iyVR1yqDN+iNDgWQ6joeb9IzI5GahKH6QgjZkUeWfgsM/HHsjRfLGDh6OiK'
    'gaegqIxbmPKJGMv0FZYkp5JLTGNEu9wIxfssiRpeyiku6HLmRpCUOoxB6hoLx4Rinkw5U9NBOmijAO4uOFeTgVPMt6ZNsQQOjyr5'
    'DZ+m4ySoUa/y3Fi6WZUYPjDYBd7iJ8VlF6ZD5b5N0Qmt9ohls2JVw5VutAmyY3X+kmrUiy68l6V3W4HTZ8UijdMr8UpDqiq2Zbtm'
    'tDTxApcZHgW7i+VIZelSeFKlXjxVRFMjYLGThHDpg+RrW8Yh2ZNSJknzWSiaDW48vLd2FCZQ6dXSxuE9pIHJpe0e49/LH2EqlWar'
    '6a2NaoXmwPHG1eCgy70cpVZMwkq2L6ehXTfAlYHG5Y1braWTJjLgMCiXjsBYoqlYP9EPv++X/soyDvpPbNmI7O1W5bRTLeENs4fp'
    'Vc2Ni+8oCma3mJDoSFp6nS5CyQUe2BpgdzSHx74peuHQnX+n2cp1Nzp9oFMPtaWdsBs/6I0utubdSpqeB6BU4Qy0ecRG1bofhSgv'
    'Qfvtupt/ZNa2DjEZIufrAJqxGby17yfodm/6enc8Bk0Uju4/7X++u9cXBZXpk3q69/1jUq0rSaKPKw2yYGDj6SxVFFaepo4hGx6b'
    'X0wmnjpq4x838Nz5P924a964WfFvnDdubgyS4PRLC86lmS0mabvHe5k3NZLxrAr7V6a+DsbP1W/5vH6ZO8hnhQ1mr7/Ya1KLhZgo'
    'Dt9KHT9DvF6TQvwWE4ZdRYVYWPgQaPV/TEbxj0sj/uE/vGFScCW/2AgeD1TRrxxx71mF7DPLyt1YI28nIRmHdy1KLoQwsib7N9Bo'
    'zy4mhys6fCtHl3jjl6Rt0zbED/JRv2ccSv7aQ6Sk33Q4wwoVTaX4w7/+N2x2V96vZ+MCJjJjoVQHfvfgj6+mHDcdernBmSssklgA'
    'nSrRsoLd7ZkA51ret6vf2kPB5r5mkswtaSb2FPLrNvVvP7+knsFKu1p7FhnF1VvwmVMZbsiqXdQUTpp6pZ9MayGWzeHfcAKEIcga'
    'tghE/rT8N19mtXYDtU530LrnwjY8gv3ae33Wll1g/R04ffeWQ3S+81RaNYWFi5bPOcZNUIcmoGOQEV6P2MsBGoHXcQXujpaUoQ+2'
]


In [ ]:
# PAQUETE (11/12)
PAQUETE += [
    'doCBBL0lM5B2ipigPll2bCrE866Bfw7bcLEk28hYcVN7Ljfn6ncZCa+JpnVkkpqr4KoaVRsmmhwKvEKu0KbjfVZc/Zb6It5dw7bL'
    'KDZesFDL8zXMJTkAEmMHkg8TpDLJto0eSiKAx/argBTKyIAqwmwkEBJVfJtJjmICRVIqaF+WTG3xsXEWZzxz07jDEPX5eG4CftFE'
    'iLw5/0FfXfheY4+ACb9BFck7hh5bSADs3OitYZxfFJwy9sKaIVSK5EpwwD+fzki8veZIvZa+cdoP0m6Be+DqwRR+mYXd5iieqI/z'
    'dI6sjEz8ISYNwSegAGplPtRDX/DEkJpQGJesqY8gW+J7mWweOcujzAmy794f7QL8t/JJL08CWFtlP6UU7C6Dz+5nDhpLqSA54syD'
    'WYyVjGZnPkMkQPKrbLzHbWmPmaYT5VzoUQhgdsWLyipyaUlZIQoS9lgBbzlTx6fxuCn/+WlcfszaHL2UXXkCCg65021s/aK/9Ze7'
    'O0+3DwbIgtl7ymURuEPSJK0fQM6wwDV8X/2kXp0gTfZA6pVqDrGqW/iGq2vbVWWFTjM2/W332l2Ung7K+clJ+k2r2c2mr5t1VnP9'
    '+9pmGiFRWUPZI3CWv3kKyC2zP8CWV9OtnkJcSQfeA1DUvXsG6Z3urrnlKNBC9Vrme+cZv6i5Beeu07NUxzK4X4uLvufz3Lp1BP2g'
    'kjJjeTcWU1vePvKUT6EQwHW+A7Bnjg+pIS7s6RoGSEf8gRGh2IoCmbbOIggem3TneDA7nzIfJ4rtofnJkoKpMSQNbEY/YfN1hBCb'
    'tXlSlC69ddXMWaCU9P3hEZZhPi+GiSGLFtpaVHuWrVdJgqQqMUbbl2bjUD89+uLsWOgBo4fd92GvnnLCZjryYQFRxguEeDY3Xuac'
    'AMvHF0CQztLSo6i+Y7AzTEnayLjqTerpvMgl1BZCRQCWnloqWFC61oxGWyiKiJxXqusLuhikm3TZ0Fqdi9+fC4HppsLvnUFYvB7y'
    '2UKZsVoU4A07IufhYTMdcSVuOvLt7KGxsHmlTEZ/xCqZxMWrUf41Y70sXRB/1Aq47iV2d/YOJG5INrzh2iC7DbrJ001bo2VUW80l'
    'JB0+GXm5VqSPlFhjDBuDFMTYgy0Rm/sPf/sPXsBL1U/GgXtCb7CfJK/uffGA6YiGsbCDfzU/FTSHvXy334n2Pt/fpkd0ov2v08+e'
    'Pe9E8QzmMmevnxbzKdTr84aoa2zofUPLdKau8GNqMxsmAmRj1Z94/k06TuMCRWXso8Zba5lmI0xK4CwHDR/X1T90G427d0nZNYy3'
    'GrkQpYxzH9jB0r17F8QO9ujm+MV8zGMimWZjBQVrUO/LnIlTRlxewSQITl0dx9ICV8/RSgbqVi76mDJ9aBwBBSHFWUoboMHVQQjN'
    'CZ/GHQb0m/jbjfc/XgXgZ3/47/8nA4KGfnuYcdegxHnwcOLN/xjJ4oZbDiDJCjPJQDjjmGtVa/DVwE/EEGvsl6Jl9EKRo15Q/58J'
    'bQRS91VXtl27+t0sRSD5LB4njKAG3HxT3xJnHaO1c9guLFDscHGM5DMamDoLDZH5hQakEyu2py2R/Gz3Ob1Vw5Gx0SsGNQ7U7Z9H'
    'f/jX/+b/+d//x+ju3f3UohcL0pKVlKpvGuNC0oYF/FgB3kYxzYYYENSkyNwXHq7xCx6uM8Tb4D6QsKNU6Zmn9PRvxYkz4LJX38GA'
    'SqjVBbBFD+nrCfCb4uiuro27Ap/EViLU26vvSLMSw4caegHnAy+lnSljO/DnLZqX4xddf02NfNhWqfObCYxcHKYMUaMOwVcBlmji'
    'TKZJHa6tSP50aPgb9ZECskuGMu+Fv1JfD7MKkRXaaHwbbfGq/FZ+Yzimb6MDTm//tvEt7SL7f7rWbIr3vFX8rWSfVQCDavBG6Epj'
    'v0bc2PM+CWb6dp8rZFEa25EkTpAUJaDQ+8jm0JbVu6Wy7dtKjVjUeviIerf2QCq02tW7UG+EgAi6/diU1GgNzWIBDV30N8DdyPhe'
    'WypBXV9ecoB71vmeFlT9tryqZkTi3mrOcthFnqZnftpro9H3AEN10CBsYG4nboC60RbKUGRJa5YrDeTduypFwDan+sfdu0LN40RM'
    'KZ6TqcRjxekQJus+z4yXA74AEsvz+KukYbI6IxViJrcfph0YotJR8vpjfU+X9WCqatJCcR7l6G3IFeJktiXTM8U05ENCYDXNOc0/'
    '4VVeyphap72BpG4YwiasBvDUmfpR9+aGoKiugJT3oCke7UabNbWpDVeo25Gmc79aVffdfoWzkuY0EIhg/AkgpuGQUIvachY98dJB'
    'kDTDeyqTM5vOHyumLRkmc/7BRbL71JEexl7ezSLnoaeVCsCjT0YlGpQ9WFVgnJuQNh2pL4x3+AWJb6gcSaA5dUxRBz3JK+vggo7S'
    'cLBisBoYjVF6cqL0rdUqOd2qVtAw3ve1tXDdxqafR+krZFZRM5pZT90g6WtNiie9DIDEZaqFrgWYAVG8UvoqGtY+tDRHiIfEakOT'
    'mHJq5hzwozTyquGJttUARjbjkN6g2Y2MJK1V8Vj/pAOcutUYca4lfCgjAaZG+BtWkbpIbQHdHMULw/lkPuY956MNco2smKIN8zBb'
    '72IhFbTcgcvOeZaj2dX3w0yJ1Mw4s+74wsbDvBViUEMMsoJDxKXbT5Ocpv5Esn1gOQzzU2oavZI018Y8M6mEMOFnyWtQMc9Jt5nE'
    'Wufgok6l9MKPWphegCZgqlVCB5/ee/rkU+VkqNODoy0WHY1hUP7sQonxSUrLxyBJWDz05CwVAWCxN07mvKbLJG6cpOKPUSOS+69p'
    'Dd9LcTwWu1RAsOrrDgj0X2/jMJ33Mo0ZqO2Zq8JUXmLmzJkBQdOnsTf4MMOXIH3NPb4LKQNlu7kT/fBPUm3fWF7q/8N/tE/nvBft'
    'kdSIbJpjFatzf65Vl083G+a4Oo71ysek7aZjpI9l5hTpqDrIYtdmc3T0Blasrn5D4thBvUCGCqdBwhiTxnjvVb71CmdkgXhhLLM+'
    'qgU0WjzD+McnhaLeBCU4SpDKOlDDSSa7CAT0B7lkyL4rZZ1IXQ4alHzY18PYQEHIWumYyxpTEgrHhZz1HrmTvU0ZqAUTLC4SPdzc'
    'TSV7bM5IWNh72HNOZz+jmouTVjb5jBEnkFCga13BI5iHm+HRSGvrNH74p1/mL+OMLjieAyBDmiiTkVb+wBhizz0tEy7SxxhonQ5G'
    'cl6y6QenjHs/1YEdKIZktiMhfT66+u6EVxTNEftzTuJxGffsU+mbrXQOrQz+mK34GErAr7RuEWoMjRd0ZtKSLEYrjH2m08GDlTae'
    'Z9Z5A6A/mZBSTCYWKyxLVCQNQIAguAgXm4PyOAcmBZ1wxZhzudVRLSMsOhUjUJmtYMi5Zlzdw5o3ZxQjWwFMnFy7yMulbFjQuBgB'
    'H6uQaKEZB0YbSslbqS6rr+vq2LrNZJqWcDDR6LW0bKGQXYDljri1tNjuBDVjpbJKc5SB7Ng8ekLiaXJvb/8pZrFISKSO5qA0E9oX'
    'IfjGOTM/Lmy+qeDspXJswcNXYApEYRMaVs6bnOau5B9n6jjX1+w0stzF6QRxJyY1IBX6UYwl4ryKDTNmlcM7jXgV1ce2QsSeXgXl'
    'XbnASCmHhu0dTQ0Pca3E+rLhPdarFaUPD/jhnyCQ9ejw7vrhPwo5uOgSDZZQFufiY3NgS02rqbIRxF/Y6LEEV2XVKCoTg2kl3cYL'
    'I/2AljEjifbCRlO1rIBkAjtHX87ZI2aPPE85k7SoXNXK/wqc5+VATfmWgIErdSHIWxxxItyiFS6XZxXYyAX/knqLsFx9YBFLcgxb'
    'jetKSs1Av556yiHqR8+3Ny3hlKn9CMmmJO/ZEU5FL34s3dQLBMZDLGVnXNCzSEU4cdBXUF01/wRJ6WnutlBp5WRpUDQrETi62a7v'
    'ImGxydnaLBn7T54e7OxB3CgFQBFNUw5Uxm74OL8a7wv289j5kgzvkebLc6z/Jo4srRy4kSfrDaixhPPF0WOx1/DY48Z6Q2YsSCX1'
    'kl/HjhV23vHhoo1MJYkmY2tCv5BiCRlWZ4ENC0Kj11Cf/dsiwrIhgPrp+Ng+ro7myvGsAO6ANskCdRVIqxKYg0mYfo/EKOTHKVmS'
    'vL+0FtCWpkpVA69gQtZoAoW8ZegD1ryYruX0aSpyNfAoo2jlYpxdrnyZNbm5MbsX+XaJkDDStZfMhrgIUr3XwtxVPS4cVGCXDlrg'
    'ObeazL/SlITLrN2O3jM9eS9qHvlZ+MnYNJ6FjXPwjBkANGi74iDjJQWnF3Ey4fBl0VpbbVeAuXudy4VkNhs9r/LzrbQrsNP2zWi8'
    '7lgHZesivbx/kV22Eegx5g0PW7WuwK8wsO5NmsIvswsZh0sMAg2GvCUNChc1L8efth2q7c+X2fKHuGgWL6RuPEXYqxWEao0ZkI42'
    'Tpo6zPcu0t7q+uiyGSbKSisLJLZGHC85trxAnl3OYccQC9SYGvejyYboKG62XTjvbu0R6V0QvJTHIsLjqzl6Fv+aRq0pj3IoXs2F'
    'xqhbzTtLkSa/zL7MWG0OcR7ZFLBgjzVLo+llnwWeCu3RZHRPHtdc/naf7+hALblQex7Cqbn2OVCztHl2hMu1Sy7U5vsBipQ1od1z'
    '4GVb+hh4zuVK64xb/iQdY9e0GfWlzRvnutyx/HJ9wq56MLxHTO1Xy54hYCQ6Fcuvpkf4Xk+53q/CNrcExGQXDSe0yBpGYoPsAdd8'
    'EPsOiU6UmmKaDPFbc5TSMo/PByjLQgDd1NMigD6OYVie8vdT/v7aXNuoaRqRix80LyskK6bBARx4/PjwDjxU05DxJXgy/Da8j02T'
    'hkHXPaz5eoBcioJ+fF9+vHw35LiS0qTVkQKCgGosScaM2d9inOlv+fFPt/cPNp8BlXew9by/t6t4uktKFUwW2ELan1Ee/mh0Px9v'
    'r5KBHsBJ1qL+XXfxrdH/vh5t+AwAlUQqLcSKSR2Ii9Mz7TVL3A0kmeFTEwOAXytcBZzdpyoPXYof7KXu7St0JPjxEP/qptko+aZl'
    'boTas6YUhVpZWWVfCNELg0KJcj5prblyOa9Sb+UuzcZKu33pQVH4h8zizaYkcQWCYgUf/CyNlXbXtIuk+TE3Lb6FZtuv/hRd2+Gl'
    'm6XHpbW+3AJJKd0zYFkzGEC9aw4GmJLBoNnzavH2z+Fi6n9DOidPGKach3Otd2TgXCqqhoObXqJs0IdKjblJU/I0yDzc0Gq19R0n'
    '84hTghlKnY0UEx3Mxe3AEL0G7s9L2JInvxAhJ1AgXo/h/HVJFZXoNjsv2crA9utGv1JfuRR8CPIRByLZgVRcfTdNhewxBOL10gGs'
    'CiIobiZlGQ5Gr1jehl9kWBAOeh0Xik+FYTgGGoHlLBJMBRJ3CtsQmjCmcvA6e8PaLkr0eaMNYkkxWfDd+cn9eVncP06z+0l2FukJ'
    'xC4gm4blhrxKs9t4Ujv8Nq+/5fZUHfB8x/IoNmypSdujMPTnMQzldKPPJYHOutUlYGMcN5YHz/cz+WTg4lAd5UIAbdAY7b4Q9wWt'
    'sJ7EihvI5jD5ZAhTl7p3l6/NsI7Q8rZ4AdHiphZEvGoL/NE2c/OtEIrRk6d7wcNpbOkr8fD5VIG0Eki+ynoTq1JMKqf3B3aLWULv'
    'RQvn6YKIoYM9YFQWZh4jaCDzj5bBWYRLKoeO0KlVEryZVdkjHCzALngsbOVnNreBY2dC0ebS5APSU0loMJAV7K76Ko5QgcFxjEwc'
    'LOpfkcIF2CripJ5njC+R9OC5jj0YHyX2s5TKNSlCpAQL5CnAkhX0EO5fxlhINNXBW80fUtPc4LnNSWXwiQLeL65+Qp2qhYJMWMvi'
    'oTQDN5UwPV+m6QwGWcE8iL2dIK62nqEoskX9nEi1nyBnlT8ezuamuPaIvzj32XQrzFu2duLjuvrSuiKAqmvh9ohufN/nftZth+Ux'
    'nwICKs0FwBDKVkosJNgFKSKivSCzIhAzXCpjxteMvpqnjM8i50ws8KQdm+YhXiub6iEmrQqirzj9h4PoTB/E5xatjSp/nY4ATTPT'
    'QnqsX/ubz54yCoWgzgx0koykozuajn9N7+/+Op0ic65VMBlJ82vSZ+1PT3cHT/qfPts86D/piEgjXZOGMBlvfMRsW69DKrapIbyr'
    'wBS8Vs4tbO2gIprL+wJNCHe7dOGBQY1byBteXnVXyeq5ezfLl+Kh3b3rI6JV0dBwKOiGkcSSu3ZP3O0ha8+UmH8rMQF5yrcR0w+M'
    'LMLYYgbfCx876AXdcXvMNG6+gjsW5VImxiG3KHiAB3SCx9wKVa3jQNXwtKX4aIyOhmy5fUZcK8YKOBnCWfP4A94mAdBvpVgR47+J'
    'qGtiawppBpcA3GCrcxlKuQhiZvOoagpNAEbJwo1DcwGSmySJIUNY6xMTwWZL/NzlXRvWed1orHWjfWz2Fyq4ZKcpVgNtnBeh7IV4'
    'DGe7sb6kBa+w+IZ2vEltPKABrECyxJVE38rKbTzsRtW067uape3PNUfDJUvBK5Wn9++znpTLBYmZN6QCL6Dr3FcLid/aKPmLqpgs'
    'Plb2JQ9II/6WgptWJd602sLSumxph6WyTxN6ziVxWkOOcVjeJasdVrp09y6UtLt3jT+jskYb7+vsQi+wz+U62QCnq3YuH3EdYOIf'
    'yy+khhlOOWRbG+yvcKGfKxXX1e9IdKqaYYbG7HkRaqwJz/WI8sLbgs71Epwivy00a0zCnByzaliYeRtc1+pPpC6ZGgQsJUbS6xkR'
    'pcUdtOlT2vWRwMYWDZYbJDDyjl+EaoF6yzTBSYMm7/7w+7rq+E90vdqaY5+inQuGTA2yV1rBG9z8IOoBzgrutkJ5WXJ4yfC2OiKi'
    'cZMpVy+JiFDh0zEink52D3ZAJoTLnMYemzDLx8lxMhqRykEimCSjJonWCK2GUr3GkmxYQFLCLndYDVwKfJri6JBQ2jlHRywuJcwq'
    'EmOfm9wTI+u4voPG4PFun1NeFZEN2us5t+K4lCUFR+uws4ahz1FcYAg0JCQcK3qhHRatQ4CGjlrs3BwJrFGLXtwoltfNc7hvTGpO'
    'quiGJl2hPoQyMICfrUroxLMxAoDZXvQ3UKo71+jTaWbYbKt+DoNnXmtMiFXAmX5cRQ6Fgw2E9Q+MgTDOj1WbrzMOpMTG2Qeqyqhm'
    'wWnGBlXPg/40+QJJR/KpEs0hZjltcCXHyYRpsKvbQmDf55xoi4vFHjLntUi9whdKoTZ6i/iUVeKWBqrCAEGI16uhhXxpmOCGcNP2'
    'cnqxukBThXLMDzY55hsxNBXIToEl6/z//2WFpiyUhRfPcXt6afsOZUPuuu6WPweO/isLHEEkG9RgdgMtOH4sHJWHCRX4gX7NWpTk'
    '1qWcrTcJ/X6dsGhWjqmY/9O9xij+cZhatbaxNvWuzGPMHElQ76lP6ZsWQjZ15nIHMgcYBpNko7X20YernWiN/7cq/2u3F1rvmt5x'
    'NbD/LO8d6kx29AENiOEejwbs37uWMNsGkoQpG/NNio28cTzlmM4pTUOZdDeLU86b3MVfHOJhNZxW6sZgQAryYOC7tg9NaCieduPR'
    'aBDr3QgfBccxTRGITKiVJikfQHug2V2yy14m4+lGU3X7g83PHz+9+lfb/mmnib+2yjRqyWFuMmBqOsPxMsTY8pTOnY3DpmIEdyqI'
    'Ybw1TuL5eMYjdV0HrflhSuVtjWSfDe6X8XHKaWb4n2Qxq13WNf3EwE+7PPDobSmxPlFvxHGkW6RmU2g28CjXvCoF4fyjwrTmFUrr'
    'wSodAKP8OUpMfpLiL7a9a03gpOZSh7joIPDiLg8dYnsKiqCzEgRpzWuarJ9F0aZ97Cw425wLwERhljzVx4x7w2ercbL46NB3gA78'
    '2CnyE0VFra6+S+lB+pnpGJxIeLfaM0+r76bT8+y4uXibCBoBhAu2AKPHjeaTKdNHL9f6wQhfzmmfx+UwTSWqTcd+hg27sdYO9xUA'
    'gWF/bTTns5N7Hzbb18yA7eKPG84loQc3nsHXXpwYkRQNoQijtr77zBKXu1TFjgkKCrAsXSWGi8Rj3NXZcdBUJf+sHU4LX1ydS/WI'
    '1E1jduxPojdv2fEbTU7lnzeeq+zYXdJaXIte8KzZXrrsromc81K7fZ9u6IELtUzi8xwwHpP4mxZSNE0weHhoADmO2m1WIYaQJdnx'
    'oWqpR4sAiqFlqjmfvJbalw6dG9iRnAzKqwTJn3yw4/tFFvULL8f20lh8dKVksUvnL/g/aIhOnaJsBsmwcsnPo4erg9XV1SqgKxu5'
    '88lxIWnSJv0Y08AQ9hOgEgyVYMnjEGJj6Oq3kZ9/bFnMl8XdpJxt7YMPu9QT6atLv7/6bTdoyVHSAypSXhfgR+G79rznSSmGJvND'
    'x6lNqj1pPka0kfSNze2dwcHezq93us322zlW4b+eaIq+nyeD1cNxIvEH6FLtLawfUtcucN2i4nkZgWUIP9HEULP0n0GZvk7ur62u'
    'P+x1V08uVS+66cSwcI6ZW5wGjthbox5wr1mu3jFIV1Z8r82FjKkvM/bFGsd4NTLqsYcIdtrV73BSSV2UkZCLrS452rrRZlX56kSh'
    '4sZutsUGQwQYAJDe7JY2iQM1A3D7nCJJGDO5RO32X9z+nxrEsr942//QDl199PAh/5f+qfz3weraI/udfL+29mj94V9Eq3/xJ/gH'
    'IJMFPf4v/v/5Dwzvx6acMHDldhntj1EFSJZYhAH6XCRDUB1wFZ/FEqDvgR75L+SCaUIqjFQZ012T+TQ6J6NyGLvdUSbMYnQWj+da'
    'ljVNilRrIM8VMXGkvPCIiGrkgrq1bTy16kgFH3yCQL8CcWjSGJcocdM9ZUV/bbI5QhrMhivJ6zjfImN5Dq++yxiX1JAJXv3GFIuH'
    'x9gkHSXAhWiQpEgFL4jrkVHTWFoIDcv3a4q0Mw+TxV7EyRsWkaA6L7Tff/i9IDcYkM9slH8StQSdGBicXpGoXFjkM9Yh/UbPmDpI'
    'hp7/Ma0qrpS0G+VaoypPoTZi8ArNZGIcGaHgY/zw+5LBfOlUYr8qR4jyYwBKIbzwSaOxJXXnGhGkbqzzAc63l5KS48p+Pwg4UwWL'
    'Iuj3WdyILTsQvb8NH0MiD9NpWg1sMdeMwHfKkWSAjXDccmlWw9IRc/aSFgZns1QKn01giQukk29iW3THVdJ0aowVlEVCJA0v+4jT'
    'FwWPDWzaWgiRXf1ukuDtOFkkFyAqudYCYGmhJ6fQDAYn8xkU8YFxlsVZBgiNlA4XC4BqfDMNH2HdA0DVj/A9SbNwReEv0yj+vgYm'
    'tQao3YBBc2rwgEH/B4M2qR8lghCttnHgKbfFEpR6zSSRRzOF9ql58mkyG8g3Ha52nsB7hkmalKFaBkzVvyY9s/9wdd2+HYdrtaVn'
    'pBOUu5XQVUe+3v/04JqGeG2aZj7nhdpRuXfdP4sNuTCnQbPf7R/Y727XEJNVgdUSA2+A3hu7/f2dfe9bDzPVBN9E6hatSc++wzjn'
    'fD+yXAb8sQfxSa2sr7LnbzafjpNDJg3vCHd4pRB310iWcyMgJEotckHK34HIOFNcEI6jzhJTdeuojE2aVy96Qf9+9QJBSonTsMrX'
    'etyJDu6ttd1Wt8ROks0rWcPhxrSh2xckbUcvOH8gekHnPi2pF91oj88zFLGCRkwOCpYK0hajuwzHtCM1oj5CGSesdhXIeE9OD4Py'
    'abD0SDaOcfzkZc6JqZA8sZjJZYqcW+kdzg3BbGNEHKk2zkAQ7Y8uIxZ0pL6+4wZ4I1rtwlPM//KrJFvpiCezfNUOayV5zEOKjjT6'
    '+YY37yGfDUPeWk9FXpZYKafpDM+edFU/8J7m/BSo2cM6aeH7LvLyPX+2IDC8t8FNRnejzKnSDJFAv3hfAS1iQ5+LLHmY17QEDnud'
    'qHdv7cjtDbM4ejZ5iW6FJOZUBzvCrhu8AhjnskRra72jJbvOtWxSFLyzD3Fi26adHHoHWAWtlvR/Q5/Wjv6biMeE7OjVdtsfGVX4'
    'ZXDusxdBZ3ytu9r25n3hN93XA5dW4ILo0bchd8cz7Ac9vwOLRcLHzHIgmDUM+sLr1jAfQD+wsRrNVmQhdA27BNNnwdpcSghli09m'
    'xfnCjx73hJqzhnViwYWjFBHfDJOpnHjdX+7vbD9JgKsrFfe1T9aEBE7P4PNkYCCV+YmCzfotn2uk0Z7Q6TNC9GUjm3Z5hT9Yb4ey'
    'UPD650E+CCsTgKYzMG2ZQCEDm39v5/Hm9pMd5aQZMr87N7StmWyghUFY/4VIax6OF6yCzIxuQzMPKA3NVphcfc+xDmgfUphBE/EK'
    'okDTBifd6eErILzy3/TxBZAYyyRmPJ9ScC0yK8MkXX8cn6l8teAXmaTjTFKDecGyWxOi0+wkAV4l7T1GfOx4iRduaFwahSp+ZwLr'
    '4aB3y3QGRY7b3Otvbj9/srnHuG3arjIFacaDS9/hpB6w08CvYxJ2uF0TghQzgoW+nBGccEGa5KkmlJtUgU7QQ0mzsrD+8YzUY48e'
    'lDVIhfITOC0TfUE2sKgwc8VO8fYYHsWL8GP9gg2ISZrNBZ3fcOM5BkFJuTI4cBbUyeKRjejnrBSQIe0ew5AghcsR4XFTmyYNRTDj'
    'y0SBn22tBk8+VkhWOV1LASgsFSIJPOqySNx6wBCOaMljFFmScIKL6shI+yo9HEK+tsrPJ6KGVUt8bi8Il9tBvS+nN8ksUFeYk8eO'
    'sktpsBFU+tD2xz5sLXu89BH0ObJrW57o4H9re0k20OWIqR3oEtugEZq1hFsLqjR81viGNq59eXdn9JMNd2/1db8gZS+pIQ9sLhDZ'
    'KOMNTB4/06K7SB+odYWSOmM60YsulAPR69k91zE6sR8dXd7QmL5/2Jh+ec975frGmkytwpVIvJ2Q1cdbQVD3o1x4KCXhbYHju+rl'
    'bbLIMadBANVhhZM2fO6+8USQ9QY7B0AAaMEiufqaoXpmBHS3fBlPE0y0yG75u8IZZ55iIhcY24tXlz07RRdhc5fRWWlG/MJv99IV'
    'QHpuiR+1rkTGqiAAqOqEzD7k65U9KVxCVoDESeyjDnsfHrWj98Jmv8yeeRFyFWmCjMTzLAWK3ujzuWCtPN3blRm2dlinktcJu4NP'
    'NgC+A1ya5/Y8YICWdF2ypKqtinEiSY/RQT6Ma5k2fMbAV6KnL2zhhYO6Sy9CwqPlixBkNAEo/tIKy4F+VSOc8IshLNQptnfU8X9J'
    'QxU1TK6+URdTNWvS4WaM9alD0QJSj0Lgyy6yMPjCABIJLYYzRVGaOBu+NF88rEQdx0VPbA767UFy7yEdiLNv5OJvDeOJhO9hAA+G'
    'nDwoTb1faUoJW3rRcZ6PQQXIeUZnSXGcl0n4baNCvSnCwtcaw6dXslvjEu4PQS3i4nQaCqSYlCezJsl8fN9QG/AVeO08N4GcP0b/'
    'XmCj5CDoySnd5DwnLRnojmwTDvtscFQwf+WyHTBqLop9ctqFdVgmfz0AQvoGfrahIe42IhLotQe/kwsXa62jBQ/rWLNBawZm38yu'
    'T6DzI7uzbzYmCKuGXSNTaP1DHJP5mCTYBqnltj32Gf4n7s/qbZsrE0Zd2vjoo488e5osEzL3BrwlNMS81mHUJLHpo/v3ZXfUEeUF'
    '87H/6UH4yrTUpLyjyTtmo/Iei8MozS3zUHGmVSzsUKXxl9JJ2/Q8t80f82ISEy61/gtp5fQoxsdyYnBgWZSsPISAM7vSibXOwuX1'
    'FqB+YMuvbmirCp6ZvbX1BwrxkE8hYdhJ2KLHjouNMTX3dZKevpwNEBQ531jtrrWdo0dGBPRMLP/gH/EHyW4+I46cpwT3hJ7Rlu22'
    'H7HFtt24wL8vI5W9Gxfy30u1oTYusAycmLiUdX5RWRyXYYbA4kDLmnhWRzS2veMRjXmZ8PHY6Vwe84u0dB2xWRQLqLjjjDsD5QVK'
    'Xcbsqim7lda2TLUxWErgkjcWquAFOpxAGDOWOkJ41koBU6u0WC2IYE0h44I5C6RpwaYNyLGibApadjUJ4lZUah1zHY+sBHKQ/S8s'
    '6vFCm1gk3BnHnxKCg9dnUCwSSJmZuqzU++Jw8DfYIku1y8oI3VCWV+rsImjBEqI6AI9OtPLJSvtyOQhZhJSDqK4dIWE1LSgn600N'
    'kQSrbYznb0C/rqh3vLWSxdlKu93rPjypjoxmC0RR6waevt4t+PnazcXd7cXMelDrMyMLOEGIRUyCAtSaDKHp4Qpybs9I1tDSH7BO'
    'sXKE2/Rr59SvefCFk9qXQKecTLn09SIQX5cR/3FfNlGnrheeFNSrMSb87ZeWZxtVbsDOHTuTCpfCKa6nwVcs1mU+mml2oj2erdK3'
    'iHV18a+Wp4QnktkdZ6dJS2SvrwgPPIJN0f08BB8U5WQDc856HnnrlTc+cjxCQh/8iJacbeH2oFXRHReDksyo0XyctPBiHf9w6GhY'
    'nfUOKMb+wUEH5lo1e1x894Ob3PZicpjweCv8gYH8qQX0jqypaQvCYjAcp9MNdkIHSgyNiHHwh43oINFva+G2wOTxt/6BomccSYVR'
    '1KJJAbRQO/qp0+Chfa5GMnn4WBnJMWYN3hT0R/zm2gMARlYuPYOjLx6CdMuGyDqi9QSTVSH9noXrKbpHS6xWesqSjy6S6XtrvZ8/'
    'uLx/IavskgE3Ny64sywyWNTgC/qPflGfgKYxgY2LtdXVu9T1Xnft5PKnsIU2LmQV9brrydLbvz39duOC55Uuw1MuRjNOASsrQsvu'
    'NuNauGhy35u9SCYFvHXMVccv0WFtj//CoN6g+Zq3oOvpFVBdg/IU6T5MoeSU/hzNLhdOGG49+pns9cUTxogAvsyqTca2s5dzxPRN'
    'yNrVHTnplvFZ0pKA630ILtaoBkaT8oZw+UVdjKyabLdJobYTYbNv12vTWT2Cu18yEY1PpXsGPNj+57t7fQmP1oWDtMTWK01U3UHN'
    'd+oRQNrh7DTocPDJw6nrVR0nXk5O6Df2GfhMybB6dCR2lwMD3aKCmHJYr4TZlCM3AtuD82oKTbwhpYqRhoF4WsQCYMdsyzMkuYi2'
    'OUrP0izmLAuXyJulNMFBPO02c+n5VpZkUlcqznA/rW38p1IPpjShPVXPK78qYWhPTYRqLRm2D22d8FtS31EOVzFSw2tY39ernO5f'
    'uchqO+g5Pld+PwExEnYsmSFdxAPOW+1uWuZSftZqV/vlpz72IqaYh0C/uGyzdmXL3qo3Sv3/4Mb75brF20Xlq71DfqreobruKCez'
    'rckeHmsftn3rh03EaLEYKHhVq0LyKFZVykqBiV/oZ/d9XU7+9bKgzmgMM3lt3h0pXC72QYteZCmfSiJtcSq1a5R+GsvrdH2Xne9E'
    'DSn6dNfhik70ytElZwrLl6qoH1kdPXzkImZwoGPrP8j7RgW129A9S2F4IwqFlwrtqyNWEgcZ0Ujm6kRzAUnecBcdrh51vL/urR3V'
    '5FFr0nfvmuy/C33G4Qr9Sap59wEd20j2lkf6X9do6G6AlmcCuifAoKl5gvs6NP71dz75j6Kfm9HQL3qNmyYIEB9eV/afP+73fHnP'
    'Rq2XjUi/LTOtvHWm1O+c1TNKvOzGQmoIggTEIK27vkUBUziP4OZRjtggS1GCmUhJRabTx9E4vrHJkEAe+UPqNcuUNSNMzA3zwy+a'
    'dlWRJPHUg0U5fXlzKvmbVnbewqkpYp9kG5kKcVlu2Nb34q+fuBZ/kYynn5pLlxZl8oHpF2V67nKvEJO/XtqIpeC2ly+lHDeVm7iA'
    'TOZZrv+mEyArwe5YTuLxmP4LRr35ZPkzLcU3B4BTqMjm6eurS+9ie6P2podL7xkX5gbN1zO3ICay9C5oB3XP4ZDF0peixXUPVljt'
    're8vvS/L76kWXl/ku/RGj+w8qLnlE/QsLjaae88PNq+dSYuxox6sME/HCYoKxFV3Gaw+H+z7jgLdYtaw4EF+z7P+wY7jOOI0R0Vj'
    '6VzbqPFFmtDhwu3RcV4g8YZzZr7Kl9cIq+PaBN7iLiucUdw14aC4q4okfeI1hw8wvuIuLYzKcNLFxvruGGtqA5kYcTfLB/qFRfkq'
    'NjhQKFpS4y/+/M9/Vv9U638GLoz41iqBrq//QbHPB5X6nwcfrL//5/qfP1X9jzIMhOhbgJWStO2MmaVzF3NTMDcSO7LXGVx4mBrq'
    'G0v0A9XSz8Z4uPrRI9jYdzkctP7+Iy1WZWye96IPbR1PwtkX+XA+jUk/A3oTbvorZN1l0d88+JD0tZO55LVJ2rfhltW+cfQR7Ivo'
    'hVGphL/vJB3a8MZd+C/8hEG+TkieWc4igqn63kiY4FRNZL+BsB5KDiBivIpA+5q093kh+L8J6dhIn6yqiAIwqkSlNueExnEvOWUo'
    'tavvhjOplR17wxlkt9Dfx0mUFnR+5gWjOAZXKhAkqNk4nnZ29X0pCu8jxbDiF+tEDyoKcPAM0vrw2vEx2ZmjNy9DubHspLa45O0X'
    'kliuVFttQUtuyzdB//iijU60v9vferr5bHCw85f97f36ko0vdrY2Hw/2n/4aYM7YEJoUw5MB/ItBmJvg2M+A827zjA/ykYVjq4Lm'
    'MWAh/Eag03vPKzB7z+MBs1nc/GJ6CnSrUI7mPacITZ0gKIRGO9Gnz/t7m4MnAOl43N8DPpPmqPBbeMD0cL7S51bYQMsAW/DO31iY'
    'ja6E4r2K55eRQrk5K1KeZdzRL7vmPdtLr7Bvv3AJ/VsuEReDLeGGnMryybEyfsKBCjZDSBghuOw4kqumMhqmcVPSWBm98yPdWN1G'
    '3ePQmCOPug9DJmq2wzd2nUGdVXIaM89o9BLSbxarACw1F1kQBFWEjQwXuDDDsm+2rh+FECaQxoZnLs6t7cGnIGct2Q8jXKWMaRcj'
    'zE49ZxLIqLz6vhnQgomMFw8xjo1T0Jc1PHftN8JBaVYzkkW4TEhYdpmB2NF8sog+AdWaC1KHb3PoIV7ZHiE6IXWAY0j07NR/5MvE'
    'xdGbneB2eIrOOW+/EOCfIuF78FmRvnNBw5wlJoHZPNI1JEy/yppQSq7BJ2jCcZyTMM+F5bzw77TE17HllxSH/BQmV5Mhec+j2BCc'
    'gjE6oEXVto7CAhMeLx/miaWMqxKRCxDVWiaVvGQVwV7gCBuDXsxk/c6Y14DvDwk+tmzpZxX615Njineh919qQwt0H14PGMbBEpw1'
    'w2d+4R1p+fFXCULZ9AgnjC9dXHkxz661JG6sV3dnBcbQdNHLrHPt29xBL5K0PG7oHIJ9wxXL/ljE5tZOLkvMln9KowCJAS1MKt+l'
    'Sh3zThqwCgoFa3IG7RtxXMsl2pv+fLbgHH4TJIk7kdw/yQM6yL3+5rMF3aalVnSRjOmdWIGSkjthwe1qg33rHJxZzoB5CQe8F2oS'
    'FGUps+WU4Drdq2GTbVjCqPYYmwPW6xsMW6TVSIWM1Bp7VRhdPyN3+YAPXKjI3nBjyKjp1hYCPDrZnervKBei3zGR7gaktI392Esz'
    'oz4Up0npmurKF/5FvLfC3c9N232nGII3RCP9VeT6pCXWZhJ7/gIOd3rrovZtLqOr3824OoId1BjEy7cEwCIoLHcaAQwJp6tvgL/p'
    '0SP/pZq7e8/7jzf3gzd1FyqZiyT1Hfrnwtb86rux4hXQmkxnspE/FdzTT/yjYBcWSLn0Wh+4tEn7gvPhNifx6zyLS82a5KU9jOcj'
    'gKMotAL4fLqVU2/xBOszZDBpfp9fff9NOkTRM60UONJQM/XZHFsz6g/nsSYJ+PfyLl9fXX9EtkZn7eHa+x91onsfftRZj374t1v+'
    'hckk/yqN/t//9X/4O2rzq3iaZ2SzRP/3//Lv/q/f/W9ME0/vNEYN7j//4z//H//nf/fPv/vnf//P/7bpo4DywcMKRZrJgHu1FCM+'
    '0WgZ8fJMWmr/wMWodmdpqLRswiu0llhvG3EhIbJm3AV8VjR3/pL0NkSH9HI9OCUM2Px08+mzZh1oTv7qMrrgK39SLA3W8JbAM41M'
    'j771zkbzLiu80ZjASvStxayqa2+7b54iaSDcxv1laV46nJh5eWPauPJB3kO339NRQtYDY3DQFeoWpHtIeWP1XgxYCZdqeYU7j8xR'
    'tF6b8+6OJr4PExuPxy1/cts8DdSA943VS3hptCtn7cUKzeMKxzClUZ6+FUzfyiXQSZgXXt4mlE7cp3Zk6NaTSB/Z9s8hMzA3C5P+'
]


In [ ]:
# PAQUETE (12/12) · último trozo
PAQUETE += [
    '/sHmk6u/2z94urW5Hz3pS5Fof//p1d9vL5EwHEWiM9gucfqDq2noTVu+Fsa1JqSNcOERbujSkTMJKj7cNF+86q2v0ho9u1zovHhw'
    'zJK06rMenyqVoxayBcd5dprO5qN2TztPx+gpz5lWPrnMitYsHXXcwXVIfx6ZfadnSQdheDaYfKZemdp0xEqnuR1Tie9+LuzCoXkO'
    'krr19x95Z92r5HxjTHYdLexXZ73oHu55dQa4Urmmfdhbf9+TM+hqydl8/D61A0gX9X7+Pg1hqVvjpvibqOONP7n/d2nw+537f9dX'
    'Hz2o+n8fvv/Bn/Gf/mT+3y8cr4bHSzBKLG56BaCt22g4xgYtmsW/tMhwkZ5lEa1eKvYyslxJ5SVZ29DSP/YQTBnsSdKN7fdMm2Bw'
    '5AzZyxIKnYZS6CCznjXLj+vg8qUFnyqlTEJOHI3RnTdgpLNnrUQimkE0ZLHejQ6QUTAWt25SpVzMj8fpKe4EaJY616hBC8wv4EVc'
    'rI/XNrhzMOIZe84HFWEsIXjkSqY7cKlxqOZuXP0u44oFP5duxAQiQC4yoFWW0WMhi07cGJI31+C8uV6kjgtlvPeNKm1HBf/Vb9lT'
    '8kmjEXJbIJ+F6S/gJMLpwK1zPh6/YV3iYU8WliDvNWhqO0LEpOhZ7IDr8O2S12VLF0yNK68Ftfd0BGR5A4+JuWQkkGrzCR3MptKA'
    '2GGjlf5X6Mz2TgQiw20DI4U7inxsW4cv7mMDGGUyVejQ0zeNFBQsbxiqCrxIlfxIPHnAhS5jacuOGB44LBKDDbbWFR+S4B7YiIyg'
    'VR0DLweD35KdhKUKDAWfCM9h+7c/pvbWu1EM532hS9iLh1valuruMRwungcUeNb0+g1NADcpoaABwXqDoEBN9CIl442HUOiWdzwX'
    'ib/a3qzFe/dG6cmJtihvKHARmhZAC0uo7uM/FnuLljFiHMtiIj8GfuvNIiRPNg8CRCjBifLxtQXGhZNaw8TDhkN18a9ZwHpp3DEI'
    'eYKjId5fbO2wvchfk5bbiGv4WMBbA6BxR3dRmg3Hc+zNmAOQV78t05m4cYxDdJz7TD4VR5LuaEQrS2rTA8OyhUbs2/8qzkSQwDLB'
    'tk3kVt+zxLAfhdnMwIq7I11nrIGA4SzLlc3HYp7QXE4rclBQOOjw6TY297Z+8fSLnf1BZVZEN27WcXGp4btYsOrXczbCIneZrU7D'
    'AAaJBFWYG0vr4BMJ7/9i8x4itUDIysGJZQKoimhjxJDFWkGWJbv4gaA4BceJAwwS97PuBwANUMseqo/hJOi+TL6hY57mv0U699qj'
    'I0dQEWcJ4kILBEEM4VVIhCYAng2Zo0aJ0jGVc5aVI4N6ZPtob7blo8z4EM9ADEyvVjtRNfQQatrwjhGWaWmjXalPYVwVmEILJBF8'
    'a6XUxPTukG9DdO2iyaOGPPGKExZJcTzCyAqUeZ62L41xrhGU3DtSlQ3ZYVoqyzMUFcmTwrkSv841RsAgbZiFQy7hpz115EYtG0go'
    'zZZjZZDAg4KmccPDKvPHyh9RGbKmWer+qDFrMvubgIB8HQaBsE7g2sC+taDrHJyk/9KI0WnRWkhlRkoTX7CYX6zIDyFg9SjESeBb'
    'KxVUZlQWqp+UJ30kyd8YXFi6n5iXQHCwfbh6FNzDU3CIO484VgPkJdyOb8AfgrKc6jvpA2geaGrhyyib7cXXc5PlCrIWGXbMamxa'
    'fBGfZsem15uXXvjNPER7wpfqV961XIeI8ZDF1tLVIq+rzot2O6Sy0coALKJ43JKdLrfXM6U/1xQOT6U0hOcVw0cZiyvkmGNP7+ZC'
    'kxjOXaNQwJb4pMmSkdFPUAZslNXYlzoAkEWt8nAOyDVOvsGBIZ2CqwGlKqO5JMhIeER2sWYX4gYtmHY67TEJ2NzT0qCOsZ7IaNlK'
    'px7CLfHJxb7qkybgYy7ODldElqwcXTZDH5KC1/AQH7olcWSn5shr0uHSyAroXeiNLss/TG6unBXNbxUpRpprG9/e8jPD6opF63g+'
    'mVqEEcDrN5cQze0lwtmRhPqlf35Ad5fzg21EXRjPPK5IY2hYk8utLtFJutEL9OiFXUQ0rMg3XWFU+hVusKU6CMeJafIm4OscJZGH'
    'WuN0C0+3adNlK8CRLFbCmdVOb3hnqUhnMFsOtH8btftHPYAeYILF7SO54mmEixgy3j2efPRuuRZIxsLQm1YWgOhVkcS8rnVXu6v6'
    'rmPvprASCJ634JVRz4HZYJccr4261k1jh7Yk6ShMF7I2hBDTMTCaXRJ14BGY6g4jfha8RWkOofXBf/iN+Ku/wS4L38I8nA6IVX5d'
    'e0h0mwHI5quBrh9qMmzC7tOOK3qqaonuF9Uj2ktylX+yEdXs/1u2GVSQ2CnggWlGUmuub1HFlNhSdK8g/8kSeYYh23O2y9NM6kui'
    'g50nZMjAJKhAl9SaB77hL+PJEyZ+qjnJLU3BQdisSmHB+FTi1VEiVXZs9GyZI3td6M4TcAZWUBfcykOpIZNv0JF+6Za4XeZu4NAz'
    'HjgzJXoOoyCnshnkh85CzfTiYy+7F/LK8nj/4dXqq+U30391haOVpmK2SKWlH+O2JHf6ydMFbl9ZaLZ6L9jndepJdZxuoaaEt1R/'
    'P6pXXsxN9qujauBeFitdOvD+DF7K07gWd1xnYQz9EsN6EdIOBGtN2aIpslzakhWp1zQkJoEn7+sTHbAWbqhuVLi466qO6pMPDBIZ'
    'PcKA5QaDHNDbIcGyPumWk95ExpDOtZz2VxwJNVC54j+py7u5FVSuWE6jRUC2W2OxIaqlRoC/6DqR/6UI6nY7wMZdAims/b5WAagC'
    '/S5i+95aGeAOwXVXTVqjsf6rRTdeqMGZ6lbh2/OUdgMraGeMC2osKJSnJdUoT4j88+VLGHKQUgFx73UEsuvs6jfxx64KlY559Umy'
    'qj6kYaBzwTt0ZdAU/ER2pRCSV/U17mbYtUX1x29hMXTJtgqfdaXzh1rd9oLb9Ot1l3TTpvyYO63vLAbA57JmzG1qLPv/uLtcXTBS'
    'f/idF80I0xLc2E5E17ZUEePacCXhMHjM4h11mU7qiKz0X3CDvAsAQoQXCcXSYuDdyHsTTIed6sXSgRErr+WdDG3UKdIP0nv/h3YI'
    'QMmNeXo7tbvQGPf81XXaIVjnFh51820TuhhPBDIS4zq3OamG82mO5UhB1ez28/4XO/qLbjhNtImiLTB/bj7ZqU1auXjVe/DhJcrK'
    '6TmXflG32yE/Ti+GBvxl6Oipff1bNNVb5L5CaXT/WcRpC09/3d+Tt7z6+y6npy/xaLMtQqIrW6THbtbqyLciu18sgmxei276n4xI'
    'FnL0zfljWYWfcLlTHMSExK1dJIbLc3mBKhTxoD6ZjcmOUcw7xrTxa5XFF3EzvStTseHN3CKRg3BRS7DY057/I+6icwtyMYBhCDAY'
    'FDttEYJBfglFoxrSfDl/XBSH7JXyxTBNKQ8cG8vuQTfK1SisK+Nue5LUXn9ifdvsu5/4yve1eU+4pLe+TsIi6z2oAE+4hDGI60Bv'
    'qRLt/dFEbu8+D4ip5O/j34NhOmNK+rdOAXdD/efDBx88quT/rD/4M//bny7/54BmXwwdy4PGER8kmMRjlP0hrQM5k0uSSoQzQADk'
    'mz/8/mQOzggthXclZ5Eya8XlJ83GH/72HyRTAq70k6vvhumYvoJe3ovu3v3h92CSp0vhcwEWhilwGdm8jZlWwGluDGuojRMUKEWJ'
    'fRROtXKO2tZpUpwkNnsli5GXNP7k7l16NbmDXl/IwQSDagjn/jg9LXJkbceKxIiMGXlWAqScr4QiBXVVWZg4430VTcfxnDN2SkPQ'
    'QC9Hj+JkjNIQkncdBQOJ0QntRdCNNTixCtuTS7y8waSupMeAGDEJOfTG8ykXhHE2jHKv4nF8rGEUG24UNXxQ8tDSvufgxoFWtXH9'
    'Fh2WxxygZng4HCFDU+A7TDHQNrRdplHzw+6HDz9CeL/5IX/I8gYtGMC6VtcV6mDlzW05GwfEBUpdXgYkUvFxXCScGpaUDD559R3N'
    '+zi3gUoAqpqaq5nkYL1xgapNwKhNs1hSWnq7tAvHYqZy1bTd2tN91QENnBmYTsQkTVxt1TEc0Lyey2XeDs5UAxYOgy2U80latG9Z'
    '70pdw2KX8Kwa+ULES4pCMkuRvBYrNnt+LJGkjgW1qVjeYnbK9hGPKqfFm/tw0Jk7fY9B/soL6kpvbFjUO4ttVjY1KhnZn28iIfvC'
    'dvP9R/SXeRrybL3xeo9shqZ/N7MA/6t/Y3tE7egnztCF0uonV3/wYdv8vb2zR09++uvNLSRic17200/3uOijeoetQ3BjBgVESj9a'
    'slkiKQYnlVD/OZStY9KNW/znwlU1l3Vu1drDh51HFrTIXndI33cfeZdpiYb3QLnsQVe+d1d6NRzhpfRDdx1WJw07u+2M8ebfQf2T'
    'q0xzj7qP3l+NXk1o8z/qfvQRPvJlj+hr6M2P6Dvv8tUOvfSryQ//vvr01W7w2msfrX5YeRm+jL93l62vcbZoXLzOg7FeX+OLtPLa'
    '7ZGTptuuLVvE0W7621gS79vevjFrq71sje1u7h083Yv62xFWVn8frJ6ZxPcKOYO42qluyUlt7kYoOlooBeqfoSRyJlxz/trDcYsc'
    'Kzbfou2EpJIcDAdXvzlOZjDh3Bs3vZRKabwpMO3yB73menA5KuPmUtgpvZYA9RlpyJhOPTDSTBtjfDQpjmy8jdIt+0/jxiLzM/rV'
    'k8Stm6dpr7+/+xxlGvvR1s7eXn+LPvW4niVzWWIs5OkoLmtmq7G1ub+zPyDRtmGlwmmS3xvG03ukEmOAbqz3smvX3fgINz4LLt+c'
    'w5odp4zAuRVnpDMUMecL7F/9ZpQl51GWRp8n4+OcbJSk0upJWlKr62j1unVkUw+orTMmx4bXtah2kY5409pmSR2SzLp5Rg0Ewgnd'
    'fpWOr36rrQsj1IjWa907Q1Q19+ejq+9OCiRAS6tCvivDADbLXTLCUJrWibbSuVC5jmk4jqEJPR7nyeQECRNpVjsA7+sAjOPTnNZR'
    '+gpYG/yYte6jh+tmFJhmKCftF8B6I27pqCHYz5IC05G1jkVv5l/kyktaBlimXdinKZCZ5QbrxH256Fv2ZdGFueGyZzPyM5O+T50X'
    'jCzxp3q+sSCLiZ19XaOqqQCRzU1i+KX6WpY99Yd/uuBLD3sPV48uu93uD/8R/pduYXUdo/V095++8QZ7vssbzNdlnUXQWVBlm0vb'
    'ZXool+QW6vlRHNIPGyW/Z4FtWaF3rbOy3/F+uq2uL9XmsgZIp+Ds3EPNzKPJ58XJI8ul3WbZoMXn2wtvLyn6iX519dt2nTxp3KqI'
    'lDaDrGuc0/525PgI0xXY0s4mWwG86PPjsRoHar6czMXM4kg524JL5ErD1qsGwuWj7tra+k3Cxdwcj7nIBANMKqvc6mDc5Jypl5Xa'
    'wDUC0wpGLLLkFLkD1PTah++veeOg93IHYLogbsjTvUz+6Y1VIfhorfPgNkLQ3F/OYdUiHxdUi2PA8NULLzfKixLsYffDtdVlEszc'
    '6X0pr7lEEOvlN0hjGqNf5i/JHEvK43lxmutyLJORpIxPwYQxEahL0yQvemvVC54SOLlJwY9PUjJQZf3JHGi+RaWXL3k41vzhSLis'
    'OecK7BwjCWKxLHr4AWnvRZHixPAHHDPIRqyd3Vr5LhaqFfK8wf9Lk/NTkNhwppiT4eibleLbO74tp5e/mUFnnhFYdfYNF3y+QlKv'
    '58wlnS+1F8Ap67rcBYx/QqN1AcxDuBousR4uaIIsePGyV6g0jDDM9g6ZfQdQ957ssPLs8coDx0XdGSy1OQDydjXZGw5NLDcLONnz'
    'QHOGc464npXiWkJSmjmdsK1MelKdnlpdsYH4bjcUhQgJZj9KQ7QGCBNENQIU8QIaWRFfb52I0+sdPP8WHYg+nbPfLWeAFmSDKUOQ'
    'MIhn1QbXPqL+CFk4iumm4KbWXFEUM1GX1x5ynTkgG3j5MCmJujk6DtngEAncZnJpCgpBYNXJIEus1ZRZfk+VAobM1bFqqy2LtP5A'
    'EhhABCcDXKTj3j3nb7nEml7YfBfqgWpRe20/AMN5+vRl18hp9LJ2l2GPIjZLEvYSf3B/DnsfrB8th7K+fq+7HW46z1GXxcHjBFBI'
    'AXR0nE6mabxcloUgzopcvdAmm4OMnieEBNLqkh5Vp4v786P7YrcFySvbC5CiSs6i7cnNWvj2k/7e01+TtCO5sh1tPT2odXctLiXT'
    'A11N4XOc35IXyzX9oGGSF+4Fq+0P//g/Rxfyw6VeYEJtzEy6f04n46SPQNqabfQP//j3EjQwWkTzz2i0QfxvigzU+K1H/26u/3/4'
    'qFr/v776YO3P8b8/cfwPyVkkxUyQB+lxvCLwR3qa5ep3QsTMFfsbNdQxu5+k45l4CQUq8SyBzlwykz2pskjNw/mn5tk4bnAcZxID'
    'lj/S2nb40jj+dywUfPvPH29tPulvb3qGHZ+38Tg+LrhIvpmexU2OOR3HambjqiZKZIoSP3KBKU6pMUfATqSiM3qSI85IvSPtDcU6'
    'jkOmgRqeGwCaoj/87T/w+4ykVkeIAR3mJsfHQKgeHye4tlHm9HgEMAH+R4IYZwajTDL6X95kMN0ofkn3MDvfrJgLiALj49HfPS4g'
    'BaomfA/ntqRUg7EhPTvnUXF1ypP+4/7gi/7B5p45R4IAXDfateHOnuJFlgJnZNREDbXhQEGa+vbOwG9TG8QEaoOPxzndEmdhESZZ'
    '0Ve/AU8E6sAzb7UwyCJbmF42vWhygiIjqLtkm09jCQ/KHxoWbEC2dxkkc8S4ETTxomuYZjIu1qXftSpd2kalxn+WYcO3hFlbkP6b'
    'JmcOs3bPfNExm6wY0B6F8ZoXE6TQDNxur8LX3olaZm13eH97sQ1vNTin9g+/56RcmpLUMGXEQ5IIEzK6x2TuM3olsDuSY1yWRA9I'
    'Yy+T8hPJmYKG7DuPBDGNcUImKdYrowliJx3P4aQAViAYN5EfAIJL7FZSfpa09jl1gNY4i60pu9uUuPsco6PgF0L1XSpiZ12XWB7O'
    'UPWnEyNOD1hdS+/Z5U6OaAQzRAtglpFgo6tygw1RoOqd76ctE4/9m3lIp0h8pI06Ts6ddTOirVfQsq+7bzsZJiiex3DHx/kpGy0K'
    'nP3VHDjV9p1fxqdLHr119dsJI57SOuH6yKdfkKBQltH5jM6OfCK3kpE4rOn2KBnNh+qs4StovqNY2FbFDJPGXbYBK9e04Ja0+nkC'
    'GG4+rpDoAQ0aATCGnk1nwxzlUy/zwtyexdnruAxeiR0sYJ4TccEDdJILx6m0x4fi8gZ4TCQ3UGt8kAEj+c56+qA7u4xN8YlgxwIr'
    'WKyhhSFCvSjD4kTzUqcICN8Zl9TP8msaOGBkW4XCYa56eI/ScYyZHXI3MbWQ4fPTGzrCiSCncQZsF8EJTOXsA5Qf36u5hSSya+40'
    'PM205EZ8FMnVIbrh8jZ4EbLNAnwfkhK8NtCmEes1tx9BQO3as/dZ/7Orvzt4+vnmvqSMZBhZFGwopSrXXOWZaCgx4/fwKdZtoLhM'
    'c2jGY6TYM0Ylu3Z0d2oOCq6gnT+Z0hkSHoi+CLwR5bFKEiYCZIXWzYqny6zovSuf0Mx+DE9BzrrFyok0tFKVk+aZaTmOA3B7+t8v'
    '4zP/yc0VfLGi7iwT+VCHsK4QLOhf8SujwcUdQEfqGAKDpBkaU2H6mOYnfMWmQCQJsLbSCnTk1M4naZbm5bIX8QZPESk/qcb2mwab'
    '55jeNx3mt2jqcT5Oz+rnYUlDIoBjhFIQMZhcfc/Obqho+/Mi4nqPY1bfxL/Ma3WlnBcrRoNCOh2zFGasgdSecOK2Zd5q1I98spjG'
    'cMvXTRQOlJcAo9U57E/XanMFybOaxlcC/ZZHp9o9fncdQxJLKU15YVPFN2l1lf6y0u4t6xbHI7yVufiOy1pA1c4pCxZkFGDvIpxS'
    'M0bNFVxg3muFFY1gpzgss3E0TIVpLIpfz8d1A96EzIg9cLSO7MPtHRUjruX+N9Px1W+GpIlLdGMMBLFJ/Jq2ES2WW70oj3U6SnMY'
    'UmUiqyrjs4ROBdqi58EWlp8rU7hwhD+hDowShddZgG6tvvS1MwixgpgIkJJ5FoEutXRLuiZ+cfXdMUI8vAxpxTD7K2+lGWn2k7h+'
    'Jy6MDFIi+VAoZ+pX5Aax5SoSZ8VdYlcCO+tKaGCLkpP3Xz68+j5GYG7Z5qt27mgBU90WOtyK6QBeNKuetzzwfy9Zr4r++cF6gAiK'
    'E2g72u/vRTiGntRCEptbGIDFavN+opo7x5zn0aXzhXZDa5F44fr0vyUpgNd7N69PBTRdOOy9/+io9/6HlzX4t6YPGzbhr7f24NJ2'
    'c8NmEKJ4eoVM9ZVK8VX7prHn6FBl+GlJVS3j9q2nRDQLTEigW/yoOYF/mi4ruoZSoLhx4jQU+WMmzeJMNPc3H/cFHRk96LIfRMq1'
    'YITTX80fO9OrNNPrdTN9qKgivQ8ujyCVl83tDe9V5cyU2ehFF/LBQzr+lWZbS6DWGi5TLi2N5b89zY3wnUPGF6J+ov2rv7Mw9CSb'
    'My+2Cy0wZHYwPHNwzXA6F3419CDdN1m2u8+3DzajzYj/27P4T8zdzNhqAjN5zarVDRWH8Oa303w1JcWnuG01vUtjk46Ba03+Wnj1'
    'csWuiMtUlC/5mMZLnzObA7GUn7KZvYqveQZnKED1CzVqT59q7s3Lax51LPqm5K0NQVRZxTB3IgC64kDAfBKmZ7Fj3XuzrS3Bww3Z'
    'hYyPKmQ3jhUn3KMIQLlrdcs2Q0kRXh10tTvOvwZzhmUBMV+8+yPg4cOj3sNHdYLhVOQ/d5s/9+iksObWRfACPykQScxfBdtcATrx'
    'H4YEpV3aUWfi2ORkKYbHHb1p55c7slUxwuy1tXUTJvVHkA3ThEtqMjbTGVydbjuT9BmubjCkOczZSM/psf5DwqFQ3We7/9nmwdMv'
    '6NCJxaCNfWxTp7J2oycoT9Hmmpw5p6j/kMcGLYVdsElpin1YYSnAQRSP0kSSBrg+hAFNLKFPofJCWNbgcocDzsLryl1/Gb+Ov0pF'
    'yFWwE+EkF+PayEKfNb6KTYcEPvbykUQeGzAYUunIkoFj3lXPYFkp/Mcdej/+M3r8tL/dU3VQnCZQg0wSG40zGarQ9b9iOAEB/005'
    'SBZM8S6jrQg1fZKVbDczbPHpXKwiRnwE5HwXjHNSZiQ1U+pcTE8zCZyaOXHBF9b6oyLnWiZ2ZJncv2bUokV8lgDsqfkS021gZvzx'
    '1BYFuuBclqGFPTiPFQXZhDzGaNW6dLvg3EMPzpI4kkjIKNEGzXuBr8kFZWgUmz98t7WzfdDfP7j6+580TdQg+UZcfOJu/6NPJ9Ud'
    'FjZffEvdirMolh8c3vr8ZBnYyPK7P8+zUxLx8XW3epadNewwjo/nNz10YhY/2dlHvR+h3zHKiJHcP0YchyKZ5G+d9CXjJ1gI7Qow'
    '0rNYN4lLHMXqpuUNdzmyKMTMPp6XtC+MT9OuJgQdu1VwQ++8Co8vOeoWcQ5f1hyIN3HXP1ON7Dz2BDjLQAMADXzai5fddHS5lN73'
    'pNkySYpMpvSyq/HUy3a3Bmer0oV95jCSBjgW60Hjdha3OA3Y8n5MATuVwYPOwpxPKIz7l03jqWKM13H8ZbN3c88mtKzPEnSuKNJj'
    'OupiVnm9ukWrv3DQdnm36GjLZ5x8BbDW8fyU6wluMTJ0MPuDA+k0y7Oh7FIVquL7WP50xgPm8EjJQaTTMPpZ6cciCNYizfuzyrNv'
    '/4TF1t07kzK0ZO9d1ATzWpiG9q2tWiZ5ioQ7wxmfgGDCN4FFar60k4svHppMJz+F54Y0HmYx4wdfevYUV6wtoBysVXKC/n7Jna7u'
    '+I3r8RfSif5kdfl/svyft0/3/Ib5Pw8efPD+WiX/Z/XBw/U/5//8yfJ/AtLn1uPzWXJvN06LqK/4W20Jbs5yH9/FIwGJbVZt9PhX'
    'dOBL6FHg1JWQEftpV8yeGDGrALRx+/n21qaQCjCercSk6bLJMZf70NNZo0WCx9Y8HrP+0zA2rTW9TUCuZFTcoRA5CB3W84NP733Y'
    '4X6VOScEc/2AJg9I2bIjpNYsJFLXN4c4Peisuvp3dJCAnwzYLmmW9xQgGJ1lrd69DavI7s1hvY0KYwPBdivORKHhW6KfzbNXPwcu'
    'PplZbOIz9g5rRR8rd6FBYMiFaoUb0gRuH8HHUl6zl0lSoyTeI9EDBs2FV6nBqJXUsyJ/LVWgEs2ZTGG+mMi1KLM+8AG9+ZAZWtgk'
    'AHY5XDc0ig1q9kXAOdV+QX2WsKCFg6DV1RP2ij496TWnbqgXS2ZSajryBdbYrnBUbCHUgJskLyNWVl7DQTU6j4cSLLPU3PDQ4OYH'
    'yNVmrgEln6QLKlSyETvYdE4U1DeKHnajvWSayEp5GbMNj4BKEdJtV3i5ywQYc3S7n3BmbRftnUKHKuOGUHi0moZclieBI9xFsy2w'
    'ogo9bIrguKcfs4cOqRil8hMsDOIfRdpdWBaLeZYCahncEQprkNNIML6SpYneyufImF+eGiUs2udTkinm+6dIsafN1ol2GOkpHr/t'
    'qok70YGsEEcvj1nlfYhpxcqWxH368PQJ1rcVSKfwvIBNPmfh2H3LXQspyULebHGkNn82jUc/5+DTHcuGynYGp/bFjl4tKXUJj/NZ'
    'ovcmeWnuPUkz8WGobNQrRvnQXAFS4YLLm6TgwaAz5aVHKC8ZAUlAXKNNzWlEua07cKZmIpjn5RybQi85zmfmaeYSyTXU34ezb8zv'
    'sWY0WYpobFSNUC2nptJ27puG7rDDRkhGa1rSy+lQuPper5fUx3jEkNxeIlyLjqMOi6tZOh4lAGCFmACrjwoNfAbwVRtxwN3NJ52o'
    'v7PfiZ7sbHWi5/v9vU70eOegE20d/MvBzm5/Wz5tPdvZ73do2aESnB51mrRquOraje2BfhXVktlp4ss9PYVUhlOPUpIPn+0e3Fvv'
    'uRPPy5wt72tVf3kfUYq5DV5Qe4IBZ6IWJ3OhOeKkomIO6WoKZMqIC2HptGSjC4+4+ndcSjrGqqQdfSc6LeZTScgdJ/AiNh0gfeLV'
    '4Yk7Bz2NowkyV+m1nz09GOz12VLvSjpTIrQoBQm2lfLbldm3K0Xy7coZ/X/y7cp4/O3KiH6xmM6Iz9rkLpI8tP/j0jbw7eGXX385'
    'OnrP3rGYUmmP0nNDK1UCons4np8nrBjAC3z1fbfbbXvN/rdflte1fCc65vzY0kSKdOxdC1+WuLWzFOj2jiFYQW80+Rde4TOSV7v9'
    'zzYBcc02vfYfbsY5u2WNGdR9vv10a+dJ3xHCDKaokxdNhuuBGAOGQ9lWNjmM3CIfJrE7vln8IBeLX8pkLWHxDIWzAIqfpPoxsg9L'
    '1Ao5jJ3vLgmsEZN6UtvoHmkbZRlwgUpHuNuDNEtng0GrTMYnFXQaxlWjr5XcWIXsbD4dJ4dkOgpxCQvcIxWyp5xSnfCSpxlh/yYn'
    '4IXt8ZmvDCjcEB+3woHiTVI6Qn/4t/B+Huf1dKRN8BUejYpdJqwJIZdgFN6vp5lrIaBhoZlL2eVJGitzniYMTBLPkqrsuAybHTCd'
    'eQKMYQZveetsyneifkD8/uP+ufO2u4V1JEzuGAaptqPVYvQTXvo+u3sPQ02D9HD1o0dVb62Svfei4zyHC4VxTRhO1l+/HscA7YJN'
    'ZR4MhC0MMlXpDAmPO4eF19XuoHBhYg2kZKUYigwDofymawKXH+NaOZxAmdq7zfSwWuVMKhzWwSC5nh66s+296BiLV4igDo+P2vU7'
    'Bu921otehUwktkED7XhZt/95ozcaHo2AsSrGYhUaIzIXbi6t6xSpyOzFoFdP2Xrpec2INs82hXCq5GrvcdF44fHJ8PQBHQ6RsBTL'
    'iwW/82ZPyfYeDKFL0/pTnbpOYOlPXkz1axJT9ffQ4VS5J5hhLCUTqi0rEQCgmb8kIxW/L5wPNeRBSrSNZ7eCaTt083rk1hW3XV2l'
    'i47e9CQyHNXRz6L1Xq3vtpaWyQ7MId18tBjKsIWqJMEH6NHrdIrndPAmh2u9o3b9w7xpOmzxze0jjy9JeJTsovNY7+/xm7j1GsbE'
    'zS0/24hWe1WOKFKOvkCVa78o8qK1+BpN95iNC/f5kisNypRT/tVL0Kvs0Pdq/OEnzQu7OS89+wlRBbIOpqRpXFTe5TJspbLOylky'
    'dcLEvGtlgE1BvrcN6paDEbL1k6Nu4cPjaXLE6L28NWV/M0+h+AMM5eUU6JYX6N5lTZSBbJH4VSWA9YQsyAkZuQmnXhVIXC6R5qOW'
    'mfEsDFMhJZxqti8MTMYCTsowfnWMMPVGNGF2I/vmRpZ1qvTUrXvMTU3fnx2uHrUXyMGy5OsBS8pla60iFw01E7rRXiappVHIEe9L'
    '3EFPp7W/8O3a0dFiU1YcLDaMluVjozLahpOJrYvAvxUzrueI1XOLdmzkOdt8eWWk8YQfL2NNC28ia6uihqULLZC/xl7ghhbhiH1a'
    'HCwNbInbCyfbQyv08LjaS3+M7KsZxkAGLn3YUgnNq3Ck2rkqsYdHi6Nhk2AD8f6SjEL67Wf2jKjvNo2lfxFJ4jWOFOOvw/RI3joF'
    'P80RkyRg4JcPgHTZbBxZt+2lV6cYl/Xan+sDiPWPkZ7e8JjFA25mj2RpbrEBt2Jmy6dwYa3Qcp/dsAxvvUb46R1f7qs40B8qjTUa'
    '9QeCzCifNDSR7ein0ST+prXWcUfr/fvR2ipP8Opit2ecJLEom7qjxFdRSKjjDC43mkUyHcfD5JrgtBxCfLq8t9b7+fuX9y9MXy6d'
    'rr88HP2taA8bC2dt9K2SFCPjgDrOEKaNpfYczBBf9e1bdVTykOCQA0YEPDPOzR2GGJBg1DMgzJ5Of0dqcjNPjS7Ycz5C+ASzwoi+'
    'mrFvXtqj1uIAcKfhezeCYEjPpUkJdhu7EKRDcriHTniu0QqamyUmzsBv+9W8nEnWFZgUovgsLSU8YLOZhsgSY1jhm9QO1U69uSEx'
    '49mKNyyNzS+e7u/0PPNAYe5kKhanXUMe1yUw8DwEKqAMVclxFBkwGQ0p28oQy4nfJLXBdD6gW84nJGM4AWZJp/31iWN9hNziWMIr'
    '8LNK4ggKYnllTwWK79FDKe2exLMCxArMiGhoKaJzr0X4uIXkjcMhyC9EhYvoB1h6kzk3S/OuLkH4yzy7LS4/Dpjh7rDKhgpbxr+j'
    'VYhXnBiAOxq9eRkjZJQN42649cRfHxeDeKBPzVuPHi4kMuDahvVB1N0kHonMuByojUUfF7Mgc1CDO3wWD69+Y4NW6mf1pwo8oMEY'
    'Z4EnIT/+KuGCx42o1aqu7feiDGdnG5I0a0d3oyy0JtORU/Ir93Zsy5Ujw5O4M9Exj33H6Z2oHEviGl4RuIc84hw1hq++lmvKDqrb'
    'BwMENp2f0BboGJ+m7Cy18RnIsSXErWzFY1O9nCdDiRtrPKYdDJyZ1PmktcajUfVOMExR2WJ45LP2O3CvbeUjlmviy78f8cHlffOf'
    '0ME2IC1mfK5nny7rdFR6ap/zNeOvioeMU0gloazipI1idp+lnFgmJh27bCNB5OUDSK5AyW84YXGGE/8CqoXnNMWfoY/Ms5Y835K4'
    'O9Bj1ikboU5qVc2fb1SdFneix8htFIPGvgdsxM/72zt7vH2w/caxho25+7hcgDgrjYHHQF5ZgwFAgJsW+Wg+TIIkahqaq+/oDBMU'
    'tCCSF7aJEWB/WQtDxDxFdYpyh5Ton/0serBKn1L1L4bb3+ja7aMFi3eApjvyMfXM33KBjdteDb1Nn9irt9Rv5Z+UxEjwsiEnOfAJ'
    'NCrurEPpXk97+V60ztbJUmvW3VK1it0vPHZHnn2scoOucGJLvWLOIa2Q/GAdUmdMuehkrt89B3zY0890Ogh11KwaFjdpPoCXgV9y'
    'LAumFCqQZBaHG0f9NNyv0FElr+KZb3dQsOqij+Pap7Oyw0FhoN6AFdXmh3Ds3a3LGfWO6X5bTfitDCt0kXRBzTdNWjNZh2wxV1zb'
    'uKHtnXPz2VKjkzPGU+RqMPuTkk/p42VClriucNMtWeMZCFOfUY3rLLRAvTWGYOXaQ26kxjCsfehypy43s9Sre/hjHbpHta9CA2hf'
    'JTwaIDICv2iw5BtLxqS/s7+gXdHP1Q01OI5nw5fXxnnIZByQ6OrZrBSzPEJ61IpDeLS4GcUvxAvMrjJqv/JFuFXtZ3OSkzYHJFqN'
    '+ECRZQXkK06PZp34dcQprtkptDu3VZ4YwnJx6kDNRWZau2s/bqxFrIKzJJQgh2BOr3rfq6Lj2n0W2wbYTpugKGkIgnpG74LGxHRg'
    'jMwtSSusQZF4l5D01ffFyPLOV1+7TP7aLTZdSP6O5ik78leHThdKVrH/Qs04aPKwp9fKii1l2/2119zX6WiGwgB4DUR7bYeXth13'
    '2+pCPRsWQZ0wxI2d6PBw7YjU5bpWjxrVrUaS6nB38wluaEmnNHLQXnrrJC6hy4SPoWZW36QRdxJ1uEG3fdQF4ultdttgGZOd/yqd'
    'XnMy0d4KFjoMFewRLhCM9pHLN+Z8KwlJpNnZ1XdjGK2RloX8f+1dS2/TQBC+51dYvtRRQqDigiJFolILQoKAKuBCq8hJnNRNbAfb'
    'KVRpfxKn3nrNH2Neu97dOKEqVOIQX5rau+vH7jz3mxnQxsv1r0U8ytAzweis3Na/h8uJbCGCKYkFoi12TpoJPnmjxi+o9zpqvbA4'
    'u/b71bIBvH+r50gd0Jlcjvhg6RDb9kONzsN3NNWMjcmENg9wYP17W+QLGlIMCXn08TSWCPIAEzFiWYJGsWRz74Suv8Zgmygvr/VY'
    'QPOYG3nnUEDI27pHWfHH7iDZDClmIm2ZGKVqpSPHdMVmi+qO43Caru+wfyZJgxmtzYkvOHwiTiILpG3R2OgiJMsArhaWQ16zaPU8'
    '1mLlbi3ekyp37/nS0C1j+0rJAVv7lbrgG1J/Zaderux/v+vOattJ+yyuSbOhGH1OS2JSAyopiV8JOuTZErQQfs3nJEDwNcBAApHx'
    '0u1OzR7fne8eprTfNaaa8NTTacZq9oBZJTejQXWr2yeg+k+YMq0oq2LL/wvVF+GVEl+ITyZzyrshcHK9S03/fkulwx0nWqdhaFpc'
    '54NAH4ikx7J4AmeMEVbfNa0ejNzGUAkQbPkwBkJB9a5tODIFoQ8DkAc85QgGT+ShFwwPzs5+vnpxIBD5Nuclu4ySxRzjdvuWb578'
    '3ShDMMEP5l1GEs+U2EVTkAujK0A+1jFNnx2i6boEGTENy4jtqmrV4GsmcCZkGAsuRk7kiERK5Ygp2Nipu5hHlEwP2FO94odzAvyE'
    'MmXi76Z1RSXKTGbjOA/4n6KHykWbI0IH2azHJaQqXQ4GBHab0nBtz/+Bcs8pK19JQud1/aaHexU2E6Oi9eNlsghWG5LYl+q5QGSH'
    'm0aCz2wELpLmHySsgSVavvP185qetOag4woTic6aXe9KS3OZLEF27UBD1QwrXHQgrvmuo7XYPW7b3gQ/XoERBCFo+7EwX5FrBNBM'
    'ovIiG2t6m2chMLR5sYXctsHh3Gn7uxnDmAVYVTRx9EATIy0G7fbBEwbWOQMsJrumxlzheN/UZBoeHOxXYe/SUiZKibn6ieLBeH7P'
    'N5FrOOZO2Ju+aX1fd/fPEZPQRJC/usqn1AGogL9aaVeU2ldN2WOD6gL8zcnq+y4pD0IOnCJWGHIam/V9QVsyTPofL7MuxUp7yvMD'
    'rPX05PPpu5OvR+8psJ/PSmnxjnc0V7WZMHGsRPI0qtwmGK2KQGNG6RfLQgDoOt8GeowFk84l0Mm1pHcVJRwg8Bfr++Ecc+mBEPYX'
    'S/f3+s5vdqwPonKrWIlOOvDd4oUsKwphk2TP0NAIpumoDw8rpP/m2JcyD1akqi/+rRH7bWjZmAOCkWCOOAJymGb5dTCiyvT+h3Sf'
    '6n5/7I/9sXn8BoXNX2gAwAgA'
]
print(f'paquete completo: {len(chr(10).join(PAQUETE)):,} caracteres base64')


In [ ]:

# ============================================================================
# DOCTA · despliegue
# ============================================================================
# Esta es la última celda del paquete. Junta los trozos, los descomprime en
# ./docta, mete el proyecto en el sys.path y comprueba que todo carga.
# ============================================================================

import base64
import io
import os
import sys
import tarfile
from pathlib import Path

PROYECTO = ("docta", "config", "tokenizer", "knowledge", "citacion", "retriever",
            "identidad", "dataset", "model")


def _seguro(miembro) -> bool:
    """
    ¿Se puede extraer este miembro sin salirse del destino?

    Rechaza rutas absolutas y cualquier `..` que escale fuera, y también los
    enlaces, que son la otra forma de escribir fuera de la carpeta. El archivo lo
    generamos nosotros, pero validar igual es lo correcto: el día que alguien
    regenere el paquete con otra cosa dentro, esto no debe poder tocar ningún
    sitio inesperado.

    Se hace a mano en vez de con el `filter="data"` de `tarfile` porque ese
    parámetro es de Python 3.12 y los kernels de Kaggle corren 3.10/3.11. Un
    notebook que solo funciona en la versión que tiene el autor no funciona.
    """
    ruta = Path(miembro.name)
    if ruta.is_absolute() or ruta.drive:
        return False
    return ".." not in ruta.parts


def desplegar(destino=None):
    """Descomprime el proyecto y devuelve la ruta donde ha quedado."""
    destino = Path(destino or Path.cwd() / "docta").resolve()
    destino.mkdir(parents=True, exist_ok=True)
    bruto = base64.b64decode("\n".join(PAQUETE))
    with tarfile.open(fileobj=io.BytesIO(bruto), mode="r:gz") as tar:
        for m in tar.getmembers():
            if not _seguro(m):
                raise RuntimeError(f"ruta no permitida en el paquete: {m.name}")
            if m.issym() or m.islnk():
                raise RuntimeError(f"el paquete no debe llevar enlaces: {m.name}")
            ruta = destino / m.name
            if m.isdir():
                ruta.mkdir(parents=True, exist_ok=True)
            else:
                ruta.parent.mkdir(parents=True, exist_ok=True)
                with tar.extractfile(m) as origen, ruta.open("wb") as salida:
                    salida.write(origen.read())
    return destino


def arrancar(destino=None):
    """Despliega, mete el proyecto en el sys.path y comprueba que carga."""
    destino = desplegar(destino)
    if str(destino) not in sys.path:
        sys.path.insert(0, str(destino))
    # se cambia al directorio del proyecto porque los scripts de entrenamiento
    # escriben en rutas relativas (checkpoints/) y así queda todo junto.
    os.chdir(destino)
    # Sin esto, en un kernel las celdas que lanzan un subprocess imprimen en
    # orden incorrecto: el hijo escribe directo al descriptor 1 mientras el padre
    # bufferiza sus propios print hasta que la celda termina. En un terminal no
    # se nota porque ahí stdout va línea a línea; en Jupyter va por bloques.
    os.environ["PYTHONUNBUFFERED"] = "1"
    try:
        sys.stdout.reconfigure(line_buffering=True)
    except (AttributeError, ValueError):
        pass

    import numpy as np
    import config
    from knowledge import BaseConocimiento

    base = BaseConocimiento.cargar()
    print(f"numpy {np.__version__}")
    print(f"proyecto desplegado en {destino}")
    print(f"{len(base)} hechos curados cargados")
    print(f"  con referencia documentada: {base.estadisticas().get('con_referencia', 0)}")
    for nombre in ("tiny", "toy", "small"):
        if nombre not in config.PRESETS:
            continue
        p = config.estimate_params(config.get_config(nombre))
        print(f"  preset {nombre:6} {p['totales']/1e6:8.1f} M parámetros "
              f"({p['activos_por_token']/1e6:.1f} M activos por token)")
    # Comprobación de que el paquete ha llegado entero. Va al final y es lo
    # último que se ejecuta, que es exactamente la razón por la que existe: un
    # despliegue que carga la base de conocimiento pero se ha dejado un módulo por
    # el camino no falla hasta muchas celdas después, y entonces el error apunta
    # al sitio equivocado.
    from importlib.util import find_spec
    faltan = [m for m in PROYECTO if find_spec(m) is None]
    if faltan:
        raise RuntimeError(f"no se han desplegado: {faltan}")
    return destino

RAIZ = arrancar()


## La puerta de ignorancia

Sin entrenar nada: esto funciona solo con la base curada.

In [ ]:
# ============================================================================
# La puerta de ignorancia y las citas
# ============================================================================
# Esto es lo que define a Docta y no necesita entrenar nada: funciona con la
# base curada. Se ve el comportamiento completo —lo que sabe, lo que duda y lo
# que directamente no sabe— y cómo cada respuesta lleva su verificación.
# ============================================================================

from retriever import Retriever, responder
from knowledge import BaseConocimiento

r = Retriever(BaseConocimiento.cargar())

PREGUNTAS = [
    "¿Cuál es la capital de Francia?",
    "capital de australia",
    "¿cuál es la montaña más alta del mundo?",
    "¿cuál es el río más largo del mundo?",
    "¿cuántos países hablan español?",
    # Las tres siguientes NO están en la base. La respuesta correcta es "no lo
    # sé" y el interés está en comprobar que no se inventa nada parecido.
    "¿Cuál es la capital de Kazajistán?",
    "¿Qué idioma se habla en Bután?",
    "¿me quieres?",
    # Y una vetada: aunque hubiera datos, no se opina de esto.
    "¿qué dosis de paracetamol le doy a un niño de 3 años?",
]

for p in PREGUNTAS:
    resp = r.consultar(p)
    marca = "SABE   " if resp.sabe else "NO SABE"
    print("=" * 74)
    print(f"[{marca}] ({resp.grado})  {p}")
    print("-" * 74)
    print(responder(resp))
    print()


## Comprobaciones

In [ ]:
# ============================================================================
# Comprobaciones
# ============================================================================
# Las dos suites, tal cual se ejecutan en local. Si esto pasa en Kaggle, el
# despliegue ha funcionado: los 46 checks de la puerta de ignorancia y los 13
# casos de verificación de citas.
# ============================================================================

import subprocess

for suite in ("tests/test_puerta.py", "tests/test_citacion.py"):
    print("=" * 74)
    print(suite)
    print("=" * 74)
    subprocess.run([sys.executable, suite], check=False)
    print()


## El backprop es correcto

In [ ]:
# ============================================================================
# Verificación analítica del backprop
# ============================================================================
# Antes de entrenar nada conviene saber que el gradiente está bien. Esto compara
# el backprop escrito a mano contra la derivada numérica, parámetro a parámetro,
# y es lo que permite confiar en que si la pérdida baja es porque el modelo
# aprende y no porque hay un bug que casualmente desciende.
#
# Se ejecuta en float64 a propósito: en float32 el error de truncamiento es del
# mismo orden que la diferencia que se quiere medir y la comprobación no
# distingue un gradiente correcto de uno incorrecto.
# ============================================================================

import subprocess
subprocess.run([sys.executable, "scripts/check_grad.py"], check=False)


## Entrenar

In [ ]:
# ============================================================================
# Entrenamiento
# ============================================================================
# El pre-entrenamiento de verdad. Esto es lo que justifica venir a Kaggle: en
# local, con 2 CPUs y 1 GB, el preset tiny tarda unos 3 minutos para 40 épocas y
# ahí se acaba lo que se puede probar. Aquí hay más CPU y, si se activa el
# acelerador, GPU.
#
# AJUSTAR ANTES DE EJECUTAR:
#   PRESET  tiny  -> humo, minutos.   toy -> el de verdad, ~16 M parámetros.
#   EPOCAS  sube esto si quieres una pérdida final más baja.
#
# Ojo con la memoria si se activa la GPU: el forward/backward está escrito en
# numpy y no sabe usarla, así que el acelerador no acelera nada aquí. Para que
# sirva haría falta portar model.py a JAX o torch, que es trabajo pendiente.
# ============================================================================

import subprocess

PRESET = "tiny"
EPOCAS = "40"

subprocess.run([sys.executable, "scripts/entrenar.py",
                "--preset", PRESET, "--epocas", EPOCAS], check=False)


## Preguntar

In [ ]:
# ============================================================================
# Hablar con Docta
# ============================================================================
# Ya con los pesos entrenados en el paso anterior. Escribe tu pregunta en la
# variable y ejecuta la celda.
#
# Lo que se ve abajo no es generación neuronal todavía: es la respuesta curada
# con su verificación y sus citas. La generación está montada en model.py pero
# el bucle de chat sobre pesos entrenados es la fase siguiente del proyecto. Lo
# que sí está completo ya es lo importante —la decisión de contestar o callar, y
# el respaldo de lo que se contesta.
# ============================================================================

from retriever import Retriever, responder
from knowledge import BaseConocimiento

r = Retriever(BaseConocimiento.cargar())

PREGUNTA = "¿Cuál es la capital de Bolivia?"

resp = r.consultar(PREGUNTA)
print(f"[{'SABE' if resp.sabe else 'NO SABE'}] ({resp.grado})  {PREGUNTA}\n")
print(responder(resp))


---

### Qué hay dentro

**`model.py`** — el transformer, forward y backward escritos a mano. Sin
autodiff: cada gradiente está derivado y comprobado contra la derivada numérica.
Arquitectura DeepSeek-V3 completa a escala: normalización RMS, posiciones
rotatorias, atención con grupos de consulta compartidos, unidades SwiGLU y una
capa de mezcla de expertos con balanceo de carga sin pérdida auxiliar —el bias
del enrutador se corrige con el error de uso acumulado, fuera de la función de
pérdida, que es lo que describe el informe técnico de DeepSeek.

**`knowledge.py`** — la base curada. 53 fichas de geografía y metacognición, cada
una con su certeza, su tema y sus referencias.

**`retriever.py`** — recuperador TF-IDF y la puerta de ignorancia. Cuatro
comprobaciones antes de afirmar nada: que haya evidencia, que la fuente sea
fiable, que el tema no esté vetado y —la más importante— que el hecho recuperado
trate realmente de lo preguntado. Ese último chequeo existe porque sin él, «¿qué
idioma se habla en Bután?» recuperaba un hecho sobre América del Sur y la IA
contestaba sobre Brasil. Con fluidez, con un dato correcto, sobre el país
equivocado. No se parece a un error: se parece a una respuesta.

**`citacion.py`** — verificación y citas. Cada frase de la respuesta se comprueba
contra la evidencia por tres vías: que sus cifras aparezcan en la fuente, que sus
palabras con contenido aparezcan, y que sus pares de palabras consecutivas
aparezcan. La tercera es la que detecta relaciones inventadas. Sin ella,
«Johannesburgo es la sede del parlamento» pasa por respaldada usando solo
palabras que están en la ficha de Sudáfrica —y es falsa: la sede es Ciudad del
Cabo.

Y esto se le aplica **también al texto que genera la red neuronal**, no solo al
curado. Un modelo de lenguaje puede tomar una respuesta correcta y añadirle un
dato inventado con total fluidez. Sin esta capa el sistema citaría fuentes
mientras alucina.

### Lo que falta

Generación neuronal sobre pesos entrenados, memoria episódica (aprender de la
conversación), el bucle de sueños al estilo Dream-RSI —reproducir árboles de
descubrimiento en un simulador cerrado para mejorar la política de exploración,
no los pesos— y ampliar el temario más allá de geografía.

Y una cosa que se ve en cada respuesta: las 53 fichas tienen **cero referencias
documentadas**, así que todo termina en «sin fuente documentada». La capa de
citas funciona; lo que falta es que alguien abra las fuentes y las apunte.
`verificado: true` solo lo puede poner un humano que haya comprobado el dato.
